# HRM on 6x6 Sudoku - Train + Evaluate (12-Hour Kaggle Session)

This notebook trains the Hierarchical Reasoning Model on 6x6 Sudoku puzzles
and evaluates it automatically at the end.

**Key fixes applied:**
- Checkpoints are saved BEFORE evaluation (prevents data loss if eval crashes)
- Evaluation is wrapped in try/except (training continues even if eval fails on T4)
- Batch size = 64 (prevents OOM on T4 GPU)

**Requirements:** GPU T4 x2 accelerator, 12-hour session.

In [ ]:
!mkdir -p dataset/raw-data models/hrm utils config/arch


In [ ]:
%%writefile download.txt
12.7s 1 0.00s - Debugger warning: It seems that frozen modules are being used, which may
12.7s 2 0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
12.7s 3 0.00s - to python to disable frozen modules.
12.7s 4 0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
13.5s 5 0.00s - Debugger warning: It seems that frozen modules are being used, which may
13.5s 6 0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
13.5s 7 0.00s - to python to disable frozen modules.
13.5s 8 0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
15.0s 9 Writing evaluate.py
15.0s 10 Writing generate_6x6_sudoku_dataset_chatgpt.py
15.0s 11 Writing pdf_text.txt
15.0s 12 Writing pretrain.py
15.0s 13 Writing puzzle_dataset.py
15.1s 14 Writing requirements.txt
15.1s 15 Writing config/cfg_pretrain.yaml
15.1s 16 Writing config/arch/hrm_v1.yaml
15.1s 17 Writing dataset/build_6x6_sudoku_dataset.py
15.1s 18 Writing dataset/build_arc_dataset.py
15.1s 19 Writing dataset/build_maze_dataset.py
15.2s 20 Writing dataset/build_sudoku_dataset.py
15.2s 21 Writing dataset/common.py
15.2s 22 Writing models/common.py
15.2s 23 Writing models/layers.py
15.2s 24 Writing models/losses.py
15.3s 25 Writing models/sparse_embedding.py
15.3s 26 Writing models/hrm/hrm_act_v1.py
15.3s 27 Writing utils/functions.py
18.9s 28 Requirement already satisfied: torch in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 1)) (2.10.0+cu128)
18.9s 29 Requirement already satisfied: einops in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 2)) (0.8.2)
18.9s 30 Requirement already satisfied: tqdm in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 3)) (4.67.3)
19.0s 31 Collecting coolname (from -r requirements.txt (line 4))
19.0s 32 Downloading coolname-5.0.0-py3-none-any.whl.metadata (4.9 kB)
19.0s 33 Requirement already satisfied: pydantic in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 5)) (2.12.3)
19.1s 34 Collecting argdantic (from -r requirements.txt (line 6))
19.1s 35 Downloading argdantic-1.3.3-py2.py3-none-any.whl.metadata (7.2 kB)
19.1s 36 Requirement already satisfied: wandb in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 7)) (0.26.1)
19.1s 37 Requirement already satisfied: omegaconf in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 8)) (2.3.0)
19.2s 38 Collecting hydra-core (from -r requirements.txt (line 9))
19.2s 39 Downloading hydra_core-1.3.4-py3-none-any.whl.metadata (5.7 kB)
19.2s 40 Requirement already satisfied: huggingface_hub in /usr/local/lib/python3.12/dist-packages (from -r requirements.txt (line 10)) (1.11.0)
19.2s 41 Requirement already satisfied: filelock in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (3.29.0)
19.2s 42 Requirement already satisfied: typing-extensions>=4.10.0 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (4.15.0)
19.2s 43 Requirement already satisfied: setuptools in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (81.0.0)
19.2s 44 Requirement already satisfied: sympy>=1.13.3 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (1.14.0)
19.2s 45 Requirement already satisfied: networkx>=2.5.1 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (3.6.1)
19.2s 46 Requirement already satisfied: jinja2 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (3.1.6)
19.2s 47 Requirement already satisfied: fsspec>=0.8.5 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (2025.3.0)
19.2s 48 Requirement already satisfied: cuda-bindings==12.9.4 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.9.4)
19.2s 49 Requirement already satisfied: nvidia-cuda-nvrtc-cu12==12.8.93 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.93)
19.2s 50 Requirement already satisfied: nvidia-cuda-runtime-cu12==12.8.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.90)
19.2s 51 Requirement already satisfied: nvidia-cuda-cupti-cu12==12.8.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.90)
19.2s 52 Requirement already satisfied: nvidia-cudnn-cu12==9.10.2.21 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (9.10.2.21)
19.2s 53 Requirement already satisfied: nvidia-cublas-cu12==12.8.4.1 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.4.1)
19.2s 54 Requirement already satisfied: nvidia-cufft-cu12==11.3.3.83 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (11.3.3.83)
19.2s 55 Requirement already satisfied: nvidia-curand-cu12==10.3.9.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (10.3.9.90)
19.2s 56 Requirement already satisfied: nvidia-cusolver-cu12==11.7.3.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (11.7.3.90)
19.2s 57 Requirement already satisfied: nvidia-cusparse-cu12==12.5.8.93 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.5.8.93)
19.2s 58 Requirement already satisfied: nvidia-cusparselt-cu12==0.7.1 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (0.7.1)
19.2s 59 Requirement already satisfied: nvidia-nccl-cu12==2.27.5 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (2.27.5)
19.2s 60 Requirement already satisfied: nvidia-nvshmem-cu12==3.4.5 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (3.4.5)
19.2s 61 Requirement already satisfied: nvidia-nvtx-cu12==12.8.90 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.90)
19.2s 62 Requirement already satisfied: nvidia-nvjitlink-cu12==12.8.93 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (12.8.93)
19.2s 63 Requirement already satisfied: nvidia-cufile-cu12==1.13.1.3 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (1.13.1.3)
19.2s 64 Requirement already satisfied: triton==3.6.0 in /usr/local/lib/python3.12/dist-packages (from torch->-r requirements.txt (line 1)) (3.6.0)
19.2s 65 Requirement already satisfied: cuda-pathfinder~=1.1 in /usr/local/lib/python3.12/dist-packages (from cuda-bindings==12.9.4->torch->-r requirements.txt (line 1)) (1.5.3)
19.3s 66 Requirement already satisfied: annotated-types>=0.6.0 in /usr/local/lib/python3.12/dist-packages (from pydantic->-r requirements.txt (line 5)) (0.7.0)
19.3s 67 Requirement already satisfied: pydantic-core==2.41.4 in /usr/local/lib/python3.12/dist-packages (from pydantic->-r requirements.txt (line 5)) (2.41.4)
19.3s 68 Requirement already satisfied: typing-inspection>=0.4.2 in /usr/local/lib/python3.12/dist-packages (from pydantic->-r requirements.txt (line 5)) (0.4.2)
19.3s 69 Requirement already satisfied: pydantic-settings<3,>=2.4.0 in /usr/local/lib/python3.12/dist-packages (from argdantic->-r requirements.txt (line 6)) (2.14.0)
19.3s 70 Requirement already satisfied: click>=8.0.1 in /usr/local/lib/python3.12/dist-packages (from wandb->-r requirements.txt (line 7)) (8.3.3)
19.3s 71 Requirement already satisfied: gitpython!=3.1.29,>=1.0.0 in /usr/local/lib/python3.12/dist-packages (from wandb->-r requirements.txt (line 7)) (3.1.47)
19.3s 72 Requirement already satisfied: packaging in /usr/local/lib/python3.12/dist-packages (from wandb->-r requirements.txt (line 7)) (26.1)
19.3s 73 Requirement already satisfied: platformdirs in /usr/local/lib/python3.12/dist-packages (from wandb->-r requirements.txt (line 7)) (4.9.6)
19.3s 74 Requirement already satisfied: protobuf!=5.28.0,!=5.29.0,<8,>4.21.0 in /usr/local/lib/python3.12/dist-packages (from wandb->-r requirements.txt (line 7)) (5.29.5)
19.3s 75 Requirement already satisfied: pyyaml in /usr/local/lib/python3.12/dist-packages (from wandb->-r requirements.txt (line 7)) (6.0.3)
19.3s 76 Requirement already satisfied: requests<3,>=2.0.0 in /usr/local/lib/python3.12/dist-packages (from wandb->-r requirements.txt (line 7)) (2.32.4)
19.3s 77 Requirement already satisfied: sentry-sdk>=2.0.0 in /usr/local/lib/python3.12/dist-packages (from wandb->-r requirements.txt (line 7)) (2.58.0)
19.3s 78 Requirement already satisfied: antlr4-python3-runtime==4.9.* in /usr/local/lib/python3.12/dist-packages (from omegaconf->-r requirements.txt (line 8)) (4.9.3)
19.3s 79 Requirement already satisfied: hf-xet<2.0.0,>=1.4.3 in /usr/local/lib/python3.12/dist-packages (from huggingface_hub->-r requirements.txt (line 10)) (1.4.3)
19.3s 80 Requirement already satisfied: httpx<1,>=0.23.0 in /usr/local/lib/python3.12/dist-packages (from huggingface_hub->-r requirements.txt (line 10)) (0.28.1)
19.3s 81 Requirement already satisfied: typer in /usr/local/lib/python3.12/dist-packages (from huggingface_hub->-r requirements.txt (line 10)) (0.24.2)
19.3s 82 Requirement already satisfied: gitdb<5,>=4.0.1 in /usr/local/lib/python3.12/dist-packages (from gitpython!=3.1.29,>=1.0.0->wandb->-r requirements.txt (line 7)) (4.0.12)
19.3s 83 Requirement already satisfied: anyio in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface_hub->-r requirements.txt (line 10)) (4.13.0)
19.3s 84 Requirement already satisfied: certifi in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface_hub->-r requirements.txt (line 10)) (2026.4.22)
19.3s 85 Requirement already satisfied: httpcore==1.* in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface_hub->-r requirements.txt (line 10)) (1.0.9)
19.3s 86 Requirement already satisfied: idna in /usr/local/lib/python3.12/dist-packages (from httpx<1,>=0.23.0->huggingface_hub->-r requirements.txt (line 10)) (3.13)
19.3s 87 Requirement already satisfied: h11>=0.16 in /usr/local/lib/python3.12/dist-packages (from httpcore==1.*->httpx<1,>=0.23.0->huggingface_hub->-r requirements.txt (line 10)) (0.16.0)
19.3s 88 Requirement already satisfied: python-dotenv>=0.21.0 in /usr/local/lib/python3.12/dist-packages (from pydantic-settings<3,>=2.4.0->argdantic->-r requirements.txt (line 6)) (1.2.2)
19.4s 89 Requirement already satisfied: charset_normalizer<4,>=2 in /usr/local/lib/python3.12/dist-packages (from requests<3,>=2.0.0->wandb->-r requirements.txt (line 7)) (3.4.7)
19.4s 90 Requirement already satisfied: urllib3<3,>=1.21.1 in /usr/local/lib/python3.12/dist-packages (from requests<3,>=2.0.0->wandb->-r requirements.txt (line 7)) (2.5.0)
19.4s 91 Requirement already satisfied: mpmath<1.4,>=1.1.0 in /usr/local/lib/python3.12/dist-packages (from sympy>=1.13.3->torch->-r requirements.txt (line 1)) (1.3.0)
19.4s 92 Requirement already satisfied: MarkupSafe>=2.0 in /usr/local/lib/python3.12/dist-packages (from jinja2->torch->-r requirements.txt (line 1)) (3.0.3)
19.4s 93 Requirement already satisfied: shellingham>=1.3.0 in /usr/local/lib/python3.12/dist-packages (from typer->huggingface_hub->-r requirements.txt (line 10)) (1.5.4)
19.4s 94 Requirement already satisfied: rich>=12.3.0 in /usr/local/lib/python3.12/dist-packages (from typer->huggingface_hub->-r requirements.txt (line 10)) (13.9.4)
19.4s 95 Requirement already satisfied: annotated-doc>=0.0.2 in /usr/local/lib/python3.12/dist-packages (from typer->huggingface_hub->-r requirements.txt (line 10)) (0.0.4)
19.4s 96 Requirement already satisfied: smmap<6,>=3.0.1 in /usr/local/lib/python3.12/dist-packages (from gitdb<5,>=4.0.1->gitpython!=3.1.29,>=1.0.0->wandb->-r requirements.txt (line 7)) (5.0.3)
19.4s 97 Requirement already satisfied: markdown-it-py>=2.2.0 in /usr/local/lib/python3.12/dist-packages (from rich>=12.3.0->typer->huggingface_hub->-r requirements.txt (line 10)) (4.0.0)
19.4s 98 Requirement already satisfied: pygments<3.0.0,>=2.13.0 in /usr/local/lib/python3.12/dist-packages (from rich>=12.3.0->typer->huggingface_hub->-r requirements.txt (line 10)) (2.20.0)
19.4s 99 Requirement already satisfied: mdurl~=0.1 in /usr/local/lib/python3.12/dist-packages (from markdown-it-py>=2.2.0->rich>=12.3.0->typer->huggingface_hub->-r requirements.txt (line 10)) (0.1.2)
19.4s 100 Downloading coolname-5.0.0-py3-none-any.whl (47 kB)
19.5s 101 [?25l   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m0.0/47.4 kB[0m [31m?[0m eta [36m-:--:--[0m
[2K   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m47.4/47.4 kB[0m [31m1.7 MB/s[0m eta [36m0:00:00[0m
19.5s 102 [?25hDownloading argdantic-1.3.3-py2.py3-none-any.whl (26 kB)
19.5s 103 Downloading hydra_core-1.3.4-py3-none-any.whl (155 kB)
19.5s 104 [?25l   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m0.0/155.5 kB[0m [31m?[0m eta [36m-:--:--[0m
[2K   [90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━[0m [32m155.5/155.5 kB[0m [31m5.9 MB/s[0m eta [36m0:00:00[0m
21.6s 105 [?25hInstalling collected packages: coolname, hydra-core, argdantic
21.9s 106 Successfully installed argdantic-1.3.3 coolname-5.0.0 hydra-core-1.3.4
23.8s 107 Generating 6x6 puzzles:   0%|                          | 0/1000 [00:00<?, ?it/s]
Generating 6x6 puzzles:   8%|█▎              | 81/1000 [00:00<00:01, 805.57it/s]
Generating 6x6 puzzles:  16%|██▍            | 162/1000 [00:00<00:01, 805.55it/s]
Generating 6x6 puzzles:  25%|███▋           | 248/1000 [00:00<00:00, 827.43it/s]
Generating 6x6 puzzles:  33%|█████          | 334/1000 [00:00<00:00, 834.97it/s]
Generating 6x6 puzzles:  42%|██████▎        | 420/1000 [00:00<00:00, 843.91it/s]
Generating 6x6 puzzles:  50%|███████▌       | 505/1000 [00:00<00:00, 842.52it/s]
Generating 6x6 puzzles:  59%|████████▊      | 590/1000 [00:00<00:00, 835.72it/s]
Generating 6x6 puzzles:  67%|██████████     | 674/1000 [00:00<00:00, 818.50it/s]
Generating 6x6 puzzles:  76%|███████████▎   | 758/1000 [00:00<00:00, 823.56it/s]
Generating 6x6 puzzles:  84%|████████████▋  | 842/1000 [00:01<00:00, 825.47it/s]
Generating 6x6 puzzles:  93%|█████████████▉ | 929/1000 [00:01<00:00, 838.19it/s]
Generating 6x6 puzzles: 100%|██████████████| 1000/1000 [00:01<00:00, 832.53it/s]
24.0s 108 Generating 6x6 puzzles:   0%|                           | 0/100 [00:00<?, ?it/s]
Generating 6x6 puzzles:  83%|██████████████   | 83/100 [00:00<00:00, 826.13it/s]
Generating 6x6 puzzles: 100%|████████████████| 100/100 [00:00<00:00, 837.39it/s]
48.0s 109 0%|                                                 | 0/78125 [00:00<?, ?it/s][34m[1mwandb[0m: Tracking run with wandb version 0.26.1
48.0s 110 [34m[1mwandb[0m: W&B syncing is set to [1m`offline`[0m in this directory. Run [1m`wandb online`[0m or set [1mWANDB_MODE=online[0m to enable cloud syncing.
48.0s 111 [34m[1mwandb[0m: Run data is saved locally in [35m[1m/kaggle/working/wandb/offline-run-20260711_160237-7dx7a8sc[0m
48.7s 112 [Rank 0, World Size 1]: Epoch 0
84.5s 113 W0711 16:03:14.673000 59 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
90.2s 114 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
90.2s 115 warnings.warn(
121.2s 116 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
121.2s 117 warnings.warn(
121.4s 118 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
121.4s 119 warnings.warn(
140.6s 120 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
140.6s 121 warnings.warn(
140.9s 122 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
140.9s 123 warnings.warn(
141.0s 124 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
141.0s 125 warnings.warn(
141.1s 126 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
141.1s 127 warnings.warn(
143.6s 128 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
143.6s 129 warnings.warn(
144.2s 130 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
144.2s 131 warnings.warn(
144.6s 132 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
144.6s 133 warnings.warn(
144.7s 134 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
144.7s 135 warnings.warn(
144.9s 136 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
144.9s 137 warnings.warn(
145.9s 138 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
145.9s 139 warnings.warn(
146.1s 140 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
146.1s 141 warnings.warn(
146.3s 142 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
146.3s 143 warnings.warn(
147.5s 144 /usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:2900: UserWarning: Tesla T4 does not support bfloat16 compilation natively, skipping
147.5s 145 warnings.warn(
760.3s 146 0%|                                   | 1/78125 [01:43<2253:38:53, 103.85s/it]
  0%|                                     | 2/78125 [01:44<933:57:13, 43.04s/it]
  0%|                                     | 3/78125 [01:44<511:54:07, 23.59s/it]
  0%|                                     | 4/78125 [01:45<313:37:37, 14.45s/it]
  0%|                                     | 5/78125 [01:45<204:01:24,  9.40s/it]
  0%|                                     | 6/78125 [01:46<137:57:44,  6.36s/it]
  0%|                                      | 7/78125 [01:46<96:01:47,  4.43s/it]
  0%|                                      | 8/78125 [01:47<68:33:08,  3.16s/it]
  0%|                                      | 9/78125 [01:47<50:10:10,  2.31s/it]
  0%|                                     | 10/78125 [01:47<37:41:29,  1.74s/it]
  0%|                                     | 11/78125 [01:48<29:08:27,  1.34s/it]
  0%|                                     | 12/78125 [01:48<23:14:03,  1.07s/it]
  0%|                                     | 13/78125 [01:49<19:08:54,  1.13it/s]
  0%|                                     | 14/78125 [01:49<16:18:51,  1.33it/s]
  0%|                                     | 15/78125 [01:50<14:20:13,  1.51it/s]
  0%|                                     | 16/78125 [01:50<12:57:59,  1.67it/s]
  0%|                                     | 17/78125 [01:51<12:00:06,  1.81it/s]
  0%|                                     | 18/78125 [01:51<11:19:37,  1.92it/s]
  0%|                                     | 19/78125 [01:51<10:52:39,  1.99it/s]
  0%|                                     | 20/78125 [01:52<10:33:07,  2.06it/s]
  0%|                                     | 21/78125 [01:52<10:18:25,  2.10it/s]
  0%|                                     | 22/78125 [01:53<10:09:41,  2.14it/s]
  0%|                                     | 23/78125 [01:53<10:03:17,  2.16it/s]
  0%|                                      | 24/78125 [01:54<9:58:32,  2.17it/s]
  0%|                                      | 25/78125 [01:54<9:55:52,  2.18it/s]
  0%|                                      | 26/78125 [01:55<9:54:02,  2.19it/s]
  0%|                                      | 27/78125 [01:55<9:52:24,  2.20it/s]
  0%|                                      | 28/78125 [01:56<9:52:06,  2.20it/s]
  0%|                                      | 29/78125 [01:56<9:51:09,  2.20it/s]
  0%|                                      | 30/78125 [01:56<9:50:42,  2.20it/s]
  0%|                                      | 31/78125 [01:57<9:50:52,  2.20it/s]
  0%|                                      | 32/78125 [01:57<9:50:04,  2.21it/s]
  0%|                                      | 33/78125 [01:58<9:50:35,  2.20it/s]
  0%|                                      | 34/78125 [01:58<9:50:26,  2.20it/s]
  0%|                                      | 35/78125 [01:59<9:50:53,  2.20it/s]
  0%|                                      | 36/78125 [01:59<9:50:46,  2.20it/s]
  0%|                                      | 37/78125 [02:00<9:50:55,  2.20it/s]
  0%|                                      | 38/78125 [02:00<9:51:11,  2.20it/s]
  0%|                                      | 39/78125 [02:01<9:52:11,  2.20it/s]
  0%|                                      | 40/78125 [02:01<9:51:54,  2.20it/s]
  0%|                                      | 41/78125 [02:01<9:52:53,  2.19it/s]
  0%|                                      | 42/78125 [02:02<9:53:02,  2.19it/s]
  0%|                                      | 43/78125 [02:02<9:54:00,  2.19it/s]
  0%|                                      | 44/78125 [02:03<9:54:28,  2.19it/s]
  0%|                                      | 45/78125 [02:03<9:54:01,  2.19it/s]
  0%|                                      | 46/78125 [02:04<9:54:54,  2.19it/s]
  0%|                                      | 47/78125 [02:04<9:55:31,  2.19it/s]
  0%|                                      | 48/78125 [02:05<9:56:12,  2.18it/s]
  0%|                                      | 49/78125 [02:05<9:56:00,  2.18it/s]
  0%|                                      | 50/78125 [02:06<9:55:35,  2.18it/s]
  0%|                                      | 51/78125 [02:06<9:56:06,  2.18it/s]
  0%|                                      | 52/78125 [02:06<9:56:18,  2.18it/s]
  0%|                                      | 53/78125 [02:07<9:56:35,  2.18it/s]
  0%|                                      | 54/78125 [02:07<9:57:42,  2.18it/s]
  0%|                                      | 55/78125 [02:08<9:57:23,  2.18it/s]
  0%|                                      | 56/78125 [02:08<9:58:06,  2.18it/s]
  0%|                                      | 57/78125 [02:09<9:58:15,  2.17it/s]
  0%|                                      | 58/78125 [02:09<9:58:24,  2.17it/s]
  0%|                                      | 59/78125 [02:10<9:58:35,  2.17it/s]
  0%|                                      | 60/78125 [02:10<9:58:52,  2.17it/s]
  0%|                                      | 61/78125 [02:11<9:58:50,  2.17it/s]
  0%|                                      | 62/78125 [02:11<9:59:19,  2.17it/s]
  0%|                                      | 63/78125 [02:12<9:59:37,  2.17it/s]
  0%|                                     | 64/78125 [02:12<10:00:15,  2.17it/s]
  0%|                                     | 65/78125 [02:12<10:00:11,  2.17it/s]
  0%|                                     | 66/78125 [02:13<10:00:52,  2.17it/s]
  0%|                                     | 67/78125 [02:13<10:01:00,  2.16it/s]
  0%|                                     | 68/78125 [02:14<10:01:09,  2.16it/s]
  0%|                                     | 69/78125 [02:14<10:02:14,  2.16it/s]
  0%|                                     | 70/78125 [02:15<10:02:13,  2.16it/s]
  0%|                                     | 71/78125 [02:15<10:03:03,  2.16it/s]
  0%|                                     | 72/78125 [02:16<10:03:19,  2.16it/s]
  0%|                                     | 73/78125 [02:16<10:03:56,  2.15it/s]
  0%|                                     | 74/78125 [02:17<10:04:03,  2.15it/s]
  0%|                                     | 75/78125 [02:17<10:04:42,  2.15it/s]
  0%|                                     | 76/78125 [02:18<10:05:03,  2.15it/s]
  0%|                                     | 77/78125 [02:18<10:05:37,  2.15it/s]
  0%|                                     | 78/78125 [02:18<10:05:39,  2.15it/s]
  0%|                                     | 79/78125 [02:19<10:06:31,  2.14it/s]
  0%|                                     | 80/78125 [02:19<10:06:23,  2.15it/s]
  0%|                                     | 81/78125 [02:20<10:06:57,  2.14it/s]
  0%|                                     | 82/78125 [02:20<10:06:51,  2.14it/s]
  0%|                                     | 83/78125 [02:21<10:06:26,  2.14it/s]
  0%|                                     | 84/78125 [02:21<10:07:37,  2.14it/s]
  0%|                                     | 85/78125 [02:22<10:07:51,  2.14it/s]
  0%|                                     | 86/78125 [02:22<10:08:06,  2.14it/s]
  0%|                                     | 87/78125 [02:23<10:08:15,  2.14it/s]
  0%|                                     | 88/78125 [02:23<10:09:01,  2.14it/s]
  0%|                                     | 89/78125 [02:24<10:09:45,  2.13it/s]
  0%|                                     | 90/78125 [02:24<10:09:55,  2.13it/s]
  0%|                                     | 91/78125 [02:25<10:09:49,  2.13it/s]
  0%|                                     | 92/78125 [02:25<10:10:22,  2.13it/s]
  0%|                                     | 93/78125 [02:26<10:10:32,  2.13it/s]
  0%|                                     | 94/78125 [02:26<10:10:56,  2.13it/s]
  0%|                                     | 95/78125 [02:26<10:11:10,  2.13it/s]
  0%|                                     | 96/78125 [02:27<10:11:53,  2.13it/s]
  0%|                                     | 97/78125 [02:27<10:11:50,  2.13it/s]
  0%|                                     | 98/78125 [02:28<10:12:12,  2.12it/s]
  0%|                                     | 99/78125 [02:28<10:12:04,  2.12it/s]
  0%|                                    | 100/78125 [02:29<10:12:06,  2.12it/s]
  0%|                                    | 101/78125 [02:29<10:13:47,  2.12it/s]
  0%|                                    | 102/78125 [02:30<10:13:13,  2.12it/s]
  0%|                                    | 103/78125 [02:30<10:13:23,  2.12it/s]
  0%|                                    | 104/78125 [02:31<10:13:57,  2.12it/s]
  0%|                                    | 105/78125 [02:31<10:14:30,  2.12it/s]
  0%|                                    | 106/78125 [02:32<10:15:34,  2.11it/s]
  0%|                                    | 107/78125 [02:32<10:16:08,  2.11it/s]
  0%|                                    | 108/78125 [02:33<10:16:22,  2.11it/s]
  0%|                                    | 109/78125 [02:33<10:17:04,  2.11it/s]
  0%|                                    | 110/78125 [02:34<10:17:24,  2.11it/s]
  0%|                                    | 111/78125 [02:34<10:17:30,  2.11it/s]
  0%|                                    | 112/78125 [02:35<10:18:38,  2.10it/s]
  0%|                                    | 113/78125 [02:35<10:19:24,  2.10it/s]
  0%|                                    | 114/78125 [02:35<10:19:29,  2.10it/s]
  0%|                                    | 115/78125 [02:36<10:20:18,  2.10it/s]
  0%|                                    | 116/78125 [02:36<10:20:33,  2.10it/s]
  0%|                                    | 117/78125 [02:37<10:20:28,  2.10it/s]
  0%|                                    | 118/78125 [02:37<10:21:48,  2.09it/s]
  0%|                                    | 119/78125 [02:38<10:21:23,  2.09it/s]
  0%|                                    | 120/78125 [02:38<10:22:01,  2.09it/s]
  0%|                                    | 121/78125 [02:39<10:21:51,  2.09it/s]
  0%|                                    | 122/78125 [02:39<10:22:01,  2.09it/s]
  0%|                                    | 123/78125 [02:40<10:22:07,  2.09it/s]
  0%|                                    | 124/78125 [02:40<10:23:05,  2.09it/s]
  0%|                                    | 125/78125 [02:41<10:23:38,  2.08it/s]
  0%|                                    | 126/78125 [02:41<10:24:06,  2.08it/s]
  0%|                                    | 127/78125 [02:42<10:25:08,  2.08it/s]
  0%|                                    | 128/78125 [02:42<10:26:08,  2.08it/s]
  0%|                                    | 129/78125 [02:43<10:26:14,  2.08it/s]
  0%|                                    | 130/78125 [02:43<10:26:42,  2.07it/s]
  0%|                                    | 131/78125 [02:44<10:27:07,  2.07it/s]
  0%|                                    | 132/78125 [02:44<10:26:53,  2.07it/s]
  0%|                                    | 133/78125 [02:45<10:27:12,  2.07it/s]
  0%|                                    | 134/78125 [02:45<10:27:43,  2.07it/s]
  0%|                                    | 135/78125 [02:46<10:27:33,  2.07it/s]
  0%|                                    | 136/78125 [02:46<10:27:46,  2.07it/s]
  0%|                                    | 137/78125 [02:47<10:28:21,  2.07it/s]
  0%|                                    | 138/78125 [02:47<10:29:01,  2.07it/s]
  0%|                                    | 139/78125 [02:47<10:29:18,  2.07it/s]
  0%|                                    | 140/78125 [02:48<10:29:25,  2.06it/s]
  0%|                                    | 141/78125 [02:48<10:29:47,  2.06it/s]
  0%|                                    | 142/78125 [02:49<10:30:56,  2.06it/s]
  0%|                                    | 143/78125 [02:49<10:31:29,  2.06it/s]
  0%|                                    | 144/78125 [02:50<10:32:22,  2.06it/s]
  0%|                                    | 145/78125 [02:50<10:33:13,  2.05it/s]
  0%|                                    | 146/78125 [02:51<10:33:15,  2.05it/s]
  0%|                                    | 147/78125 [02:51<10:33:48,  2.05it/s]
  0%|                                    | 148/78125 [02:52<10:34:22,  2.05it/s]
  0%|                                    | 149/78125 [02:52<10:34:18,  2.05it/s]
  0%|                                    | 150/78125 [02:53<10:34:57,  2.05it/s]
  0%|                                    | 151/78125 [02:53<10:34:48,  2.05it/s]
  0%|                                    | 152/78125 [02:54<10:34:44,  2.05it/s]
  0%|                                    | 153/78125 [02:54<10:35:26,  2.05it/s]
  0%|                                    | 154/78125 [02:55<10:35:51,  2.04it/s]
  0%|                                    | 155/78125 [02:55<10:36:06,  2.04it/s]
  0%|                                    | 156/78125 [02:56<10:36:55,  2.04it/s]
  0%|                                    | 157/78125 [02:56<10:37:26,  2.04it/s]
  0%|                                    | 158/78125 [02:57<10:38:03,  2.04it/s]
  0%|                                    | 159/78125 [02:57<10:38:17,  2.04it/s]
  0%|                                    | 160/78125 [02:58<10:37:44,  2.04it/s]
  0%|                                    | 161/78125 [02:58<10:38:01,  2.04it/s]
  0%|                                    | 162/78125 [02:59<10:39:19,  2.03it/s]
  0%|                                    | 163/78125 [02:59<10:39:27,  2.03it/s]
  0%|                                    | 164/78125 [03:00<10:39:58,  2.03it/s]
  0%|                                    | 165/78125 [03:00<10:39:58,  2.03it/s]
  0%|                                    | 166/78125 [03:01<10:40:44,  2.03it/s]
  0%|                                    | 167/78125 [03:01<10:41:13,  2.03it/s]
  0%|                                    | 168/78125 [03:02<10:41:41,  2.02it/s]
  0%|                                    | 169/78125 [03:02<10:39:31,  2.03it/s]
  0%|                                    | 170/78125 [03:03<10:40:17,  2.03it/s]
  0%|                                    | 171/78125 [03:03<10:40:53,  2.03it/s]
  0%|                                    | 172/78125 [03:04<10:41:35,  2.02it/s]
  0%|                                    | 173/78125 [03:04<10:42:35,  2.02it/s]
  0%|                                    | 174/78125 [03:05<10:43:25,  2.02it/s]
  0%|                                    | 175/78125 [03:05<10:43:13,  2.02it/s]
  0%|                                    | 176/78125 [03:06<10:43:58,  2.02it/s]
  0%|                                    | 177/78125 [03:06<10:44:25,  2.02it/s]
  0%|                                    | 178/78125 [03:07<10:44:30,  2.02it/s]
  0%|                                    | 179/78125 [03:07<10:45:29,  2.01it/s]
  0%|                                    | 180/78125 [03:08<10:45:46,  2.01it/s]
  0%|                                    | 181/78125 [03:08<10:46:02,  2.01it/s]
  0%|                                    | 182/78125 [03:09<10:46:23,  2.01it/s]
  0%|                                    | 183/78125 [03:09<10:48:13,  2.00it/s]
  0%|                                    | 184/78125 [03:10<10:48:17,  2.00it/s]
  0%|                                    | 185/78125 [03:10<10:48:47,  2.00it/s]
  0%|                                    | 186/78125 [03:11<10:48:56,  2.00it/s]
  0%|                                    | 187/78125 [03:11<10:49:06,  2.00it/s]
  0%|                                    | 188/78125 [03:12<10:49:20,  2.00it/s]
  0%|                                    | 189/78125 [03:12<10:50:01,  2.00it/s]
  0%|                                    | 190/78125 [03:13<10:50:21,  2.00it/s]
  0%|                                    | 191/78125 [03:13<10:50:40,  2.00it/s]
  0%|                                    | 192/78125 [03:14<10:50:49,  2.00it/s]
  0%|                                    | 193/78125 [03:14<10:51:11,  1.99it/s]
  0%|                                    | 194/78125 [03:15<10:51:28,  1.99it/s]
  0%|                                    | 195/78125 [03:15<10:51:20,  1.99it/s]
  0%|                                    | 196/78125 [03:16<10:51:57,  1.99it/s]
  0%|                                    | 197/78125 [03:16<10:52:22,  1.99it/s]
  0%|                                    | 198/78125 [03:17<10:53:02,  1.99it/s]
  0%|                                    | 199/78125 [03:17<10:53:26,  1.99it/s]
  0%|                                    | 200/78125 [03:18<10:53:28,  1.99it/s]
  0%|                                    | 201/78125 [03:18<10:54:02,  1.99it/s]
  0%|                                    | 202/78125 [03:19<10:54:39,  1.98it/s]
  0%|                                    | 203/78125 [03:19<10:54:34,  1.98it/s]
  0%|                                    | 204/78125 [03:20<10:54:42,  1.98it/s]
  0%|                                    | 205/78125 [03:20<10:54:50,  1.98it/s]
  0%|                                    | 206/78125 [03:21<10:55:06,  1.98it/s]
  0%|                                    | 207/78125 [03:21<10:55:00,  1.98it/s]
  0%|                                    | 208/78125 [03:22<10:55:01,  1.98it/s]
  0%|                                    | 209/78125 [03:22<10:55:56,  1.98it/s]
  0%|                                    | 210/78125 [03:23<10:55:28,  1.98it/s]
  0%|                                    | 211/78125 [03:23<10:54:18,  1.98it/s]
  0%|                                    | 212/78125 [03:24<10:54:10,  1.99it/s]
  0%|                                    | 213/78125 [03:24<10:55:25,  1.98it/s]
  0%|                                    | 214/78125 [03:25<10:54:41,  1.98it/s]
  0%|                                    | 215/78125 [03:25<10:54:02,  1.99it/s]
  0%|                                    | 216/78125 [03:26<10:53:33,  1.99it/s]
  0%|                                    | 217/78125 [03:26<10:53:34,  1.99it/s]
  0%|                                    | 218/78125 [03:27<10:53:53,  1.99it/s]
  0%|                                    | 219/78125 [03:27<10:54:09,  1.98it/s]
  0%|                                    | 220/78125 [03:28<10:54:18,  1.98it/s]
  0%|                                    | 221/78125 [03:28<10:53:54,  1.99it/s]
  0%|                                    | 222/78125 [03:29<10:53:10,  1.99it/s]
  0%|                                    | 223/78125 [03:29<10:52:12,  1.99it/s]
  0%|                                    | 224/78125 [03:30<10:50:19,  2.00it/s]
  0%|                                    | 225/78125 [03:30<10:50:19,  2.00it/s]
  0%|                                    | 226/78125 [03:31<10:50:09,  2.00it/s]
  0%|                                    | 227/78125 [03:31<10:49:49,  2.00it/s]
  0%|                                    | 228/78125 [03:32<10:49:11,  2.00it/s]
  0%|                                    | 229/78125 [03:32<10:48:03,  2.00it/s]
  0%|                                    | 230/78125 [03:33<10:47:41,  2.00it/s]
  0%|                                    | 231/78125 [03:33<10:46:27,  2.01it/s]
  0%|                                    | 232/78125 [03:34<10:45:31,  2.01it/s]
  0%|                                    | 233/78125 [03:34<10:45:41,  2.01it/s]
  0%|                                    | 234/78125 [03:35<10:44:57,  2.01it/s]
  0%|                                    | 235/78125 [03:35<10:44:02,  2.02it/s]
  0%|                                    | 236/78125 [03:36<10:44:25,  2.01it/s]
  0%|                                    | 237/78125 [03:36<10:44:18,  2.01it/s]
  0%|                                    | 238/78125 [03:37<10:43:42,  2.02it/s]
  0%|                                    | 239/78125 [03:37<10:43:19,  2.02it/s]
  0%|                                    | 240/78125 [03:38<10:43:09,  2.02it/s]
  0%|                                    | 241/78125 [03:38<10:42:58,  2.02it/s]
  0%|                                    | 242/78125 [03:39<10:42:23,  2.02it/s]
  0%|                                    | 243/78125 [03:39<10:41:36,  2.02it/s]
  0%|                                    | 244/78125 [03:40<10:41:43,  2.02it/s]
  0%|                                    | 245/78125 [03:40<10:42:06,  2.02it/s]
  0%|                                    | 246/78125 [03:41<10:42:10,  2.02it/s]
  0%|                                    | 247/78125 [03:41<10:42:04,  2.02it/s]
  0%|                                    | 248/78125 [03:42<10:42:01,  2.02it/s]
  0%|                                    | 249/78125 [03:42<10:41:24,  2.02it/s]
  0%|                                    | 250/78125 [03:43<10:40:18,  2.03it/s]
  0%|                                    | 251/78125 [03:43<10:40:39,  2.03it/s]
  0%|                                    | 252/78125 [03:44<10:41:03,  2.02it/s]
  0%|                                    | 253/78125 [03:44<10:41:07,  2.02it/s]
  0%|                                    | 254/78125 [03:45<10:40:51,  2.03it/s]
  0%|                                    | 255/78125 [03:45<10:40:44,  2.03it/s]
  0%|                                    | 256/78125 [03:46<10:40:23,  2.03it/s]
  0%|                                    | 257/78125 [03:46<10:40:45,  2.03it/s]
  0%|                                    | 258/78125 [03:47<10:38:24,  2.03it/s]
  0%|                                    | 259/78125 [03:47<10:38:49,  2.03it/s]
  0%|                                    | 260/78125 [03:48<10:39:10,  2.03it/s]
  0%|                                    | 261/78125 [03:48<10:39:19,  2.03it/s]
  0%|                                    | 262/78125 [03:49<10:39:40,  2.03it/s]
  0%|                                    | 263/78125 [03:49<10:38:52,  2.03it/s]
  0%|                                    | 264/78125 [03:50<10:38:51,  2.03it/s]
  0%|                                    | 265/78125 [03:50<10:39:06,  2.03it/s]
  0%|                                    | 266/78125 [03:51<10:39:09,  2.03it/s]
  0%|                                    | 267/78125 [03:51<10:39:33,  2.03it/s]
  0%|                                    | 268/78125 [03:52<10:39:28,  2.03it/s]
  0%|                                    | 269/78125 [03:52<10:38:24,  2.03it/s]
  0%|                                    | 270/78125 [03:53<10:37:31,  2.04it/s]
  0%|                                    | 271/78125 [03:53<10:37:21,  2.04it/s]
  0%|▏                                   | 272/78125 [03:53<10:38:28,  2.03it/s]
  0%|▏                                   | 273/78125 [03:54<10:38:08,  2.03it/s]
  0%|▏                                   | 274/78125 [03:54<10:38:33,  2.03it/s]
  0%|▏                                   | 275/78125 [03:55<10:38:22,  2.03it/s]
  0%|▏                                   | 276/78125 [03:55<10:38:13,  2.03it/s]
  0%|▏                                   | 277/78125 [03:56<10:38:13,  2.03it/s]
  0%|▏                                   | 278/78125 [03:56<10:35:50,  2.04it/s]
  0%|▏                                   | 279/78125 [03:57<10:36:59,  2.04it/s]
  0%|▏                                   | 280/78125 [03:57<10:36:14,  2.04it/s]
  0%|▏                                   | 281/78125 [03:58<10:35:23,  2.04it/s]
  0%|▏                                   | 282/78125 [03:58<10:36:05,  2.04it/s]
  0%|▏                                   | 283/78125 [03:59<10:34:58,  2.04it/s]
  0%|▏                                   | 284/78125 [03:59<10:36:06,  2.04it/s]
  0%|▏                                   | 285/78125 [04:00<10:37:01,  2.04it/s]
  0%|▏                                   | 286/78125 [04:00<10:36:32,  2.04it/s]
  0%|▏                                   | 287/78125 [04:01<10:35:16,  2.04it/s]
  0%|▏                                   | 288/78125 [04:01<10:35:46,  2.04it/s]
  0%|▏                                   | 289/78125 [04:02<10:36:35,  2.04it/s]
  0%|▏                                   | 290/78125 [04:02<10:36:45,  2.04it/s]
  0%|▏                                   | 291/78125 [04:03<10:37:00,  2.04it/s]
  0%|▏                                   | 292/78125 [04:03<10:35:17,  2.04it/s]
  0%|▏                                   | 293/78125 [04:04<10:35:14,  2.04it/s]
  0%|▏                                   | 294/78125 [04:04<10:35:15,  2.04it/s]
  0%|▏                                   | 295/78125 [04:05<10:34:01,  2.05it/s]
  0%|▏                                   | 296/78125 [04:05<10:34:03,  2.05it/s]
  0%|▏                                   | 297/78125 [04:06<10:35:14,  2.04it/s]
  0%|▏                                   | 298/78125 [04:06<10:34:28,  2.04it/s]
  0%|▏                                   | 299/78125 [04:07<10:34:00,  2.05it/s]
  0%|▏                                   | 300/78125 [04:07<10:35:50,  2.04it/s]
  0%|▏                                   | 301/78125 [04:08<10:36:08,  2.04it/s]
  0%|▏                                   | 302/78125 [04:08<10:35:51,  2.04it/s]
  0%|▏                                   | 303/78125 [04:09<10:36:22,  2.04it/s]
  0%|▏                                   | 304/78125 [04:09<10:35:55,  2.04it/s]
  0%|▏                                   | 305/78125 [04:10<10:35:20,  2.04it/s]
  0%|▏                                   | 306/78125 [04:10<10:37:01,  2.04it/s]
  0%|▏                                   | 307/78125 [04:11<10:37:17,  2.04it/s]
  0%|▏                                   | 308/78125 [04:11<10:34:56,  2.04it/s]
  0%|▏                                   | 309/78125 [04:12<10:35:04,  2.04it/s]
  0%|▏                                   | 310/78125 [04:12<10:36:12,  2.04it/s]
  0%|▏                                   | 311/78125 [04:13<10:36:11,  2.04it/s]
  0%|▏                                   | 312/78125 [04:13<10:36:32,  2.04it/s]
  0%|▏                                   | 313/78125 [04:14<10:37:23,  2.03it/s]
  0%|▏                                   | 314/78125 [04:14<10:37:03,  2.04it/s]
  0%|▏                                   | 315/78125 [04:15<10:36:52,  2.04it/s]
  0%|▏                                   | 316/78125 [04:15<10:35:14,  2.04it/s]
  0%|▏                                   | 317/78125 [04:16<10:35:47,  2.04it/s]
  0%|▏                                   | 318/78125 [04:16<10:36:12,  2.04it/s]
  0%|▏                                   | 319/78125 [04:17<10:37:21,  2.03it/s]
  0%|▏                                   | 320/78125 [04:17<10:37:33,  2.03it/s]
  0%|▏                                   | 321/78125 [04:18<10:37:18,  2.03it/s]
  0%|▏                                   | 322/78125 [04:18<10:35:22,  2.04it/s]
  0%|▏                                   | 323/78125 [04:19<10:35:54,  2.04it/s]
  0%|▏                                   | 324/78125 [04:19<10:36:38,  2.04it/s]
  0%|▏                                   | 325/78125 [04:19<10:38:24,  2.03it/s]
  0%|▏                                   | 326/78125 [04:20<10:38:55,  2.03it/s]
  0%|▏                                   | 327/78125 [04:20<10:39:32,  2.03it/s]
  0%|▏                                   | 328/78125 [04:21<10:39:18,  2.03it/s]
  0%|▏                                   | 329/78125 [04:21<10:37:00,  2.04it/s]
  0%|▏                                   | 330/78125 [04:22<10:37:43,  2.03it/s]
  0%|▏                                   | 331/78125 [04:22<10:38:23,  2.03it/s]
  0%|▏                                   | 332/78125 [04:23<10:38:31,  2.03it/s]
  0%|▏                                   | 333/78125 [04:23<10:38:17,  2.03it/s]
  0%|▏                                   | 334/78125 [04:24<10:38:10,  2.03it/s]
  0%|▏                                   | 335/78125 [04:24<10:38:10,  2.03it/s]
  0%|▏                                   | 336/78125 [04:25<10:37:45,  2.03it/s]
  0%|▏                                   | 337/78125 [04:25<10:37:03,  2.04it/s]
  0%|▏                                   | 338/78125 [04:26<10:37:35,  2.03it/s]
  0%|▏                                   | 339/78125 [04:26<10:37:50,  2.03it/s]
  0%|▏                                   | 340/78125 [04:27<10:38:19,  2.03it/s]
  0%|▏                                   | 341/78125 [04:27<10:39:04,  2.03it/s]
  0%|▏                                   | 342/78125 [04:28<10:39:56,  2.03it/s]
  0%|▏                                   | 343/78125 [04:28<10:39:07,  2.03it/s]
  0%|▏                                   | 344/78125 [04:29<10:39:48,  2.03it/s]
  0%|▏                                   | 345/78125 [04:29<10:39:52,  2.03it/s]
  0%|▏                                   | 346/78125 [04:30<10:39:37,  2.03it/s]
  0%|▏                                   | 347/78125 [04:30<10:39:42,  2.03it/s]
  0%|▏                                   | 348/78125 [04:31<10:39:43,  2.03it/s]
  0%|▏                                   | 349/78125 [04:31<10:40:17,  2.02it/s]
  0%|▏                                   | 350/78125 [04:32<10:39:52,  2.03it/s]
  0%|▏                                   | 351/78125 [04:32<10:40:16,  2.02it/s]
  0%|▏                                   | 352/78125 [04:33<10:39:17,  2.03it/s]
  0%|▏                                   | 353/78125 [04:33<10:39:12,  2.03it/s]
  0%|▏                                   | 354/78125 [04:34<10:39:29,  2.03it/s]
  0%|▏                                   | 355/78125 [04:34<10:40:11,  2.02it/s]
  0%|▏                                   | 356/78125 [04:35<10:40:02,  2.03it/s]
  0%|▏                                   | 357/78125 [04:35<10:40:32,  2.02it/s]
  0%|▏                                   | 358/78125 [04:36<10:40:15,  2.02it/s]
  0%|▏                                   | 359/78125 [04:36<10:40:04,  2.02it/s]
  0%|▏                                   | 360/78125 [04:37<10:39:53,  2.03it/s]
  0%|▏                                   | 361/78125 [04:37<10:40:14,  2.02it/s]
  0%|▏                                   | 362/78125 [04:38<10:40:41,  2.02it/s]
  0%|▏                                   | 363/78125 [04:38<10:40:50,  2.02it/s]
  0%|▏                                   | 364/78125 [04:39<10:41:19,  2.02it/s]
  0%|▏                                   | 365/78125 [04:39<10:41:08,  2.02it/s]
  0%|▏                                   | 366/78125 [04:40<10:41:16,  2.02it/s]
  0%|▏                                   | 367/78125 [04:40<10:41:22,  2.02it/s]
  0%|▏                                   | 368/78125 [04:41<10:42:37,  2.02it/s]
  0%|▏                                   | 369/78125 [04:41<10:42:20,  2.02it/s]
  0%|▏                                   | 370/78125 [04:42<10:41:37,  2.02it/s]
  0%|▏                                   | 371/78125 [04:42<10:41:29,  2.02it/s]
  0%|▏                                   | 372/78125 [04:43<10:41:42,  2.02it/s]
  0%|▏                                   | 373/78125 [04:43<10:41:48,  2.02it/s]
  0%|▏                                   | 374/78125 [04:44<10:41:52,  2.02it/s]
  0%|▏                                   | 375/78125 [04:44<10:42:58,  2.02it/s]
  0%|▏                                   | 376/78125 [04:45<10:43:07,  2.01it/s]
  0%|▏                                   | 377/78125 [04:45<10:43:07,  2.01it/s]
  0%|▏                                   | 378/78125 [04:46<10:44:01,  2.01it/s]
  0%|▏                                   | 379/78125 [04:46<10:43:31,  2.01it/s]
  0%|▏                                   | 380/78125 [04:47<10:42:41,  2.02it/s]
  0%|▏                                   | 381/78125 [04:47<10:42:49,  2.02it/s]
  0%|▏                                   | 382/78125 [04:48<10:43:05,  2.01it/s]
  0%|▏                                   | 383/78125 [04:48<10:43:30,  2.01it/s]
  0%|▏                                   | 384/78125 [04:49<10:44:43,  2.01it/s]
  0%|▏                                   | 385/78125 [04:49<10:44:57,  2.01it/s]
  0%|▏                                   | 386/78125 [04:50<10:44:35,  2.01it/s]
  0%|▏                                   | 387/78125 [04:50<10:44:34,  2.01it/s]
  0%|▏                                   | 388/78125 [04:51<10:43:54,  2.01it/s]
  0%|▏                                   | 389/78125 [04:51<10:42:44,  2.02it/s]
  0%|▏                                   | 390/78125 [04:52<10:42:31,  2.02it/s]
  1%|▏                                   | 391/78125 [04:52<10:43:03,  2.01it/s]
  1%|▏                                   | 392/78125 [04:53<10:42:07,  2.02it/s]
  1%|▏                                   | 393/78125 [04:53<10:42:32,  2.02it/s]
  1%|▏                                   | 394/78125 [04:54<10:43:07,  2.01it/s]
  1%|▏                                   | 395/78125 [04:54<10:43:47,  2.01it/s]
  1%|▏                                   | 396/78125 [04:55<10:43:36,  2.01it/s]
  1%|▏                                   | 397/78125 [04:55<10:43:08,  2.01it/s]
  1%|▏                                   | 398/78125 [04:56<10:43:19,  2.01it/s]
  1%|▏                                   | 399/78125 [04:56<10:43:23,  2.01it/s]
  1%|▏                                   | 400/78125 [04:57<10:42:52,  2.02it/s]
  1%|▏                                   | 401/78125 [04:57<10:42:16,  2.02it/s]
  1%|▏                                   | 402/78125 [04:58<10:42:02,  2.02it/s]
  1%|▏                                   | 403/78125 [04:58<10:42:10,  2.02it/s]
  1%|▏                                   | 404/78125 [04:59<10:42:04,  2.02it/s]
  1%|▏                                   | 405/78125 [04:59<10:42:22,  2.02it/s]
  1%|▏                                   | 406/78125 [05:00<10:41:53,  2.02it/s]
  1%|▏                                   | 407/78125 [05:00<10:42:42,  2.02it/s]
  1%|▏                                   | 408/78125 [05:01<10:43:19,  2.01it/s]
  1%|▏                                   | 409/78125 [05:01<10:42:18,  2.02it/s]
  1%|▏                                   | 410/78125 [05:02<10:41:58,  2.02it/s]
  1%|▏                                   | 411/78125 [05:02<10:41:43,  2.02it/s]
  1%|▏                                   | 412/78125 [05:03<10:41:55,  2.02it/s]
  1%|▏                                   | 413/78125 [05:03<10:41:50,  2.02it/s]
  1%|▏                                   | 414/78125 [05:04<10:41:55,  2.02it/s]
  1%|▏                                   | 415/78125 [05:04<10:41:45,  2.02it/s]
  1%|▏                                   | 416/78125 [05:05<10:41:06,  2.02it/s]
  1%|▏                                   | 417/78125 [05:05<10:40:10,  2.02it/s]
  1%|▏                                   | 418/78125 [05:06<10:41:12,  2.02it/s]
  1%|▏                                   | 419/78125 [05:06<10:40:58,  2.02it/s]
  1%|▏                                   | 420/78125 [05:07<10:40:38,  2.02it/s]
  1%|▏                                   | 421/78125 [05:07<10:40:51,  2.02it/s]
  1%|▏                                   | 422/78125 [05:07<10:41:54,  2.02it/s]
  1%|▏                                   | 423/78125 [05:08<10:41:32,  2.02it/s]
  1%|▏                                   | 424/78125 [05:08<10:41:15,  2.02it/s]
  1%|▏                                   | 425/78125 [05:09<10:41:11,  2.02it/s]
  1%|▏                                   | 426/78125 [05:09<10:40:24,  2.02it/s]
  1%|▏                                   | 427/78125 [05:10<10:40:58,  2.02it/s]
  1%|▏                                   | 428/78125 [05:10<10:41:15,  2.02it/s]
  1%|▏                                   | 429/78125 [05:11<10:42:05,  2.02it/s]
  1%|▏                                   | 430/78125 [05:11<10:42:16,  2.02it/s]
  1%|▏                                   | 431/78125 [05:12<10:42:32,  2.02it/s]
  1%|▏                                   | 432/78125 [05:12<10:42:39,  2.01it/s]
  1%|▏                                   | 433/78125 [05:13<10:41:26,  2.02it/s]
  1%|▏                                   | 434/78125 [05:13<10:40:15,  2.02it/s]
  1%|▏                                   | 435/78125 [05:14<10:40:29,  2.02it/s]
  1%|▏                                   | 436/78125 [05:14<10:40:22,  2.02it/s]
  1%|▏                                   | 437/78125 [05:15<10:39:15,  2.03it/s]
  1%|▏                                   | 438/78125 [05:15<10:39:05,  2.03it/s]
  1%|▏                                   | 439/78125 [05:16<10:38:54,  2.03it/s]
  1%|▏                                   | 440/78125 [05:16<10:39:50,  2.02it/s]
  1%|▏                                   | 441/78125 [05:17<10:39:38,  2.02it/s]
  1%|▏                                   | 442/78125 [05:17<10:38:54,  2.03it/s]
  1%|▏                                   | 443/78125 [05:18<10:39:01,  2.03it/s]
  1%|▏                                   | 444/78125 [05:18<10:37:53,  2.03it/s]
  1%|▏                                   | 445/78125 [05:19<10:38:32,  2.03it/s]
  1%|▏                                   | 446/78125 [05:19<10:37:30,  2.03it/s]
  1%|▏                                   | 447/78125 [05:20<10:38:54,  2.03it/s]
  1%|▏                                   | 448/78125 [05:20<10:39:15,  2.03it/s]
  1%|▏                                   | 449/78125 [05:21<10:38:31,  2.03it/s]
  1%|▏                                   | 450/78125 [05:21<10:38:52,  2.03it/s]
  1%|▏                                   | 451/78125 [05:22<10:39:06,  2.03it/s]
  1%|▏                                   | 452/78125 [05:22<10:39:04,  2.03it/s]
  1%|▏                                   | 453/78125 [05:23<10:39:44,  2.02it/s]
  1%|▏                                   | 454/78125 [05:23<10:38:57,  2.03it/s]
  1%|▏                                   | 455/78125 [05:24<10:39:42,  2.02it/s]
  1%|▏                                   | 456/78125 [05:24<10:39:26,  2.02it/s]
  1%|▏                                   | 457/78125 [05:25<10:39:14,  2.02it/s]
  1%|▏                                   | 458/78125 [05:25<10:39:05,  2.03it/s]
  1%|▏                                   | 459/78125 [05:26<10:39:33,  2.02it/s]
  1%|▏                                   | 460/78125 [05:26<10:37:29,  2.03it/s]
  1%|▏                                   | 461/78125 [05:27<10:37:59,  2.03it/s]
  1%|▏                                   | 462/78125 [05:27<10:38:41,  2.03it/s]
  1%|▏                                   | 463/78125 [05:28<10:39:10,  2.03it/s]
  1%|▏                                   | 464/78125 [05:28<10:38:37,  2.03it/s]
  1%|▏                                   | 465/78125 [05:29<10:38:38,  2.03it/s]
  1%|▏                                   | 466/78125 [05:29<10:37:20,  2.03it/s]
  1%|▏                                   | 467/78125 [05:30<10:37:47,  2.03it/s]
  1%|▏                                   | 468/78125 [05:30<10:37:55,  2.03it/s]
  1%|▏                                   | 469/78125 [05:31<10:38:31,  2.03it/s]
  1%|▏                                   | 470/78125 [05:31<10:37:24,  2.03it/s]
  1%|▏                                   | 471/78125 [05:32<10:37:24,  2.03it/s]
  1%|▏                                   | 472/78125 [05:32<10:37:09,  2.03it/s]
  1%|▏                                   | 473/78125 [05:33<10:38:19,  2.03it/s]
  1%|▏                                   | 474/78125 [05:33<10:38:31,  2.03it/s]
  1%|▏                                   | 475/78125 [05:34<10:38:42,  2.03it/s]
  1%|▏                                   | 476/78125 [05:34<10:38:05,  2.03it/s]
  1%|▏                                   | 477/78125 [05:35<10:38:25,  2.03it/s]
  1%|▏                                   | 478/78125 [05:35<10:38:15,  2.03it/s]
  1%|▏                                   | 479/78125 [05:36<10:38:47,  2.03it/s]
  1%|▏                                   | 480/78125 [05:36<10:38:11,  2.03it/s]
  1%|▏                                   | 481/78125 [05:37<10:38:20,  2.03it/s]
  1%|▏                                   | 482/78125 [05:37<10:37:45,  2.03it/s]
  1%|▏                                   | 483/78125 [05:38<10:37:41,  2.03it/s]
  1%|▏                                   | 484/78125 [05:38<10:38:24,  2.03it/s]
  1%|▏                                   | 485/78125 [05:39<10:38:14,  2.03it/s]
  1%|▏                                   | 486/78125 [05:39<10:38:49,  2.03it/s]
  1%|▏                                   | 487/78125 [05:40<10:38:54,  2.03it/s]
  1%|▏                                   | 488/78125 [05:40<10:39:04,  2.02it/s]
  1%|▏                                   | 489/78125 [05:41<10:37:57,  2.03it/s]
  1%|▏                                   | 490/78125 [05:41<10:37:52,  2.03it/s]
  1%|▏                                   | 491/78125 [05:42<10:38:20,  2.03it/s]
  1%|▏                                   | 492/78125 [05:42<10:38:17,  2.03it/s]
  1%|▏                                   | 493/78125 [05:43<10:39:09,  2.02it/s]
  1%|▏                                   | 494/78125 [05:43<10:38:47,  2.03it/s]
  1%|▏                                   | 495/78125 [05:44<10:37:22,  2.03it/s]
  1%|▏                                   | 496/78125 [05:44<10:37:37,  2.03it/s]
  1%|▏                                   | 497/78125 [05:45<10:36:52,  2.03it/s]
  1%|▏                                   | 498/78125 [05:45<10:37:40,  2.03it/s]
  1%|▏                                   | 499/78125 [05:46<10:38:00,  2.03it/s]
  1%|▏                                   | 500/78125 [05:46<10:38:45,  2.03it/s]
  1%|▏                                   | 501/78125 [05:46<10:39:17,  2.02it/s]
  1%|▏                                   | 502/78125 [05:47<10:38:47,  2.03it/s]
  1%|▏                                   | 503/78125 [05:47<10:38:59,  2.02it/s]
  1%|▏                                   | 504/78125 [05:48<10:39:22,  2.02it/s]
  1%|▏                                   | 505/78125 [05:48<10:39:35,  2.02it/s]
  1%|▏                                   | 506/78125 [05:49<10:39:19,  2.02it/s]
  1%|▏                                   | 507/78125 [05:49<10:38:50,  2.02it/s]
  1%|▏                                   | 508/78125 [05:50<10:39:03,  2.02it/s]
  1%|▏                                   | 509/78125 [05:50<10:37:46,  2.03it/s]
  1%|▏                                   | 510/78125 [05:51<10:37:52,  2.03it/s]
  1%|▏                                   | 511/78125 [05:51<10:37:27,  2.03it/s]
  1%|▏                                   | 512/78125 [05:52<10:37:44,  2.03it/s]
  1%|▏                                   | 513/78125 [05:52<10:38:39,  2.03it/s]
  1%|▏                                   | 514/78125 [05:53<10:38:35,  2.03it/s]
  1%|▏                                   | 515/78125 [05:53<10:38:51,  2.02it/s]
  1%|▏                                   | 516/78125 [05:54<10:39:24,  2.02it/s]
  1%|▏                                   | 517/78125 [05:54<10:38:57,  2.02it/s]
  1%|▏                                   | 518/78125 [05:55<10:38:44,  2.03it/s]
  1%|▏                                   | 519/78125 [05:55<10:38:44,  2.02it/s]
  1%|▏                                   | 520/78125 [05:56<10:38:15,  2.03it/s]
  1%|▏                                   | 521/78125 [05:56<10:38:41,  2.03it/s]
  1%|▏                                   | 522/78125 [05:57<10:39:02,  2.02it/s]
  1%|▏                                   | 523/78125 [05:57<10:38:00,  2.03it/s]
  1%|▏                                   | 524/78125 [05:58<10:37:32,  2.03it/s]
  1%|▏                                   | 525/78125 [05:58<10:37:56,  2.03it/s]
  1%|▏                                   | 526/78125 [05:59<10:38:34,  2.03it/s]
  1%|▏                                   | 527/78125 [05:59<10:37:54,  2.03it/s]
  1%|▏                                   | 528/78125 [06:00<10:38:35,  2.03it/s]
  1%|▏                                   | 529/78125 [06:00<10:38:32,  2.03it/s]
  1%|▏                                   | 530/78125 [06:01<10:38:17,  2.03it/s]
  1%|▏                                   | 531/78125 [06:01<10:38:25,  2.03it/s]
  1%|▏                                   | 532/78125 [06:02<10:38:57,  2.02it/s]
  1%|▏                                   | 533/78125 [06:02<10:39:12,  2.02it/s]
  1%|▏                                   | 534/78125 [06:03<10:39:10,  2.02it/s]
  1%|▏                                   | 535/78125 [06:03<10:39:00,  2.02it/s]
  1%|▏                                   | 536/78125 [06:04<10:39:16,  2.02it/s]
  1%|▏                                   | 537/78125 [06:04<10:40:04,  2.02it/s]
  1%|▏                                   | 538/78125 [06:05<10:39:39,  2.02it/s]
  1%|▏                                   | 539/78125 [06:05<10:39:54,  2.02it/s]
  1%|▏                                   | 540/78125 [06:06<10:39:34,  2.02it/s]
  1%|▏                                   | 541/78125 [06:06<10:40:12,  2.02it/s]
  1%|▏                                   | 542/78125 [06:07<10:40:06,  2.02it/s]
  1%|▎                                   | 543/78125 [06:07<10:39:57,  2.02it/s]
  1%|▎                                   | 544/78125 [06:08<10:39:55,  2.02it/s]
  1%|▎                                   | 545/78125 [06:08<10:38:39,  2.02it/s]
  1%|▎                                   | 546/78125 [06:09<10:38:27,  2.03it/s]
  1%|▎                                   | 547/78125 [06:09<10:39:12,  2.02it/s]
  1%|▎                                   | 548/78125 [06:10<10:39:26,  2.02it/s]
  1%|▎                                   | 549/78125 [06:10<10:39:37,  2.02it/s]
  1%|▎                                   | 550/78125 [06:11<10:39:47,  2.02it/s]
  1%|▎                                   | 551/78125 [06:11<10:39:54,  2.02it/s]
  1%|▎                                   | 552/78125 [06:12<10:39:00,  2.02it/s]
  1%|▎                                   | 553/78125 [06:12<10:39:11,  2.02it/s]
  1%|▎                                   | 554/78125 [06:13<10:39:06,  2.02it/s]
  1%|▎                                   | 555/78125 [06:13<10:38:26,  2.02it/s]
  1%|▎                                   | 556/78125 [06:14<10:38:46,  2.02it/s]
  1%|▎                                   | 557/78125 [06:14<10:39:10,  2.02it/s]
  1%|▎                                   | 558/78125 [06:15<10:39:36,  2.02it/s]
  1%|▎                                   | 559/78125 [06:15<10:39:44,  2.02it/s]
  1%|▎                                   | 560/78125 [06:16<10:40:50,  2.02it/s]
  1%|▎                                   | 561/78125 [06:16<10:40:21,  2.02it/s]
  1%|▎                                   | 562/78125 [06:17<10:40:12,  2.02it/s]
  1%|▎                                   | 563/78125 [06:17<10:39:20,  2.02it/s]
  1%|▎                                   | 564/78125 [06:18<10:39:54,  2.02it/s]
  1%|▎                                   | 565/78125 [06:18<10:40:07,  2.02it/s]
  1%|▎                                   | 566/78125 [06:19<10:39:00,  2.02it/s]
  1%|▎                                   | 567/78125 [06:19<10:39:42,  2.02it/s]
  1%|▎                                   | 568/78125 [06:20<10:39:52,  2.02it/s]
  1%|▎                                   | 569/78125 [06:20<10:40:26,  2.02it/s]
  1%|▎                                   | 570/78125 [06:21<10:39:56,  2.02it/s]
  1%|▎                                   | 571/78125 [06:21<10:40:24,  2.02it/s]
  1%|▎                                   | 572/78125 [06:22<10:40:19,  2.02it/s]
  1%|▎                                   | 573/78125 [06:22<10:39:44,  2.02it/s]
  1%|▎                                   | 574/78125 [06:23<10:38:52,  2.02it/s]
  1%|▎                                   | 575/78125 [06:23<10:39:40,  2.02it/s]
  1%|▎                                   | 576/78125 [06:24<10:39:47,  2.02it/s]
  1%|▎                                   | 577/78125 [06:24<10:38:55,  2.02it/s]
  1%|▎                                   | 578/78125 [06:25<10:38:04,  2.03it/s]
  1%|▎                                   | 579/78125 [06:25<10:38:33,  2.02it/s]
  1%|▎                                   | 580/78125 [06:26<10:39:21,  2.02it/s]
  1%|▎                                   | 581/78125 [06:26<10:39:22,  2.02it/s]
  1%|▎                                   | 582/78125 [06:27<10:38:50,  2.02it/s]
  1%|▎                                   | 583/78125 [06:27<10:39:46,  2.02it/s]
  1%|▎                                   | 584/78125 [06:28<10:39:26,  2.02it/s]
  1%|▎                                   | 585/78125 [06:28<10:39:09,  2.02it/s]
  1%|▎                                   | 586/78125 [06:29<10:37:56,  2.03it/s]
  1%|▎                                   | 587/78125 [06:29<10:38:12,  2.02it/s]
  1%|▎                                   | 588/78125 [06:29<10:38:09,  2.03it/s]
  1%|▎                                   | 589/78125 [06:30<10:38:52,  2.02it/s]
  1%|▎                                   | 590/78125 [06:30<10:38:51,  2.02it/s]
  1%|▎                                   | 591/78125 [06:31<10:39:12,  2.02it/s]
  1%|▎                                   | 592/78125 [06:31<10:38:59,  2.02it/s]
  1%|▎                                   | 593/78125 [06:32<10:39:12,  2.02it/s]
  1%|▎                                   | 594/78125 [06:32<10:39:08,  2.02it/s]
  1%|▎                                   | 595/78125 [06:33<10:39:19,  2.02it/s]
  1%|▎                                   | 596/78125 [06:33<10:38:33,  2.02it/s]
  1%|▎                                   | 597/78125 [06:34<10:38:28,  2.02it/s]
  1%|▎                                   | 598/78125 [06:34<10:38:00,  2.03it/s]
  1%|▎                                   | 599/78125 [06:35<10:38:23,  2.02it/s]
  1%|▎                                   | 600/78125 [06:35<10:37:50,  2.03it/s]
  1%|▎                                   | 601/78125 [06:36<10:38:04,  2.02it/s]
  1%|▎                                   | 602/78125 [06:36<10:38:28,  2.02it/s]
  1%|▎                                   | 603/78125 [06:37<10:38:39,  2.02it/s]
  1%|▎                                   | 604/78125 [06:37<10:40:01,  2.02it/s]
  1%|▎                                   | 605/78125 [06:38<10:39:29,  2.02it/s]
  1%|▎                                   | 606/78125 [06:38<10:39:09,  2.02it/s]
  1%|▎                                   | 607/78125 [06:39<10:39:04,  2.02it/s]
  1%|▎                                   | 608/78125 [06:39<10:39:23,  2.02it/s]
  1%|▎                                   | 609/78125 [06:40<10:38:34,  2.02it/s]
  1%|▎                                   | 610/78125 [06:40<10:38:38,  2.02it/s]
  1%|▎                                   | 611/78125 [06:41<10:39:34,  2.02it/s]
  1%|▎                                   | 612/78125 [06:41<10:40:00,  2.02it/s]
  1%|▎                                   | 613/78125 [06:42<10:40:07,  2.02it/s]
  1%|▎                                   | 614/78125 [06:42<10:40:38,  2.02it/s]
  1%|▎                                   | 615/78125 [06:43<10:40:10,  2.02it/s]
  1%|▎                                   | 616/78125 [06:43<10:40:24,  2.02it/s]
  1%|▎                                   | 617/78125 [06:44<10:40:52,  2.02it/s]
  1%|▎                                   | 618/78125 [06:44<10:40:57,  2.02it/s]
  1%|▎                                   | 619/78125 [06:45<10:40:32,  2.02it/s]
  1%|▎                                   | 620/78125 [06:45<10:39:54,  2.02it/s]
  1%|▎                                   | 621/78125 [06:46<10:39:47,  2.02it/s]
  1%|▎                                   | 622/78125 [06:46<10:39:34,  2.02it/s]
  1%|▎                                   | 623/78125 [06:47<10:38:15,  2.02it/s]
  1%|▎                                   | 624/78125 [06:47<10:38:45,  2.02it/s]
  1%|▎                                   | 625/78125 [06:48<10:38:41,  2.02it/s]
  1%|▎                                   | 626/78125 [06:48<10:39:07,  2.02it/s]
  1%|▎                                   | 627/78125 [06:49<10:38:55,  2.02it/s]
  1%|▎                                   | 628/78125 [06:49<10:38:59,  2.02it/s]
  1%|▎                                   | 629/78125 [06:50<10:39:00,  2.02it/s]
  1%|▎                                   | 630/78125 [06:50<10:38:45,  2.02it/s]
  1%|▎                                   | 631/78125 [06:51<10:39:00,  2.02it/s]
  1%|▎                                   | 632/78125 [06:51<10:38:58,  2.02it/s]
  1%|▎                                   | 633/78125 [06:52<10:38:33,  2.02it/s]
  1%|▎                                   | 634/78125 [06:52<10:39:07,  2.02it/s]
  1%|▎                                   | 635/78125 [06:53<10:39:17,  2.02it/s]
  1%|▎                                   | 636/78125 [06:53<10:39:28,  2.02it/s]
  1%|▎                                   | 637/78125 [06:54<10:39:41,  2.02it/s]
  1%|▎                                   | 638/78125 [06:54<10:40:07,  2.02it/s]
  1%|▎                                   | 639/78125 [06:55<10:38:49,  2.02it/s]
  1%|▎                                   | 640/78125 [06:55<10:39:18,  2.02it/s]
  1%|▎                                   | 641/78125 [06:56<10:38:23,  2.02it/s]
  1%|▎                                   | 642/78125 [06:56<10:38:13,  2.02it/s]
  1%|▎                                   | 643/78125 [06:57<10:38:38,  2.02it/s]
  1%|▎                                   | 644/78125 [06:57<10:39:23,  2.02it/s]
  1%|▎                                   | 645/78125 [06:58<10:39:52,  2.02it/s]
  1%|▎                                   | 646/78125 [06:58<10:39:06,  2.02it/s]
  1%|▎                                   | 647/78125 [06:59<10:39:00,  2.02it/s]
  1%|▎                                   | 648/78125 [06:59<10:39:46,  2.02it/s]
  1%|▎                                   | 649/78125 [07:00<10:38:52,  2.02it/s]
  1%|▎                                   | 650/78125 [07:00<10:39:02,  2.02it/s]
  1%|▎                                   | 651/78125 [07:01<10:38:51,  2.02it/s]
  1%|▎                                   | 652/78125 [07:01<10:39:22,  2.02it/s]
  1%|▎                                   | 653/78125 [07:02<10:39:06,  2.02it/s]
  1%|▎                                   | 654/78125 [07:02<10:38:48,  2.02it/s]
  1%|▎                                   | 655/78125 [07:03<10:39:01,  2.02it/s]
  1%|▎                                   | 656/78125 [07:03<10:39:18,  2.02it/s]
  1%|▎                                   | 657/78125 [07:04<10:38:40,  2.02it/s]
  1%|▎                                   | 658/78125 [07:04<10:39:09,  2.02it/s]
  1%|▎                                   | 659/78125 [07:05<10:39:06,  2.02it/s]
  1%|▎                                   | 660/78125 [07:05<10:39:01,  2.02it/s]
  1%|▎                                   | 661/78125 [07:06<10:38:31,  2.02it/s]
  1%|▎                                   | 662/78125 [07:06<10:38:06,  2.02it/s]
  1%|▎                                   | 663/78125 [07:07<10:38:12,  2.02it/s]
  1%|▎                                   | 664/78125 [07:07<10:38:54,  2.02it/s]
  1%|▎                                   | 665/78125 [07:08<10:39:07,  2.02it/s]
  1%|▎                                   | 666/78125 [07:08<10:39:12,  2.02it/s]
  1%|▎                                   | 667/78125 [07:09<10:40:20,  2.02it/s]
  1%|▎                                   | 668/78125 [07:09<10:39:47,  2.02it/s]
  1%|▎                                   | 669/78125 [07:10<10:40:16,  2.02it/s]
  1%|▎                                   | 670/78125 [07:10<10:39:48,  2.02it/s]
  1%|▎                                   | 671/78125 [07:11<10:39:11,  2.02it/s]
  1%|▎                                   | 672/78125 [07:11<10:39:10,  2.02it/s]
  1%|▎                                   | 673/78125 [07:12<10:38:29,  2.02it/s]
  1%|▎                                   | 674/78125 [07:12<10:39:00,  2.02it/s]
  1%|▎                                   | 675/78125 [07:13<10:38:50,  2.02it/s]
  1%|▎                                   | 676/78125 [07:13<10:38:33,  2.02it/s]
  1%|▎                                   | 677/78125 [07:14<10:38:41,  2.02it/s]
  1%|▎                                   | 678/78125 [07:14<10:39:08,  2.02it/s]
  1%|▎                                   | 679/78125 [07:15<10:39:08,  2.02it/s]
  1%|▎                                   | 680/78125 [07:15<10:40:18,  2.02it/s]
  1%|▎                                   | 681/78125 [07:16<10:39:56,  2.02it/s]
  1%|▎                                   | 682/78125 [07:16<10:40:08,  2.02it/s]
  1%|▎                                   | 683/78125 [07:17<10:40:25,  2.02it/s]
  1%|▎                                   | 684/78125 [07:17<10:39:39,  2.02it/s]
  1%|▎                                   | 685/78125 [07:18<10:40:06,  2.02it/s]
  1%|▎                                   | 686/78125 [07:18<10:40:00,  2.02it/s]
  1%|▎                                   | 687/78125 [07:19<10:39:58,  2.02it/s]
  1%|▎                                   | 688/78125 [07:19<10:40:06,  2.02it/s]
  1%|▎                                   | 689/78125 [07:19<10:39:53,  2.02it/s]
  1%|▎                                   | 690/78125 [07:20<10:39:44,  2.02it/s]
  1%|▎                                   | 691/78125 [07:20<10:39:50,  2.02it/s]
  1%|▎                                   | 692/78125 [07:21<10:38:54,  2.02it/s]
  1%|▎                                   | 693/78125 [07:21<10:39:42,  2.02it/s]
  1%|▎                                   | 694/78125 [07:22<10:39:59,  2.02it/s]
  1%|▎                                   | 695/78125 [07:22<10:40:19,  2.02it/s]
  1%|▎                                   | 696/78125 [07:23<10:40:17,  2.02it/s]
  1%|▎                                   | 697/78125 [07:23<10:39:10,  2.02it/s]
  1%|▎                                   | 698/78125 [07:24<10:39:08,  2.02it/s]
  1%|▎                                   | 699/78125 [07:24<10:39:43,  2.02it/s]
  1%|▎                                   | 700/78125 [07:25<10:40:03,  2.02it/s]
  1%|▎                                   | 701/78125 [07:25<10:40:29,  2.01it/s]
  1%|▎                                   | 702/78125 [07:26<10:40:17,  2.02it/s]
  1%|▎                                   | 703/78125 [07:26<10:40:15,  2.02it/s]
  1%|▎                                   | 704/78125 [07:27<10:40:29,  2.01it/s]
  1%|▎                                   | 705/78125 [07:27<10:39:20,  2.02it/s]
  1%|▎                                   | 706/78125 [07:28<10:39:34,  2.02it/s]
  1%|▎                                   | 707/78125 [07:28<10:39:42,  2.02it/s]
  1%|▎                                   | 708/78125 [07:29<10:39:50,  2.02it/s]
  1%|▎                                   | 709/78125 [07:29<10:40:00,  2.02it/s]
  1%|▎                                   | 710/78125 [07:30<10:40:01,  2.02it/s]
  1%|▎                                   | 711/78125 [07:30<10:40:17,  2.02it/s]
  1%|▎                                   | 712/78125 [07:31<10:40:12,  2.02it/s]
  1%|▎                                   | 713/78125 [07:31<10:39:54,  2.02it/s]
  1%|▎                                   | 714/78125 [07:32<10:39:22,  2.02it/s]
  1%|▎                                   | 715/78125 [07:32<10:39:30,  2.02it/s]
  1%|▎                                   | 716/78125 [07:33<10:39:49,  2.02it/s]
  1%|▎                                   | 717/78125 [07:33<10:39:50,  2.02it/s]
  1%|▎                                   | 718/78125 [07:34<10:40:09,  2.02it/s]
  1%|▎                                   | 719/78125 [07:34<10:40:13,  2.02it/s]
  1%|▎                                   | 720/78125 [07:35<10:39:44,  2.02it/s]
  1%|▎                                   | 721/78125 [07:35<10:39:54,  2.02it/s]
  1%|▎                                   | 722/78125 [07:36<10:39:29,  2.02it/s]
  1%|▎                                   | 723/78125 [07:36<10:40:02,  2.02it/s]
  1%|▎                                   | 724/78125 [07:37<10:41:02,  2.01it/s]
  1%|▎                                   | 725/78125 [07:37<10:40:21,  2.01it/s]
  1%|▎                                   | 726/78125 [07:38<10:39:30,  2.02it/s]
  1%|▎                                   | 727/78125 [07:38<10:40:07,  2.02it/s]
  1%|▎                                   | 728/78125 [07:39<10:40:08,  2.02it/s]
  1%|▎                                   | 729/78125 [07:39<10:39:41,  2.02it/s]
  1%|▎                                   | 730/78125 [07:40<10:39:53,  2.02it/s]
  1%|▎                                   | 731/78125 [07:40<10:39:32,  2.02it/s]
  1%|▎                                   | 732/78125 [07:41<10:40:10,  2.01it/s]
  1%|▎                                   | 733/78125 [07:41<10:40:32,  2.01it/s]
  1%|▎                                   | 734/78125 [07:42<10:40:18,  2.01it/s]
  1%|▎                                   | 735/78125 [07:42<10:40:51,  2.01it/s]
  1%|▎                                   | 736/78125 [07:43<10:41:02,  2.01it/s]
  1%|▎                                   | 737/78125 [07:43<10:39:50,  2.02it/s]
  1%|▎                                   | 738/78125 [07:44<10:40:08,  2.01it/s]
  1%|▎                                   | 739/78125 [07:44<10:40:34,  2.01it/s]
  1%|▎                                   | 740/78125 [07:45<10:39:49,  2.02it/s]
  1%|▎                                   | 741/78125 [07:45<10:39:00,  2.02it/s]
  1%|▎                                   | 742/78125 [07:46<10:38:49,  2.02it/s]
  1%|▎                                   | 743/78125 [07:46<10:39:07,  2.02it/s]
  1%|▎                                   | 744/78125 [07:47<10:39:53,  2.02it/s]
  1%|▎                                   | 745/78125 [07:47<10:39:12,  2.02it/s]
  1%|▎                                   | 746/78125 [07:48<10:38:58,  2.02it/s]
  1%|▎                                   | 747/78125 [07:48<10:39:11,  2.02it/s]
  1%|▎                                   | 748/78125 [07:49<10:39:03,  2.02it/s]
  1%|▎                                   | 749/78125 [07:49<10:39:03,  2.02it/s]
  1%|▎                                   | 750/78125 [07:50<10:39:11,  2.02it/s]
  1%|▎                                   | 751/78125 [07:50<10:39:10,  2.02it/s]
  1%|▎                                   | 752/78125 [07:51<10:39:51,  2.02it/s]
  1%|▎                                   | 753/78125 [07:51<10:38:46,  2.02it/s]
  1%|▎                                   | 754/78125 [07:52<10:38:36,  2.02it/s]
  1%|▎                                   | 755/78125 [07:52<10:38:38,  2.02it/s]
  1%|▎                                   | 756/78125 [07:53<10:38:31,  2.02it/s]
  1%|▎                                   | 757/78125 [07:53<10:38:36,  2.02it/s]
  1%|▎                                   | 758/78125 [07:54<10:39:05,  2.02it/s]
  1%|▎                                   | 759/78125 [07:54<10:39:45,  2.02it/s]
  1%|▎                                   | 760/78125 [07:55<10:39:16,  2.02it/s]
  1%|▎                                   | 761/78125 [07:55<10:38:31,  2.02it/s]
  1%|▎                                   | 762/78125 [07:56<10:38:52,  2.02it/s]
  1%|▎                                   | 763/78125 [07:56<10:38:59,  2.02it/s]
  1%|▎                                   | 764/78125 [07:57<10:38:47,  2.02it/s]
  1%|▎                                   | 765/78125 [07:57<10:39:27,  2.02it/s]
  1%|▎                                   | 766/78125 [07:58<10:39:30,  2.02it/s]
  1%|▎                                   | 767/78125 [07:58<10:39:33,  2.02it/s]
  1%|▎                                   | 768/78125 [07:59<10:39:46,  2.02it/s]
  1%|▎                                   | 769/78125 [07:59<10:39:07,  2.02it/s]
  1%|▎                                   | 770/78125 [08:00<10:39:21,  2.02it/s]
  1%|▎                                   | 771/78125 [08:00<10:39:47,  2.02it/s]
  1%|▎                                   | 772/78125 [08:01<10:39:50,  2.01it/s]
  1%|▎                                   | 773/78125 [08:01<10:39:52,  2.01it/s]
  1%|▎                                   | 774/78125 [08:02<10:39:47,  2.02it/s]
  1%|▎                                   | 775/78125 [08:02<10:39:34,  2.02it/s]
  1%|▎                                   | 776/78125 [08:03<10:38:35,  2.02it/s]
  1%|▎                                   | 777/78125 [08:03<10:38:08,  2.02it/s]
  1%|▎                                   | 778/78125 [08:04<10:38:31,  2.02it/s]
  1%|▎                                   | 779/78125 [08:04<10:38:57,  2.02it/s]
  1%|▎                                   | 780/78125 [08:05<10:39:06,  2.02it/s]
  1%|▎                                   | 781/78125 [08:05<10:39:07,  2.02it/s]
  1%|▎                                   | 782/78125 [08:06<10:38:26,  2.02it/s]
  1%|▎                                   | 783/78125 [08:06<10:38:24,  2.02it/s]
  1%|▎                                   | 784/78125 [08:07<10:38:05,  2.02it/s]
  1%|▎                                   | 785/78125 [08:07<10:37:42,  2.02it/s]
  1%|▎                                   | 786/78125 [08:08<10:37:35,  2.02it/s]
  1%|▎                                   | 787/78125 [08:08<10:38:00,  2.02it/s]
  1%|▎                                   | 788/78125 [08:09<10:37:52,  2.02it/s]
  1%|▎                                   | 789/78125 [08:09<10:38:07,  2.02it/s]
  1%|▎                                   | 790/78125 [08:10<10:38:16,  2.02it/s]
  1%|▎                                   | 791/78125 [08:10<10:38:08,  2.02it/s]
  1%|▎                                   | 792/78125 [08:11<10:37:40,  2.02it/s]
  1%|▎                                   | 793/78125 [08:11<10:38:16,  2.02it/s]
  1%|▎                                   | 794/78125 [08:12<10:38:22,  2.02it/s]
  1%|▎                                   | 795/78125 [08:12<10:38:08,  2.02it/s]
  1%|▎                                   | 796/78125 [08:13<10:38:12,  2.02it/s]
  1%|▎                                   | 797/78125 [08:13<10:37:52,  2.02it/s]
  1%|▎                                   | 798/78125 [08:14<10:37:34,  2.02it/s]
  1%|▎                                   | 799/78125 [08:14<10:37:22,  2.02it/s]
  1%|▎                                   | 800/78125 [08:15<10:36:52,  2.02it/s]
  1%|▎                                   | 801/78125 [08:15<10:36:14,  2.03it/s]
  1%|▎                                   | 802/78125 [08:16<10:36:07,  2.03it/s]
  1%|▎                                   | 803/78125 [08:16<10:36:34,  2.02it/s]
  1%|▎                                   | 804/78125 [08:16<10:37:13,  2.02it/s]
  1%|▎                                   | 805/78125 [08:17<10:37:31,  2.02it/s]
  1%|▎                                   | 806/78125 [08:17<10:36:49,  2.02it/s]
  1%|▎                                   | 807/78125 [08:18<10:37:25,  2.02it/s]
  1%|▎                                   | 808/78125 [08:18<10:37:12,  2.02it/s]
  1%|▎                                   | 809/78125 [08:19<10:37:37,  2.02it/s]
  1%|▎                                   | 810/78125 [08:19<10:37:13,  2.02it/s]
  1%|▎                                   | 811/78125 [08:20<10:37:07,  2.02it/s]
  1%|▎                                   | 812/78125 [08:20<10:37:18,  2.02it/s]
  1%|▎                                   | 813/78125 [08:21<10:37:22,  2.02it/s]
  1%|▍                                   | 814/78125 [08:21<10:37:10,  2.02it/s]
  1%|▍                                   | 815/78125 [08:22<10:38:09,  2.02it/s]
  1%|▍                                   | 816/78125 [08:22<10:38:29,  2.02it/s]
  1%|▍                                   | 817/78125 [08:23<10:37:32,  2.02it/s]
  1%|▍                                   | 818/78125 [08:23<10:37:58,  2.02it/s]
  1%|▍                                   | 819/78125 [08:24<10:38:40,  2.02it/s]
  1%|▍                                   | 820/78125 [08:24<10:38:17,  2.02it/s]
  1%|▍                                   | 821/78125 [08:25<10:37:57,  2.02it/s]
  1%|▍                                   | 822/78125 [08:25<10:37:30,  2.02it/s]
  1%|▍                                   | 823/78125 [08:26<10:37:57,  2.02it/s]
  1%|▍                                   | 824/78125 [08:26<10:38:10,  2.02it/s]
  1%|▍                                   | 825/78125 [08:27<10:37:57,  2.02it/s]
  1%|▍                                   | 826/78125 [08:27<10:37:31,  2.02it/s]
  1%|▍                                   | 827/78125 [08:28<10:37:11,  2.02it/s]
  1%|▍                                   | 828/78125 [08:28<10:37:27,  2.02it/s]
  1%|▍                                   | 829/78125 [08:29<10:37:50,  2.02it/s]
  1%|▍                                   | 830/78125 [08:29<10:38:10,  2.02it/s]
  1%|▍                                   | 831/78125 [08:30<10:38:27,  2.02it/s]
  1%|▍                                   | 832/78125 [08:30<10:38:03,  2.02it/s]
  1%|▍                                   | 833/78125 [08:31<10:38:29,  2.02it/s]
  1%|▍                                   | 834/78125 [08:31<10:38:09,  2.02it/s]
  1%|▍                                   | 835/78125 [08:32<10:37:57,  2.02it/s]
  1%|▍                                   | 836/78125 [08:32<10:38:38,  2.02it/s]
  1%|▍                                   | 837/78125 [08:33<10:38:16,  2.02it/s]
  1%|▍                                   | 838/78125 [08:33<10:38:13,  2.02it/s]
  1%|▍                                   | 839/78125 [08:34<10:38:31,  2.02it/s]
  1%|▍                                   | 840/78125 [08:34<10:38:24,  2.02it/s]
  1%|▍                                   | 841/78125 [08:35<10:38:34,  2.02it/s]
  1%|▍                                   | 842/78125 [08:35<10:38:54,  2.02it/s]
  1%|▍                                   | 843/78125 [08:36<10:38:18,  2.02it/s]
  1%|▍                                   | 844/78125 [08:36<10:38:44,  2.02it/s]
  1%|▍                                   | 845/78125 [08:37<10:38:43,  2.02it/s]
  1%|▍                                   | 846/78125 [08:37<10:38:54,  2.02it/s]
  1%|▍                                   | 847/78125 [08:38<10:38:18,  2.02it/s]
  1%|▍                                   | 848/78125 [08:38<10:38:03,  2.02it/s]
  1%|▍                                   | 849/78125 [08:39<10:37:37,  2.02it/s]
  1%|▍                                   | 850/78125 [08:39<10:38:15,  2.02it/s]
  1%|▍                                   | 851/78125 [08:40<10:38:41,  2.02it/s]
  1%|▍                                   | 852/78125 [08:40<10:39:27,  2.01it/s]
  1%|▍                                   | 853/78125 [08:41<10:39:22,  2.01it/s]
  1%|▍                                   | 854/78125 [08:41<10:38:49,  2.02it/s]
  1%|▍                                   | 855/78125 [08:42<10:38:26,  2.02it/s]
  1%|▍                                   | 856/78125 [08:42<10:38:05,  2.02it/s]
  1%|▍                                   | 857/78125 [08:43<10:37:52,  2.02it/s]
  1%|▍                                   | 858/78125 [08:43<10:37:58,  2.02it/s]
  1%|▍                                   | 859/78125 [08:44<10:38:13,  2.02it/s]
  1%|▍                                   | 860/78125 [08:44<10:38:25,  2.02it/s]
  1%|▍                                   | 861/78125 [08:45<10:38:24,  2.02it/s]
  1%|▍                                   | 862/78125 [08:45<10:38:14,  2.02it/s]
  1%|▍                                   | 863/78125 [08:46<10:38:46,  2.02it/s]
  1%|▍                                   | 864/78125 [08:46<10:38:12,  2.02it/s]
  1%|▍                                   | 865/78125 [08:47<10:38:20,  2.02it/s]
  1%|▍                                   | 866/78125 [08:47<10:38:17,  2.02it/s]
  1%|▍                                   | 867/78125 [08:48<10:37:44,  2.02it/s]
  1%|▍                                   | 868/78125 [08:48<10:38:32,  2.02it/s]
  1%|▍                                   | 869/78125 [08:49<10:38:43,  2.02it/s]
  1%|▍                                   | 870/78125 [08:49<10:38:37,  2.02it/s]
  1%|▍                                   | 871/78125 [08:50<10:38:40,  2.02it/s]
  1%|▍                                   | 872/78125 [08:50<10:38:28,  2.02it/s]
  1%|▍                                   | 873/78125 [08:51<10:38:47,  2.02it/s]
  1%|▍                                   | 874/78125 [08:51<10:39:43,  2.01it/s]
  1%|▍                                   | 875/78125 [08:52<10:39:41,  2.01it/s]
  1%|▍                                   | 876/78125 [08:52<10:40:04,  2.01it/s]
  1%|▍                                   | 877/78125 [08:53<10:40:15,  2.01it/s]
  1%|▍                                   | 878/78125 [08:53<10:40:05,  2.01it/s]
  1%|▍                                   | 879/78125 [08:54<10:39:10,  2.01it/s]
  1%|▍                                   | 880/78125 [08:54<10:38:40,  2.02it/s]
  1%|▍                                   | 881/78125 [08:55<10:38:05,  2.02it/s]
  1%|▍                                   | 882/78125 [08:55<10:37:52,  2.02it/s]
  1%|▍                                   | 883/78125 [08:56<10:38:43,  2.02it/s]
  1%|▍                                   | 884/78125 [08:56<10:38:27,  2.02it/s]
  1%|▍                                   | 885/78125 [08:57<10:37:45,  2.02it/s]
  1%|▍                                   | 886/78125 [08:57<10:38:14,  2.02it/s]
  1%|▍                                   | 887/78125 [08:58<10:38:19,  2.02it/s]
  1%|▍                                   | 888/78125 [08:58<10:38:48,  2.02it/s]
  1%|▍                                   | 889/78125 [08:59<10:38:23,  2.02it/s]
  1%|▍                                   | 890/78125 [08:59<10:38:57,  2.01it/s]
  1%|▍                                   | 891/78125 [09:00<10:39:05,  2.01it/s]
  1%|▍                                   | 892/78125 [09:00<10:38:45,  2.02it/s]
  1%|▍                                   | 893/78125 [09:01<10:38:26,  2.02it/s]
  1%|▍                                   | 894/78125 [09:01<10:38:48,  2.01it/s]
  1%|▍                                   | 895/78125 [09:02<10:39:17,  2.01it/s]
  1%|▍                                   | 896/78125 [09:02<10:39:16,  2.01it/s]
  1%|▍                                   | 897/78125 [09:03<10:38:32,  2.02it/s]
  1%|▍                                   | 898/78125 [09:03<10:38:39,  2.02it/s]
  1%|▍                                   | 899/78125 [09:04<10:38:32,  2.02it/s]
  1%|▍                                   | 900/78125 [09:04<10:39:04,  2.01it/s]
  1%|▍                                   | 901/78125 [09:05<10:38:59,  2.01it/s]
  1%|▍                                   | 902/78125 [09:05<10:38:17,  2.02it/s]
  1%|▍                                   | 903/78125 [09:06<10:38:37,  2.02it/s]
  1%|▍                                   | 904/78125 [09:06<10:38:24,  2.02it/s]
  1%|▍                                   | 905/78125 [09:07<10:39:11,  2.01it/s]
  1%|▍                                   | 906/78125 [09:07<10:38:37,  2.02it/s]
  1%|▍                                   | 907/78125 [09:08<10:39:08,  2.01it/s]
  1%|▍                                   | 908/78125 [09:08<10:38:13,  2.02it/s]
  1%|▍                                   | 909/78125 [09:09<10:37:57,  2.02it/s]
  1%|▍                                   | 910/78125 [09:09<10:38:15,  2.02it/s]
  1%|▍                                   | 911/78125 [09:10<10:37:27,  2.02it/s]
  1%|▍                                   | 912/78125 [09:10<10:37:53,  2.02it/s]
  1%|▍                                   | 913/78125 [09:11<10:37:14,  2.02it/s]
  1%|▍                                   | 914/78125 [09:11<10:38:02,  2.02it/s]
  1%|▍                                   | 915/78125 [09:12<10:38:30,  2.02it/s]
  1%|▍                                   | 916/78125 [09:12<10:38:56,  2.01it/s]
  1%|▍                                   | 917/78125 [09:13<10:38:55,  2.01it/s]
  1%|▍                                   | 918/78125 [09:13<10:38:07,  2.02it/s]
  1%|▍                                   | 919/78125 [09:14<10:38:18,  2.02it/s]
  1%|▍                                   | 920/78125 [09:14<10:39:02,  2.01it/s]
  1%|▍                                   | 921/78125 [09:15<10:38:35,  2.01it/s]
  1%|▍                                   | 922/78125 [09:15<10:37:19,  2.02it/s]
  1%|▍                                   | 923/78125 [09:15<10:37:45,  2.02it/s]
  1%|▍                                   | 924/78125 [09:16<10:37:07,  2.02it/s]
  1%|▍                                   | 925/78125 [09:16<10:37:04,  2.02it/s]
  1%|▍                                   | 926/78125 [09:17<10:37:15,  2.02it/s]
  1%|▍                                   | 927/78125 [09:17<10:37:30,  2.02it/s]
  1%|▍                                   | 928/78125 [09:18<10:37:30,  2.02it/s]
  1%|▍                                   | 929/78125 [09:18<10:36:54,  2.02it/s]
  1%|▍                                   | 930/78125 [09:19<10:37:11,  2.02it/s]
  1%|▍                                   | 931/78125 [09:19<10:37:35,  2.02it/s]
  1%|▍                                   | 932/78125 [09:20<10:37:53,  2.02it/s]
  1%|▍                                   | 933/78125 [09:20<10:37:38,  2.02it/s]
  1%|▍                                   | 934/78125 [09:21<10:37:30,  2.02it/s]
  1%|▍                                   | 935/78125 [09:21<10:37:11,  2.02it/s]
  1%|▍                                   | 936/78125 [09:22<10:37:58,  2.02it/s]
  1%|▍                                   | 937/78125 [09:22<10:36:56,  2.02it/s]
  1%|▍                                   | 938/78125 [09:23<10:37:25,  2.02it/s]
  1%|▍                                   | 939/78125 [09:23<10:37:42,  2.02it/s]
  1%|▍                                   | 940/78125 [09:24<10:36:58,  2.02it/s]
  1%|▍                                   | 941/78125 [09:24<10:36:57,  2.02it/s]
  1%|▍                                   | 942/78125 [09:25<10:36:43,  2.02it/s]
  1%|▍                                   | 943/78125 [09:25<10:37:15,  2.02it/s]
  1%|▍                                   | 944/78125 [09:26<10:36:11,  2.02it/s]
  1%|▍                                   | 945/78125 [09:26<10:36:12,  2.02it/s]
  1%|▍                                   | 946/78125 [09:27<10:36:52,  2.02it/s]
  1%|▍                                   | 947/78125 [09:27<10:37:34,  2.02it/s]
  1%|▍                                   | 948/78125 [09:28<10:36:54,  2.02it/s]
  1%|▍                                   | 949/78125 [09:28<10:37:05,  2.02it/s]
  1%|▍                                   | 950/78125 [09:29<10:37:15,  2.02it/s]
  1%|▍                                   | 951/78125 [09:29<10:37:07,  2.02it/s]
  1%|▍                                   | 952/78125 [09:30<10:37:54,  2.02it/s]
  1%|▍                                   | 953/78125 [09:30<10:37:43,  2.02it/s]
  1%|▍                                   | 954/78125 [09:31<10:37:32,  2.02it/s]
  1%|▍                                   | 955/78125 [09:31<10:38:05,  2.02it/s]
  1%|▍                                   | 956/78125 [09:32<10:37:00,  2.02it/s]
  1%|▍                                   | 957/78125 [09:32<10:37:45,  2.02it/s]
  1%|▍                                   | 958/78125 [09:33<10:37:51,  2.02it/s]
  1%|▍                                   | 959/78125 [09:33<10:37:14,  2.02it/s]
  1%|▍                                   | 960/78125 [09:34<10:37:21,  2.02it/s]
  1%|▍                                   | 961/78125 [09:34<10:37:38,  2.02it/s]
  1%|▍                                   | 962/78125 [09:35<10:37:28,  2.02it/s]
  1%|▍                                   | 963/78125 [09:35<10:37:21,  2.02it/s]
  1%|▍                                   | 964/78125 [09:36<10:37:35,  2.02it/s]
  1%|▍                                   | 965/78125 [09:36<10:37:10,  2.02it/s]
  1%|▍                                   | 966/78125 [09:37<10:36:59,  2.02it/s]
  1%|▍                                   | 967/78125 [09:37<10:36:44,  2.02it/s]
  1%|▍                                   | 968/78125 [09:38<10:37:07,  2.02it/s]
  1%|▍                                   | 969/78125 [09:38<10:36:42,  2.02it/s]
  1%|▍                                   | 970/78125 [09:39<10:36:36,  2.02it/s]
  1%|▍                                   | 971/78125 [09:39<10:37:52,  2.02it/s]
  1%|▍                                   | 972/78125 [09:40<10:38:19,  2.01it/s]
  1%|▍                                   | 973/78125 [09:40<10:38:42,  2.01it/s]
  1%|▍                                   | 974/78125 [09:41<10:38:23,  2.01it/s]
  1%|▍                                   | 975/78125 [09:41<10:38:36,  2.01it/s]
  1%|▍                                   | 976/78125 [09:42<10:39:00,  2.01it/s]
  1%|▍                                   | 977/78125 [09:42<10:37:56,  2.02it/s]
  1%|▍                                   | 978/78125 [09:43<10:37:42,  2.02it/s]
  1%|▍                                   | 979/78125 [09:43<10:37:52,  2.02it/s]
  1%|▍                                   | 980/78125 [09:44<10:38:08,  2.01it/s]
  1%|▍                                   | 981/78125 [09:44<10:38:27,  2.01it/s]
  1%|▍                                   | 982/78125 [09:45<10:37:54,  2.02it/s]
  1%|▍                                   | 983/78125 [09:45<10:38:09,  2.01it/s]
  1%|▍                                   | 984/78125 [09:46<10:38:36,  2.01it/s]
  1%|▍                                   | 985/78125 [09:46<10:38:52,  2.01it/s]
  1%|▍                                   | 986/78125 [09:47<10:38:12,  2.01it/s]
  1%|▍                                   | 987/78125 [09:47<10:38:52,  2.01it/s]
  1%|▍                                   | 988/78125 [09:48<10:37:54,  2.02it/s]
  1%|▍                                   | 989/78125 [09:48<10:38:13,  2.01it/s]
  1%|▍                                   | 990/78125 [09:49<10:38:18,  2.01it/s]
  1%|▍                                   | 991/78125 [09:49<10:38:36,  2.01it/s]
  1%|▍                                   | 992/78125 [09:50<10:39:05,  2.01it/s]
  1%|▍                                   | 993/78125 [09:50<10:37:48,  2.02it/s]
  1%|▍                                   | 994/78125 [09:51<10:37:46,  2.02it/s]
  1%|▍                                   | 995/78125 [09:51<10:37:52,  2.02it/s]
  1%|▍                                   | 996/78125 [09:52<10:37:44,  2.02it/s]
  1%|▍                                   | 997/78125 [09:52<10:36:56,  2.02it/s]
  1%|▍                                   | 998/78125 [09:53<10:36:26,  2.02it/s]
  1%|▍                                   | 999/78125 [09:53<10:36:46,  2.02it/s]
  1%|▍                                  | 1000/78125 [09:54<10:37:29,  2.02it/s]
  1%|▍                                  | 1001/78125 [09:54<10:37:46,  2.02it/s]
  1%|▍                                  | 1002/78125 [09:55<10:37:52,  2.02it/s]
  1%|▍                                  | 1003/78125 [09:55<10:37:30,  2.02it/s]
  1%|▍                                  | 1004/78125 [09:56<10:37:08,  2.02it/s]
  1%|▍                                  | 1005/78125 [09:56<10:37:15,  2.02it/s]
  1%|▍                                  | 1006/78125 [09:57<10:38:13,  2.01it/s]
  1%|▍                                  | 1007/78125 [09:57<10:38:16,  2.01it/s]
  1%|▍                                  | 1008/78125 [09:58<10:37:58,  2.01it/s]
  1%|▍                                  | 1009/78125 [09:58<10:37:32,  2.02it/s]
  1%|▍                                  | 1010/78125 [09:59<10:38:26,  2.01it/s]
  1%|▍                                  | 1011/78125 [09:59<10:38:41,  2.01it/s]
  1%|▍                                  | 1012/78125 [10:00<10:38:50,  2.01it/s]
  1%|▍                                  | 1013/78125 [10:00<10:39:01,  2.01it/s]
  1%|▍                                  | 1014/78125 [10:01<10:40:03,  2.01it/s]
  1%|▍                                  | 1015/78125 [10:01<10:39:59,  2.01it/s]
  1%|▍                                  | 1016/78125 [10:02<10:40:33,  2.01it/s]
  1%|▍                                  | 1017/78125 [10:02<10:39:22,  2.01it/s]
  1%|▍                                  | 1018/78125 [10:03<10:39:25,  2.01it/s]
  1%|▍                                  | 1019/78125 [10:03<10:39:24,  2.01it/s]
  1%|▍                                  | 1020/78125 [10:04<10:39:14,  2.01it/s]
  1%|▍                                  | 1021/78125 [10:04<10:39:27,  2.01it/s]
  1%|▍                                  | 1022/78125 [10:05<10:39:51,  2.01it/s]
  1%|▍                                  | 1023/78125 [10:05<10:39:13,  2.01it/s]
  1%|▍                                  | 1024/78125 [10:06<10:39:08,  2.01it/s]
  1%|▍                                  | 1025/78125 [10:06<10:37:57,  2.01it/s]
  1%|▍                                  | 1026/78125 [10:07<10:38:59,  2.01it/s]
  1%|▍                                  | 1027/78125 [10:07<10:38:27,  2.01it/s]
  1%|▍                                  | 1028/78125 [10:08<10:39:27,  2.01it/s]
  1%|▍                                  | 1029/78125 [10:08<10:39:59,  2.01it/s]
  1%|▍                                  | 1030/78125 [10:09<10:40:23,  2.01it/s]
  1%|▍                                  | 1031/78125 [10:09<10:39:24,  2.01it/s]
  1%|▍                                  | 1032/78125 [10:10<10:38:59,  2.01it/s]
  1%|▍                                  | 1033/78125 [10:10<10:38:44,  2.01it/s]
  1%|▍                                  | 1034/78125 [10:11<10:38:34,  2.01it/s]
  1%|▍                                  | 1035/78125 [10:11<10:37:51,  2.01it/s]
  1%|▍                                  | 1036/78125 [10:12<10:37:56,  2.01it/s]
  1%|▍                                  | 1037/78125 [10:12<10:38:29,  2.01it/s]
  1%|▍                                  | 1038/78125 [10:13<10:38:27,  2.01it/s]
  1%|▍                                  | 1039/78125 [10:13<10:38:44,  2.01it/s]
  1%|▍                                  | 1040/78125 [10:14<10:38:01,  2.01it/s]
  1%|▍                                  | 1041/78125 [10:14<10:37:31,  2.02it/s]
  1%|▍                                  | 1042/78125 [10:15<10:37:41,  2.01it/s]
  1%|▍                                  | 1043/78125 [10:15<10:38:21,  2.01it/s]
  1%|▍                                  | 1044/78125 [10:16<10:37:59,  2.01it/s]
  1%|▍                                  | 1045/78125 [10:16<10:38:12,  2.01it/s]
  1%|▍                                  | 1046/78125 [10:17<10:37:30,  2.02it/s]
  1%|▍                                  | 1047/78125 [10:17<10:38:05,  2.01it/s]
  1%|▍                                  | 1048/78125 [10:18<10:38:29,  2.01it/s]
  1%|▍                                  | 1049/78125 [10:18<10:38:35,  2.01it/s]
  1%|▍                                  | 1050/78125 [10:19<10:38:11,  2.01it/s]
  1%|▍                                  | 1051/78125 [10:19<10:37:39,  2.01it/s]
  1%|▍                                  | 1052/78125 [10:20<10:38:00,  2.01it/s]
  1%|▍                                  | 1053/78125 [10:20<10:37:41,  2.01it/s]
  1%|▍                                  | 1054/78125 [10:21<10:36:58,  2.02it/s]
  1%|▍                                  | 1055/78125 [10:21<10:37:07,  2.02it/s]
  1%|▍                                  | 1056/78125 [10:21<10:38:05,  2.01it/s]
  1%|▍                                  | 1057/78125 [10:22<10:36:34,  2.02it/s]
  1%|▍                                  | 1058/78125 [10:22<10:37:03,  2.02it/s]
  1%|▍                                  | 1059/78125 [10:23<10:37:48,  2.01it/s]
  1%|▍                                  | 1060/78125 [10:23<10:37:24,  2.02it/s]
  1%|▍                                  | 1061/78125 [10:24<10:37:41,  2.01it/s]
  1%|▍                                  | 1062/78125 [10:24<10:37:53,  2.01it/s]
  1%|▍                                  | 1063/78125 [10:25<10:37:27,  2.01it/s]
  1%|▍                                  | 1064/78125 [10:25<10:37:29,  2.01it/s]
  1%|▍                                  | 1065/78125 [10:26<10:37:28,  2.01it/s]
  1%|▍                                  | 1066/78125 [10:26<10:37:31,  2.01it/s]
  1%|▍                                  | 1067/78125 [10:27<10:37:19,  2.02it/s]
  1%|▍                                  | 1068/78125 [10:27<10:37:21,  2.02it/s]
  1%|▍                                  | 1069/78125 [10:28<10:37:37,  2.01it/s]
  1%|▍                                  | 1070/78125 [10:28<10:37:55,  2.01it/s]
  1%|▍                                  | 1071/78125 [10:29<10:37:40,  2.01it/s]
  1%|▍                                  | 1072/78125 [10:29<10:37:30,  2.01it/s]
  1%|▍                                  | 1073/78125 [10:30<10:36:23,  2.02it/s]
  1%|▍                                  | 1074/78125 [10:30<10:36:12,  2.02it/s]
  1%|▍                                  | 1075/78125 [10:31<10:36:37,  2.02it/s]
  1%|▍                                  | 1076/78125 [10:31<10:37:04,  2.02it/s]
  1%|▍                                  | 1077/78125 [10:32<10:37:10,  2.02it/s]
  1%|▍                                  | 1078/78125 [10:32<10:37:47,  2.01it/s]
  1%|▍                                  | 1079/78125 [10:33<10:37:38,  2.01it/s]
  1%|▍                                  | 1080/78125 [10:33<10:37:33,  2.01it/s]
  1%|▍                                  | 1081/78125 [10:34<10:37:29,  2.01it/s]
  1%|▍                                  | 1082/78125 [10:34<10:37:44,  2.01it/s]
  1%|▍                                  | 1083/78125 [10:35<10:37:53,  2.01it/s]
  1%|▍                                  | 1084/78125 [10:35<10:36:43,  2.02it/s]
  1%|▍                                  | 1085/78125 [10:36<10:36:34,  2.02it/s]
  1%|▍                                  | 1086/78125 [10:36<10:36:33,  2.02it/s]
  1%|▍                                  | 1087/78125 [10:37<10:37:00,  2.02it/s]
  1%|▍                                  | 1088/78125 [10:37<10:37:14,  2.01it/s]
  1%|▍                                  | 1089/78125 [10:38<10:36:17,  2.02it/s]
  1%|▍                                  | 1090/78125 [10:38<10:36:10,  2.02it/s]
  1%|▍                                  | 1091/78125 [10:39<10:37:29,  2.01it/s]
  1%|▍                                  | 1092/78125 [10:39<10:36:57,  2.02it/s]
  1%|▍                                  | 1093/78125 [10:40<10:37:19,  2.01it/s]
  1%|▍                                  | 1094/78125 [10:40<10:37:19,  2.01it/s]
  1%|▍                                  | 1095/78125 [10:41<10:37:11,  2.01it/s]
  1%|▍                                  | 1096/78125 [10:41<10:37:40,  2.01it/s]
  1%|▍                                  | 1097/78125 [10:42<10:37:14,  2.01it/s]
  1%|▍                                  | 1098/78125 [10:42<10:36:30,  2.02it/s]
  1%|▍                                  | 1099/78125 [10:43<10:36:46,  2.02it/s]
  1%|▍                                  | 1100/78125 [10:43<10:36:32,  2.02it/s]
  1%|▍                                  | 1101/78125 [10:44<10:35:59,  2.02it/s]
  1%|▍                                  | 1102/78125 [10:44<10:36:12,  2.02it/s]
  1%|▍                                  | 1103/78125 [10:45<10:36:44,  2.02it/s]
  1%|▍                                  | 1104/78125 [10:45<10:37:15,  2.01it/s]
  1%|▍                                  | 1105/78125 [10:46<10:36:27,  2.02it/s]
  1%|▍                                  | 1106/78125 [10:46<10:36:18,  2.02it/s]
  1%|▍                                  | 1107/78125 [10:47<10:36:23,  2.02it/s]
  1%|▍                                  | 1108/78125 [10:47<10:35:59,  2.02it/s]
  1%|▍                                  | 1109/78125 [10:48<10:36:14,  2.02it/s]
  1%|▍                                  | 1110/78125 [10:48<10:36:41,  2.02it/s]
  1%|▍                                  | 1111/78125 [10:49<10:36:49,  2.02it/s]
  1%|▍                                  | 1112/78125 [10:49<10:36:45,  2.02it/s]
  1%|▍                                  | 1113/78125 [10:50<10:37:42,  2.01it/s]
  1%|▍                                  | 1114/78125 [10:50<10:36:46,  2.02it/s]
  1%|▍                                  | 1115/78125 [10:51<10:37:09,  2.01it/s]
  1%|▍                                  | 1116/78125 [10:51<10:38:01,  2.01it/s]
  1%|▌                                  | 1117/78125 [10:52<10:36:55,  2.02it/s]
  1%|▌                                  | 1118/78125 [10:52<10:36:28,  2.02it/s]
  1%|▌                                  | 1119/78125 [10:53<10:36:18,  2.02it/s]
  1%|▌                                  | 1120/78125 [10:53<10:36:17,  2.02it/s]
  1%|▌                                  | 1121/78125 [10:54<10:34:23,  2.02it/s]
  1%|▌                                  | 1122/78125 [10:54<10:35:09,  2.02it/s]
  1%|▌                                  | 1123/78125 [10:55<10:35:06,  2.02it/s]
  1%|▌                                  | 1124/78125 [10:55<10:35:26,  2.02it/s]
  1%|▌                                  | 1125/78125 [10:56<10:35:58,  2.02it/s]
  1%|▌                                  | 1126/78125 [10:56<10:36:06,  2.02it/s]
  1%|▌                                  | 1127/78125 [10:57<10:36:17,  2.02it/s]
  1%|▌                                  | 1128/78125 [10:57<10:36:23,  2.02it/s]
  1%|▌                                  | 1129/78125 [10:58<10:35:42,  2.02it/s]
  1%|▌                                  | 1130/78125 [10:58<10:36:21,  2.02it/s]
  1%|▌                                  | 1131/78125 [10:59<10:36:58,  2.01it/s]
  1%|▌                                  | 1132/78125 [10:59<10:36:26,  2.02it/s]
  1%|▌                                  | 1133/78125 [11:00<10:36:10,  2.02it/s]
  1%|▌                                  | 1134/78125 [11:00<10:36:03,  2.02it/s]
  1%|▌                                  | 1135/78125 [11:01<10:36:36,  2.02it/s]
  1%|▌                                  | 1136/78125 [11:01<10:36:50,  2.01it/s]
  1%|▌                                  | 1137/78125 [11:02<10:36:06,  2.02it/s]
  1%|▌                                  | 1138/78125 [11:02<10:35:34,  2.02it/s]
  1%|▌                                  | 1139/78125 [11:03<10:35:26,  2.02it/s]
  1%|▌                                  | 1140/78125 [11:03<10:35:25,  2.02it/s]
  1%|▌                                  | 1141/78125 [11:04<10:34:59,  2.02it/s]
  1%|▌                                  | 1142/78125 [11:04<10:35:27,  2.02it/s]
  1%|▌                                  | 1143/78125 [11:05<10:35:02,  2.02it/s]
  1%|▌                                  | 1144/78125 [11:05<10:35:13,  2.02it/s]
  1%|▌                                  | 1145/78125 [11:06<10:36:02,  2.02it/s]
  1%|▌                                  | 1146/78125 [11:06<10:35:48,  2.02it/s]
  1%|▌                                  | 1147/78125 [11:07<10:35:17,  2.02it/s]
  1%|▌                                  | 1148/78125 [11:07<10:35:28,  2.02it/s]
  1%|▌                                  | 1149/78125 [11:08<10:34:58,  2.02it/s]
  1%|▌                                  | 1150/78125 [11:08<10:34:28,  2.02it/s]
  1%|▌                                  | 1151/78125 [11:09<10:34:23,  2.02it/s]
  1%|▌                                  | 1152/78125 [11:09<10:35:02,  2.02it/s]
  1%|▌                                  | 1153/78125 [11:10<10:35:21,  2.02it/s]
  1%|▌                                  | 1154/78125 [11:10<10:35:20,  2.02it/s]
  1%|▌                                  | 1155/78125 [11:11<10:35:42,  2.02it/s]
  1%|▌                                  | 1156/78125 [11:11<10:36:08,  2.02it/s]
  1%|▌                                  | 1157/78125 [11:12<10:36:17,  2.02it/s]
  1%|▌                                  | 1158/78125 [11:12<10:35:48,  2.02it/s]
  1%|▌                                  | 1159/78125 [11:13<10:36:19,  2.02it/s]
  1%|▌                                  | 1160/78125 [11:13<10:36:32,  2.02it/s]
  1%|▌                                  | 1161/78125 [11:14<10:36:40,  2.01it/s]
  1%|▌                                  | 1162/78125 [11:14<10:37:03,  2.01it/s]
  1%|▌                                  | 1163/78125 [11:15<10:36:16,  2.02it/s]
  1%|▌                                  | 1164/78125 [11:15<10:35:59,  2.02it/s]
  1%|▌                                  | 1165/78125 [11:16<10:35:43,  2.02it/s]
  1%|▌                                  | 1166/78125 [11:16<10:35:51,  2.02it/s]
  1%|▌                                  | 1167/78125 [11:17<10:35:40,  2.02it/s]
  1%|▌                                  | 1168/78125 [11:17<10:35:09,  2.02it/s]
  1%|▌                                  | 1169/78125 [11:18<10:34:39,  2.02it/s]
  1%|▌                                  | 1170/78125 [11:18<10:35:18,  2.02it/s]
  1%|▌                                  | 1171/78125 [11:19<10:35:17,  2.02it/s]
  2%|▌                                  | 1172/78125 [11:19<10:35:22,  2.02it/s]
  2%|▌                                  | 1173/78125 [11:20<10:35:35,  2.02it/s]
  2%|▌                                  | 1174/78125 [11:20<10:35:11,  2.02it/s]
  2%|▌                                  | 1175/78125 [11:21<10:35:52,  2.02it/s]
  2%|▌                                  | 1176/78125 [11:21<10:35:56,  2.02it/s]
  2%|▌                                  | 1177/78125 [11:21<10:35:43,  2.02it/s]
  2%|▌                                  | 1178/78125 [11:22<10:36:21,  2.02it/s]
  2%|▌                                  | 1179/78125 [11:22<10:35:51,  2.02it/s]
  2%|▌                                  | 1180/78125 [11:23<10:35:06,  2.02it/s]
  2%|▌                                  | 1181/78125 [11:23<10:36:32,  2.01it/s]
  2%|▌                                  | 1182/78125 [11:24<10:36:50,  2.01it/s]
  2%|▌                                  | 1183/78125 [11:24<10:36:41,  2.01it/s]
  2%|▌                                  | 1184/78125 [11:25<10:36:54,  2.01it/s]
  2%|▌                                  | 1185/78125 [11:25<10:36:09,  2.02it/s]
  2%|▌                                  | 1186/78125 [11:26<10:35:36,  2.02it/s]
  2%|▌                                  | 1187/78125 [11:26<10:36:03,  2.02it/s]
  2%|▌                                  | 1188/78125 [11:27<10:36:18,  2.02it/s]
  2%|▌                                  | 1189/78125 [11:27<10:36:31,  2.01it/s]
  2%|▌                                  | 1190/78125 [11:28<10:35:56,  2.02it/s]
  2%|▌                                  | 1191/78125 [11:28<10:35:46,  2.02it/s]
  2%|▌                                  | 1192/78125 [11:29<10:35:18,  2.02it/s]
  2%|▌                                  | 1193/78125 [11:29<10:34:32,  2.02it/s]
  2%|▌                                  | 1194/78125 [11:30<10:35:04,  2.02it/s]
  2%|▌                                  | 1195/78125 [11:30<10:35:22,  2.02it/s]
  2%|▌                                  | 1196/78125 [11:31<10:35:01,  2.02it/s]
  2%|▌                                  | 1197/78125 [11:31<10:35:13,  2.02it/s]
  2%|▌                                  | 1198/78125 [11:32<10:35:25,  2.02it/s]
  2%|▌                                  | 1199/78125 [11:32<10:34:51,  2.02it/s]
  2%|▌                                  | 1200/78125 [11:33<10:35:16,  2.02it/s]
  2%|▌                                  | 1201/78125 [11:33<10:35:00,  2.02it/s]
  2%|▌                                  | 1202/78125 [11:34<10:35:38,  2.02it/s]
  2%|▌                                  | 1203/78125 [11:34<10:35:13,  2.02it/s]
  2%|▌                                  | 1204/78125 [11:35<10:35:28,  2.02it/s]
  2%|▌                                  | 1205/78125 [11:35<10:35:26,  2.02it/s]
  2%|▌                                  | 1206/78125 [11:36<10:35:28,  2.02it/s]
  2%|▌                                  | 1207/78125 [11:36<10:35:15,  2.02it/s]
  2%|▌                                  | 1208/78125 [11:37<10:35:15,  2.02it/s]
  2%|▌                                  | 1209/78125 [11:37<10:35:21,  2.02it/s]
  2%|▌                                  | 1210/78125 [11:38<10:35:19,  2.02it/s]
  2%|▌                                  | 1211/78125 [11:38<10:35:38,  2.02it/s]
  2%|▌                                  | 1212/78125 [11:39<10:35:32,  2.02it/s]
  2%|▌                                  | 1213/78125 [11:39<10:35:44,  2.02it/s]
  2%|▌                                  | 1214/78125 [11:40<10:35:01,  2.02it/s]
  2%|▌                                  | 1215/78125 [11:40<10:35:21,  2.02it/s]
  2%|▌                                  | 1216/78125 [11:41<10:35:17,  2.02it/s]
  2%|▌                                  | 1217/78125 [11:41<10:35:46,  2.02it/s]
  2%|▌                                  | 1218/78125 [11:42<10:34:52,  2.02it/s]
  2%|▌                                  | 1219/78125 [11:42<10:34:43,  2.02it/s]
  2%|▌                                  | 1220/78125 [11:43<10:34:06,  2.02it/s]
  2%|▌                                  | 1221/78125 [11:43<10:33:59,  2.02it/s]
  2%|▌                                  | 1222/78125 [11:44<10:34:22,  2.02it/s]
  2%|▌                                  | 1223/78125 [11:44<10:34:37,  2.02it/s]
  2%|▌                                  | 1224/78125 [11:45<10:35:14,  2.02it/s]
  2%|▌                                  | 1225/78125 [11:45<10:35:32,  2.02it/s]
  2%|▌                                  | 1226/78125 [11:46<10:35:36,  2.02it/s]
  2%|▌                                  | 1227/78125 [11:46<10:35:14,  2.02it/s]
  2%|▌                                  | 1228/78125 [11:47<10:35:27,  2.02it/s]
  2%|▌                                  | 1229/78125 [11:47<10:35:37,  2.02it/s]
  2%|▌                                  | 1230/78125 [11:48<10:35:55,  2.02it/s]
  2%|▌                                  | 1231/78125 [11:48<10:36:02,  2.01it/s]
  2%|▌                                  | 1232/78125 [11:49<10:35:12,  2.02it/s]
  2%|▌                                  | 1233/78125 [11:49<10:35:17,  2.02it/s]
  2%|▌                                  | 1234/78125 [11:50<10:34:37,  2.02it/s]
  2%|▌                                  | 1235/78125 [11:50<10:34:01,  2.02it/s]
  2%|▌                                  | 1236/78125 [11:51<10:33:58,  2.02it/s]
  2%|▌                                  | 1237/78125 [11:51<10:34:13,  2.02it/s]
  2%|▌                                  | 1238/78125 [11:52<10:34:49,  2.02it/s]
  2%|▌                                  | 1239/78125 [11:52<10:35:15,  2.02it/s]
  2%|▌                                  | 1240/78125 [11:53<10:35:26,  2.02it/s]
  2%|▌
760.3s 147 | 1241/78125 [11:53<10:35:14,  2.02it/s]
  2%|▌                                  | 1242/78125 [11:54<10:35:36,  2.02it/s]
  2%|▌                                  | 1243/78125 [11:54<10:36:22,  2.01it/s]
  2%|▌                                  | 1244/78125 [11:55<10:36:02,  2.01it/s]
  2%|▌                                  | 1245/78125 [11:55<10:35:39,  2.02it/s]
  2%|▌                                  | 1246/78125 [11:56<10:34:56,  2.02it/s]
  2%|▌                                  | 1247/78125 [11:56<10:35:05,  2.02it/s]
  2%|▌                                  | 1248/78125 [11:57<10:34:38,  2.02it/s]
  2%|▌                                  | 1249/78125 [11:57<10:34:32,  2.02it/s]
  2%|▌                                  | 1250/78125 [11:58<10:35:07,  2.02it/s]
  2%|▌                                  | 1251/78125 [11:58<10:34:18,  2.02it/s]
  2%|▌                                  | 1252/78125 [11:59<10:33:37,  2.02it/s]
  2%|▌                                  | 1253/78125 [11:59<10:33:56,  2.02it/s]
  2%|▌                                  | 1254/78125 [12:00<10:34:31,  2.02it/s]
  2%|▌                                  | 1255/78125 [12:00<10:35:33,  2.02it/s]
  2%|▌                                  | 1256/78125 [12:01<10:35:56,  2.01it/s]
  2%|▌                                  | 1257/78125 [12:01<10:36:17,  2.01it/s]
  2%|▌                                  | 1258/78125 [12:02<10:36:14,  2.01it/s]
  2%|▌                                  | 1259/78125 [12:02<10:36:17,  2.01it/s]
  2%|▌                                  | 1260/78125 [12:03<10:35:33,  2.02it/s]
  2%|▌                                  | 1261/78125 [12:03<10:35:31,  2.02it/s]
  2%|▌                                  | 1262/78125 [12:04<10:34:57,  2.02it/s]
  2%|▌                                  | 1263/78125 [12:04<10:35:21,  2.02it/s]
  2%|▌                                  | 1264/78125 [12:05<10:34:30,  2.02it/s]
  2%|▌                                  | 1265/78125 [12:05<10:35:03,  2.02it/s]
  2%|▌                                  | 1266/78125 [12:06<10:35:17,  2.02it/s]
  2%|▌                                  | 1267/78125 [12:06<10:35:02,  2.02it/s]
  2%|▌                                  | 1268/78125 [12:07<10:35:10,  2.02it/s]
  2%|▌                                  | 1269/78125 [12:07<10:34:58,  2.02it/s]
  2%|▌                                  | 1270/78125 [12:08<10:34:52,  2.02it/s]
  2%|▌                                  | 1271/78125 [12:08<10:34:50,  2.02it/s]
  2%|▌                                  | 1272/78125 [12:09<10:35:00,  2.02it/s]
  2%|▌                                  | 1273/78125 [12:09<10:35:01,  2.02it/s]
  2%|▌                                  | 1274/78125 [12:10<10:34:29,  2.02it/s]
  2%|▌                                  | 1275/78125 [12:10<10:34:28,  2.02it/s]
  2%|▌                                  | 1276/78125 [12:11<10:34:40,  2.02it/s]
  2%|▌                                  | 1277/78125 [12:11<10:34:46,  2.02it/s]
  2%|▌                                  | 1278/78125 [12:12<10:34:51,  2.02it/s]
  2%|▌                                  | 1279/78125 [12:12<10:34:52,  2.02it/s]
  2%|▌                                  | 1280/78125 [12:13<10:34:43,  2.02it/s]
  2%|▌                                  | 1281/78125 [12:13<10:34:13,  2.02it/s]
  2%|▌                                  | 1282/78125 [12:14<10:33:43,  2.02it/s]
  2%|▌                                  | 1283/78125 [12:14<10:33:31,  2.02it/s]
  2%|▌                                  | 1284/78125 [12:15<10:33:54,  2.02it/s]
  2%|▌                                  | 1285/78125 [12:15<10:33:32,  2.02it/s]
  2%|▌                                  | 1286/78125 [12:16<10:33:46,  2.02it/s]
  2%|▌                                  | 1287/78125 [12:16<10:33:23,  2.02it/s]
  2%|▌                                  | 1288/78125 [12:17<10:33:20,  2.02it/s]
  2%|▌                                  | 1289/78125 [12:17<10:33:16,  2.02it/s]
  2%|▌                                  | 1290/78125 [12:17<10:33:22,  2.02it/s]
  2%|▌                                  | 1291/78125 [12:18<10:33:12,  2.02it/s]
  2%|▌                                  | 1292/78125 [12:18<10:32:54,  2.02it/s]
  2%|▌                                  | 1293/78125 [12:19<10:34:09,  2.02it/s]
  2%|▌                                  | 1294/78125 [12:19<10:34:32,  2.02it/s]
  2%|▌                                  | 1295/78125 [12:20<10:33:31,  2.02it/s]
  2%|▌                                  | 1296/78125 [12:20<10:34:16,  2.02it/s]
  2%|▌                                  | 1297/78125 [12:21<10:34:15,  2.02it/s]
  2%|▌                                  | 1298/78125 [12:21<10:34:12,  2.02it/s]
  2%|▌                                  | 1299/78125 [12:22<10:34:19,  2.02it/s]
  2%|▌                                  | 1300/78125 [12:22<10:34:26,  2.02it/s]
  2%|▌                                  | 1301/78125 [12:23<10:34:02,  2.02it/s]
  2%|▌                                  | 1302/78125 [12:23<10:33:44,  2.02it/s]
  2%|▌                                  | 1303/78125 [12:24<10:33:43,  2.02it/s]
  2%|▌                                  | 1304/78125 [12:24<10:34:26,  2.02it/s]
  2%|▌                                  | 1305/78125 [12:25<10:34:43,  2.02it/s]
  2%|▌                                  | 1306/78125 [12:25<10:33:54,  2.02it/s]
  2%|▌                                  | 1307/78125 [12:26<10:34:33,  2.02it/s]
  2%|▌                                  | 1308/78125 [12:26<10:34:29,  2.02it/s]
  2%|▌                                  | 1309/78125 [12:27<10:34:29,  2.02it/s]
  2%|▌                                  | 1310/78125 [12:27<10:34:18,  2.02it/s]
  2%|▌                                  | 1311/78125 [12:28<10:34:46,  2.02it/s]
  2%|▌                                  | 1312/78125 [12:28<10:34:40,  2.02it/s]
  2%|▌                                  | 1313/78125 [12:29<10:35:05,  2.02it/s]
  2%|▌                                  | 1314/78125 [12:29<10:35:10,  2.02it/s]
  2%|▌                                  | 1315/78125 [12:30<10:34:44,  2.02it/s]
  2%|▌                                  | 1316/78125 [12:30<10:34:52,  2.02it/s]
  2%|▌                                  | 1317/78125 [12:31<10:34:45,  2.02it/s]
  2%|▌                                  | 1318/78125 [12:31<10:35:10,  2.02it/s]
  2%|▌                                  | 1319/78125 [12:32<10:34:32,  2.02it/s]
  2%|▌                                  | 1320/78125 [12:32<10:35:20,  2.01it/s]
  2%|▌                                  | 1321/78125 [12:33<10:34:40,  2.02it/s]
  2%|▌                                  | 1322/78125 [12:33<10:34:46,  2.02it/s]
  2%|▌                                  | 1323/78125 [12:34<10:34:27,  2.02it/s]
  2%|▌                                  | 1324/78125 [12:34<10:34:09,  2.02it/s]
  2%|▌                                  | 1325/78125 [12:35<10:33:42,  2.02it/s]
  2%|▌                                  | 1326/78125 [12:35<10:33:44,  2.02it/s]
  2%|▌                                  | 1327/78125 [12:36<10:34:09,  2.02it/s]
  2%|▌                                  | 1328/78125 [12:36<10:34:08,  2.02it/s]
  2%|▌                                  | 1329/78125 [12:37<10:34:11,  2.02it/s]
  2%|▌                                  | 1330/78125 [12:37<10:34:20,  2.02it/s]
  2%|▌                                  | 1331/78125 [12:38<10:33:28,  2.02it/s]
  2%|▌                                  | 1332/78125 [12:38<10:33:31,  2.02it/s]
  2%|▌                                  | 1333/78125 [12:39<10:33:52,  2.02it/s]
  2%|▌                                  | 1334/78125 [12:39<10:33:26,  2.02it/s]
  2%|▌                                  | 1335/78125 [12:40<10:34:27,  2.02it/s]
  2%|▌                                  | 1336/78125 [12:40<10:35:00,  2.02it/s]
  2%|▌                                  | 1337/78125 [12:41<10:34:42,  2.02it/s]
  2%|▌                                  | 1338/78125 [12:41<10:35:25,  2.01it/s]
  2%|▌                                  | 1339/78125 [12:42<10:34:32,  2.02it/s]
  2%|▌                                  | 1340/78125 [12:42<10:34:08,  2.02it/s]
  2%|▌                                  | 1341/78125 [12:43<10:34:09,  2.02it/s]
  2%|▌                                  | 1342/78125 [12:43<10:33:09,  2.02it/s]
  2%|▌                                  | 1343/78125 [12:44<10:32:47,  2.02it/s]
  2%|▌                                  | 1344/78125 [12:44<10:33:24,  2.02it/s]
  2%|▌                                  | 1345/78125 [12:45<10:32:57,  2.02it/s]
  2%|▌                                  | 1346/78125 [12:45<10:33:18,  2.02it/s]
  2%|▌                                  | 1347/78125 [12:46<10:33:33,  2.02it/s]
  2%|▌                                  | 1348/78125 [12:46<10:33:18,  2.02it/s]
  2%|▌                                  | 1349/78125 [12:47<10:33:17,  2.02it/s]
  2%|▌                                  | 1350/78125 [12:47<10:33:18,  2.02it/s]
  2%|▌                                  | 1351/78125 [12:48<10:33:34,  2.02it/s]
  2%|▌                                  | 1352/78125 [12:48<10:33:43,  2.02it/s]
  2%|▌                                  | 1353/78125 [12:49<10:34:24,  2.02it/s]
  2%|▌                                  | 1354/78125 [12:49<10:34:22,  2.02it/s]
  2%|▌                                  | 1355/78125 [12:50<10:34:20,  2.02it/s]
  2%|▌                                  | 1356/78125 [12:50<10:34:06,  2.02it/s]
  2%|▌                                  | 1357/78125 [12:51<10:33:34,  2.02it/s]
  2%|▌                                  | 1358/78125 [12:51<10:33:45,  2.02it/s]
  2%|▌                                  | 1359/78125 [12:52<10:33:45,  2.02it/s]
  2%|▌                                  | 1360/78125 [12:52<10:34:29,  2.02it/s]
  2%|▌                                  | 1361/78125 [12:53<10:34:55,  2.02it/s]
  2%|▌                                  | 1362/78125 [12:53<10:34:27,  2.02it/s]
  2%|▌                                  | 1363/78125 [12:54<10:33:38,  2.02it/s]
  2%|▌                                  | 1364/78125 [12:54<10:33:58,  2.02it/s]
  2%|▌                                  | 1365/78125 [12:55<10:34:45,  2.02it/s]
  2%|▌                                  | 1366/78125 [12:55<10:34:19,  2.02it/s]
  2%|▌                                  | 1367/78125 [12:56<10:34:42,  2.02it/s]
  2%|▌                                  | 1368/78125 [12:56<10:34:58,  2.01it/s]
  2%|▌                                  | 1369/78125 [12:57<10:34:35,  2.02it/s]
  2%|▌                                  | 1370/78125 [12:57<10:35:03,  2.01it/s]
  2%|▌                                  | 1371/78125 [12:58<10:34:22,  2.02it/s]
  2%|▌                                  | 1372/78125 [12:58<10:33:57,  2.02it/s]
  2%|▌                                  | 1373/78125 [12:59<10:34:32,  2.02it/s]
  2%|▌                                  | 1374/78125 [12:59<10:34:12,  2.02it/s]
  2%|▌                                  | 1375/78125 [13:00<10:33:35,  2.02it/s]
  2%|▌                                  | 1376/78125 [13:00<10:32:59,  2.02it/s]
  2%|▌                                  | 1377/78125 [13:01<10:33:31,  2.02it/s]
  2%|▌                                  | 1378/78125 [13:01<10:33:36,  2.02it/s]
  2%|▌                                  | 1379/78125 [13:02<10:33:37,  2.02it/s]
  2%|▌                                  | 1380/78125 [13:02<10:33:50,  2.02it/s]
  2%|▌                                  | 1381/78125 [13:03<10:33:37,  2.02it/s]
  2%|▌                                  | 1382/78125 [13:03<10:33:48,  2.02it/s]
  2%|▌                                  | 1383/78125 [13:04<10:32:58,  2.02it/s]
  2%|▌                                  | 1384/78125 [13:04<10:32:18,  2.02it/s]
  2%|▌                                  | 1385/78125 [13:05<10:32:50,  2.02it/s]
  2%|▌                                  | 1386/78125 [13:05<10:33:01,  2.02it/s]
  2%|▌                                  | 1387/78125 [13:06<10:33:25,  2.02it/s]
  2%|▌                                  | 1388/78125 [13:06<10:33:38,  2.02it/s]
  2%|▌                                  | 1389/78125 [13:07<10:33:38,  2.02it/s]
  2%|▌                                  | 1390/78125 [13:07<10:33:53,  2.02it/s]
  2%|▌                                  | 1391/78125 [13:08<10:34:11,  2.02it/s]
  2%|▌                                  | 1392/78125 [13:08<10:34:09,  2.02it/s]
  2%|▌                                  | 1393/78125 [13:09<10:33:58,  2.02it/s]
  2%|▌                                  | 1394/78125 [13:09<10:33:17,  2.02it/s]
  2%|▌                                  | 1395/78125 [13:10<10:32:50,  2.02it/s]
  2%|▋                                  | 1396/78125 [13:10<10:33:20,  2.02it/s]
  2%|▋                                  | 1397/78125 [13:11<10:33:21,  2.02it/s]
  2%|▋                                  | 1398/78125 [13:11<10:33:23,  2.02it/s]
  2%|▋                                  | 1399/78125 [13:11<10:32:37,  2.02it/s]
  2%|▋                                  | 1400/78125 [13:12<10:33:54,  2.02it/s]
  2%|▋                                  | 1401/78125 [13:12<10:33:47,  2.02it/s]
  2%|▋                                  | 1402/78125 [13:13<10:34:00,  2.02it/s]
  2%|▋                                  | 1403/78125 [13:13<10:33:48,  2.02it/s]
  2%|▋                                  | 1404/78125 [13:14<10:34:25,  2.02it/s]
  2%|▋                                  | 1405/78125 [13:14<10:34:13,  2.02it/s]
  2%|▋                                  | 1406/78125 [13:15<10:33:33,  2.02it/s]
  2%|▋                                  | 1407/78125 [13:15<10:33:28,  2.02it/s]
  2%|▋                                  | 1408/78125 [13:16<10:34:19,  2.02it/s]
  2%|▋                                  | 1409/78125 [13:16<10:33:22,  2.02it/s]
  2%|▋                                  | 1410/78125 [13:17<10:33:58,  2.02it/s]
  2%|▋                                  | 1411/78125 [13:17<10:33:39,  2.02it/s]
  2%|▋                                  | 1412/78125 [13:18<10:33:17,  2.02it/s]
  2%|▋                                  | 1413/78125 [13:18<10:33:03,  2.02it/s]
  2%|▋                                  | 1414/78125 [13:19<10:33:17,  2.02it/s]
  2%|▋                                  | 1415/78125 [13:19<10:33:37,  2.02it/s]
  2%|▋                                  | 1416/78125 [13:20<10:34:30,  2.01it/s]
  2%|▋                                  | 1417/78125 [13:20<10:34:15,  2.02it/s]
  2%|▋                                  | 1418/78125 [13:21<10:33:56,  2.02it/s]
  2%|▋                                  | 1419/78125 [13:21<10:34:27,  2.02it/s]
  2%|▋                                  | 1420/78125 [13:22<10:34:12,  2.02it/s]
  2%|▋                                  | 1421/78125 [13:22<10:33:52,  2.02it/s]
  2%|▋                                  | 1422/78125 [13:23<10:33:57,  2.02it/s]
  2%|▋                                  | 1423/78125 [13:23<10:34:57,  2.01it/s]
  2%|▋                                  | 1424/78125 [13:24<10:34:39,  2.01it/s]
  2%|▋                                  | 1425/78125 [13:24<10:34:00,  2.02it/s]
  2%|▋                                  | 1426/78125 [13:25<10:33:52,  2.02it/s]
  2%|▋                                  | 1427/78125 [13:25<10:34:02,  2.02it/s]
  2%|▋                                  | 1428/78125 [13:26<10:34:08,  2.02it/s]
  2%|▋                                  | 1429/78125 [13:26<10:33:49,  2.02it/s]
  2%|▋                                  | 1430/78125 [13:27<10:33:40,  2.02it/s]
  2%|▋                                  | 1431/78125 [13:27<10:34:18,  2.02it/s]
  2%|▋                                  | 1432/78125 [13:28<10:34:25,  2.01it/s]
  2%|▋                                  | 1433/78125 [13:28<10:33:44,  2.02it/s]
  2%|▋                                  | 1434/78125 [13:29<10:33:42,  2.02it/s]
  2%|▋                                  | 1435/78125 [13:29<10:34:10,  2.02it/s]
  2%|▋                                  | 1436/78125 [13:30<10:34:07,  2.02it/s]
  2%|▋                                  | 1437/78125 [13:30<10:33:03,  2.02it/s]
  2%|▋                                  | 1438/78125 [13:31<10:33:33,  2.02it/s]
  2%|▋                                  | 1439/78125 [13:31<10:33:00,  2.02it/s]
  2%|▋                                  | 1440/78125 [13:32<10:32:47,  2.02it/s]
  2%|▋                                  | 1441/78125 [13:32<10:33:01,  2.02it/s]
  2%|▋                                  | 1442/78125 [13:33<10:33:18,  2.02it/s]
  2%|▋                                  | 1443/78125 [13:33<10:33:41,  2.02it/s]
  2%|▋                                  | 1444/78125 [13:34<10:34:05,  2.02it/s]
  2%|▋                                  | 1445/78125 [13:34<10:34:52,  2.01it/s]
  2%|▋                                  | 1446/78125 [13:35<10:34:16,  2.01it/s]
  2%|▋                                  | 1447/78125 [13:35<10:34:00,  2.02it/s]
  2%|▋                                  | 1448/78125 [13:36<10:34:35,  2.01it/s]
  2%|▋                                  | 1449/78125 [13:36<10:33:58,  2.02it/s]
  2%|▋                                  | 1450/78125 [13:37<10:34:00,  2.02it/s]
  2%|▋                                  | 1451/78125 [13:37<10:33:50,  2.02it/s]
  2%|▋                                  | 1452/78125 [13:38<10:33:58,  2.02it/s]
  2%|▋                                  | 1453/78125 [13:38<10:33:27,  2.02it/s]
  2%|▋                                  | 1454/78125 [13:39<10:33:15,  2.02it/s]
  2%|▋                                  | 1455/78125 [13:39<10:32:20,  2.02it/s]
  2%|▋                                  | 1456/78125 [13:40<10:32:44,  2.02it/s]
  2%|▋                                  | 1457/78125 [13:40<10:32:42,  2.02it/s]
  2%|▋                                  | 1458/78125 [13:41<10:32:14,  2.02it/s]
  2%|▋                                  | 1459/78125 [13:41<10:32:26,  2.02it/s]
  2%|▋                                  | 1460/78125 [13:42<10:32:45,  2.02it/s]
  2%|▋                                  | 1461/78125 [13:42<10:33:39,  2.02it/s]
  2%|▋                                  | 1462/78125 [13:43<10:33:18,  2.02it/s]
  2%|▋                                  | 1463/78125 [13:43<10:33:15,  2.02it/s]
  2%|▋                                  | 1464/78125 [13:44<10:32:37,  2.02it/s]
  2%|▋                                  | 1465/78125 [13:44<10:33:47,  2.02it/s]
  2%|▋                                  | 1466/78125 [13:45<10:33:28,  2.02it/s]
  2%|▋                                  | 1467/78125 [13:45<10:33:12,  2.02it/s]
  2%|▋                                  | 1468/78125 [13:46<10:33:37,  2.02it/s]
  2%|▋                                  | 1469/78125 [13:46<10:33:04,  2.02it/s]
  2%|▋                                  | 1470/78125 [13:47<10:33:24,  2.02it/s]
  2%|▋                                  | 1471/78125 [13:47<10:33:28,  2.02it/s]
  2%|▋                                  | 1472/78125 [13:48<10:33:04,  2.02it/s]
  2%|▋                                  | 1473/78125 [13:48<10:32:26,  2.02it/s]
  2%|▋                                  | 1474/78125 [13:49<10:32:59,  2.02it/s]
  2%|▋                                  | 1475/78125 [13:49<10:32:36,  2.02it/s]
  2%|▋                                  | 1476/78125 [13:50<10:33:36,  2.02it/s]
  2%|▋                                  | 1477/78125 [13:50<10:33:21,  2.02it/s]
  2%|▋                                  | 1478/78125 [13:51<10:33:13,  2.02it/s]
  2%|▋                                  | 1479/78125 [13:51<10:32:33,  2.02it/s]
  2%|▋                                  | 1480/78125 [13:52<10:32:42,  2.02it/s]
  2%|▋                                  | 1481/78125 [13:52<10:32:49,  2.02it/s]
  2%|▋                                  | 1482/78125 [13:53<10:32:24,  2.02it/s]
  2%|▋                                  | 1483/78125 [13:53<10:32:51,  2.02it/s]
  2%|▋                                  | 1484/78125 [13:54<10:32:50,  2.02it/s]
  2%|▋                                  | 1485/78125 [13:54<10:32:32,  2.02it/s]
  2%|▋                                  | 1486/78125 [13:55<10:31:57,  2.02it/s]
  2%|▋                                  | 1487/78125 [13:55<10:31:57,  2.02it/s]
  2%|▋                                  | 1488/78125 [13:56<10:32:21,  2.02it/s]
  2%|▋                                  | 1489/78125 [13:56<10:32:14,  2.02it/s]
  2%|▋                                  | 1490/78125 [13:57<10:32:02,  2.02it/s]
  2%|▋                                  | 1491/78125 [13:57<10:32:19,  2.02it/s]
  2%|▋                                  | 1492/78125 [13:58<10:32:26,  2.02it/s]
  2%|▋                                  | 1493/78125 [13:58<10:32:28,  2.02it/s]
  2%|▋                                  | 1494/78125 [13:59<10:32:27,  2.02it/s]
  2%|▋                                  | 1495/78125 [13:59<10:31:59,  2.02it/s]
  2%|▋                                  | 1496/78125 [14:00<10:31:59,  2.02it/s]
  2%|▋                                  | 1497/78125 [14:00<10:32:42,  2.02it/s]
  2%|▋                                  | 1498/78125 [14:01<10:32:31,  2.02it/s]
  2%|▋                                  | 1499/78125 [14:01<10:32:42,  2.02it/s]
  2%|▋                                  | 1500/78125 [14:02<10:33:04,  2.02it/s]
  2%|▋                                  | 1501/78125 [14:02<10:32:46,  2.02it/s]
  2%|▋                                  | 1502/78125 [14:03<10:32:42,  2.02it/s]
  2%|▋                                  | 1503/78125 [14:03<10:32:31,  2.02it/s]
  2%|▋                                  | 1504/78125 [14:04<10:32:30,  2.02it/s]
  2%|▋                                  | 1505/78125 [14:04<10:32:32,  2.02it/s]
  2%|▋                                  | 1506/78125 [14:05<10:32:39,  2.02it/s]
  2%|▋                                  | 1507/78125 [14:05<10:33:34,  2.02it/s]
  2%|▋                                  | 1508/78125 [14:06<10:33:11,  2.02it/s]
  2%|▋                                  | 1509/78125 [14:06<10:33:34,  2.02it/s]
  2%|▋                                  | 1510/78125 [14:07<10:33:10,  2.02it/s]
  2%|▋                                  | 1511/78125 [14:07<10:32:50,  2.02it/s]
  2%|▋                                  | 1512/78125 [14:08<10:32:54,  2.02it/s]
  2%|▋                                  | 1513/78125 [14:08<10:32:47,  2.02it/s]
  2%|▋                                  | 1514/78125 [14:08<10:32:39,  2.02it/s]
  2%|▋                                  | 1515/78125 [14:09<10:32:11,  2.02it/s]
  2%|▋                                  | 1516/78125 [14:09<10:31:50,  2.02it/s]
  2%|▋                                  | 1517/78125 [14:10<10:31:46,  2.02it/s]
  2%|▋                                  | 1518/78125 [14:10<10:32:05,  2.02it/s]
  2%|▋                                  | 1519/78125 [14:11<10:31:37,  2.02it/s]
  2%|▋                                  | 1520/78125 [14:11<10:32:16,  2.02it/s]
  2%|▋                                  | 1521/78125 [14:12<10:32:35,  2.02it/s]
  2%|▋                                  | 1522/78125 [14:12<10:32:22,  2.02it/s]
  2%|▋                                  | 1523/78125 [14:13<10:32:34,  2.02it/s]
  2%|▋                                  | 1524/78125 [14:13<10:33:09,  2.02it/s]
  2%|▋                                  | 1525/78125 [14:14<10:33:21,  2.02it/s]
  2%|▋                                  | 1526/78125 [14:14<10:32:34,  2.02it/s]
  2%|▋                                  | 1527/78125 [14:15<10:32:18,  2.02it/s]
  2%|▋                                  | 1528/78125 [14:15<10:32:47,  2.02it/s]
  2%|▋                                  | 1529/78125 [14:16<10:32:36,  2.02it/s]
  2%|▋                                  | 1530/78125 [14:16<10:32:23,  2.02it/s]
  2%|▋                                  | 1531/78125 [14:17<10:32:07,  2.02it/s]
  2%|▋                                  | 1532/78125 [14:17<10:32:14,  2.02it/s]
  2%|▋                                  | 1533/78125 [14:18<10:31:55,  2.02it/s]
  2%|▋                                  | 1534/78125 [14:18<10:32:08,  2.02it/s]
  2%|▋                                  | 1535/78125 [14:19<10:31:52,  2.02it/s]
  2%|▋                                  | 1536/78125 [14:19<10:31:36,  2.02it/s]
  2%|▋                                  | 1537/78125 [14:20<10:32:22,  2.02it/s]
  2%|▋                                  | 1538/78125 [14:20<10:32:18,  2.02it/s]
  2%|▋                                  | 1539/78125 [14:21<10:32:11,  2.02it/s]
  2%|▋                                  | 1540/78125 [14:21<10:32:35,  2.02it/s]
  2%|▋                                  | 1541/78125 [14:22<10:32:12,  2.02it/s]
  2%|▋                                  | 1542/78125 [14:22<10:32:08,  2.02it/s]
  2%|▋                                  | 1543/78125 [14:23<10:31:40,  2.02it/s]
  2%|▋                                  | 1544/78125 [14:23<10:31:38,  2.02it/s]
  2%|▋                                  | 1545/78125 [14:24<10:31:50,  2.02it/s]
  2%|▋                                  | 1546/78125 [14:24<10:31:34,  2.02it/s]
  2%|▋                                  | 1547/78125 [14:25<10:31:57,  2.02it/s]
  2%|▋                                  | 1548/78125 [14:25<10:32:05,  2.02it/s]
  2%|▋                                  | 1549/78125 [14:26<10:32:23,  2.02it/s]
  2%|▋                                  | 1550/78125 [14:26<10:31:52,  2.02it/s]
  2%|▋                                  | 1551/78125 [14:27<10:32:09,  2.02it/s]
  2%|▋                                  | 1552/78125 [14:27<10:32:39,  2.02it/s]
  2%|▋                                  | 1553/78125 [14:28<10:32:18,  2.02it/s]
  2%|▋                                  | 1554/78125 [14:28<10:32:35,  2.02it/s]
  2%|▋                                  | 1555/78125 [14:29<10:31:29,  2.02it/s]
  2%|▋                                  | 1556/78125 [14:29<10:32:01,  2.02it/s]
  2%|▋                                  | 1557/78125 [14:30<10:31:46,  2.02it/s]
  2%|▋                                  | 1558/78125 [14:30<10:32:08,  2.02it/s]
  2%|▋                                  | 1559/78125 [14:31<10:32:10,  2.02it/s]
  2%|▋                                  | 1560/78125 [14:31<10:31:40,  2.02it/s]
  2%|▋                                  | 1561/78125 [14:32<10:31:44,  2.02it/s]
  2%|▋                                  | 1562/78125 [14:32<10:32:16,  2.02it/s]
  2%|▋                                  | 1563/78125 [14:33<10:32:14,  2.02it/s]
  2%|▋                                  | 1564/78125 [14:33<10:31:32,  2.02it/s]
  2%|▋                                  | 1565/78125 [14:34<10:32:10,  2.02it/s]
  2%|▋                                  | 1566/78125 [14:34<10:31:48,  2.02it/s]
  2%|▋                                  | 1567/78125 [14:35<10:31:59,  2.02it/s]
  2%|▋                                  | 1568/78125 [14:35<10:32:27,  2.02it/s]
  2%|▋                                  | 1569/78125 [14:36<10:32:04,  2.02it/s]
  2%|▋                                  | 1570/78125 [14:36<10:32:01,  2.02it/s]
  2%|▋                                  | 1571/78125 [14:37<10:31:27,  2.02it/s]
  2%|▋                                  | 1572/78125 [14:37<10:32:17,  2.02it/s]
  2%|▋                                  | 1573/78125 [14:38<10:32:27,  2.02it/s]
  2%|▋                                  | 1574/78125 [14:38<10:31:42,  2.02it/s]
  2%|▋                                  | 1575/78125 [14:39<10:31:23,  2.02it/s]
  2%|▋                                  | 1576/78125 [14:39<10:31:18,  2.02it/s]
  2%|▋                                  | 1577/78125 [14:40<10:32:03,  2.02it/s]
  2%|▋                                  | 1578/78125 [14:40<10:33:01,  2.02it/s]
  2%|▋                                  | 1579/78125 [14:41<10:32:46,  2.02it/s]
  2%|▋                                  | 1580/78125 [14:41<10:31:44,  2.02it/s]
  2%|▋                                  | 1581/78125 [14:42<10:32:13,  2.02it/s]
  2%|▋                                  | 1582/78125 [14:42<10:32:44,  2.02it/s]
  2%|▋                                  | 1583/78125 [14:43<10:32:37,  2.02it/s]
  2%|▋                                  | 1584/78125 [14:43<10:32:18,  2.02it/s]
  2%|▋                                  | 1585/78125 [14:44<10:32:57,  2.02it/s]
  2%|▋                                  | 1586/78125 [14:44<10:32:13,  2.02it/s]
  2%|▋                                  | 1587/78125 [14:45<10:32:50,  2.02it/s]
  2%|▋                                  | 1588/78125 [14:45<10:32:19,  2.02it/s]
  2%|▋                                  | 1589/78125 [14:46<10:32:24,  2.02it/s]
  2%|▋                                  | 1590/78125 [14:46<10:32:50,  2.02it/s]
  2%|▋                                  | 1591/78125 [14:47<10:32:49,  2.02it/s]
  2%|▋                                  | 1592/78125 [14:47<10:32:23,  2.02it/s]
  2%|▋                                  | 1593/78125 [14:48<10:31:38,  2.02it/s]
  2%|▋                                  | 1594/78125 [14:48<10:31:52,  2.02it/s]
  2%|▋                                  | 1595/78125 [14:49<10:32:08,  2.02it/s]
  2%|▋                                  | 1596/78125 [14:49<10:31:59,  2.02it/s]
  2%|▋                                  | 1597/78125 [14:50<10:31:14,  2.02it/s]
  2%|▋                                  | 1598/78125 [14:50<10:31:01,  2.02it/s]
  2%|▋                                  | 1599/78125 [14:51<10:31:14,  2.02it/s]
  2%|▋                                  | 1600/78125 [14:51<10:31:22,  2.02it/s]
  2%|▋                                  | 1601/78125 [14:52<10:30:42,  2.02it/s]
  2%|▋                                  | 1602/78125 [14:52<10:31:38,  2.02it/s]
  2%|▋                                  | 1603/78125 [14:53<10:32:05,  2.02it/s]
  2%|▋                                  | 1604/78125 [14:53<10:31:49,  2.02it/s]
  2%|▋                                  | 1605/78125 [14:54<10:31:25,  2.02it/s]
  2%|▋                                  | 1606/78125 [14:54<10:31:37,  2.02it/s]
  2%|▋                                  | 1607/78125 [14:55<10:31:56,  2.02it/s]
  2%|▋                                  | 1608/78125 [14:55<10:31:45,  2.02it/s]
  2%|▋                                  | 1609/78125 [14:56<10:32:00,  2.02it/s]
  2%|▋                                  | 1610/78125 [14:56<10:31:21,  2.02it/s]
  2%|▋                                  | 1611/78125 [14:57<10:30:41,  2.02it/s]
  2%|▋                                  | 1612/78125 [14:57<10:30:57,  2.02it/s]
  2%|▋                                  | 1613/78125 [14:58<10:31:13,  2.02it/s]
  2%|▋                                  | 1614/78125 [14:58<10:30:28,  2.02it/s]
  2%|▋                                  | 1615/78125 [14:59<10:30:29,  2.02it/s]
  2%|▋                                  | 1616/78125 [14:59<10:31:24,  2.02it/s]
  2%|▋                                  | 1617/78125 [15:00<10:32:18,  2.02it/s]
  2%|▋                                  | 1618/78125 [15:00<10:32:12,  2.02it/s]
  2%|▋                                  | 1619/78125 [15:01<10:31:54,  2.02it/s]
  2%|▋                                  | 1620/78125 [15:01<10:31:33,  2.02it/s]
  2%|▋                                  | 1621/78125 [15:01<10:31:49,  2.02it/s]
  2%|▋                                  | 1622/78125 [15:02<10:32:03,  2.02it/s]
  2%|▋                                  | 1623/78125 [15:02<10:31:51,  2.02it/s]
  2%|▋                                  | 1624/78125 [15:03<10:31:58,  2.02it/s]
  2%|▋                                  | 1625/78125 [15:03<10:31:26,  2.02it/s]
  2%|▋                                  | 1626/78125 [15:04<10:31:15,  2.02it/s]
  2%|▋                                  | 1627/78125 [15:04<10:31:59,  2.02it/s]
  2%|▋                                  | 1628/78125 [15:05<10:32:08,  2.02it/s]
  2%|▋                                  | 1629/78125 [15:05<10:31:42,  2.02it/s]
  2%|▋                                  | 1630/78125 [15:06<10:31:39,  2.02it/s]
  2%|▋                                  | 1631/78125 [15:06<10:31:41,  2.02it/s]
  2%|▋                                  | 1632/78125 [15:07<10:32:27,  2.02it/s]
  2%|▋                                  | 1633/78125 [15:07<10:32:38,  2.02it/s]
  2%|▋                                  | 1634/78125 [15:08<10:32:30,  2.02it/s]
  2%|▋                                  | 1635/78125 [15:08<10:32:49,  2.01it/s]
  2%|▋                                  | 1636/78125 [15:09<10:32:14,  2.02it/s]
  2%|▋                                  | 1637/78125 [15:09<10:32:12,  2.02it/s]
  2%|▋                                  | 1638/78125 [15:10<10:31:55,  2.02it/s]
  2%|▋                                  | 1639/78125 [15:10<10:32:06,  2.02it/s]
  2%|▋                                  | 1640/78125 [15:11<10:32:15,  2.02it/s]
  2%|▋                                  | 1641/78125 [15:11<10:32:13,  2.02it/s]
  2%|▋                                  | 1642/78125 [15:12<10:31:41,  2.02it/s]
  2%|▋                                  | 1643/78125 [15:12<10:32:10,  2.02it/s]
  2%|▋                                  | 1644/78125 [15:13<10:31:08,  2.02it/s]
  2%|▋                                  | 1645/78125 [15:13<10:31:31,  2.02it/s]
  2%|▋                                  | 1646/78125 [15:14<10:32:03,  2.02it/s]
  2%|▋                                  | 1647/78125 [15:14<10:33:00,  2.01it/s]
  2%|▋                                  | 1648/78125 [15:15<10:32:49,  2.01it/s]
  2%|▋                                  | 1649/78125 [15:15<10:32:00,  2.02it/s]
  2%|▋                                  | 1650/78125 [15:16<10:32:23,  2.02it/s]
  2%|▋                                  | 1651/78125 [15:16<10:32:39,  2.01it/s]
  2%|▋                                  | 1652/78125 [15:17<10:32:15,  2.02it/s]
  2%|▋                                  | 1653/78125 [15:17<10:32:40,  2.01it/s]
  2%|▋                                  | 1654/78125 [15:18<10:31:43,  2.02it/s]
  2%|▋                                  | 1655/78125 [15:18<10:32:01,  2.02it/s]
  2%|▋                                  | 1656/78125 [15:19<10:31:39,  2.02it/s]
  2%|▋                                  | 1657/78125 [15:19<10:30:58,  2.02it/s]
  2%|▋                                  | 1658/78125 [15:20<10:30:37,  2.02it/s]
  2%|▋                                  | 1659/78125 [15:20<10:30:52,  2.02it/s]
  2%|▋                                  | 1660/78125 [15:21<10:30:44,  2.02it/s]
  2%|▋                                  | 1661/78125 [15:21<10:31:22,  2.02it/s]
  2%|▋                                  | 1662/78125 [15:22<10:31:51,  2.02it/s]
  2%|▋                                  | 1663/78125 [15:22<10:31:49,  2.02it/s]
  2%|▋                                  | 1664/78125 [15:23<10:31:19,  2.02it/s]
  2%|▋                                  | 1665/78125 [15:23<10:30:59,  2.02it/s]
  2%|▋                                  | 1666/78125 [15:24<10:30:28,  2.02it/s]
  2%|▋                                  | 1667/78125 [15:24<10:31:08,  2.02it/s]
  2%|▋                                  | 1668/78125 [15:25<10:31:11,  2.02it/s]
  2%|▋                                  | 1669/78125 [15:25<10:31:30,  2.02it/s]
  2%|▋                                  | 1670/78125 [15:26<10:32:17,  2.02it/s]
  2%|▋                                  | 1671/78125 [15:26<10:31:38,  2.02it/s]
  2%|▋                                  | 1672/78125 [15:27<10:31:46,  2.02it/s]
  2%|▋                                  | 1673/78125 [15:27<10:31:40,  2.02it/s]
  2%|▋                                  | 1674/78125 [15:28<10:31:51,  2.02it/s]
  2%|▊                                  | 1675/78125 [15:28<10:32:19,  2.02it/s]
  2%|▊                                  | 1676/78125 [15:29<10:32:01,  2.02it/s]
  2%|▊                                  | 1677/78125 [15:29<10:31:50,  2.02it/s]
  2%|▊                                  | 1678/78125 [15:30<10:31:59,  2.02it/s]
  2%|▊                                  | 1679/78125 [15:30<10:31:32,  2.02it/s]
  2%|▊                                  | 1680/78125 [15:31<10:31:21,  2.02it/s]
  2%|▊                                  | 1681/78125 [15:31<10:31:30,  2.02it/s]
  2%|▊                                  | 1682/78125 [15:32<10:31:53,  2.02it/s]
  2%|▊                                  | 1683/78125 [15:32<10:31:57,  2.02it/s]
  2%|▊                                  | 1684/78125 [15:33<10:32:16,  2.01it/s]
  2%|▊                                  | 1685/78125 [15:33<10:32:45,  2.01it/s]
  2%|▊                                  | 1686/78125 [15:34<10:32:08,  2.02it/s]
  2%|▊                                  | 1687/78125 [15:34<10:32:12,  2.02it/s]
  2%|▊                                  | 1688/78125 [15:35<10:33:04,  2.01it/s]
  2%|▊                                  | 1689/78125 [15:35<10:32:35,  2.01it/s]
  2%|▊                                  | 1690/78125 [15:36<10:33:00,  2.01it/s]
  2%|▊                                  | 1691/78125 [15:36<10:32:34,  2.01it/s]
  2%|▊                                  | 1692/78125 [15:37<10:32:43,  2.01it/s]
  2%|▊                                  | 1693/78125 [15:37<10:32:16,  2.01it/s]
  2%|▊                                  | 1694/78125 [15:38<10:31:50,  2.02it/s]
  2%|▊                                  | 1695/78125 [15:38<10:31:45,  2.02it/s]
  2%|▊                                  | 1696/78125 [15:39<10:30:59,  2.02it/s]
  2%|▊                                  | 1697/78125 [15:39<10:31:48,  2.02it/s]
  2%|▊                                  | 1698/78125 [15:40<10:31:35,  2.02it/s]
  2%|▊                                  | 1699/78125 [15:40<10:31:55,  2.02it/s]
  2%|▊                                  | 1700/78125 [15:41<10:31:54,  2.02it/s]
  2%|▊                                  | 1701/78125 [15:41<10:31:53,  2.02it/s]
  2%|▊                                  | 1702/78125 [15:42<10:32:22,  2.01it/s]
  2%|▊                                  | 1703/78125 [15:42<10:32:05,  2.02it/s]
  2%|▊                                  | 1704/78125 [15:43<10:31:32,  2.02it/s]
  2%|▊                                  | 1705/78125 [15:43<10:31:37,  2.02it/s]
  2%|▊                                  | 1706/78125 [15:44<10:31:26,  2.02it/s]
  2%|▊                                  | 1707/78125 [15:44<10:32:01,  2.02it/s]
  2%|▊                                  | 1708/78125 [15:45<10:32:01,  2.02it/s]
  2%|▊                                  | 1709/78125 [15:45<10:32:42,  2.01it/s]
  2%|▊                                  | 1710/78125 [15:46<10:32:43,  2.01it/s]
  2%|▊                                  | 1711/78125 [15:46<10:32:15,  2.01it/s]
  2%|▊                                  | 1712/78125 [15:47<10:31:33,  2.02it/s]
  2%|▊                                  | 1713/78125 [15:47<10:31:40,  2.02it/s]
  2%|▊                                  | 1714/78125 [15:48<10:31:50,  2.02it/s]
  2%|▊                                  | 1715/78125 [15:48<10:31:44,  2.02it/s]
  2%|▊                                  | 1716/78125 [15:49<10:31:29,  2.02it/s]
  2%|▊                                  | 1717/78125 [15:49<10:31:52,  2.02it/s]
  2%|▊                                  | 1718/78125 [15:50<10:31:57,  2.02it/s]
  2%|▊                                  | 1719/78125 [15:50<10:31:45,  2.02it/s]
  2%|▊                                  | 1720/78125 [15:51<10:30:47,  2.02it/s]
  2%|▊                                  | 1721/78125 [15:51<10:31:33,  2.02it/s]
  2%|▊                                  | 1722/78125 [15:52<10:31:32,  2.02it/s]
  2%|▊                                  | 1723/78125 [15:52<10:31:38,  2.02it/s]
  2%|▊                                  | 1724/78125 [15:53<10:30:45,  2.02it/s]
  2%|▊                                  | 1725/78125 [15:53<10:30:21,  2.02it/s]
  2%|▊                                  | 1726/78125 [15:54<10:30:44,  2.02it/s]
  2%|▊                                  | 1727/78125 [15:54<10:30:35,  2.02it/s]
  2%|▊                                  | 1728/78125 [15:55<10:30:40,  2.02it/s]
  2%|▊                                  | 1729/78125 [15:55<10:30:30,  2.02it/s]
  2%|▊                                  | 1730/78125 [15:56<10:30:34,  2.02it/s]
  2%|▊                                  | 1731/78125 [15:56<10:30:18,  2.02it/s]
  2%|▊                                  | 1732/78125 [15:57<10:30:16,  2.02it/s]
  2%|▊                                  | 1733/78125 [15:57<10:30:00,  2.02it/s]
  2%|▊                                  | 1734/78125 [15:58<10:30:23,  2.02it/s]
  2%|▊                                  | 1735/78125 [15:58<10:30:22,  2.02it/s]
  2%|▊                                  | 1736/78125 [15:59<10:30:47,  2.02it/s]
  2%|▊                                  | 1737/78125 [15:59<10:31:03,  2.02it/s]
  2%|▊                                  | 1738/78125 [16:00<10:30:24,  2.02it/s]
  2%|▊                                  | 1739/78125 [16:00<10:30:31,  2.02it/s]
  2%|▊                                  | 1740/78125 [16:00<10:30:10,  2.02it/s]
  2%|▊                                  | 1741/78125 [16:01<10:29:37,  2.02it/s]
  2%|▊                                  | 1742/78125 [16:01<10:29:35,  2.02it/s]
  2%|▊                                  | 1743/78125 [16:02<10:29:32,  2.02it/s]
  2%|▊                                  | 1744/78125 [16:02<10:29:01,  2.02it/s]
  2%|▊                                  | 1745/78125 [16:03<10:30:12,  2.02it/s]
  2%|▊                                  | 1746/78125 [16:03<10:30:26,  2.02it/s]
  2%|▊                                  | 1747/78125 [16:04<10:29:57,  2.02it/s]
  2%|▊                                  | 1748/78125 [16:04<10:29:54,  2.02it/s]
  2%|▊                                  | 1749/78125 [16:05<10:30:12,  2.02it/s]
  2%|▊                                  | 1750/78125 [16:05<10:30:28,  2.02it/s]
  2%|▊                                  | 1751/78125 [16:06<10:30:26,  2.02it/s]
  2%|▊                                  | 1752/78125 [16:06<10:29:59,  2.02it/s]
  2%|▊                                  | 1753/78125 [16:07<10:30:07,  2.02it/s]
  2%|▊                                  | 1754/78125 [16:07<10:29:54,  2.02it/s]
  2%|▊                                  | 1755/78125 [16:08<10:30:09,  2.02it/s]
  2%|▊                                  | 1756/78125 [16:08<10:31:01,  2.02it/s]
  2%|▊                                  | 1757/78125 [16:09<10:30:37,  2.02it/s]
  2%|▊                                  | 1758/78125 [16:09<10:30:52,  2.02it/s]
  2%|▊                                  | 1759/78125 [16:10<10:30:53,  2.02it/s]
  2%|▊                                  | 1760/78125 [16:10<10:30:14,  2.02it/s]
  2%|▊                                  | 1761/78125 [16:11<10:29:54,  2.02it/s]
  2%|▊                                  | 1762/78125 [16:11<10:29:02,  2.02it/s]
  2%|▊                                  | 1763/78125 [16:12<10:29:14,  2.02it/s]
  2%|▊                                  | 1764/78125 [16:12<10:29:37,  2.02it/s]
  2%|▊                                  | 1765/78125 [16:13<10:29:57,  2.02it/s]
  2%|▊                                  | 1766/78125 [16:13<10:29:13,  2.02it/s]
  2%|▊                                  | 1767/78125 [16:14<10:28:58,  2.02it/s]
  2%|▊                                  | 1768/78125 [16:14<10:28:12,  2.03it/s]
  2%|▊                                  | 1769/78125 [16:15<10:28:43,  2.02it/s]
  2%|▊                                  | 1770/78125 [16:15<10:29:26,  2.02it/s]
  2%|▊                                  | 1771/78125 [16:16<10:29:44,  2.02it/s]
  2%|▊                                  | 1772/78125 [16:16<10:30:02,  2.02it/s]
  2%|▊                                  | 1773/78125 [16:17<10:29:33,  2.02it/s]
  2%|▊                                  | 1774/78125 [16:17<10:29:09,  2.02it/s]
  2%|▊                                  | 1775/78125 [16:18<10:29:55,  2.02it/s]
  2%|▊                                  | 1776/78125 [16:18<10:29:40,  2.02it/s]
  2%|▊                                  | 1777/78125 [16:19<10:29:00,  2.02it/s]
  2%|▊                                  | 1778/78125 [16:19<10:28:50,  2.02it/s]
  2%|▊                                  | 1779/78125 [16:20<10:29:04,  2.02it/s]
  2%|▊                                  | 1780/78125 [16:20<10:29:26,  2.02it/s]
  2%|▊                                  | 1781/78125 [16:21<10:29:23,  2.02it/s]
  2%|▊                                  | 1782/78125 [16:21<10:29:43,  2.02it/s]
  2%|▊                                  | 1783/78125 [16:22<10:29:52,  2.02it/s]
  2%|▊                                  | 1784/78125 [16:22<10:30:25,  2.02it/s]
  2%|▊                                  | 1785/78125 [16:23<10:30:09,  2.02it/s]
  2%|▊                                  | 1786/78125 [16:23<10:30:20,  2.02it/s]
  2%|▊                                  | 1787/78125 [16:24<10:30:16,  2.02it/s]
  2%|▊                                  | 1788/78125 [16:24<10:30:03,  2.02it/s]
  2%|▊                                  | 1789/78125 [16:25<10:30:23,  2.02it/s]
  2%|▊                                  | 1790/78125 [16:25<10:30:33,  2.02it/s]
  2%|▊                                  | 1791/78125 [16:26<10:30:21,  2.02it/s]
  2%|▊                                  | 1792/78125 [16:26<10:29:40,  2.02it/s]
  2%|▊                                  | 1793/78125 [16:27<10:29:51,  2.02it/s]
  2%|▊                                  | 1794/78125 [16:27<10:30:02,  2.02it/s]
  2%|▊                                  | 1795/78125 [16:28<10:30:03,  2.02it/s]
  2%|▊                                  | 1796/78125 [16:28<10:30:02,  2.02it/s]
  2%|▊                                  | 1797/78125 [16:29<10:29:05,  2.02it/s]
  2%|▊                                  | 1798/78125 [16:29<10:29:04,  2.02it/s]
  2%|▊                                  | 1799/78125 [16:30<10:29:04,  2.02it/s]
  2%|▊                                  | 1800/78125 [16:30<10:29:32,  2.02it/s]
  2%|▊                                  | 1801/78125 [16:31<10:29:08,  2.02it/s]
  2%|▊                                  | 1802/78125 [16:31<10:29:54,  2.02it/s]
  2%|▊                                  | 1803/78125 [16:32<10:29:16,  2.02it/s]
  2%|▊                                  | 1804/78125 [16:32<10:29:29,  2.02it/s]
  2%|▊                                  | 1805/78125 [16:33<10:29:38,  2.02it/s]
  2%|▊                                  | 1806/78125 [16:33<10:29:37,  2.02it/s]
  2%|▊                                  | 1807/78125 [16:34<10:27:56,  2.03it/s]
  2%|▊                                  | 1808/78125 [16:34<10:27:49,  2.03it/s]
  2%|▊                                  | 1809/78125 [16:35<10:28:36,  2.02it/s]
  2%|▊                                  | 1810/78125 [16:35<10:29:15,  2.02it/s]
  2%|▊                                  | 1811/78125 [16:36<10:29:44,  2.02it/s]
  2%|▊                                  | 1812/78125 [16:36<10:29:16,  2.02it/s]
  2%|▊                                  | 1813/78125 [16:37<10:29:37,  2.02it/s]
  2%|▊                                  | 1814/78125 [16:37<10:29:58,  2.02it/s]
  2%|▊                                  | 1815/78125 [16:38<10:29:24,  2.02it/s]
  2%|▊                                  | 1816/78125 [16:38<10:29:21,  2.02it/s]
  2%|▊                                  | 1817/78125 [16:39<10:29:02,  2.02it/s]
  2%|▊                                  | 1818/78125 [16:39<10:29:06,  2.02it/s]
  2%|▊                                  | 1819/78125 [16:40<10:29:22,  2.02it/s]
  2%|▊                                  | 1820/78125 [16:40<10:29:03,  2.02it/s]
  2%|▊                                  | 1821/78125 [16:41<10:28:20,  2.02it/s]
  2%|▊                                  | 1822/78125 [16:41<10:29:00,  2.02it/s]
  2%|▊                                  | 1823/78125 [16:42<10:28:58,  2.02it/s]
  2%|▊                                  | 1824/78125 [16:42<10:28:30,  2.02it/s]
  2%|▊                                  | 1825/78125 [16:43<10:28:47,  2.02it/s]
  2%|▊                                  | 1826/78125 [16:43<10:28:10,  2.02it/s]
  2%|▊                                  | 1827/78125 [16:44<10:28:46,  2.02it/s]
  2%|▊                                  | 1828/78125 [16:44<10:28:55,  2.02it/s]
  2%|▊                                  | 1829/78125 [16:45<10:28:48,  2.02it/s]
  2%|▊                                  | 1830/78125 [16:45<10:28:17,  2.02it/s]
  2%|▊                                  | 1831/78125 [16:46<10:27:51,  2.03it/s]
  2%|▊                                  | 1832/78125 [16:46<10:28:07,  2.02it/s]
  2%|▊                                  | 1833/78125 [16:47<10:28:05,  2.02it/s]
  2%|▊                                  | 1834/78125 [16:47<10:28:42,  2.02it/s]
  2%|▊                                  | 1835/78125 [16:48<10:28:36,  2.02it/s]
  2%|▊                                  | 1836/78125 [16:48<10:29:41,  2.02it/s]
  2%|▊                                  | 1837/78125 [16:48<10:29:35,  2.02it/s]
  2%|▊                                  | 1838/78125 [16:49<10:29:48,  2.02it/s]
  2%|▊                                  | 1839/78125 [16:49<10:29:31,  2.02it/s]
  2%|▊                                  | 1840/78125 [16:50<10:29:53,  2.02it/s]
  2%|▊                                  | 1841/78125 [16:50<10:28:54,  2.02it/s]
  2%|▊                                  | 1842/78125 [16:51<10:28:29,  2.02it/s]
  2%|▊                                  | 1843/78125 [16:51<10:28:38,  2.02it/s]
  2%|▊                                  | 1844/78125 [16:52<10:29:02,  2.02it/s]
  2%|▊                                  | 1845/78125 [16:52<10:28:24,  2.02it/s]
  2%|▊                                  | 1846/78125 [16:53<10:29:06,  2.02it/s]
  2%|▊                                  | 1847/78125 [16:53<10:29:00,  2.02it/s]
  2%|▊                                  | 1848/78125 [16:54<10:28:36,  2.02it/s]
  2%|▊                                  | 1849/78125 [16:54<10:28:21,  2.02it/s]
  2%|▊                                  | 1850/78125 [16:55<10:28:33,  2.02it/s]
  2%|▊                                  | 1851/78125 [16:55<10:28:22,  2.02it/s]
  2%|▊                                  | 1852/78125 [16:56<10:28:35,  2.02it/s]
  2%|▊                                  | 1853/78125 [16:56<10:28:38,  2.02it/s]
  2%|▊                                  | 1854/78125 [16:57<10:29:05,  2.02it/s]
  2%|▊                                  | 1855/78125 [16:57<10:28:42,  2.02it/s]
  2%|▊                                  | 1856/78125 [16:58<10:28:39,  2.02it/s]
  2%|▊                                  | 1857/78125 [16:58<10:29:15,  2.02it/s]
  2%|▊                                  | 1858/78125 [16:59<10:29:29,  2.02it/s]
  2%|▊                                  | 1859/78125 [16:59<10:29:01,  2.02it/s]
  2%|▊                                  | 1860/78125 [17:00<10:29:03,  2.02it/s]
  2%|▊                                  | 1861/78125 [17:00<10:29:22,  2.02it/s]
  2%|▊                                  | 1862/78125 [17:01<10:28:30,  2.02it/s]
  2%|▊                                  | 1863/78125 [17:01<10:28:35,  2.02it/s]
  2%|▊                                  | 1864/78125 [17:02<10:28:19,  2.02it/s]
  2%|▊                                  | 1865/78125 [17:02<10:28:15,  2.02it/s]
  2%|▊                                  | 1866/78125 [17:03<10:27:52,  2.02it/s]
  2%|▊                                  | 1867/78125 [17:03<10:28:23,  2.02it/s]
  2%|▊                                  | 1868/78125 [17:04<10:29:20,  2.02it/s]
  2%|▊                                  | 1869/78125 [17:04<10:29:25,  2.02it/s]
  2%|▊                                  | 1870/78125 [17:05<10:28:50,  2.02it/s]
  2%|▊                                  | 1871/78125 [17:05<10:28:22,  2.02it/s]
  2%|▊                                  | 1872/78125 [17:06<10:27:54,  2.02it/s]
  2%|▊                                  | 1873/78125 [17:06<10:28:36,  2.02it/s]
  2%|▊                                  | 1874/78125 [17:07<10:28:47,  2.02it/s]
  2%|▊                                  | 1875/78125 [17:07<10:28:41,  2.02it/s]
  2%|▊                                  | 1876/78125 [17:08<10:27:52,  2.02it/s]
  2%|▊                                  | 1877/78125 [17:08<10:27:44,  2.02it/s]
  2%|▊                                  | 1878/78125 [17:09<10:28:03,  2.02it/s]
  2%|▊                                  | 1879/78125 [17:09<10:28:12,  2.02it/s]
  2%|▊                                  | 1880/78125 [17:10<10:28:39,  2.02it/s]
  2%|▊                                  | 1881/78125 [17:10<10:28:55,  2.02it/s]
  2%|▊                                  | 1882/78125 [17:11<10:28:04,  2.02it/s]
  2%|▊                                  | 1883/78125 [17:11<10:28:37,  2.02it/s]
  2%|▊                                  | 1884/78125 [17:12<10:28:32,  2.02it/s]
  2%|▊                                  | 1885/78125 [17:12<10:29:22,  2.02it/s]
  2%|▊                                  | 1886/78125 [17:13<10:28:56,  2.02it/s]
  2%|▊                                  | 1887/78125 [17:13<10:28:23,  2.02it/s]
  2%|▊                                  | 1888/78125 [17:14<10:28:01,  2.02it/s]
  2%|▊                                  | 1889/78125 [17:14<10:29:03,  2.02it/s]
  2%|▊                                  | 1890/78125 [17:15<10:28:43,  2.02it/s]
  2%|▊                                  | 1891/78125 [17:15<10:28:44,  2.02it/s]
  2%|▊                                  | 1892/78125 [17:16<10:28:26,  2.02it/s]
  2%|▊                                  | 1893/78125 [17:16<10:28:46,  2.02it/s]
  2%|▊                                  | 1894/78125 [17:17<10:29:22,  2.02it/s]
  2%|▊                                  | 1895/78125 [17:17<10:28:26,  2.02it/s]
  2%|▊                                  | 1896/78125 [17:18<10:28:21,  2.02it/s]
  2%|▊                                  | 1897/78125 [17:18<10:27:31,  2.02it/s]
  2%|▊                                  | 1898/78125 [17:19<10:27:47,  2.02it/s]
  2%|▊                                  | 1899/78125 [17:19<10:28:21,  2.02it/s]
  2%|▊                                  | 1900/78125 [17:20<10:28:56,  2.02it/s]
  2%|▊                                  | 1901/78125 [17:20<10:28:36,  2.02it/s]
  2%|▊                                  | 1902/78125 [17:21<10:28:50,  2.02it/s]
  2%|▊                                  | 1903/78125 [17:21<10:28:54,  2.02it/s]
  2%|▊                                  | 1904/78125 [17:22<10:28:54,  2.02it/s]
  2%|▊                                  | 1905/78125 [17:22<10:28:44,  2.02it/s]
  2%|▊                                  | 1906/78125 [17:23<10:28:55,  2.02it/s]
  2%|▊                                  | 1907/78125 [17:23<10:28:50,  2.02it/s]
  2%|▊                                  | 1908/78125 [17:24<10:28:48,  2.02it/s]
  2%|▊                                  | 1909/78125 [17:24<10:28:23,  2.02it/s]
  2%|▊                                  | 1910/78125 [17:25<10:28:19,  2.02it/s]
  2%|▊                                  | 1911/78125 [17:25<10:27:57,  2.02it/s]
  2%|▊                                  | 1912/78125 [17:26<10:27:37,  2.02it/s]
  2%|▊                                  | 1913/78125 [17:26<10:28:08,  2.02it/s]
  2%|▊                                  | 1914/78125 [17:27<10:28:52,  2.02it/s]
  2%|▊                                  | 1915/78125 [17:27<10:29:30,  2.02it/s]
  2%|▊                                  | 1916/78125 [17:28<10:29:11,  2.02it/s]
  2%|▊                                  | 1917/78125 [17:28<10:29:18,  2.02it/s]
  2%|▊                                  | 1918/78125 [17:29<10:28:52,  2.02it/s]
  2%|▊                                  | 1919/78125 [17:29<10:28:39,  2.02it/s]
  2%|▊                                  | 1920/78125 [17:30<10:29:06,  2.02it/s]
  2%|▊                                  | 1921/78125 [17:30<10:28:13,  2.02it/s]
  2%|▊                                  | 1922/78125 [17:31<10:27:52,  2.02it/s]
  2%|▊                                  | 1923/78125 [17:31<10:28:15,  2.02it/s]
  2%|▊                                  | 1924/78125 [17:32<10:28:01,  2.02it/s]
  2%|▊                                  | 1925/78125 [17:32<10:27:28,  2.02it/s]
  2%|▊                                  | 1926/78125 [17:33<10:27:12,  2.02it/s]
  2%|▊                                  | 1927/78125 [17:33<10:28:00,  2.02it/s]
  2%|▊                                  | 1928/78125 [17:34<10:27:56,  2.02it/s]
  2%|▊                                  | 1929/78125 [17:34<10:28:48,  2.02it/s]
  2%|▊                                  | 1930/78125 [17:34<10:28:13,  2.02it/s]
  2%|▊                                  | 1931/78125 [17:35<10:27:50,  2.02it/s]
  2%|▊                                  | 1932/78125 [17:35<10:28:20,  2.02it/s]
  2%|▊                                  | 1933/78125 [17:36<10:28:39,  2.02it/s]
  2%|▊                                  | 1934/78125 [17:36<10:28:41,  2.02it/s]
  2%|▊                                  | 1935/78125 [17:37<10:28:41,  2.02it/s]
  2%|▊                                  | 1936/78125 [17:37<10:28:45,  2.02it/s]
  2%|▊                                  | 1937/78125 [17:38<10:28:15,  2.02it/s]
  2%|▊                                  | 1938/78125 [17:38<10:28:54,  2.02it/s]
  2%|▊                                  | 1939/78125 [17:39<10:28:59,  2.02it/s]
  2%|▊                                  | 1940/78125 [17:39<10:29:02,  2.02it/s]
  2%|▊                                  | 1941/78125 [17:40<10:29:13,  2.02it/s]
  2%|▊                                  | 1942/78125 [17:40<10:29:23,  2.02it/s]
  2%|▊                                  | 1943/78125 [17:41<10:29:22,  2.02it/s]
  2%|▊                                  | 1944/78125 [17:41<10:29:45,  2.02it/s]
  2%|▊                                  | 1945/78125 [17:42<10:28:52,  2.02it/s]
  2%|▊                                  | 1946/78125 [17:42<10:29:00,  2.02it/s]
  2%|▊                                  | 1947/78125 [17:43<10:28:31,  2.02it/s]
  2%|▊                                  | 1948/78125 [17:43<10:28:33,  2.02it/s]
  2%|▊                                  | 1949/78125 [17:44<10:29:24,  2.02it/s]
  2%|▊                                  | 1950/78125 [17:44<10:29:17,  2.02it/s]
  2%|▊                                  | 1951/78125 [17:45<10:28:41,  2.02it/s]
  2%|▊                                  | 1952/78125 [17:45<10:28:36,  2.02it/s]
  2%|▊                                  | 1953/78125 [17:46<10:28:53,  2.02it/s]
  3%|▉                                  | 1954/78125 [17:46<10:28:44,  2.02it/s]
  3%|▉                                  | 1955/78125 [17:47<10:28:57,  2.02it/s]
  3%|▉                                  | 1956/78125 [17:47<10:28:40,  2.02it/s]
  3%|▉                                  | 1957/78125 [17:48<10:29:08,  2.02it/s]
  3%|▉                                  | 1958/78125 [17:48<10:29:36,  2.02it/s]
  3%|▉                                  | 1959/78125 [17:49<10:29:22,  2.02it/s]
  3%|▉                                  | 1960/78125 [17:49<10:29:01,  2.02it/s]
  3%|▉                                  | 1961/78125 [17:50<10:29:15,  2.02it/s]
  3%|▉                                  | 1962/78125 [17:50<10:29:49,  2.02it/s]
  3%|▉                                  | 1963/78125 [17:51<10:29:48,  2.02it/s]
  3%|▉                                  | 1964/78125 [17:51<10:28:33,  2.02it/s]
  3%|▉                                  | 1965/78125 [17:52<10:28:59,  2.02it/s]
  3%|▉                                  | 1966/78125 [17:52<10:29:00,  2.02it/s]
  3%|▉                                  | 1967/78125 [17:53<10:29:12,  2.02it/s]
  3%|▉                                  | 1968/78125 [17:53<10:29:16,  2.02it/s]
  3%|▉                                  | 1969/78125 [17:54<10:29:14,  2.02it/s]
  3%|▉                                  | 1970/78125 [17:54<10:29:04,  2.02it/s]
  3%|▉                                  | 1971/78125 [17:55<10:29:34,  2.02it/s]
  3%|▉                                  | 1972/78125 [17:55<10:28:56,  2.02it/s]
  3%|▉                                  | 1973/78125 [17:56<10:28:42,  2.02it/s]
  3%|▉                                  | 1974/78125 [17:56<10:28:05,  2.02it/s]
  3%|▉                                  | 1975/78125 [17:57<10:27:46,  2.02it/s]
  3%|▉                                  | 1976/78125 [17:57<10:28:47,  2.02it/s]
  3%|▉                                  | 1977/78125 [17:58<10:29:11,  2.02it/s]
  3%|▉                                  | 1978/78125 [17:58<10:30:59,  2.01it/s]
  3%|▉                                  | 1979/78125 [17:59<10:30:57,  2.01it/s]
  3%|▉                                  | 1980/78125 [17:59<10:30:18,  2.01it/s]
  3%|▉                                  | 1981/78125 [18:00<10:29:11,  2.02it/s]
  3%|▉                                  | 1982/78125 [18:00<10:28:55,  2.02it/s]
  3%|▉                                  | 1983/78125 [18:01<10:28:36,  2.02it/s]
  3%|▉                                  | 1984/78125 [18:01<10:28:10,  2.02it/s]
  3%|▉                                  | 1985/78125 [18:02<10:27:29,  2.02it/s]
  3%|▉                                  | 1986/78125 [18:02<10:27:37,  2.02it/s]
  3%|▉                                  | 1987/78125 [18:03<10:27:14,  2.02it/s]
  3%|▉                                  | 1988/78125 [18:03<10:28:18,  2.02it/s]
  3%|▉                                  | 1989/78125 [18:04<10:28:58,  2.02it/s]
  3%|▉                                  | 1990/78125 [18:04<10:29:48,  2.01it/s]
  3%|▉                                  | 1991/78125 [18:05<10:29:21,  2.02it/s]
  3%|▉                                  | 1992/78125 [18:05<10:29:17,  2.02it/s]
  3%|▉                                  | 1993/78125 [18:06<10:28:50,  2.02it/s]
  3%|▉                                  | 1994/78125 [18:06<10:29:08,  2.02it/s]
  3%|▉                                  | 1995/78125 [18:07<10:29:10,  2.02it/s]
  3%|▉                                  | 1996/78125 [18:07<10:28:58,  2.02it/s]
  3%|▉                                  | 1997/78125 [18:08<10:29:27,  2.02it/s]
  3%|▉                                  | 1998/78125 [18:08<10:28:49,  2.02it/s]
  3%|▉                                  | 1999/78125 [18:09<10:29:13,  2.02it/s]
  3%|▉                                  | 2000/78125 [18:09<10:28:45,  2.02it/s]
  3%|▉                                  | 2001/78125 [18:10<10:28:09,  2.02it/s]
  3%|▉                                  | 2002/78125 [18:10<10:29:02,  2.02it/s]
  3%|▉                                  | 2003/78125 [18:11<10:30:23,  2.01it/s]
  3%|▉                                  | 2004/78125 [18:11<10:31:11,  2.01it/s]
  3%|▉                                  | 2005/78125 [18:12<10:30:10,  2.01it/s]
  3%|▉                                  | 2006/78125 [18:12<10:28:40,  2.02it/s]
  3%|▉                                  | 2007/78125 [18:13<10:28:36,  2.02it/s]
  3%|▉                                  | 2008/78125 [18:13<10:29:28,  2.02it/s]
  3%|▉                                  | 2009/78125 [18:14<10:29:27,  2.02it/s]
  3%|▉                                  | 2010/78125 [18:14<10:29:27,  2.02it/s]
  3%|▉                                  | 2011/78125 [18:15<10:28:49,  2.02it/s]
  3%|▉                                  | 2012/78125 [18:15<10:28:37,  2.02it/s]
  3%|▉                                  | 2013/78125 [18:16<10:28:57,  2.02it/s]
  3%|▉                                  | 2014/78125 [18:16<10:28:38,  2.02it/s]
  3%|▉                                  | 2015/78125 [18:17<10:28:30,  2.02it/s]
  3%|▉                                  | 2016/78125 [18:17<10:28:49,  2.02it/s]
  3%|▉                                  | 2017/78125 [18:18<10:28:49,  2.02it/s]
  3%|▉                                  | 2018/78125 [18:18<10:28:30,  2.02it/s]
  3%|▉                                  | 2019/78125 [18:19<10:28:19,  2.02it/s]
  3%|▉                                  | 2020/78125 [18:19<10:28:06,  2.02it/s]
  3%|▉                                  | 2021/78125 [18:20<10:28:10,  2.02it/s]
  3%|▉                                  | 2022/78125 [18:20<10:27:57,  2.02it/s]
  3%|▉                                  | 2023/78125 [18:21<10:28:12,  2.02it/s]
  3%|▉                                  | 2024/78125 [18:21<10:28:51,  2.02it/s]
  3%|▉                                  | 2025/78125 [18:22<10:28:58,  2.02it/s]
  3%|▉                                  | 2026/78125 [18:22<10:29:41,  2.01it/s]
  3%|▉                                  | 2027/78125 [18:23<10:28:45,  2.02it/s]
  3%|▉                                  | 2028/78125 [18:23<10:28:30,  2.02it/s]
  3%|▉                                  | 2029/78125 [18:24<10:28:18,  2.02it/s]
  3%|▉                                  | 2030/78125 [18:24<10:28:55,  2.02it/s]
  3%|▉                                  | 2031/78125 [18:25<10:28:30,  2.02it/s]
  3%|▉                                  | 2032/78125 [18:25<10:28:42,  2.02it/s]
  3%|▉                                  | 2033/78125 [18:26<10:28:49,  2.02it/s]
  3%|▉                                  | 2034/78125 [18:26<10:29:16,  2.02it/s]
  3%|▉                                  | 2035/78125 [18:27<10:28:38,  2.02it/s]
  3%|▉                                  | 2036/78125 [18:27<10:29:19,  2.02it/s]
  3%|▉                                  | 2037/78125 [18:28<10:28:28,  2.02it/s]
  3%|▉                                  | 2038/78125 [18:28<10:28:54,  2.02it/s]
  3%|▉                                  | 2039/78125 [18:29<10:29:36,  2.01it/s]
  3%|▉                                  | 2040/78125 [18:29<10:29:50,  2.01it/s]
  3%|▉                                  | 2041/78125 [18:30<10:29:09,  2.02it/s]
  3%|▉                                  | 2042/78125 [18:30<10:29:38,  2.01it/s]
  3%|▉                                  | 2043/78125 [18:31<10:29:30,  2.01it/s]
  3%|▉                                  | 2044/78125 [18:31<10:29:02,  2.02it/s]
  3%|▉                                  | 2045/78125 [18:32<10:29:32,  2.01it/s]
  3%|▉                                  | 2046/78125 [18:32<10:29:11,  2.02it/s]
  3%|▉                                  | 2047/78125 [18:32<10:29:02,  2.02it/s]
  3%|▉                                  | 2048/78125 [18:33<10:29:57,  2.01it/s]
  3%|▉                                  | 2049/78125 [18:33<10:29:28,  2.01it/s]
  3%|▉                                  | 2050/78125 [18:34<10:29:34,  2.01it/s]
  3%|▉                                  | 2051/78125 [18:34<10:29:52,  2.01it/s]
  3%|▉                                  | 2052/78125 [18:35<10:29:30,  2.01it/s]
  3%|▉                                  | 2053/78125 [18:35<10:29:16,  2.01it/s]
  3%|▉                                  | 2054/78125 [18:36<10:29:36,  2.01it/s]
  3%|▉                                  | 2055/78125 [18:36<10:29:24,  2.01it/s]
  3%|▉                                  | 2056/78125 [18:37<10:29:33,  2.01it/s]
  3%|▉                                  | 2057/78125 [18:37<10:29:22,  2.01it/s]
  3%|▉                                  | 2058/78125 [18:38<10:29:06,  2.02it/s]
  3%|▉                                  | 2059/78125 [18:38<10:28:06,  2.02it/s]
  3%|▉                                  | 2060/78125 [18:39<10:28:55,  2.02it/s]
  3%|▉                                  | 2061/78125 [18:39<10:27:35,  2.02it/s]
  3%|▉                                  | 2062/78125 [18:40<10:27:45,  2.02it/s]
  3%|▉                                  | 2063/78125 [18:40<10:28:14,  2.02it/s]
  3%|▉                                  | 2064/78125 [18:41<10:28:51,  2.02it/s]
  3%|▉                                  | 2065/78125 [18:41<10:28:37,  2.02it/s]
  3%|▉                                  | 2066/78125 [18:42<10:28:35,  2.02it/s]
  3%|▉                                  | 2067/78125 [18:42<10:28:14,  2.02it/s]
  3%|▉                                  | 2068/78125 [18:43<10:28:18,  2.02it/s]
  3%|▉                                  | 2069/78125 [18:43<10:28:14,  2.02it/s]
  3%|▉                                  | 2070/78125 [18:44<10:28:40,  2.02it/s]
  3%|▉                                  | 2071/78125 [18:44<10:28:12,  2.02it/s]
  3%|▉                                  | 2072/78125 [18:45<10:28:17,  2.02it/s]
  3%|▉                                  | 2073/78125 [18:45<10:28:10,  2.02it/s]
  3%|▉                                  | 2074/78125 [18:46<10:28:22,  2.02it/s]
  3%|▉                                  | 2075/78125 [18:46<10:27:52,  2.02it/s]
  3%|▉                                  | 2076/78125 [18:47<10:28:29,  2.02it/s]
  3%|▉                                  | 2077/78125 [18:47<10:28:40,  2.02it/s]
  3%|▉                                  | 2078/78125 [18:48<10:28:53,  2.02it/s]
  3%|▉                                  | 2079/78125 [18:48<10:29:16,  2.01it/s]
  3%|▉                                  | 2080/78125 [18:49<10:29:13,  2.01it/s]
  3%|▉                                  | 2081/78125 [18:49<10:29:20,  2.01it/s]
  3%|▉                                  | 2082/78125 [18:50<10:28:26,  2.02it/s]
  3%|▉                                  | 2083/78125 [18:50<10:28:39,  2.02it/s]
  3%|▉                                  | 2084/78125 [18:51<10:28:48,  2.02it/s]
  3%|▉                                  | 2085/78125 [18:51<10:29:35,  2.01it/s]
  3%|▉                                  | 2086/78125 [18:52<10:29:36,  2.01it/s]
  3%|▉                                  | 2087/78125 [18:52<10:29:21,  2.01it/s]
  3%|▉                                  | 2088/78125 [18:53<10:28:41,  2.02it/s]
  3%|▉                                  | 2089/78125 [18:53<10:29:12,  2.01it/s]
  3%|▉                                  | 2090/78125 [18:54<10:28:37,  2.02it/s]
  3%|▉                                  | 2091/78125 [18:54<10:28:15,  2.02it/s]
  3%|▉                                  | 2092/78125 [18:55<10:28:21,  2.02it/s]
  3%|▉                                  | 2093/78125 [18:55<10:28:38,  2.02it/s]
  3%|▉                                  | 2094/78125 [18:56<10:28:24,  2.02it/s]
  3%|▉                                  | 2095/78125 [18:56<10:28:48,  2.02it/s]
  3%|▉                                  | 2096/78125 [18:57<10:29:16,  2.01it/s]
  3%|▉                                  | 2097/78125 [18:57<10:27:44,  2.02it/s]
  3%|▉                                  | 2098/78125 [18:58<10:27:47,  2.02it/s]
  3%|▉                                  | 2099/78125 [18:58<10:28:01,  2.02it/s]
  3%|▉                                  | 2100/78125 [18:59<10:27:47,  2.02it/s]
  3%|▉                                  | 2101/78125 [18:59<10:27:48,  2.02it/s]
  3%|▉                                  | 2102/78125 [19:00<10:28:13,  2.02it/s]
  3%|▉                                  | 2103/78125 [19:00<10:27:01,  2.02it/s]
  3%|▉                                  | 2104/78125 [19:01<10:26:36,  2.02it/s]
  3%|▉                                  | 2105/78125 [19:01<10:26:29,  2.02it/s]
  3%|▉                                  | 2106/78125 [19:02<10:26:56,  2.02it/s]
  3%|▉                                  | 2107/78125 [19:02<10:28:05,  2.02it/s]
  3%|▉                                  | 2108/78125 [19:03<10:28:09,  2.02it/s]
  3%|▉                                  | 2109/78125 [19:03<10:28:04,  2.02it/s]
  3%|▉                                  | 2110/78125 [19:04<10:27:57,  2.02it/s]
  3%|▉                                  | 2111/78125 [19:04<10:28:47,  2.01it/s]
  3%|▉                                  | 2112/78125 [19:05<10:29:47,  2.01it/s]
  3%|▉                                  | 2113/78125 [19:05<10:29:32,  2.01it/s]
  3%|▉                                  | 2114/78125 [19:06<10:28:52,  2.01it/s]
  3%|▉                                  | 2115/78125 [19:06<10:28:50,  2.01it/s]
  3%|▉                                  | 2116/78125 [19:07<10:29:12,  2.01it/s]
  3%|▉                                  | 2117/78125 [19:07<10:28:53,  2.01it/s]
  3%|▉                                  | 2118/78125 [19:08<10:29:29,  2.01it/s]
  3%|▉                                  | 2119/78125 [19:08<10:29:15,  2.01it/s]
  3%|▉                                  | 2120/78125 [19:09<10:28:54,  2.01it/s]
  3%|▉                                  | 2121/78125 [19:09<10:29:11,  2.01it/s]
  3%|▉                                  | 2122/78125 [19:10<10:29:47,  2.01it/s]
  3%|▉                                  | 2123/78125 [19:10<10:29:23,  2.01it/s]
  3%|▉                                  | 2124/78125 [19:11<10:29:03,  2.01it/s]
  3%|▉                                  | 2125/78125 [19:11<10:28:28,  2.02it/s]
  3%|▉                                  | 2126/78125 [19:12<10:28:33,  2.02it/s]
  3%|▉                                  | 2127/78125 [19:12<10:28:32,  2.02it/s]
  3%|▉                                  | 2128/78125 [19:13<10:28:40,  2.01it/s]
  3%|▉                                  | 2129/78125 [19:13<10:28:42,  2.01it/s]
  3%|▉                                  | 2130/78125 [19:14<10:28:36,  2.01it/s]
  3%|▉                                  | 2131/78125 [19:14<10:28:26,  2.02it/s]
  3%|▉                                  | 2132/78125 [19:15<10:28:48,  2.01it/s]
  3%|▉                                  | 2133/78125 [19:15<10:27:52,  2.02it/s]
  3%|▉                                  | 2134/78125 [19:16<10:28:17,  2.02it/s]
  3%|▉                                  | 2135/78125 [19:16<10:28:02,  2.02it/s]
  3%|▉                                  | 2136/78125 [19:17<10:28:56,  2.01it/s]
  3%|▉                                  | 2137/78125 [19:17<10:29:20,  2.01it/s]
  3%|▉                                  | 2138/78125 [19:18<10:30:14,  2.01it/s]
  3%|▉                                  | 2139/78125 [19:18<10:30:13,  2.01it/s]
  3%|▉                                  | 2140/78125 [19:19<10:30:28,  2.01it/s]
  3%|▉                                  | 2141/78125 [19:19<10:29:25,  2.01it/s]
  3%|▉                                  | 2142/78125 [19:20<10:28:57,  2.01it/s]
  3%|▉                                  | 2143/78125 [19:20<10:28:32,  2.01it/s]
  3%|▉                                  | 2144/78125 [19:21<10:28:24,  2.02it/s]
  3%|▉                                  | 2145/78125 [19:21<10:28:33,  2.01it/s]
  3%|▉                                  | 2146/78125 [19:22<10:28:37,  2.01it/s]
  3%|▉                                  | 2147/78125 [19:22<10:28:55,  2.01it/s]
  3%|▉                                  | 2148/78125 [19:23<10:29:08,  2.01it/s]
  3%|▉                                  | 2149/78125 [19:23<10:28:39,  2.01it/s]
  3%|▉                                  | 2150/78125 [19:24<10:28:59,  2.01it/s]
  3%|▉                                  | 2151/78125 [19:24<10:28:50,  2.01it/s]
  3%|▉                                  | 2152/78125 [19:25<10:28:10,  2.02it/s]
  3%|▉                                  | 2153/78125 [19:25<10:28:27,  2.01it/s]
  3%|▉                                  | 2154/78125 [19:26<10:27:34,  2.02it/s]
  3%|▉                                  | 2155/78125 [19:26<10:28:18,  2.02it/s]
  3%|▉                                  | 2156/78125 [19:27<10:28:51,  2.01it/s]
  3%|▉                                  | 2157/78125 [19:27<10:29:29,  2.01it/s]
  3%|▉                                  | 2158/78125 [19:28<10:28:02,  2.02it/s]
  3%|▉                                  | 2159/78125 [19:28<10:27:34,  2.02it/s]
  3%|▉                                  | 2160/78125 [19:29<10:27:59,  2.02it/s]
  3%|▉                                  | 2161/78125 [19:29<10:27:49,  2.02it/s]
  3%|▉                                  | 2162/78125 [19:30<10:27:42,  2.02it/s]
  3%|▉                                  | 2163/78125 [19:30<10:28:02,  2.02it/s]
  3%|▉                                  | 2164/78125 [19:31<10:28:50,  2.01it/s]
  3%|▉                                  | 2165/78125 [19:31<10:28:46,  2.01it/s]
  3%|▉                                  | 2166/78125 [19:32<10:27:51,  2.02it/s]
  3%|▉                                  | 2167/78125 [19:32<10:27:30,  2.02it/s]
  3%|▉                                  | 2168/78125 [19:33<10:27:19,  2.02it/s]
  3%|▉                                  | 2169/78125 [19:33<10:28:11,  2.02it/s]
  3%|▉                                  | 2170/78125 [19:34<10:28:39,  2.01it/s]
  3%|▉                                  | 2171/78125 [19:34<10:28:55,  2.01it/s]
  3%|▉                                  | 2172/78125 [19:35<10:29:06,  2.01it/s]
  3%|▉                                  | 2173/78125 [19:35<10:28:34,  2.01it/s]
  3%|▉                                  | 2174/78125 [19:36<10:28:37,  2.01it/s]
  3%|▉                                  | 2175/78125 [19:36<10:28:45,  2.01it/s]
  3%|▉                                  | 2176/78125 [19:37<10:29:10,  2.01it/s]
  3%|▉                                  | 2177/78125 [19:37<10:28:20,  2.01it/s]
  3%|▉                                  | 2178/78125 [19:38<10:27:57,  2.02it/s]
  3%|▉                                  | 2179/78125 [19:38<10:28:15,  2.01it/s]
  3%|▉                                  | 2180/78125 [19:38<10:28:17,  2.01it/s]
  3%|▉                                  | 2181/78125 [19:39<10:28:14,  2.01it/s]
  3%|▉                                  | 2182/78125 [19:39<10:28:37,  2.01it/s]
  3%|▉                                  | 2183/78125 [19:40<10:26:49,  2.02it/s]
  3%|▉                                  | 2184/78125 [19:40<10:27:30,  2.02it/s]
  3%|▉                                  | 2185/78125 [19:41<10:28:23,  2.01it/s]
  3%|▉                                  | 2186/78125 [19:41<10:28:13,  2.01it/s]
  3%|▉                                  | 2187/78125 [19:42<10:27:48,  2.02it/s]
  3%|▉                                  | 2188/78125 [19:42<10:28:24,  2.01it/s]
  3%|▉                                  | 2189/78125 [19:43<10:28:16,  2.01it/s]
  3%|▉                                  | 2190/78125 [19:43<10:27:34,  2.02it/s]
  3%|▉                                  | 2191/78125 [19:44<10:28:06,  2.01it/s]
  3%|▉                                  | 2192/78125 [19:44<10:28:49,  2.01it/s]
  3%|▉                                  | 2193/78125 [19:45<10:28:38,  2.01it/s]
  3%|▉                                  | 2194/78125 [19:45<10:27:38,  2.02it/s]
  3%|▉                                  | 2195/78125 [19:46<10:27:46,  2.02it/s]
  3%|▉                                  | 2196/78125 [19:46<10:28:08,  2.01it/s]
  3%|▉                                  | 2197/78125 [19:47<10:28:03,  2.01it/s]
  3%|▉                                  | 2198/78125 [19:47<10:28:30,  2.01it/s]
  3%|▉                                  | 2199/78125 [19:48<10:27:57,  2.02it/s]
  3%|▉                                  | 2200/78125 [19:48<10:27:36,  2.02it/s]
  3%|▉                                  | 2201/78125 [19:49<10:27:37,  2.02it/s]
  3%|▉                                  | 2202/78125 [19:49<10:28:12,  2.01it/s]
  3%|▉                                  | 2203/78125 [19:50<10:26:45,  2.02it/s]
  3%|▉                                  | 2204/78125 [19:50<10:26:23,  2.02it/s]
  3%|▉                                  | 2205/78125 [19:51<10:26:11,  2.02it/s]
  3%|▉                                  | 2206/78125 [19:51<10:26:15,  2.02it/s]
  3%|▉                                  | 2207/78125 [19:52<10:26:26,  2.02it/s]
  3%|▉                                  | 2208/78125 [19:52<10:26:30,  2.02it/s]
  3%|▉                                  | 2209/78125 [19:53<10:26:24,  2.02it/s]
  3%|▉                                  | 2210/78125 [19:53<10:26:23,  2.02it/s]
  3%|▉                                  | 2211/78125 [19:54<10:26:14,  2.02it/s]
  3%|▉                                  | 2212/78125 [19:54<10:25:48,  2.02it/s]
  3%|▉                                  | 2213/78125 [19:55<10:27:05,  2.02it/s]
  3%|▉                                  | 2214/78125 [19:55<10:26:32,  2.02it/s]
  3%|▉                                  | 2215/78125 [19:56<10:27:25,  2.02it/s]
  3%|▉                                  | 2216/78125 [19:56<10:28:05,  2.01it/s]
  3%|▉                                  | 2217/78125 [19:57<10:27:58,  2.01it/s]
  3%|▉                                  | 2218/78125 [19:57<10:27:51,  2.01it/s]
  3%|▉                                  | 2219/78125 [19:58<10:27:37,  2.02it/s]
  3%|▉                                  | 2220/78125 [19:58<10:26:56,  2.02it/s]
  3%|▉                                  | 2221/78125 [19:59<10:27:07,  2.02it/s]
  3%|▉                                  | 2222/78125 [19:59<10:27:17,  2.02it/s]
  3%|▉                                  | 2223/78125 [20:00<10:26:31,  2.02it/s]
  3%|▉                                  | 2224/78125 [20:00<10:27:12,  2.02it/s]
  3%|▉                                  | 2225/78125 [20:01<10:27:26,  2.02it/s]
  3%|▉                                  | 2226/78125 [20:01<10:27:21,  2.02it/s]
  3%|▉                                  | 2227/78125 [20:02<10:27:29,  2.02it/s]
  3%|▉                                  | 2228/78125 [20:02<10:26:35,  2.02it/s]
  3%|▉                                  | 2229/78125 [20:03<10:26:44,  2.02it/s]
  3%|▉                                  | 2230/78125 [20:03<10:25:54,  2.02it/s]
  3%|▉                                  | 2231/78125 [20:04<10:26:47,  2.02it/s]
  3%|▉                                  | 2232/78125 [20:04<10:27:23,  2.02it/s]
  3%|█                                  | 2233/78125 [20:05<10:27:59,  2.01it/s]
  3%|█                                  | 2234/78125 [20:05<10:27:47,  2.01it/s]
  3%|█                                  | 2235/78125 [20:06<10:27:28,  2.02it/s]
  3%|█                                  | 2236/78125 [20:06<10:27:03,  2.02it/s]
  3%|█                                  | 2237/78125 [20:07<10:27:25,  2.02it/s]
  3%|█                                  | 2238/78125 [20:07<10:27:38,  2.02it/s]
  3%|█                                  | 2239/78125 [20:08<10:27:37,  2.02it/s]
  3%|█                                  | 2240/78125 [20:08<10:27:24,  2.02it/s]
  3%|█                                  | 2241/78125 [20:09<10:26:27,  2.02it/s]
  3%|█                                  | 2242/78125 [20:09<10:26:30,  2.02it/s]
  3%|█                                  | 2243/78125 [20:10<10:27:04,  2.02it/s]
  3%|█                                  | 2244/78125 [20:10<10:27:26,  2.02it/s]
  3%|█                                  | 2245/78125 [20:11<10:26:53,  2.02it/s]
  3%|█                                  | 2246/78125 [20:11<10:27:04,  2.02it/s]
  3%|█                                  | 2247/78125 [20:12<10:26:57,  2.02it/s]
  3%|█                                  | 2248/78125 [20:12<10:26:34,  2.02it/s]
  3%|█                                  | 2249/78125 [20:13<10:26:22,  2.02it/s]
  3%|█                                  | 2250/78125 [20:13<10:26:20,  2.02it/s]
  3%|█                                  | 2251/78125 [20:14<10:26:25,  2.02it/s]
  3%|█                                  | 2252/78125 [20:14<10:27:19,  2.02it/s]
  3%|█                                  | 2253/78125 [20:15<10:27:36,  2.01it/s]
  3%|█                                  | 2254/78125 [20:15<10:27:03,  2.02it/s]
  3%|█                                  | 2255/78125 [20:16<10:27:12,  2.02it/s]
  3%|█                                  | 2256/78125 [20:16<10:27:22,  2.02it/s]
  3%|█                                  | 2257/78125 [20:17<10:27:03,  2.02it/s]
  3%|█                                  | 2258/78125 [20:17<10:27:10,  2.02it/s]
  3%|█                                  | 2259/78125 [20:18<10:26:38,  2.02it/s]
  3%|█                                  | 2260/78125 [20:18<10:27:11,  2.02it/s]
  3%|█                                  | 2261/78125 [20:19<10:27:11,  2.02it/s]
  3%|█                                  | 2262/78125 [20:19<10:27:15,  2.02it/s]
  3%|█                                  | 2263/78125 [20:20<10:26:15,  2.02it/s]
  3%|█                                  | 2264/78125 [20:20<10:26:44,  2.02it/s]
  3%|█                                  | 2265/78125 [20:21<10:26:52,  2.02it/s]
  3%|█                                  | 2266/78125 [20:21<10:26:48,  2.02it/s]
  3%|█                                  | 2267/78125 [20:22<10:26:26,  2.02it/s]
  3%|█                                  | 2268/78125 [20:22<10:26:39,  2.02it/s]
  3%|█                                  | 2269/78125 [20:23<10:27:06,  2.02it/s]
  3%|█                                  | 2270/78125 [20:23<10:27:23,  2.02it/s]
  3%|█                                  | 2271/78125 [20:24<10:27:24,  2.01it/s]
  3%|█                                  | 2272/78125 [20:24<10:27:20,  2.02it/s]
  3%|█                                  | 2273/78125 [20:25<10:27:29,  2.01it/s]
  3%|█                                  | 2274/78125 [20:25<10:27:43,  2.01it/s]
  3%|█                                  | 2275/78125 [20:26<10:27:21,  2.02it/s]
  3%|█                                  | 2276/78125 [20:26<10:26:02,  2.02it/s]
  3%|█                                  | 2277/78125 [20:27<10:26:30,  2.02it/s]
  3%|█                                  | 2278/78125 [20:27<10:26:20,  2.02it/s]
  3%|█                                  | 2279/78125 [20:28<10:26:49,  2.02it/s]
  3%|█                                  | 2280/78125 [20:28<10:26:01,  2.02it/s]
  3%|█                                  | 2281/78125 [20:29<10:26:19,  2.02it/s]
  3%|█                                  | 2282/78125 [20:29<10:26:30,  2.02it/s]
  3%|█                                  | 2283/78125 [20:30<10:26:28,  2.02it/s]
  3%|█                                  | 2284/78125 [20:30<10:27:05,  2.02it/s]
  3%|█                                  | 2285/78125 [20:31<10:27:28,  2.01it/s]
  3%|█                                  | 2286/78125 [20:31<10:27:39,  2.01it/s]
  3%|█                                  | 2287/78125 [20:32<10:26:57,  2.02it/s]
  3%|█                                  | 2288/78125 [20:32<10:27:19,  2.01it/s]
  3%|█                                  | 2289/78125 [20:33<10:26:37,  2.02it/s]
  3%|█                                  | 2290/78125 [20:33<10:25:48,  2.02it/s]
  3%|█                                  | 2291/78125 [20:34<10:26:22,  2.02it/s]
  3%|█                                  | 2292/78125 [20:34<10:26:20,  2.02it/s]
  3%|█                                  | 2293/78125 [20:35<10:26:03,  2.02it/s]
  3%|█                                  | 2294/78125 [20:35<10:26:05,  2.02it/s]
  3%|█                                  | 2295/78125 [20:36<10:25:53,  2.02it/s]
  3%|█                                  | 2296/78125 [20:36<10:26:38,  2.02it/s]
  3%|█                                  | 2297/78125 [20:37<10:26:19,  2.02it/s]
  3%|█                                  | 2298/78125 [20:37<10:27:05,  2.02it/s]
  3%|█                                  | 2299/78125 [20:38<10:26:55,  2.02it/s]
  3%|█                                  | 2300/78125 [20:38<10:26:41,  2.02it/s]
  3%|█                                  | 2301/78125 [20:38<10:26:40,  2.02it/s]
  3%|█                                  | 2302/78125 [20:39<10:27:27,  2.01it/s]
  3%|█                                  | 2303/78125 [20:39<10:27:17,  2.01it/s]
  3%|█                                  | 2304/78125 [20:40<10:26:31,  2.02it/s]
  3%|█                                  | 2305/78125 [20:40<10:27:36,  2.01it/s]
  3%|█                                  | 2306/78125 [20:41<10:27:38,  2.01it/s]
  3%|█                                  | 2307/78125 [20:41<10:26:41,  2.02it/s]
  3%|█                                  | 2308/78125 [20:42<10:26:46,  2.02it/s]
  3%|█                                  | 2309/78125 [20:42<10:26:06,  2.02it/s]
  3%|█                                  | 2310/78125 [20:43<10:26:47,  2.02it/s]
  3%|█                                  | 2311/78125 [20:43<10:26:09,  2.02it/s]
  3%|█                                  | 2312/78125 [20:44<10:26:03,  2.02it/s]
  3%|█                                  | 2313/78125 [20:44<10:26:11,  2.02it/s]
  3%|█                                  | 2314/78125 [20:45<10:26:31,  2.02it/s]
  3%|█                                  | 2315/78125 [20:45<10:26:37,  2.02it/s]
  3%|█                                  | 2316/78125 [20:46<10:27:32,  2.01it/s]
  3%|█                                  | 2317/78125 [20:46<10:26:52,  2.02it/s]
  3%|█                                  | 2318/78125 [20:47<10:26:46,  2.02it/s]
  3%|█                                  | 2319/78125 [20:47<10:27:03,  2.01it/s]
  3%|█                                  | 2320/78125 [20:48<10:26:26,  2.02it/s]
  3%|█                                  | 2321/78125 [20:48<10:26:23,  2.02it/s]
  3%|█                                  | 2322/78125 [20:49<10:26:36,  2.02it/s]
  3%|█                                  | 2323/78125 [20:49<10:26:54,  2.02it/s]
  3%|█                                  | 2324/78125 [20:50<10:27:20,  2.01it/s]
  3%|█                                  | 2325/78125 [20:50<10:26:35,  2.02it/s]
  3%|█                                  | 2326/78125 [20:51<10:26:12,  2.02it/s]
  3%|█                                  | 2327/78125 [20:51<10:26:12,  2.02it/s]
  3%|█                                  | 2328/78125 [20:52<10:26:18,  2.02it/s]
  3%|█                                  | 2329/78125 [20:52<10:25:52,  2.02it/s]
  3%|█                                  | 2330/78125 [20:53<10:25:47,  2.02it/s]
  3%|█                                  | 2331/78125 [20:53<10:26:41,  2.02it/s]
  3%|█                                  | 2332/78125 [20:54<10:26:49,  2.02it/s]
  3%|█                                  | 2333/78125 [20:54<10:26:32,  2.02it/s]
  3%|█                                  | 2334/78125 [20:55<10:26:11,  2.02it/s]
  3%|█                                  | 2335/78125 [20:55<10:25:57,  2.02it/s]
  3%|█                                  | 2336/78125 [20:56<10:26:52,  2.01it/s]
  3%|█                                  | 2337/78125 [20:56<10:27:01,  2.01it/s]
  3%|█                                  | 2338/78125 [20:57<10:26:35,  2.02it/s]
  3%|█                                  | 2339/78125 [20:57<10:26:13,  2.02it/s]
  3%|█                                  | 2340/78125 [20:58<10:26:40,  2.02it/s]
  3%|█                                  | 2341/78125 [20:58<10:26:43,  2.02it/s]
  3%|█                                  | 2342/78125 [20:59<10:25:53,  2.02it/s]
  3%|█                                  | 2343/78125 [20:59<10:26:07,  2.02it/s]
  3%|█                                  | 2344/78125 [21:00<10:25:04,  2.02it/s]
  3%|█                                  | 2345/78125 [21:00<10:25:12,  2.02it/s]
  3%|█                                  | 2346/78125 [21:01<10:25:50,  2.02it/s]
  3%|█                                  | 2347/78125 [21:01<10:26:13,  2.02it/s]
  3%|█                                  | 2348/78125 [21:02<10:26:14,  2.02it/s]
  3%|█                                  | 2349/78125 [21:02<10:25:59,  2.02it/s]
  3%|█                                  | 2350/78125 [21:03<10:26:08,  2.02it/s]
  3%|█                                  | 2351/78125 [21:03<10:25:48,  2.02it/s]
  3%|█                                  | 2352/78125 [21:04<10:25:53,  2.02it/s]
  3%|█                                  | 2353/78125 [21:04<10:26:06,  2.02it/s]
  3%|█                                  | 2354/78125 [21:05<10:25:21,  2.02it/s]
  3%|█                                  | 2355/78125 [21:05<10:25:32,  2.02it/s]
  3%|█                                  | 2356/78125 [21:06<10:25:27,  2.02it/s]
  3%|█                                  | 2357/78125 [21:06<10:25:40,  2.02it/s]
  3%|█                                  | 2358/78125 [21:07<10:25:54,  2.02it/s]
  3%|█                                  | 2359/78125 [21:07<10:26:23,  2.02it/s]
  3%|█                                  | 2360/78125 [21:08<10:27:21,  2.01it/s]
  3%|█                                  | 2361/78125 [21:08<10:27:15,  2.01it/s]
  3%|█                                  | 2362/78125 [21:09<10:26:17,  2.02it/s]
  3%|█                                  | 2363/78125 [21:09<10:26:36,  2.02it/s]
  3%|█                                  | 2364/78125 [21:10<10:27:06,  2.01it/s]
  3%|█                                  | 2365/78125 [21:10<10:27:28,  2.01it/s]
  3%|█                                  | 2366/78125 [21:11<10:27:06,  2.01it/s]
  3%|█                                  | 2367/78125 [21:11<10:27:15,  2.01it/s]
  3%|█                                  | 2368/78125 [21:12<10:27:04,  2.01it/s]
  3%|█                                  | 2369/78125 [21:12<10:26:54,  2.01it/s]
  3%|█                                  | 2370/78125 [21:13<10:26:25,  2.02it/s]
  3%|█                                  | 2371/78125 [21:13<10:26:50,  2.01it/s]
  3%|█                                  | 2372/78125 [21:14<10:26:51,  2.01it/s]
  3%|█                                  | 2373/78125 [21:14<10:26:48,  2.01it/s]
  3%|█                                  | 2374/78125 [21:15<10:27:15,  2.01it/s]
  3%|█                                  | 2375/78125 [21:15<10:26:56,  2.01it/s]
  3%|█                                  | 2376/78125 [21:16<10:27:20,  2.01it/s]
  3%|█                                  | 2377/78125 [21:16<10:27:27,  2.01it/s]
  3%|█                                  | 2378/78125 [21:17<10:26:43,  2.01it/s]
  3%|█                                  | 2379/78125 [21:17<10:26:36,  2.01it/s]
  3%|█                                  | 2380/78125 [21:18<10:27:01,  2.01it/s]
  3%|█                                  | 2381/78125 [21:18<10:27:09,  2.01it/s]
  3%|█                                  | 2382/78125 [21:19<10:27:14,  2.01it/s]
  3%|█                                  | 2383/78125 [21:19<10:26:54,  2.01it/s]
  3%|█                                  | 2384/78125 [21:20<10:26:57,  2.01it/s]
  3%|█                                  | 2385/78125 [21:20<10:26:21,  2.02it/s]
  3%|█                                  | 2386/78125 [21:21<10:26:14,  2.02it/s]
  3%|█                                  | 2387/78125 [21:21<10:26:58,  2.01it/s]
  3%|█                                  | 2388/78125 [21:22<10:26:32,  2.01it/s]
  3%|█                                  | 2389/78125 [21:22<10:26:28,  2.01it/s]
  3%|█                                  | 2390/78125 [21:23<10:26:34,  2.01it/s]
  3%|█                                  | 2391/78125 [21:23<10:26:33,  2.01it/s]
  3%|█                                  | 2392/78125 [21:24<10:26:38,  2.01it/s]
  3%|█                                  | 2393/78125 [21:24<10:27:21,  2.01it/s]
  3%|█                                  | 2394/78125 [21:25<10:26:17,  2.02it/s]
  3%|█                                  | 2395/78125 [21:25<10:26:45,  2.01it/s]
  3%|█                                  | 2396/78125 [21:26<10:27:03,  2.01it/s]
  3%|█                                  | 2397/78125 [21:26<10:27:11,  2.01it/s]
  3%|█                                  | 2398/78125 [21:27<10:27:06,  2.01it/s]
  3%|█                                  | 2399/78125 [21:27<10:27:12,  2.01it/s]
  3%|█                                  | 2400/78125 [21:28<10:27:22,  2.01it/s]
  3%|█                                  | 2401/78125 [21:28<10:27:27,  2.01it/s]
  3%|█                                  | 2402/78125 [21:29<10:26:59,  2.01it/s]
  3%|█                                  | 2403/78125 [21:29<10:25:53,  2.02it/s]
  3%|█                                  | 2404/78125 [21:30<10:25:43,  2.02it/s]
  3%|█                                  | 2405/78125 [21:30<10:25:58,  2.02it/s]
  3%|█                                  | 2406/78125 [21:31<10:26:25,  2.01it/s]
  3%|█                                  | 2407/78125 [21:31<10:26:30,  2.01it/s]
  3%|█                                  | 2408/78125 [21:32<10:26:34,  2.01it/s]
  3%|█                                  | 2409/78125 [21:32<10:26:20,  2.01it/s]
  3%|█                                  | 2410/78125 [21:33<10:26:18,  2.01it/s]
  3%|█                                  | 2411/78125 [21:33<10:25:59,  2.02it/s]
  3%|█                                  | 2412/78125 [21:34<10:26:16,  2.01it/s]
  3%|█                                  | 2413/78125 [21:34<10:25:34,  2.02it/s]
  3%|█                                  | 2414/78125 [21:35<10:25:07,  2.02it/s]
  3%|█                                  | 2415/78125 [21:35<10:25:31,  2.02it/s]
  3%|█                                  | 2416/78125 [21:36<10:25:08,  2.02it/s]
  3%|█                                  | 2417/78125 [21:36<10:26:15,  2.01it/s]
  3%|█                                  | 2418/78125 [21:37<10:26:17,  2.01it/s]
  3%|█                                  | 2419/78125 [21:37<10:26:29,  2.01it/s]
  3%|█                                  | 2420/78125 [21:38<10:26:24,  2.01it/s]
  3%|█                                  | 2421/78125 [21:38<10:27:42,  2.01it/s]
  3%|█                                  | 2422/78125 [21:39<10:27:38,  2.01it/s]
  3%|█                                  | 2423/78125 [21:39<10:26:57,  2.01it/s]
  3%|█                                  | 2424/78125 [21:40<10:25:36,  2.02it/s]
  3%|█                                  | 2425/78125 [21:40<10:25:54,  2.02it/s]
  3%|█                                  | 2426/78125 [21:41<10:25:37,  2.02it/s]
  3%|█                                  | 2427/78125 [21:41<10:25:11,  2.02it/s]
  3%|█                                  | 2428/78125 [21:42<10:25:51,  2.02it/s]
  3%|█                                  | 2429/78125 [21:42<10:26:01,  2.02it/s]
  3%|█                                  | 2430/78125 [21:43<10:26:39,  2.01it/s]
  3%|█                                  | 2431/78125 [21:43<10:26:05,  2.02it/s]
  3%|█                                  | 2432/78125 [21:43<10:26:10,  2.01it/s]
  3%|█                                  | 2433/78125 [21:44<10:26:23,  2.01it/s]
  3%|█                                  | 2434/78125 [21:44<10:26:04,  2.01it/s]
  3%|█                                  | 2435/78125 [21:45<10:25:29,  2.02it/s]
  3%|█                                  | 2436/78125 [21:45<10:25:55,  2.02it/s]
  3%|█                                  | 2437/78125 [21:46<10:24:51,  2.02it/s]
  3%|█                                  | 2438/78125 [21:46<10:25:47,  2.02it/s]
  3%|█                                  | 2439/78125 [21:47<10:25:33,  2.02it/s]
  3%|█                                  | 2440/78125 [21:47<10:26:04,  2.01it/s]
  3%|█                                  | 2441/78125 [21:48<10:25:54,  2.02it/s]
  3%|█                                  | 2442/78125 [21:48<10:26:15,  2.01it/s]
  3%|█                                  | 2443/78125 [21:49<10:26:18,  2.01it/s]
  3%|█                                  | 2444/78125 [21:49<10:26:05,  2.01it/s]
  3%|█                                  | 2445/78125 [21:50<10:26:27,  2.01it/s]
  3%|█                                  | 2446/78125 [21:50<10:25:55,  2.02it/s]
  3%|█                                  | 2447/78125 [21:51<10:25:58,  2.01it/s]
  3%|█                                  | 2448/78125 [21:51<10:26:07,  2.01it/s]
  3%|█                                  | 2449/78125 [21:52<10:26:23,  2.01it/s]
  3%|█                                  | 2450/78125 [21:52<10:26:25,  2.01it/s]
  3%|█                                  | 2451/78125 [21:53<10:26:13,  2.01it/s]
  3%|█                                  | 2452/78125 [21:53<10:25:53,  2.02it/s]
  3%|█                                  | 2453/78125 [21:54<10:25:41,  2.02it/s]
  3%|█                                  | 2454/78125 [21:54<10:26:32,  2.01it/s]
  3%|█                                  | 2455/78125 [21:55<10:25:34,  2.02it/s]
  3%|█                                  | 2456/78125 [21:55<10:26:05,  2.01it/s]
  3%|█                                  | 2457/78125 [21:56<10:26:14,  2.01it/s]
  3%|█                                  | 2458/78125 [21:56<10:26:09,  2.01it/s]
  3%|█                                  | 2459/78125 [21:57<10:26:18,  2.01it/s]
  3%|█                                  | 2460/78125 [21:57<10:25:51,  2.01it/s]
  3%|█                                  | 2461/78125 [21:58<10:26:25,  2.01it/s]
  3%|█                                  | 2462/78125 [21:58<10:26:20,  2.01it/s]
  3%|█                                  | 2463/78125 [21:59<10:26:00,  2.01it/s]
  3%|█                                  | 2464/78125 [21:59<10:25:42,  2.02it/s]
  3%|█                                  | 2465/78125 [22:00<10:26:15,  2.01it/s]
  3%|█                                  | 2466/78125 [22:00<10:26:22,  2.01it/s]
  3%|█                                  | 2467/78125 [22:01<10:26:41,  2.01it/s]
  3%|█                                  | 2468/78125 [22:01<10:27:02,  2.01it/s]
  3%|█                                  | 2469/78125 [22:02<10:26:51,  2.01it/s]
  3%|█                                  | 2470/78125 [22:02<10:26:16,  2.01it/s]
  3%|█                                  | 2471/78125 [22:03<10:26:28,  2.01it/s]
  3%|█                                  | 2472/78125 [22:03<10:25:30,  2.02it/s]
  3%|█                                  | 2473/78125 [22:04<10:25:31,  2.02it/s]
  3%|█                                  | 2474/78125 [22:04<10:24:59,  2.02it/s]
760.3s 148 3%|█                                  | 2475/78125 [22:05<10:24:51,  2.02it/s]
  3%|█                                  | 2476/78125 [22:05<10:25:13,  2.02it/s]
  3%|█                                  | 2477/78125 [22:06<10:25:44,  2.01it/s]
  3%|█                                  | 2478/78125 [22:06<10:25:50,  2.01it/s]
  3%|█                                  | 2479/78125 [22:07<10:25:53,  2.01it/s]
  3%|█                                  | 2480/78125 [22:07<10:26:39,  2.01it/s]
  3%|█                                  | 2481/78125 [22:08<10:27:14,  2.01it/s]
  3%|█                                  | 2482/78125 [22:08<10:26:31,  2.01it/s]
  3%|█                                  | 2483/78125 [22:09<10:26:24,  2.01it/s]
  3%|█                                  | 2484/78125 [22:09<10:26:29,  2.01it/s]
  3%|█                                  | 2485/78125 [22:10<10:26:07,  2.01it/s]
  3%|█                                  | 2486/78125 [22:10<10:26:06,  2.01it/s]
  3%|█                                  | 2487/78125 [22:11<10:26:08,  2.01it/s]
  3%|█                                  | 2488/78125 [22:11<10:25:56,  2.01it/s]
  3%|█                                  | 2489/78125 [22:12<10:26:17,  2.01it/s]
  3%|█                                  | 2490/78125 [22:12<10:26:01,  2.01it/s]
  3%|█                                  | 2491/78125 [22:13<10:25:11,  2.02it/s]
  3%|█                                  | 2492/78125 [22:13<10:25:08,  2.02it/s]
  3%|█                                  | 2493/78125 [22:14<10:24:59,  2.02it/s]
  3%|█                                  | 2494/78125 [22:14<10:25:56,  2.01it/s]
  3%|█                                  | 2495/78125 [22:15<10:25:49,  2.01it/s]
  3%|█                                  | 2496/78125 [22:15<10:25:34,  2.01it/s]
  3%|█                                  | 2497/78125 [22:16<10:25:06,  2.02it/s]
  3%|█                                  | 2498/78125 [22:16<10:24:54,  2.02it/s]
  3%|█                                  | 2499/78125 [22:17<10:25:09,  2.02it/s]
  3%|█                                  | 2500/78125 [22:17<10:25:12,  2.02it/s]
  3%|█                                  | 2501/78125 [22:18<10:25:48,  2.01it/s]
  3%|█                                  | 2502/78125 [22:18<10:25:56,  2.01it/s]
  3%|█                                  | 2503/78125 [22:19<10:25:31,  2.01it/s]
  3%|█                                  | 2504/78125 [22:19<10:25:04,  2.02it/s]
  3%|█                                  | 2505/78125 [22:20<10:25:10,  2.02it/s]
  3%|█                                  | 2506/78125 [22:20<10:25:24,  2.02it/s]
  3%|█                                  | 2507/78125 [22:21<10:25:02,  2.02it/s]
  3%|█                                  | 2508/78125 [22:21<10:25:45,  2.01it/s]
  3%|█                                  | 2509/78125 [22:22<10:25:34,  2.01it/s]
  3%|█                                  | 2510/78125 [22:22<10:25:56,  2.01it/s]
  3%|█                                  | 2511/78125 [22:23<10:26:03,  2.01it/s]
  3%|█▏                                 | 2512/78125 [22:23<10:24:59,  2.02it/s]
  3%|█▏                                 | 2513/78125 [22:24<10:24:58,  2.02it/s]
  3%|█▏                                 | 2514/78125 [22:24<10:25:17,  2.02it/s]
  3%|█▏                                 | 2515/78125 [22:25<10:25:27,  2.01it/s]
  3%|█▏                                 | 2516/78125 [22:25<10:25:39,  2.01it/s]
  3%|█▏                                 | 2517/78125 [22:26<10:25:57,  2.01it/s]
  3%|█▏                                 | 2518/78125 [22:26<10:25:47,  2.01it/s]
  3%|█▏                                 | 2519/78125 [22:27<10:26:11,  2.01it/s]
  3%|█▏                                 | 2520/78125 [22:27<10:26:16,  2.01it/s]
  3%|█▏                                 | 2521/78125 [22:28<10:26:11,  2.01it/s]
  3%|█▏                                 | 2522/78125 [22:28<10:26:22,  2.01it/s]
  3%|█▏                                 | 2523/78125 [22:29<10:26:01,  2.01it/s]
  3%|█▏                                 | 2524/78125 [22:29<10:25:11,  2.02it/s]
  3%|█▏                                 | 2525/78125 [22:30<10:25:13,  2.02it/s]
  3%|█▏                                 | 2526/78125 [22:30<10:25:31,  2.01it/s]
  3%|█▏                                 | 2527/78125 [22:31<10:25:35,  2.01it/s]
  3%|█▏                                 | 2528/78125 [22:31<10:25:27,  2.01it/s]
  3%|█▏                                 | 2529/78125 [22:32<10:26:15,  2.01it/s]
  3%|█▏                                 | 2530/78125 [22:32<10:25:55,  2.01it/s]
  3%|█▏                                 | 2531/78125 [22:33<10:25:57,  2.01it/s]
  3%|█▏                                 | 2532/78125 [22:33<10:25:58,  2.01it/s]
  3%|█▏                                 | 2533/78125 [22:34<10:25:27,  2.01it/s]
  3%|█▏                                 | 2534/78125 [22:34<10:25:12,  2.02it/s]
  3%|█▏                                 | 2535/78125 [22:35<10:25:14,  2.01it/s]
  3%|█▏                                 | 2536/78125 [22:35<10:25:02,  2.02it/s]
  3%|█▏                                 | 2537/78125 [22:36<10:25:09,  2.02it/s]
  3%|█▏                                 | 2538/78125 [22:36<10:25:50,  2.01it/s]
  3%|█▏                                 | 2539/78125 [22:37<10:26:01,  2.01it/s]
  3%|█▏                                 | 2540/78125 [22:37<10:25:11,  2.01it/s]
  3%|█▏                                 | 2541/78125 [22:38<10:25:36,  2.01it/s]
  3%|█▏                                 | 2542/78125 [22:38<10:25:32,  2.01it/s]
  3%|█▏                                 | 2543/78125 [22:39<10:25:24,  2.01it/s]
  3%|█▏                                 | 2544/78125 [22:39<10:24:25,  2.02it/s]
  3%|█▏                                 | 2545/78125 [22:40<10:23:45,  2.02it/s]
  3%|█▏                                 | 2546/78125 [22:40<10:24:13,  2.02it/s]
  3%|█▏                                 | 2547/78125 [22:41<10:25:05,  2.02it/s]
  3%|█▏                                 | 2548/78125 [22:41<10:24:54,  2.02it/s]
  3%|█▏                                 | 2549/78125 [22:42<10:25:34,  2.01it/s]
  3%|█▏                                 | 2550/78125 [22:42<10:24:30,  2.02it/s]
  3%|█▏                                 | 2551/78125 [22:43<10:24:51,  2.02it/s]
  3%|█▏                                 | 2552/78125 [22:43<10:25:11,  2.01it/s]
  3%|█▏                                 | 2553/78125 [22:44<10:24:58,  2.02it/s]
  3%|█▏                                 | 2554/78125 [22:44<10:25:01,  2.02it/s]
  3%|█▏                                 | 2555/78125 [22:45<10:25:14,  2.01it/s]
  3%|█▏                                 | 2556/78125 [22:45<10:24:36,  2.02it/s]
  3%|█▏                                 | 2557/78125 [22:46<10:24:49,  2.02it/s]
  3%|█▏                                 | 2558/78125 [22:46<10:25:07,  2.01it/s]
  3%|█▏                                 | 2559/78125 [22:47<10:24:50,  2.02it/s]
  3%|█▏                                 | 2560/78125 [22:47<10:25:24,  2.01it/s]
  3%|█▏                                 | 2561/78125 [22:48<10:24:42,  2.02it/s]
  3%|█▏                                 | 2562/78125 [22:48<10:23:46,  2.02it/s]
  3%|█▏                                 | 2563/78125 [22:49<10:24:32,  2.02it/s]
  3%|█▏                                 | 2564/78125 [22:49<10:24:49,  2.02it/s]
  3%|█▏                                 | 2565/78125 [22:50<10:26:02,  2.01it/s]
  3%|█▏                                 | 2566/78125 [22:50<10:24:36,  2.02it/s]
  3%|█▏                                 | 2567/78125 [22:51<10:24:31,  2.02it/s]
  3%|█▏                                 | 2568/78125 [22:51<10:25:24,  2.01it/s]
  3%|█▏                                 | 2569/78125 [22:52<10:25:48,  2.01it/s]
  3%|█▏                                 | 2570/78125 [22:52<10:25:32,  2.01it/s]
  3%|█▏                                 | 2571/78125 [22:52<10:25:32,  2.01it/s]
  3%|█▏                                 | 2572/78125 [22:53<10:25:12,  2.01it/s]
  3%|█▏                                 | 2573/78125 [22:53<10:25:10,  2.01it/s]
  3%|█▏                                 | 2574/78125 [22:54<10:25:19,  2.01it/s]
  3%|█▏                                 | 2575/78125 [22:54<10:24:57,  2.01it/s]
  3%|█▏                                 | 2576/78125 [22:55<10:25:44,  2.01it/s]
  3%|█▏                                 | 2577/78125 [22:55<10:25:38,  2.01it/s]
  3%|█▏                                 | 2578/78125 [22:56<10:25:05,  2.01it/s]
  3%|█▏                                 | 2579/78125 [22:56<10:25:36,  2.01it/s]
  3%|█▏                                 | 2580/78125 [22:57<10:24:52,  2.01it/s]
  3%|█▏                                 | 2581/78125 [22:57<10:24:40,  2.02it/s]
  3%|█▏                                 | 2582/78125 [22:58<10:24:24,  2.02it/s]
  3%|█▏                                 | 2583/78125 [22:58<10:24:48,  2.02it/s]
  3%|█▏                                 | 2584/78125 [22:59<10:25:18,  2.01it/s]
  3%|█▏                                 | 2585/78125 [22:59<10:24:46,  2.02it/s]
  3%|█▏                                 | 2586/78125 [23:00<10:24:15,  2.02it/s]
  3%|█▏                                 | 2587/78125 [23:00<10:24:12,  2.02it/s]
  3%|█▏                                 | 2588/78125 [23:01<10:24:18,  2.02it/s]
  3%|█▏                                 | 2589/78125 [23:01<10:24:19,  2.02it/s]
  3%|█▏                                 | 2590/78125 [23:02<10:24:51,  2.01it/s]
  3%|█▏                                 | 2591/78125 [23:02<10:25:03,  2.01it/s]
  3%|█▏                                 | 2592/78125 [23:03<10:25:05,  2.01it/s]
  3%|█▏                                 | 2593/78125 [23:03<10:25:13,  2.01it/s]
  3%|█▏                                 | 2594/78125 [23:04<10:24:59,  2.01it/s]
  3%|█▏                                 | 2595/78125 [23:04<10:24:47,  2.01it/s]
  3%|█▏                                 | 2596/78125 [23:05<10:26:07,  2.01it/s]
  3%|█▏                                 | 2597/78125 [23:05<10:25:21,  2.01it/s]
  3%|█▏                                 | 2598/78125 [23:06<10:25:44,  2.01it/s]
  3%|█▏                                 | 2599/78125 [23:06<10:25:50,  2.01it/s]
  3%|█▏                                 | 2600/78125 [23:07<10:25:40,  2.01it/s]
  3%|█▏                                 | 2601/78125 [23:07<10:26:06,  2.01it/s]
  3%|█▏                                 | 2602/78125 [23:08<10:25:38,  2.01it/s]
  3%|█▏                                 | 2603/78125 [23:08<10:25:33,  2.01it/s]
  3%|█▏                                 | 2604/78125 [23:09<10:25:10,  2.01it/s]
  3%|█▏                                 | 2605/78125 [23:09<10:24:43,  2.01it/s]
  3%|█▏                                 | 2606/78125 [23:10<10:24:31,  2.02it/s]
  3%|█▏                                 | 2607/78125 [23:10<10:24:55,  2.01it/s]
  3%|█▏                                 | 2608/78125 [23:11<10:25:18,  2.01it/s]
  3%|█▏                                 | 2609/78125 [23:11<10:25:03,  2.01it/s]
  3%|█▏                                 | 2610/78125 [23:12<10:24:24,  2.02it/s]
  3%|█▏                                 | 2611/78125 [23:12<10:25:04,  2.01it/s]
  3%|█▏                                 | 2612/78125 [23:13<10:25:10,  2.01it/s]
  3%|█▏                                 | 2613/78125 [23:13<10:25:14,  2.01it/s]
  3%|█▏                                 | 2614/78125 [23:14<10:24:41,  2.01it/s]
  3%|█▏                                 | 2615/78125 [23:14<10:24:39,  2.01it/s]
  3%|█▏                                 | 2616/78125 [23:15<10:24:10,  2.02it/s]
  3%|█▏                                 | 2617/78125 [23:15<10:24:24,  2.02it/s]
  3%|█▏                                 | 2618/78125 [23:16<10:25:12,  2.01it/s]
  3%|█▏                                 | 2619/78125 [23:16<10:24:28,  2.02it/s]
  3%|█▏                                 | 2620/78125 [23:17<10:24:14,  2.02it/s]
  3%|█▏                                 | 2621/78125 [23:17<10:23:02,  2.02it/s]
  3%|█▏                                 | 2622/78125 [23:18<10:23:03,  2.02it/s]
  3%|█▏                                 | 2623/78125 [23:18<10:23:04,  2.02it/s]
  3%|█▏                                 | 2624/78125 [23:19<10:23:52,  2.02it/s]
  3%|█▏                                 | 2625/78125 [23:19<10:23:24,  2.02it/s]
  3%|█▏                                 | 2626/78125 [23:20<10:24:02,  2.02it/s]
  3%|█▏                                 | 2627/78125 [23:20<10:23:17,  2.02it/s]
  3%|█▏                                 | 2628/78125 [23:21<10:23:26,  2.02it/s]
  3%|█▏                                 | 2629/78125 [23:21<10:23:35,  2.02it/s]
  3%|█▏                                 | 2630/78125 [23:22<10:23:32,  2.02it/s]
  3%|█▏                                 | 2631/78125 [23:22<10:23:34,  2.02it/s]
  3%|█▏                                 | 2632/78125 [23:23<10:23:51,  2.02it/s]
  3%|█▏                                 | 2633/78125 [23:23<10:23:52,  2.02it/s]
  3%|█▏                                 | 2634/78125 [23:24<10:24:10,  2.02it/s]
  3%|█▏                                 | 2635/78125 [23:24<10:23:59,  2.02it/s]
  3%|█▏                                 | 2636/78125 [23:25<10:24:26,  2.01it/s]
  3%|█▏                                 | 2637/78125 [23:25<10:24:19,  2.02it/s]
  3%|█▏                                 | 2638/78125 [23:26<10:24:41,  2.01it/s]
  3%|█▏                                 | 2639/78125 [23:26<10:24:34,  2.01it/s]
  3%|█▏                                 | 2640/78125 [23:27<10:24:40,  2.01it/s]
  3%|█▏                                 | 2641/78125 [23:27<10:24:32,  2.01it/s]
  3%|█▏                                 | 2642/78125 [23:28<10:25:08,  2.01it/s]
  3%|█▏                                 | 2643/78125 [23:28<10:24:53,  2.01it/s]
  3%|█▏                                 | 2644/78125 [23:29<10:24:35,  2.01it/s]
  3%|█▏                                 | 2645/78125 [23:29<10:24:26,  2.01it/s]
  3%|█▏                                 | 2646/78125 [23:30<10:24:25,  2.01it/s]
  3%|█▏                                 | 2647/78125 [23:30<10:24:00,  2.02it/s]
  3%|█▏                                 | 2648/78125 [23:31<10:24:25,  2.01it/s]
  3%|█▏                                 | 2649/78125 [23:31<10:24:18,  2.01it/s]
  3%|█▏                                 | 2650/78125 [23:32<10:23:44,  2.02it/s]
  3%|█▏                                 | 2651/78125 [23:32<10:23:22,  2.02it/s]
  3%|█▏                                 | 2652/78125 [23:33<10:23:10,  2.02it/s]
  3%|█▏                                 | 2653/78125 [23:33<10:23:58,  2.02it/s]
  3%|█▏                                 | 2654/78125 [23:34<10:24:02,  2.02it/s]
  3%|█▏                                 | 2655/78125 [23:34<10:24:10,  2.02it/s]
  3%|█▏                                 | 2656/78125 [23:35<10:23:48,  2.02it/s]
  3%|█▏                                 | 2657/78125 [23:35<10:23:06,  2.02it/s]
  3%|█▏                                 | 2658/78125 [23:36<10:23:12,  2.02it/s]
  3%|█▏                                 | 2659/78125 [23:36<10:23:15,  2.02it/s]
  3%|█▏                                 | 2660/78125 [23:37<10:23:31,  2.02it/s]
  3%|█▏                                 | 2661/78125 [23:37<10:23:52,  2.02it/s]
  3%|█▏                                 | 2662/78125 [23:38<10:24:38,  2.01it/s]
  3%|█▏                                 | 2663/78125 [23:38<10:24:33,  2.01it/s]
  3%|█▏                                 | 2664/78125 [23:39<10:25:03,  2.01it/s]
  3%|█▏                                 | 2665/78125 [23:39<10:24:59,  2.01it/s]
  3%|█▏                                 | 2666/78125 [23:40<10:24:34,  2.01it/s]
  3%|█▏                                 | 2667/78125 [23:40<10:23:55,  2.02it/s]
  3%|█▏                                 | 2668/78125 [23:41<10:23:39,  2.02it/s]
  3%|█▏                                 | 2669/78125 [23:41<10:22:26,  2.02it/s]
  3%|█▏                                 | 2670/78125 [23:42<10:24:00,  2.02it/s]
  3%|█▏                                 | 2671/78125 [23:42<10:23:43,  2.02it/s]
  3%|█▏                                 | 2672/78125 [23:43<10:23:53,  2.02it/s]
  3%|█▏                                 | 2673/78125 [23:43<10:23:39,  2.02it/s]
  3%|█▏                                 | 2674/78125 [23:44<10:23:43,  2.02it/s]
  3%|█▏                                 | 2675/78125 [23:44<10:24:47,  2.01it/s]
  3%|█▏                                 | 2676/78125 [23:45<10:24:11,  2.01it/s]
  3%|█▏                                 | 2677/78125 [23:45<10:23:34,  2.02it/s]
  3%|█▏                                 | 2678/78125 [23:46<10:23:43,  2.02it/s]
  3%|█▏                                 | 2679/78125 [23:46<10:23:56,  2.02it/s]
  3%|█▏                                 | 2680/78125 [23:47<10:24:17,  2.01it/s]
  3%|█▏                                 | 2681/78125 [23:47<10:24:24,  2.01it/s]
  3%|█▏                                 | 2682/78125 [23:48<10:23:38,  2.02it/s]
  3%|█▏                                 | 2683/78125 [23:48<10:23:27,  2.02it/s]
  3%|█▏                                 | 2684/78125 [23:49<10:23:13,  2.02it/s]
  3%|█▏                                 | 2685/78125 [23:49<10:23:03,  2.02it/s]
  3%|█▏                                 | 2686/78125 [23:50<10:22:57,  2.02it/s]
  3%|█▏                                 | 2687/78125 [23:50<10:23:00,  2.02it/s]
  3%|█▏                                 | 2688/78125 [23:51<10:23:32,  2.02it/s]
  3%|█▏                                 | 2689/78125 [23:51<10:23:46,  2.02it/s]
  3%|█▏                                 | 2690/78125 [23:52<10:23:50,  2.02it/s]
  3%|█▏                                 | 2691/78125 [23:52<10:24:16,  2.01it/s]
  3%|█▏                                 | 2692/78125 [23:53<10:23:58,  2.01it/s]
  3%|█▏                                 | 2693/78125 [23:53<10:24:19,  2.01it/s]
  3%|█▏                                 | 2694/78125 [23:54<10:24:46,  2.01it/s]
  3%|█▏                                 | 2695/78125 [23:54<10:24:36,  2.01it/s]
  3%|█▏                                 | 2696/78125 [23:55<10:24:34,  2.01it/s]
  3%|█▏                                 | 2697/78125 [23:55<10:23:59,  2.01it/s]
  3%|█▏                                 | 2698/78125 [23:56<10:24:07,  2.01it/s]
  3%|█▏                                 | 2699/78125 [23:56<10:23:42,  2.02it/s]
  3%|█▏                                 | 2700/78125 [23:57<10:23:28,  2.02it/s]
  3%|█▏                                 | 2701/78125 [23:57<10:22:59,  2.02it/s]
  3%|█▏                                 | 2702/78125 [23:57<10:23:06,  2.02it/s]
  3%|█▏                                 | 2703/78125 [23:58<10:23:34,  2.02it/s]
  3%|█▏                                 | 2704/78125 [23:58<10:23:32,  2.02it/s]
  3%|█▏                                 | 2705/78125 [23:59<10:23:12,  2.02it/s]
  3%|█▏                                 | 2706/78125 [23:59<10:23:05,  2.02it/s]
  3%|█▏                                 | 2707/78125 [24:00<10:23:14,  2.02it/s]
  3%|█▏                                 | 2708/78125 [24:00<10:23:12,  2.02it/s]
  3%|█▏                                 | 2709/78125 [24:01<10:23:21,  2.02it/s]
  3%|█▏                                 | 2710/78125 [24:01<10:23:42,  2.02it/s]
  3%|█▏                                 | 2711/78125 [24:02<10:23:48,  2.01it/s]
  3%|█▏                                 | 2712/78125 [24:02<10:23:43,  2.02it/s]
  3%|█▏                                 | 2713/78125 [24:03<10:23:41,  2.02it/s]
  3%|█▏                                 | 2714/78125 [24:03<10:23:53,  2.01it/s]
  3%|█▏                                 | 2715/78125 [24:04<10:23:51,  2.01it/s]
  3%|█▏                                 | 2716/78125 [24:04<10:24:04,  2.01it/s]
  3%|█▏                                 | 2717/78125 [24:05<10:24:07,  2.01it/s]
  3%|█▏                                 | 2718/78125 [24:05<10:24:19,  2.01it/s]
  3%|█▏                                 | 2719/78125 [24:06<10:23:34,  2.02it/s]
  3%|█▏                                 | 2720/78125 [24:06<10:23:09,  2.02it/s]
  3%|█▏                                 | 2721/78125 [24:07<10:23:09,  2.02it/s]
  3%|█▏                                 | 2722/78125 [24:07<10:23:41,  2.01it/s]
  3%|█▏                                 | 2723/78125 [24:08<10:23:59,  2.01it/s]
  3%|█▏                                 | 2724/78125 [24:08<10:24:56,  2.01it/s]
  3%|█▏                                 | 2725/78125 [24:09<10:25:04,  2.01it/s]
  3%|█▏                                 | 2726/78125 [24:09<10:24:21,  2.01it/s]
  3%|█▏                                 | 2727/78125 [24:10<10:23:43,  2.01it/s]
  3%|█▏                                 | 2728/78125 [24:10<10:23:39,  2.01it/s]
  3%|█▏                                 | 2729/78125 [24:11<10:23:12,  2.02it/s]
  3%|█▏                                 | 2730/78125 [24:11<10:23:31,  2.02it/s]
  3%|█▏                                 | 2731/78125 [24:12<10:23:48,  2.01it/s]
  3%|█▏                                 | 2732/78125 [24:12<10:23:36,  2.01it/s]
  3%|█▏                                 | 2733/78125 [24:13<10:23:07,  2.02it/s]
  3%|█▏                                 | 2734/78125 [24:13<10:23:05,  2.02it/s]
  4%|█▏                                 | 2735/78125 [24:14<10:23:00,  2.02it/s]
  4%|█▏                                 | 2736/78125 [24:14<10:22:54,  2.02it/s]
  4%|█▏                                 | 2737/78125 [24:15<10:23:18,  2.02it/s]
  4%|█▏                                 | 2738/78125 [24:15<10:23:26,  2.02it/s]
  4%|█▏                                 | 2739/78125 [24:16<10:23:00,  2.02it/s]
  4%|█▏                                 | 2740/78125 [24:16<10:22:55,  2.02it/s]
  4%|█▏                                 | 2741/78125 [24:17<10:22:59,  2.02it/s]
  4%|█▏                                 | 2742/78125 [24:17<10:22:54,  2.02it/s]
  4%|█▏                                 | 2743/78125 [24:18<10:24:03,  2.01it/s]
  4%|█▏                                 | 2744/78125 [24:18<10:24:02,  2.01it/s]
  4%|█▏                                 | 2745/78125 [24:19<10:23:57,  2.01it/s]
  4%|█▏                                 | 2746/78125 [24:19<10:22:42,  2.02it/s]
  4%|█▏                                 | 2747/78125 [24:20<10:22:37,  2.02it/s]
  4%|█▏                                 | 2748/78125 [24:20<10:23:02,  2.02it/s]
  4%|█▏                                 | 2749/78125 [24:21<10:22:15,  2.02it/s]
  4%|█▏                                 | 2750/78125 [24:21<10:23:19,  2.02it/s]
  4%|█▏                                 | 2751/78125 [24:22<10:22:58,  2.02it/s]
  4%|█▏                                 | 2752/78125 [24:22<10:23:07,  2.02it/s]
  4%|█▏                                 | 2753/78125 [24:23<10:23:48,  2.01it/s]
  4%|█▏                                 | 2754/78125 [24:23<10:23:56,  2.01it/s]
  4%|█▏                                 | 2755/78125 [24:24<10:23:12,  2.02it/s]
  4%|█▏                                 | 2756/78125 [24:24<10:22:38,  2.02it/s]
  4%|█▏                                 | 2757/78125 [24:25<10:22:36,  2.02it/s]
  4%|█▏                                 | 2758/78125 [24:25<10:22:28,  2.02it/s]
  4%|█▏                                 | 2759/78125 [24:26<10:23:04,  2.02it/s]
  4%|█▏                                 | 2760/78125 [24:26<10:23:52,  2.01it/s]
  4%|█▏                                 | 2761/78125 [24:27<10:22:33,  2.02it/s]
  4%|█▏                                 | 2762/78125 [24:27<10:22:12,  2.02it/s]
  4%|█▏                                 | 2763/78125 [24:28<10:22:27,  2.02it/s]
  4%|█▏                                 | 2764/78125 [24:28<10:22:28,  2.02it/s]
  4%|█▏                                 | 2765/78125 [24:29<10:22:46,  2.02it/s]
  4%|█▏                                 | 2766/78125 [24:29<10:22:09,  2.02it/s]
  4%|█▏                                 | 2767/78125 [24:30<10:23:02,  2.02it/s]
  4%|█▏                                 | 2768/78125 [24:30<10:23:21,  2.01it/s]
  4%|█▏                                 | 2769/78125 [24:31<10:22:57,  2.02it/s]
  4%|█▏                                 | 2770/78125 [24:31<10:22:55,  2.02it/s]
  4%|█▏                                 | 2771/78125 [24:32<10:22:49,  2.02it/s]
  4%|█▏                                 | 2772/78125 [24:32<10:22:39,  2.02it/s]
  4%|█▏                                 | 2773/78125 [24:33<10:22:43,  2.02it/s]
  4%|█▏                                 | 2774/78125 [24:33<10:22:33,  2.02it/s]
  4%|█▏                                 | 2775/78125 [24:34<10:22:32,  2.02it/s]
  4%|█▏                                 | 2776/78125 [24:34<10:23:11,  2.02it/s]
  4%|█▏                                 | 2777/78125 [24:35<10:23:39,  2.01it/s]
  4%|█▏                                 | 2778/78125 [24:35<10:23:41,  2.01it/s]
  4%|█▏                                 | 2779/78125 [24:36<10:23:46,  2.01it/s]
  4%|█▏                                 | 2780/78125 [24:36<10:23:00,  2.02it/s]
  4%|█▏                                 | 2781/78125 [24:37<10:23:11,  2.02it/s]
  4%|█▏                                 | 2782/78125 [24:37<10:23:06,  2.02it/s]
  4%|█▏                                 | 2783/78125 [24:38<10:23:26,  2.01it/s]
  4%|█▏                                 | 2784/78125 [24:38<10:23:25,  2.01it/s]
  4%|█▏                                 | 2785/78125 [24:39<10:23:22,  2.01it/s]
  4%|█▏                                 | 2786/78125 [24:39<10:23:09,  2.01it/s]
  4%|█▏                                 | 2787/78125 [24:40<10:22:59,  2.02it/s]
  4%|█▏                                 | 2788/78125 [24:40<10:23:22,  2.01it/s]
  4%|█▏                                 | 2789/78125 [24:41<10:23:40,  2.01it/s]
  4%|█▏                                 | 2790/78125 [24:41<10:22:41,  2.02it/s]
  4%|█▎                                 | 2791/78125 [24:42<10:22:37,  2.02it/s]
  4%|█▎                                 | 2792/78125 [24:42<10:21:59,  2.02it/s]
  4%|█▎                                 | 2793/78125 [24:43<10:22:46,  2.02it/s]
  4%|█▎                                 | 2794/78125 [24:43<10:21:59,  2.02it/s]
  4%|█▎                                 | 2795/78125 [24:44<10:22:09,  2.02it/s]
  4%|█▎                                 | 2796/78125 [24:44<10:22:23,  2.02it/s]
  4%|█▎                                 | 2797/78125 [24:45<10:21:58,  2.02it/s]
  4%|█▎                                 | 2798/78125 [24:45<10:21:50,  2.02it/s]
  4%|█▎                                 | 2799/78125 [24:46<10:21:27,  2.02it/s]
  4%|█▎                                 | 2800/78125 [24:46<10:21:51,  2.02it/s]
  4%|█▎                                 | 2801/78125 [24:47<10:22:19,  2.02it/s]
  4%|█▎                                 | 2802/78125 [24:47<10:21:41,  2.02it/s]
  4%|█▎                                 | 2803/78125 [24:48<10:23:02,  2.01it/s]
  4%|█▎                                 | 2804/78125 [24:48<10:23:43,  2.01it/s]
  4%|█▎                                 | 2805/78125 [24:49<10:24:01,  2.01it/s]
  4%|█▎                                 | 2806/78125 [24:49<10:24:17,  2.01it/s]
  4%|█▎                                 | 2807/78125 [24:50<10:24:36,  2.01it/s]
  4%|█▎                                 | 2808/78125 [24:50<10:24:06,  2.01it/s]
  4%|█▎                                 | 2809/78125 [24:51<10:23:33,  2.01it/s]
  4%|█▎                                 | 2810/78125 [24:51<10:23:03,  2.01it/s]
  4%|█▎                                 | 2811/78125 [24:52<10:23:16,  2.01it/s]
  4%|█▎                                 | 2812/78125 [24:52<10:22:50,  2.02it/s]
  4%|█▎                                 | 2813/78125 [24:53<10:23:14,  2.01it/s]
  4%|█▎                                 | 2814/78125 [24:53<10:22:26,  2.02it/s]
  4%|█▎                                 | 2815/78125 [24:54<10:22:01,  2.02it/s]
  4%|█▎                                 | 2816/78125 [24:54<10:22:45,  2.02it/s]
  4%|█▎                                 | 2817/78125 [24:55<10:23:18,  2.01it/s]
  4%|█▎                                 | 2818/78125 [24:55<10:22:26,  2.02it/s]
  4%|█▎                                 | 2819/78125 [24:56<10:22:51,  2.02it/s]
  4%|█▎                                 | 2820/78125 [24:56<10:21:52,  2.02it/s]
  4%|█▎                                 | 2821/78125 [24:57<10:22:26,  2.02it/s]
  4%|█▎                                 | 2822/78125 [24:57<10:22:58,  2.01it/s]
  4%|█▎                                 | 2823/78125 [24:58<10:22:07,  2.02it/s]
  4%|█▎                                 | 2824/78125 [24:58<10:22:46,  2.02it/s]
  4%|█▎                                 | 2825/78125 [24:59<10:23:32,  2.01it/s]
  4%|█▎                                 | 2826/78125 [24:59<10:22:53,  2.01it/s]
  4%|█▎                                 | 2827/78125 [25:00<10:22:36,  2.02it/s]
  4%|█▎                                 | 2828/78125 [25:00<10:22:34,  2.02it/s]
  4%|█▎                                 | 2829/78125 [25:01<10:22:49,  2.01it/s]
  4%|█▎                                 | 2830/78125 [25:01<10:22:30,  2.02it/s]
  4%|█▎                                 | 2831/78125 [25:02<10:22:23,  2.02it/s]
  4%|█▎                                 | 2832/78125 [25:02<10:22:34,  2.02it/s]
  4%|█▎                                 | 2833/78125 [25:02<10:21:49,  2.02it/s]
  4%|█▎                                 | 2834/78125 [25:03<10:21:58,  2.02it/s]
  4%|█▎                                 | 2835/78125 [25:03<10:21:48,  2.02it/s]
  4%|█▎                                 | 2836/78125 [25:04<10:22:06,  2.02it/s]
  4%|█▎                                 | 2837/78125 [25:04<10:22:32,  2.02it/s]
  4%|█▎                                 | 2838/78125 [25:05<10:23:19,  2.01it/s]
  4%|█▎                                 | 2839/78125 [25:05<10:23:52,  2.01it/s]
  4%|█▎                                 | 2840/78125 [25:06<10:23:08,  2.01it/s]
  4%|█▎                                 | 2841/78125 [25:06<10:22:34,  2.02it/s]
  4%|█▎                                 | 2842/78125 [25:07<10:22:06,  2.02it/s]
  4%|█▎                                 | 2843/78125 [25:07<10:22:30,  2.02it/s]
  4%|█▎                                 | 2844/78125 [25:08<10:23:10,  2.01it/s]
  4%|█▎                                 | 2845/78125 [25:08<10:23:10,  2.01it/s]
  4%|█▎                                 | 2846/78125 [25:09<10:22:34,  2.02it/s]
  4%|█▎                                 | 2847/78125 [25:09<10:22:37,  2.02it/s]
  4%|█▎                                 | 2848/78125 [25:10<10:22:59,  2.01it/s]
  4%|█▎                                 | 2849/78125 [25:10<10:22:37,  2.02it/s]
  4%|█▎                                 | 2850/78125 [25:11<10:22:18,  2.02it/s]
  4%|█▎                                 | 2851/78125 [25:11<10:22:26,  2.02it/s]
  4%|█▎                                 | 2852/78125 [25:12<10:22:49,  2.01it/s]
  4%|█▎                                 | 2853/78125 [25:12<10:22:18,  2.02it/s]
  4%|█▎                                 | 2854/78125 [25:13<10:22:23,  2.02it/s]
  4%|█▎                                 | 2855/78125 [25:13<10:22:24,  2.02it/s]
  4%|█▎                                 | 2856/78125 [25:14<10:22:46,  2.01it/s]
  4%|█▎                                 | 2857/78125 [25:14<10:22:47,  2.01it/s]
  4%|█▎                                 | 2858/78125 [25:15<10:22:23,  2.02it/s]
  4%|█▎                                 | 2859/78125 [25:15<10:23:14,  2.01it/s]
  4%|█▎                                 | 2860/78125 [25:16<10:23:21,  2.01it/s]
  4%|█▎                                 | 2861/78125 [25:16<10:22:37,  2.01it/s]
  4%|█▎                                 | 2862/78125 [25:17<10:23:02,  2.01it/s]
  4%|█▎                                 | 2863/78125 [25:17<10:22:19,  2.02it/s]
  4%|█▎                                 | 2864/78125 [25:18<10:22:12,  2.02it/s]
  4%|█▎                                 | 2865/78125 [25:18<10:22:23,  2.02it/s]
  4%|█▎                                 | 2866/78125 [25:19<10:21:35,  2.02it/s]
  4%|█▎                                 | 2867/78125 [25:19<10:21:09,  2.02it/s]
  4%|█▎                                 | 2868/78125 [25:20<10:21:01,  2.02it/s]
  4%|█▎                                 | 2869/78125 [25:20<10:21:56,  2.02it/s]
  4%|█▎                                 | 2870/78125 [25:21<10:22:00,  2.02it/s]
  4%|█▎                                 | 2871/78125 [25:21<10:21:42,  2.02it/s]
  4%|█▎                                 | 2872/78125 [25:22<10:21:50,  2.02it/s]
  4%|█▎                                 | 2873/78125 [25:22<10:21:29,  2.02it/s]
  4%|█▎                                 | 2874/78125 [25:23<10:22:01,  2.02it/s]
  4%|█▎                                 | 2875/78125 [25:23<10:22:15,  2.02it/s]
  4%|█▎                                 | 2876/78125 [25:24<10:22:46,  2.01it/s]
  4%|█▎                                 | 2877/78125 [25:24<10:22:43,  2.01it/s]
  4%|█▎                                 | 2878/78125 [25:25<10:22:50,  2.01it/s]
  4%|█▎                                 | 2879/78125 [25:25<10:22:56,  2.01it/s]
  4%|█▎                                 | 2880/78125 [25:26<10:22:32,  2.01it/s]
  4%|█▎                                 | 2881/78125 [25:26<10:22:34,  2.01it/s]
  4%|█▎                                 | 2882/78125 [25:27<10:21:57,  2.02it/s]
  4%|█▎                                 | 2883/78125 [25:27<10:21:57,  2.02it/s]
  4%|█▎                                 | 2884/78125 [25:28<10:22:01,  2.02it/s]
  4%|█▎                                 | 2885/78125 [25:28<10:21:19,  2.02it/s]
  4%|█▎                                 | 2886/78125 [25:29<10:21:49,  2.02it/s]
  4%|█▎                                 | 2887/78125 [25:29<10:21:43,  2.02it/s]
  4%|█▎                                 | 2888/78125 [25:30<10:21:21,  2.02it/s]
  4%|█▎                                 | 2889/78125 [25:30<10:21:47,  2.02it/s]
  4%|█▎                                 | 2890/78125 [25:31<10:22:25,  2.01it/s]
  4%|█▎                                 | 2891/78125 [25:31<10:22:06,  2.02it/s]
  4%|█▎                                 | 2892/78125 [25:32<10:22:09,  2.02it/s]
  4%|█▎                                 | 2893/78125 [25:32<10:21:38,  2.02it/s]
  4%|█▎                                 | 2894/78125 [25:33<10:21:30,  2.02it/s]
  4%|█▎                                 | 2895/78125 [25:33<10:20:54,  2.02it/s]
  4%|█▎                                 | 2896/78125 [25:34<10:21:18,  2.02it/s]
  4%|█▎                                 | 2897/78125 [25:34<10:22:06,  2.02it/s]
  4%|█▎                                 | 2898/78125 [25:35<10:22:08,  2.02it/s]
  4%|█▎                                 | 2899/78125 [25:35<10:22:12,  2.02it/s]
  4%|█▎                                 | 2900/78125 [25:36<10:22:01,  2.02it/s]
  4%|█▎                                 | 2901/78125 [25:36<10:22:03,  2.02it/s]
  4%|█▎                                 | 2902/78125 [25:37<10:22:34,  2.01it/s]
  4%|█▎                                 | 2903/78125 [25:37<10:22:06,  2.02it/s]
  4%|█▎                                 | 2904/78125 [25:38<10:21:43,  2.02it/s]
  4%|█▎                                 | 2905/78125 [25:38<10:21:52,  2.02it/s]
  4%|█▎                                 | 2906/78125 [25:39<10:22:20,  2.01it/s]
  4%|█▎                                 | 2907/78125 [25:39<10:21:52,  2.02it/s]
  4%|█▎                                 | 2908/78125 [25:40<10:21:40,  2.02it/s]
  4%|█▎                                 | 2909/78125 [25:40<10:22:08,  2.02it/s]
  4%|█▎                                 | 2910/78125 [25:41<10:22:02,  2.02it/s]
  4%|█▎                                 | 2911/78125 [25:41<10:21:55,  2.02it/s]
  4%|█▎                                 | 2912/78125 [25:42<10:21:47,  2.02it/s]
  4%|█▎                                 | 2913/78125 [25:42<10:21:26,  2.02it/s]
  4%|█▎                                 | 2914/78125 [25:43<10:22:08,  2.01it/s]
  4%|█▎                                 | 2915/78125 [25:43<10:21:55,  2.02it/s]
  4%|█▎                                 | 2916/78125 [25:44<10:22:31,  2.01it/s]
  4%|█▎                                 | 2917/78125 [25:44<10:23:06,  2.01it/s]
  4%|█▎                                 | 2918/78125 [25:45<10:22:45,  2.01it/s]
  4%|█▎                                 | 2919/78125 [25:45<10:22:31,  2.01it/s]
  4%|█▎                                 | 2920/78125 [25:46<10:22:08,  2.01it/s]
  4%|█▎                                 | 2921/78125 [25:46<10:22:14,  2.01it/s]
  4%|█▎                                 | 2922/78125 [25:47<10:22:44,  2.01it/s]
  4%|█▎                                 | 2923/78125 [25:47<10:22:46,  2.01it/s]
  4%|█▎                                 | 2924/78125 [25:48<10:23:01,  2.01it/s]
  4%|█▎                                 | 2925/78125 [25:48<10:22:37,  2.01it/s]
  4%|█▎                                 | 2926/78125 [25:49<10:22:48,  2.01it/s]
  4%|█▎                                 | 2927/78125 [25:49<10:23:04,  2.01it/s]
  4%|█▎                                 | 2928/78125 [25:50<10:22:52,  2.01it/s]
  4%|█▎                                 | 2929/78125 [25:50<10:22:43,  2.01it/s]
  4%|█▎                                 | 2930/78125 [25:51<10:22:38,  2.01it/s]
  4%|█▎                                 | 2931/78125 [25:51<10:22:00,  2.01it/s]
  4%|█▎                                 | 2932/78125 [25:52<10:22:20,  2.01it/s]
  4%|█▎                                 | 2933/78125 [25:52<10:21:43,  2.02it/s]
  4%|█▎                                 | 2934/78125 [25:53<10:21:23,  2.02it/s]
  4%|█▎                                 | 2935/78125 [25:53<10:22:13,  2.01it/s]
  4%|█▎                                 | 2936/78125 [25:54<10:22:40,  2.01it/s]
  4%|█▎                                 | 2937/78125 [25:54<10:22:19,  2.01it/s]
  4%|█▎                                 | 2938/78125 [25:55<10:22:21,  2.01it/s]
  4%|█▎                                 | 2939/78125 [25:55<10:22:39,  2.01it/s]
  4%|█▎                                 | 2940/78125 [25:56<10:21:56,  2.01it/s]
  4%|█▎                                 | 2941/78125 [25:56<10:21:48,  2.02it/s]
  4%|█▎                                 | 2942/78125 [25:57<10:21:37,  2.02it/s]
  4%|█▎                                 | 2943/78125 [25:57<10:21:22,  2.02it/s]
  4%|█▎                                 | 2944/78125 [25:58<10:21:34,  2.02it/s]
  4%|█▎                                 | 2945/78125 [25:58<10:21:49,  2.02it/s]
  4%|█▎                                 | 2946/78125 [25:59<10:21:28,  2.02it/s]
  4%|█▎                                 | 2947/78125 [25:59<10:21:20,  2.02it/s]
  4%|█▎                                 | 2948/78125 [26:00<10:21:50,  2.01it/s]
  4%|█▎                                 | 2949/78125 [26:00<10:21:37,  2.02it/s]
  4%|█▎                                 | 2950/78125 [26:01<10:21:56,  2.01it/s]
  4%|█▎                                 | 2951/78125 [26:01<10:21:52,  2.01it/s]
  4%|█▎                                 | 2952/78125 [26:02<10:22:41,  2.01it/s]
  4%|█▎                                 | 2953/78125 [26:02<10:22:10,  2.01it/s]
  4%|█▎                                 | 2954/78125 [26:03<10:22:37,  2.01it/s]
  4%|█▎                                 | 2955/78125 [26:03<10:22:27,  2.01it/s]
  4%|█▎                                 | 2956/78125 [26:04<10:22:34,  2.01it/s]
  4%|█▎                                 | 2957/78125 [26:04<10:22:24,  2.01it/s]
  4%|█▎                                 | 2958/78125 [26:05<10:22:52,  2.01it/s]
  4%|█▎                                 | 2959/78125 [26:05<10:22:37,  2.01it/s]
  4%|█▎                                 | 2960/78125 [26:06<10:22:10,  2.01it/s]
  4%|█▎                                 | 2961/78125 [26:06<10:22:46,  2.01it/s]
  4%|█▎                                 | 2962/78125 [26:07<10:22:19,  2.01it/s]
  4%|█▎                                 | 2963/78125 [26:07<10:22:50,  2.01it/s]
  4%|█▎                                 | 2964/78125 [26:08<10:23:07,  2.01it/s]
  4%|█▎                                 | 2965/78125 [26:08<10:22:48,  2.01it/s]
  4%|█▎                                 | 2966/78125 [26:09<10:22:29,  2.01it/s]
  4%|█▎                                 | 2967/78125 [26:09<10:22:54,  2.01it/s]
  4%|█▎                                 | 2968/78125 [26:10<10:22:55,  2.01it/s]
  4%|█▎                                 | 2969/78125 [26:10<10:23:06,  2.01it/s]
  4%|█▎                                 | 2970/78125 [26:10<10:22:44,  2.01it/s]
  4%|█▎                                 | 2971/78125 [26:11<10:22:34,  2.01it/s]
  4%|█▎                                 | 2972/78125 [26:11<10:22:17,  2.01it/s]
  4%|█▎                                 | 2973/78125 [26:12<10:21:27,  2.02it/s]
  4%|█▎                                 | 2974/78125 [26:12<10:21:31,  2.02it/s]
  4%|█▎                                 | 2975/78125 [26:13<10:20:46,  2.02it/s]
  4%|█▎                                 | 2976/78125 [26:13<10:20:52,  2.02it/s]
  4%|█▎                                 | 2977/78125 [26:14<10:20:48,  2.02it/s]
  4%|█▎                                 | 2978/78125 [26:14<10:21:14,  2.02it/s]
  4%|█▎                                 | 2979/78125 [26:15<10:20:55,  2.02it/s]
  4%|█▎                                 | 2980/78125 [26:15<10:21:36,  2.01it/s]
  4%|█▎                                 | 2981/78125 [26:16<10:21:46,  2.01it/s]
  4%|█▎                                 | 2982/78125 [26:16<10:21:24,  2.02it/s]
  4%|█▎                                 | 2983/78125 [26:17<10:21:09,  2.02it/s]
  4%|█▎                                 | 2984/78125 [26:17<10:20:38,  2.02it/s]
  4%|█▎                                 | 2985/78125 [26:18<10:21:03,  2.02it/s]
  4%|█▎                                 | 2986/78125 [26:18<10:21:24,  2.02it/s]
  4%|█▎                                 | 2987/78125 [26:19<10:22:18,  2.01it/s]
  4%|█▎                                 | 2988/78125 [26:19<10:21:31,  2.01it/s]
  4%|█▎                                 | 2989/78125 [26:20<10:21:12,  2.02it/s]
  4%|█▎                                 | 2990/78125 [26:20<10:21:07,  2.02it/s]
  4%|█▎                                 | 2991/78125 [26:21<10:21:08,  2.02it/s]
  4%|█▎                                 | 2992/78125 [26:21<10:21:50,  2.01it/s]
  4%|█▎                                 | 2993/78125 [26:22<10:21:32,  2.01it/s]
  4%|█▎                                 | 2994/78125 [26:22<10:21:30,  2.01it/s]
  4%|█▎                                 | 2995/78125 [26:23<10:21:15,  2.02it/s]
  4%|█▎                                 | 2996/78125 [26:23<10:21:19,  2.02it/s]
  4%|█▎                                 | 2997/78125 [26:24<10:22:14,  2.01it/s]
  4%|█▎                                 | 2998/78125 [26:24<10:22:20,  2.01it/s]
  4%|█▎                                 | 2999/78125 [26:25<10:23:03,  2.01it/s]
  4%|█▎                                 | 3000/78125 [26:25<10:22:41,  2.01it/s]
  4%|█▎                                 | 3001/78125 [26:26<10:23:20,  2.01it/s]
  4%|█▎                                 | 3002/78125 [26:26<10:22:21,  2.01it/s]
  4%|█▎                                 | 3003/78125 [26:27<10:21:47,  2.01it/s]
  4%|█▎                                 | 3004/78125 [26:27<10:21:57,  2.01it/s]
  4%|█▎                                 | 3005/78125 [26:28<10:21:26,  2.01it/s]
  4%|█▎                                 | 3006/78125 [26:28<10:21:45,  2.01it/s]
  4%|█▎                                 | 3007/78125 [26:29<10:21:16,  2.02it/s]
  4%|█▎                                 | 3008/78125 [26:29<10:20:57,  2.02it/s]
  4%|█▎                                 | 3009/78125 [26:30<10:20:56,  2.02it/s]
  4%|█▎                                 | 3010/78125 [26:30<10:20:41,  2.02it/s]
  4%|█▎                                 | 3011/78125 [26:31<10:21:12,  2.02it/s]
  4%|█▎                                 | 3012/78125 [26:31<10:21:30,  2.01it/s]
  4%|█▎                                 | 3013/78125 [26:32<10:21:04,  2.02it/s]
  4%|█▎                                 | 3014/78125 [26:32<10:21:13,  2.02it/s]
  4%|█▎                                 | 3015/78125 [26:33<10:20:44,  2.02it/s]
  4%|█▎                                 | 3016/78125 [26:33<10:20:37,  2.02it/s]
  4%|█▎                                 | 3017/78125 [26:34<10:21:32,  2.01it/s]
  4%|█▎                                 | 3018/78125 [26:34<10:21:37,  2.01it/s]
  4%|█▎                                 | 3019/78125 [26:35<10:20:59,  2.02it/s]
  4%|█▎                                 | 3020/78125 [26:35<10:20:52,  2.02it/s]
  4%|█▎                                 | 3021/78125 [26:36<10:20:41,  2.02it/s]
  4%|█▎                                 | 3022/78125 [26:36<10:20:44,  2.02it/s]
  4%|█▎                                 | 3023/78125 [26:37<10:21:02,  2.02it/s]
  4%|█▎                                 | 3024/78125 [26:37<10:21:39,  2.01it/s]
  4%|█▎                                 | 3025/78125 [26:38<10:20:52,  2.02it/s]
  4%|█▎                                 | 3026/78125 [26:38<10:21:13,  2.01it/s]
  4%|█▎                                 | 3027/78125 [26:39<10:21:48,  2.01it/s]
  4%|█▎                                 | 3028/78125 [26:39<10:21:47,  2.01it/s]
  4%|█▎                                 | 3029/78125 [26:40<10:21:50,  2.01it/s]
  4%|█▎                                 | 3030/78125 [26:40<10:21:31,  2.01it/s]
  4%|█▎                                 | 3031/78125 [26:41<10:21:16,  2.01it/s]
  4%|█▎                                 | 3032/78125 [26:41<10:20:26,  2.02it/s]
  4%|█▎                                 | 3033/78125 [26:42<10:20:26,  2.02it/s]
  4%|█▎                                 | 3034/78125 [26:42<10:21:16,  2.01it/s]
  4%|█▎                                 | 3035/78125 [26:43<10:21:00,  2.02it/s]
  4%|█▎                                 | 3036/78125 [26:43<10:21:20,  2.01it/s]
  4%|█▎                                 | 3037/78125 [26:44<10:21:36,  2.01it/s]
  4%|█▎                                 | 3038/78125 [26:44<10:21:21,  2.01it/s]
  4%|█▎                                 | 3039/78125 [26:45<10:20:38,  2.02it/s]
  4%|█▎                                 | 3040/78125 [26:45<10:21:13,  2.01it/s]
  4%|█▎                                 | 3041/78125 [26:46<10:21:10,  2.01it/s]
  4%|█▎                                 | 3042/78125 [26:46<10:20:50,  2.02it/s]
  4%|█▎                                 | 3043/78125 [26:47<10:20:34,  2.02it/s]
  4%|█▎                                 | 3044/78125 [26:47<10:20:40,  2.02it/s]
  4%|█▎                                 | 3045/78125 [26:48<10:20:27,  2.02it/s]
  4%|█▎                                 | 3046/78125 [26:48<10:20:04,  2.02it/s]
  4%|█▎                                 | 3047/78125 [26:49<10:20:05,  2.02it/s]
  4%|█▎                                 | 3048/78125 [26:49<10:20:09,  2.02it/s]
  4%|█▎                                 | 3049/78125 [26:50<10:20:26,  2.02it/s]
  4%|█▎                                 | 3050/78125 [26:50<10:20:33,  2.02it/s]
  4%|█▎                                 | 3051/78125 [26:51<10:20:05,  2.02it/s]
  4%|█▎                                 | 3052/78125 [26:51<10:20:51,  2.02it/s]
  4%|█▎                                 | 3053/78125 [26:52<10:21:10,  2.01it/s]
  4%|█▎                                 | 3054/78125 [26:52<10:21:31,  2.01it/s]
  4%|█▎                                 | 3055/78125 [26:53<10:21:06,  2.01it/s]
  4%|█▎                                 | 3056/78125 [26:53<10:21:20,  2.01it/s]
  4%|█▎                                 | 3057/78125 [26:54<10:21:12,  2.01it/s]
  4%|█▎                                 | 3058/78125 [26:54<10:20:54,  2.01it/s]
  4%|█▎                                 | 3059/78125 [26:55<10:21:30,  2.01it/s]
  4%|█▎                                 | 3060/78125 [26:55<10:21:34,  2.01it/s]
  4%|█▎                                 | 3061/78125 [26:56<10:20:44,  2.02it/s]
  4%|█▎                                 | 3062/78125 [26:56<10:20:58,  2.01it/s]
  4%|█▎                                 | 3063/78125 [26:57<10:21:26,  2.01it/s]
  4%|█▎                                 | 3064/78125 [26:57<10:20:35,  2.02it/s]
  4%|█▎                                 | 3065/78125 [26:58<10:20:47,  2.02it/s]
  4%|█▎                                 | 3066/78125 [26:58<10:20:51,  2.01it/s]
  4%|█▎                                 | 3067/78125 [26:59<10:21:12,  2.01it/s]
  4%|█▎                                 | 3068/78125 [26:59<10:21:20,  2.01it/s]
  4%|█▎                                 | 3069/78125 [27:00<10:21:16,  2.01it/s]
  4%|█▍                                 | 3070/78125 [27:00<10:20:21,  2.02it/s]
  4%|█▍                                 | 3071/78125 [27:01<10:19:53,  2.02it/s]
  4%|█▍                                 | 3072/78125 [27:01<10:20:12,  2.02it/s]
  4%|█▍                                 | 3073/78125 [27:02<10:19:54,  2.02it/s]
  4%|█▍                                 | 3074/78125 [27:02<10:20:09,  2.02it/s]
  4%|█▍                                 | 3075/78125 [27:03<10:19:57,  2.02it/s]
  4%|█▍                                 | 3076/78125 [27:03<10:20:33,  2.02it/s]
  4%|█▍                                 | 3077/78125 [27:04<10:20:03,  2.02it/s]
  4%|█▍                                 | 3078/78125 [27:04<10:20:32,  2.02it/s]
  4%|█▍                                 | 3079/78125 [27:05<10:20:54,  2.01it/s]
  4%|█▍                                 | 3080/78125 [27:05<10:20:07,  2.02it/s]
  4%|█▍                                 | 3081/78125 [27:06<10:20:19,  2.02it/s]
  4%|█▍                                 | 3082/78125 [27:06<10:20:43,  2.01it/s]
  4%|█▍                                 | 3083/78125 [27:07<10:21:03,  2.01it/s]
  4%|█▍                                 | 3084/78125 [27:07<10:20:58,  2.01it/s]
  4%|█▍                                 | 3085/78125 [27:08<10:20:07,  2.02it/s]
  4%|█▍                                 | 3086/78125 [27:08<10:20:33,  2.02it/s]
  4%|█▍                                 | 3087/78125 [27:09<10:20:58,  2.01it/s]
  4%|█▍                                 | 3088/78125 [27:09<10:20:46,  2.01it/s]
  4%|█▍                                 | 3089/78125 [27:10<10:20:53,  2.01it/s]
  4%|█▍                                 | 3090/78125 [27:10<10:20:09,  2.02it/s]
  4%|█▍                                 | 3091/78125 [27:11<10:20:39,  2.01it/s]
  4%|█▍                                 | 3092/78125 [27:11<10:20:05,  2.02it/s]
  4%|█▍                                 | 3093/78125 [27:12<10:20:24,  2.02it/s]
  4%|█▍                                 | 3094/78125 [27:12<10:19:29,  2.02it/s]
  4%|█▍                                 | 3095/78125 [27:13<10:19:56,  2.02it/s]
  4%|█▍                                 | 3096/78125 [27:13<10:20:15,  2.02it/s]
  4%|█▍                                 | 3097/78125 [27:14<10:20:35,  2.01it/s]
  4%|█▍                                 | 3098/78125 [27:14<10:20:10,  2.02it/s]
  4%|█▍                                 | 3099/78125 [27:15<10:20:01,  2.02it/s]
  4%|█▍                                 | 3100/78125 [27:15<10:19:44,  2.02it/s]
  4%|█▍                                 | 3101/78125 [27:15<10:20:10,  2.02it/s]
  4%|█▍                                 | 3102/78125 [27:16<10:20:04,  2.02it/s]
  4%|█▍                                 | 3103/78125 [27:16<10:19:42,  2.02it/s]
  4%|█▍                                 | 3104/78125 [27:17<10:19:32,  2.02it/s]
  4%|█▍                                 | 3105/78125 [27:17<10:19:15,  2.02it/s]
  4%|█▍                                 | 3106/78125 [27:18<10:19:52,  2.02it/s]
  4%|█▍                                 | 3107/78125 [27:18<10:20:19,  2.02it/s]
  4%|█▍                                 | 3108/78125 [27:19<10:20:40,  2.01it/s]
  4%|█▍                                 | 3109/78125 [27:19<10:20:10,  2.02it/s]
  4%|█▍                                 | 3110/78125 [27:20<10:19:44,  2.02it/s]
  4%|█▍                                 | 3111/78125 [27:20<10:20:09,  2.02it/s]
  4%|█▍                                 | 3112/78125 [27:21<10:20:05,  2.02it/s]
  4%|█▍                                 | 3113/78125 [27:21<10:20:25,  2.02it/s]
  4%|█▍                                 | 3114/78125 [27:22<10:19:33,  2.02it/s]
  4%|█▍                                 | 3115/78125 [27:22<10:20:06,  2.02it/s]
  4%|█▍                                 | 3116/78125 [27:23<10:20:14,  2.02it/s]
  4%|█▍                                 | 3117/78125 [27:23<10:19:44,  2.02it/s]
  4%|█▍                                 | 3118/78125 [27:24<10:19:44,  2.02it/s]
  4%|█▍                                 | 3119/78125 [27:24<10:19:33,  2.02it/s]
  4%|█▍                                 | 3120/78125 [27:25<10:19:38,  2.02it/s]
  4%|█▍                                 | 3121/78125 [27:25<10:19:45,  2.02it/s]
  4%|█▍                                 | 3122/78125 [27:26<10:20:15,  2.02it/s]
  4%|█▍                                 | 3123/78125 [27:26<10:20:18,  2.02it/s]
  4%|█▍                                 | 3124/78125 [27:27<10:20:28,  2.01it/s]
  4%|█▍                                 | 3125/78125 [27:27<10:20:21,  2.01it/s]
  4%|█▍                                 | 3126/78125 [27:28<10:19:53,  2.02it/s]
  4%|█▍                                 | 3127/78125 [27:28<10:20:00,  2.02it/s]
  4%|█▍                                 | 3128/78125 [27:29<10:19:31,  2.02it/s]
  4%|█▍                                 | 3129/78125 [27:29<10:19:04,  2.02it/s]
  4%|█▍                                 | 3130/78125 [27:30<10:19:35,  2.02it/s]
  4%|█▍                                 | 3131/78125 [27:30<10:19:28,  2.02it/s]
  4%|█▍                                 | 3132/78125 [27:31<10:19:26,  2.02it/s]
  4%|█▍                                 | 3133/78125 [27:31<10:19:11,  2.02it/s]
  4%|█▍                                 | 3134/78125 [27:32<10:19:27,  2.02it/s]
  4%|█▍                                 | 3135/78125 [27:32<10:19:24,  2.02it/s]
  4%|█▍                                 | 3136/78125 [27:33<10:20:08,  2.02it/s]
  4%|█▍                                 | 3137/78125 [27:33<10:19:54,  2.02it/s]
  4%|█▍                                 | 3138/78125 [27:34<10:19:38,  2.02it/s]
  4%|█▍                                 | 3139/78125 [27:34<10:19:48,  2.02it/s]
  4%|█▍                                 | 3140/78125 [27:35<10:19:26,  2.02it/s]
  4%|█▍                                 | 3141/78125 [27:35<10:19:04,  2.02it/s]
  4%|█▍                                 | 3142/78125 [27:36<10:18:36,  2.02it/s]
  4%|█▍                                 | 3143/78125 [27:36<10:17:36,  2.02it/s]
  4%|█▍                                 | 3144/78125 [27:37<10:18:25,  2.02it/s]
  4%|█▍                                 | 3145/78125 [27:37<10:18:49,  2.02it/s]
  4%|█▍                                 | 3146/78125 [27:38<10:18:23,  2.02it/s]
  4%|█▍                                 | 3147/78125 [27:38<10:17:31,  2.02it/s]
  4%|█▍                                 | 3148/78125 [27:39<10:17:58,  2.02it/s]
  4%|█▍                                 | 3149/78125 [27:39<10:17:45,  2.02it/s]
  4%|█▍                                 | 3150/78125 [27:40<10:17:35,  2.02it/s]
  4%|█▍                                 | 3151/78125 [27:40<10:17:40,  2.02it/s]
  4%|█▍                                 | 3152/78125 [27:41<10:17:20,  2.02it/s]
  4%|█▍                                 | 3153/78125 [27:41<10:17:47,  2.02it/s]
  4%|█▍                                 | 3154/78125 [27:42<10:17:19,  2.02it/s]
  4%|█▍                                 | 3155/78125 [27:42<10:16:49,  2.03it/s]
  4%|█▍                                 | 3156/78125 [27:43<10:17:19,  2.02it/s]
  4%|█▍                                 | 3157/78125 [27:43<10:17:13,  2.02it/s]
  4%|█▍                                 | 3158/78125 [27:44<10:17:47,  2.02it/s]
  4%|█▍                                 | 3159/78125 [27:44<10:18:18,  2.02it/s]
  4%|█▍                                 | 3160/78125 [27:45<10:17:54,  2.02it/s]
  4%|█▍                                 | 3161/78125 [27:45<10:18:14,  2.02it/s]
  4%|█▍                                 | 3162/78125 [27:46<10:18:11,  2.02it/s]
  4%|█▍                                 | 3163/78125 [27:46<10:18:04,  2.02it/s]
  4%|█▍                                 | 3164/78125 [27:47<10:17:44,  2.02it/s]
  4%|█▍                                 | 3165/78125 [27:47<10:17:38,  2.02it/s]
  4%|█▍                                 | 3166/78125 [27:48<10:17:16,  2.02it/s]
  4%|█▍                                 | 3167/78125 [27:48<10:17:18,  2.02it/s]
  4%|█▍                                 | 3168/78125 [27:49<10:17:33,  2.02it/s]
  4%|█▍                                 | 3169/78125 [27:49<10:17:20,  2.02it/s]
  4%|█▍                                 | 3170/78125 [27:50<10:17:22,  2.02it/s]
  4%|█▍                                 | 3171/78125 [27:50<10:17:46,  2.02it/s]
  4%|█▍                                 | 3172/78125 [27:51<10:17:55,  2.02it/s]
  4%|█▍                                 | 3173/78125 [27:51<10:18:14,  2.02it/s]
  4%|█▍                                 | 3174/78125 [27:52<10:18:07,  2.02it/s]
  4%|█▍                                 | 3175/78125 [27:52<10:18:51,  2.02it/s]
  4%|█▍                                 | 3176/78125 [27:53<10:19:12,  2.02it/s]
  4%|█▍                                 | 3177/78125 [27:53<10:19:16,  2.02it/s]
  4%|█▍                                 | 3178/78125 [27:54<10:19:14,  2.02it/s]
  4%|█▍                                 | 3179/78125 [27:54<10:18:56,  2.02it/s]
  4%|█▍                                 | 3180/78125 [27:55<10:19:11,  2.02it/s]
  4%|█▍                                 | 3181/78125 [27:55<10:19:06,  2.02it/s]
  4%|█▍                                 | 3182/78125 [27:56<10:19:07,  2.02it/s]
  4%|█▍                                 | 3183/78125 [27:56<10:19:28,  2.02it/s]
  4%|█▍                                 | 3184/78125 [27:57<10:19:21,  2.02it/s]
  4%|█▍                                 | 3185/78125 [27:57<10:18:58,  2.02it/s]
  4%|█▍                                 | 3186/78125 [27:58<10:19:08,  2.02it/s]
  4%|█▍                                 | 3187/78125 [27:58<10:19:24,  2.02it/s]
  4%|█▍                                 | 3188/78125 [27:59<10:19:05,  2.02it/s]
  4%|█▍                                 | 3189/78125 [27:59<10:20:15,  2.01it/s]
  4%|█▍                                 | 3190/78125 [28:00<10:20:06,  2.01it/s]
  4%|█▍                                 | 3191/78125 [28:00<10:19:59,  2.01it/s]
  4%|█▍                                 | 3192/78125 [28:01<10:19:03,  2.02it/s]
  4%|█▍                                 | 3193/78125 [28:01<10:18:56,  2.02it/s]
  4%|█▍                                 | 3194/78125 [28:02<10:19:04,  2.02it/s]
  4%|█▍                                 | 3195/78125 [28:02<10:19:30,  2.02it/s]
  4%|█▍                                 | 3196/78125 [28:03<10:19:30,  2.02it/s]
  4%|█▍                                 | 3197/78125 [28:03<10:19:38,  2.02it/s]
  4%|█▍                                 | 3198/78125 [28:04<10:20:07,  2.01it/s]
  4%|█▍                                 | 3199/78125 [28:04<10:20:33,  2.01it/s]
  4%|█▍                                 | 3200/78125 [28:05<10:20:29,  2.01it/s]
  4%|█▍                                 | 3201/78125 [28:05<10:20:34,  2.01it/s]
  4%|█▍                                 | 3202/78125 [28:06<10:20:30,  2.01it/s]
  4%|█▍                                 | 3203/78125 [28:06<10:20:15,  2.01it/s]
  4%|█▍                                 | 3204/78125 [28:07<10:19:31,  2.02it/s]
  4%|█▍                                 | 3205/78125 [28:07<10:19:53,  2.01it/s]
  4%|█▍                                 | 3206/78125 [28:08<10:20:14,  2.01it/s]
  4%|█▍                                 | 3207/78125 [28:08<10:20:54,  2.01it/s]
  4%|█▍                                 | 3208/78125 [28:09<10:20:40,  2.01it/s]
  4%|█▍                                 | 3209/78125 [28:09<10:19:14,  2.02it/s]
  4%|█▍                                 | 3210/78125 [28:10<10:20:14,  2.01it/s]
  4%|█▍                                 | 3211/78125 [28:10<10:20:23,  2.01it/s]
  4%|█▍                                 | 3212/78125 [28:11<10:21:09,  2.01it/s]
  4%|█▍                                 | 3213/78125 [28:11<10:20:43,  2.01it/s]
  4%|█▍                                 | 3214/78125 [28:12<10:21:25,  2.01it/s]
  4%|█▍                                 | 3215/78125 [28:12<10:21:06,  2.01it/s]
  4%|█▍                                 | 3216/78125 [28:13<10:21:11,  2.01it/s]
  4%|█▍                                 | 3217/78125 [28:13<10:21:23,  2.01it/s]
  4%|█▍                                 | 3218/78125 [28:13<10:20:59,  2.01it/s]
  4%|█▍                                 | 3219/78125 [28:14<10:20:35,  2.01it/s]
  4%|█▍                                 | 3220/78125 [28:14<10:20:36,  2.01it/s]
  4%|█▍                                 | 3221/78125 [28:15<10:20:43,  2.01it/s]
  4%|█▍                                 | 3222/78125 [28:15<10:21:17,  2.01it/s]
  4%|█▍                                 | 3223/78125 [28:16<10:20:51,  2.01it/s]
  4%|█▍                                 | 3224/78125 [28:16<10:20:38,  2.01it/s]
  4%|█▍                                 | 3225/78125 [28:17<10:20:21,  2.01it/s]
  4%|█▍                                 | 3226/78125 [28:17<10:20:32,  2.01it/s]
  4%|█▍                                 | 3227/78125 [28:18<10:19:54,  2.01it/s]
  4%|█▍                                 | 3228/78125 [28:18<10:20:15,  2.01it/s]
  4%|█▍                                 | 3229/78125 [28:19<10:20:41,  2.01it/s]
  4%|█▍                                 | 3230/78125 [28:19<10:20:37,  2.01it/s]
  4%|█▍                                 | 3231/78125 [28:20<10:20:45,  2.01it/s]
  4%|█▍                                 | 3232/78125 [28:20<10:21:20,  2.01it/s]
  4%|█▍                                 | 3233/78125 [28:21<10:21:46,  2.01it/s]
  4%|█▍                                 | 3234/78125 [28:21<10:22:47,  2.00it/s]
  4%|█▍                                 | 3235/78125 [28:22<10:22:17,  2.01it/s]
  4%|█▍                                 | 3236/78125 [28:22<10:21:56,  2.01it/s]
  4%|█▍                                 | 3237/78125 [28:23<10:21:51,  2.01it/s]
  4%|█▍                                 | 3238/78125 [28:23<10:21:50,  2.01it/s]
  4%|█▍                                 | 3239/78125 [28:24<10:21:49,  2.01it/s]
  4%|█▍                                 | 3240/78125 [28:24<10:22:13,  2.01it/s]
  4%|█▍                                 | 3241/78125 [28:25<10:21:21,  2.01it/s]
  4%|█▍                                 | 3242/78125 [28:25<10:19:58,  2.01it/s]
  4%|█▍                                 | 3243/78125 [28:26<10:20:31,  2.01it/s]
  4%|█▍                                 | 3244/78125 [28:26<10:21:01,  2.01it/s]
  4%|█▍                                 | 3245/78125 [28:27<10:21:43,  2.01it/s]
  4%|█▍                                 | 3246/78125 [28:27<10:21:04,  2.01it/s]
  4%|█▍                                 | 3247/78125 [28:28<10:21:18,  2.01it/s]
  4%|█▍                                 | 3248/78125 [28:28<10:22:03,  2.01it/s]
  4%|█▍                                 | 3249/78125 [28:29<10:22:07,  2.01it/s]
  4%|█▍                                 | 3250/78125 [28:29<10:21:57,  2.01it/s]
  4%|█▍                                 | 3251/78125 [28:30<10:22:23,  2.00it/s]
  4%|█▍                                 | 3252/78125 [28:30<10:22:08,  2.01it/s]
  4%|█▍                                 | 3253/78125 [28:31<10:22:53,  2.00it/s]
  4%|█▍                                 | 3254/78125 [28:31<10:23:03,  2.00it/s]
  4%|█▍                                 | 3255/78125 [28:32<10:22:17,  2.01it/s]
  4%|█▍                                 | 3256/78125 [28:32<10:22:34,  2.00it/s]
  4%|█▍                                 | 3257/78125 [28:33<10:22:22,  2.00it/s]
  4%|█▍                                 | 3258/78125 [28:33<10:22:03,  2.01it/s]
  4%|█▍                                 | 3259/78125 [28:34<10:21:47,  2.01it/s]
  4%|█▍                                 | 3260/78125 [28:34<10:21:01,  2.01it/s]
  4%|█▍                                 | 3261/78125 [28:35<10:20:09,  2.01it/s]
  4%|█▍                                 | 3262/78125 [28:35<10:20:39,  2.01it/s]
  4%|█▍                                 | 3263/78125 [28:36<10:20:55,  2.01it/s]
  4%|█▍                                 | 3264/78125 [28:36<10:21:53,  2.01it/s]
  4%|█▍                                 | 3265/78125 [28:37<10:22:17,  2.00it/s]
  4%|█▍                                 | 3266/78125 [28:37<10:22:20,  2.00it/s]
  4%|█▍                                 | 3267/78125 [28:38<10:22:29,  2.00it/s]
  4%|█▍                                 | 3268/78125 [28:38<10:22:33,  2.00it/s]
  4%|█▍                                 | 3269/78125 [28:39<10:22:20,  2.00it/s]
  4%|█▍                                 | 3270/78125 [28:39<10:22:14,  2.00it/s]
  4%|█▍                                 | 3271/78125 [28:40<10:22:05,  2.01it/s]
  4%|█▍                                 | 3272/78125 [28:40<10:22:15,  2.00it/s]
  4%|█▍                                 | 3273/78125 [28:41<10:22:46,  2.00it/s]
  4%|█▍                                 | 3274/78125 [28:41<10:22:05,  2.01it/s]
  4%|█▍                                 | 3275/78125 [28:42<10:22:16,  2.00it/s]
  4%|█▍                                 | 3276/78125 [28:42<10:22:10,  2.01it/s]
  4%|█▍                                 | 3277/78125 [28:43<10:22:12,  2.00it/s]
  4%|█▍                                 | 3278/78125 [28:43<10:22:02,  2.01it/s]
  4%|█▍                                 | 3279/78125 [28:44<10:21:54,  2.01it/s]
  4%|█▍                                 | 3280/78125 [28:44<10:22:24,  2.00it/s]
  4%|█▍                                 | 3281/78125 [28:45<10:22:10,  2.00it/s]
  4%|█▍                                 | 3282/78125 [28:45<10:20:24,  2.01it/s]
  4%|█▍                                 | 3283/78125 [28:46<10:20:53,  2.01it/s]
  4%|█▍                                 | 3284/78125 [28:46<10:20:30,  2.01it/s]
  4%|█▍                                 | 3285/78125 [28:47<10:20:36,  2.01it/s]
  4%|█▍                                 | 3286/78125 [28:47<10:21:01,  2.01it/s]
  4%|█▍                                 | 3287/78125 [28:48<10:20:25,  2.01it/s]
  4%|█▍                                 | 3288/78125 [28:48<10:20:57,  2.01it/s]
  4%|█▍                                 | 3289/78125 [28:49<10:20:28,  2.01it/s]
  4%|█▍                                 | 3290/78125 [28:49<10:20:49,  2.01it/s]
  4%|█▍                                 | 3291/78125 [28:50<10:20:47,  2.01it/s]
  4%|█▍                                 | 3292/78125 [28:50<10:21:47,  2.01it/s]
  4%|█▍                                 | 3293/78125 [28:51<10:21:40,  2.01it/s]
  4%|█▍                                 | 3294/78125 [28:51<10:20:37,  2.01it/s]
  4%|█▍                                 | 3295/78125 [28:52<10:20:22,  2.01it/s]
  4%|█▍                                 | 3296/78125 [28:52<10:19:56,  2.01it/s]
  4%|█▍                                 | 3297/78125 [28:53<10:20:48,  2.01it/s]
  4%|█▍                                 | 3298/78125 [28:53<10:21:05,  2.01it/s]
  4%|█▍                                 | 3299/78125 [28:54<10:21:03,  2.01it/s]
  4%|█▍                                 | 3300/78125 [28:54<10:20:38,  2.01it/s]
  4%|█▍                                 | 3301/78125 [28:55<10:20:55,  2.01it/s]
  4%|█▍                                 | 3302/78125 [28:55<10:20:26,  2.01it/s]
  4%|█▍                                 | 3303/78125 [28:56<10:19:46,  2.01it/s]
  4%|█▍                                 | 3304/78125 [28:56<10:20:05,  2.01it/s]
  4%|█▍                                 | 3305/78125 [28:57<10:19:58,  2.01it/s]
  4%|█▍                                 | 3306/78125 [28:57<10:19:19,  2.01it/s]
  4%|█▍                                 | 3307/78125 [28:58<10:20:07,  2.01it/s]
  4%|█▍                                 | 3308/78125 [28:58<10:20:11,  2.01it/s]
  4%|█▍                                 | 3309/78125 [28:59<10:20:02,  2.01it/s]
  4%|█▍                                 | 3310/78125 [28:59<10:19:32,  2.01it/s]
  4%|█▍                                 | 3311/78125 [29:00<10:19:31,  2.01it/s]
  4%|█▍                                 | 3312/78125 [29:00<10:19:39,  2.01it/s]
  4%|█▍                                 | 3313/78125 [29:01<10:19:24,  2.01it/s]
  4%|█▍                                 | 3314/78125 [29:01<10:19:32,  2.01it/s]
  4%|█▍                                 | 3315/78125 [29:02<10:18:58,  2.01it/s]
  4%|█▍                                 | 3316/78125 [29:02<10:19:04,  2.01it/s]
  4%|█▍                                 | 3317/78125 [29:03<10:18:31,  2.02it/s]
  4%|█▍                                 | 3318/78125 [29:03<10:19:08,  2.01it/s]
  4%|█▍                                 | 3319/78125 [29:04<10:18:49,  2.01it/s]
  4%|█▍                                 | 3320/78125 [29:04<10:17:52,  2.02it/s]
  4%|█▍                                 | 3321/78125 [29:05<10:17:37,  2.02it/s]
  4%|█▍                                 | 3322/78125 [29:05<10:17:40,  2.02it/s]
  4%|█▍                                 | 3323/78125 [29:06<10:18:15,  2.02it/s]
  4%|█▍                                 | 3324/78125 [29:06<10:17:52,  2.02it/s]
  4%|█▍                                 | 3325/78125 [29:07<10:18:02,  2.02it/s]
  4%|█▍                                 | 3326/78125 [29:07<10:17:44,  2.02it/s]
  4%|█▍                                 | 3327/78125 [29:08<10:17:39,  2.02it/s]
  4%|█▍                                 | 3328/78125 [29:08<10:18:23,  2.02it/s]
  4%|█▍                                 | 3329/78125 [29:09<10:18:28,  2.02it/s]
  4%|█▍                                 | 3330/78125 [29:09<10:18:46,  2.01it/s]
  4%|█▍                                 | 3331/78125 [29:10<10:17:47,  2.02it/s]
  4%|█▍                                 | 3332/78125 [29:10<10:17:24,  2.02it/s]
  4%|█▍                                 | 3333/78125 [29:11<10:17:57,  2.02it/s]
  4%|█▍                                 | 3334/78125 [29:11<10:17:38,  2.02it/s]
  4%|█▍                                 | 3335/78125 [29:12<10:17:01,  2.02it/s]
  4%|█▍                                 | 3336/78125 [29:12<10:16:31,  2.02it/s]
  4%|█▍                                 | 3337/78125 [29:13<10:16:43,  2.02it/s]
  4%|█▍                                 | 3338/78125 [29:13<10:16:55,  2.02it/s]
  4%|█▍                                 | 3339/78125 [29:14<10:17:24,  2.02it/s]
  4%|█▍                                 | 3340/78125 [29:14<10:17:32,  2.02it/s]
  4%|█▍                                 | 3341/78125 [29:15<10:17:12,  2.02it/s]
  4%|█▍                                 | 3342/78125 [29:15<10:17:11,  2.02it/s]
  4%|█▍                                 | 3343/78125 [29:16<10:17:14,  2.02it/s]
  4%|█▍                                 | 3344/78125 [29:16<10:17:24,  2.02it/s]
  4%|█▍                                 | 3345/78125 [29:17<10:17:01,  2.02it/s]
  4%|█▍                                 | 3346/78125 [29:17<10:17:18,  2.02it/s]
  4%|█▍                                 | 3347/78125 [29:18<10:17:24,  2.02it/s]
  4%|█▍                                 | 3348/78125 [29:18<10:17:28,  2.02it/s]
  4%|█▌                                 | 3349/78125 [29:19<10:17:59,  2.02it/s]
  4%|█▌                                 | 3350/78125 [29:19<10:17:49,  2.02it/s]
  4%|█▌                                 | 3351/78125 [29:20<10:17:34,  2.02it/s]
  4%|█▌                                 | 3352/78125 [29:20<10:17:19,  2.02it/s]
  4%|█▌                                 | 3353/78125 [29:21<10:17:09,  2.02it/s]
  4%|█▌                                 | 3354/78125 [29:21<10:17:10,  2.02it/s]
  4%|█▌                                 | 3355/78125 [29:22<10:16:46,  2.02it/s]
  4%|█▌                                 | 3356/78125 [29:22<10:16:23,  2.02it/s]
  4%|█▌                                 | 3357/78125 [29:23<10:16:24,  2.02it/s]
  4%|█▌                                 | 3358/78125 [29:23<10:16:32,  2.02it/s]
  4%|█▌                                 | 3359/78125 [29:24<10:16:31,  2.02it/s]
  4%|█▌                                 | 3360/78125 [29:24<10:16:31,  2.02it/s]
  4%|█▌                                 | 3361/78125 [29:25<10:15:47,  2.02it/s]
  4%|█▌                                 | 3362/78125 [29:25<10:15:12,  2.03it/s]
  4%|█▌                                 | 3363/78125 [29:26<10:15:10,  2.03it/s]
  4%|█▌                                 | 3364/78125 [29:26<10:15:27,  2.02it/s]
  4%|█▌                                 | 3365/78125 [29:27<10:14:55,  2.03it/s]
  4%|█▌                                 | 3366/78125 [29:27<10:14:40,  2.03it/s]
  4%|█▌                                 | 3367/78125 [29:28<10:15:08,  2.03it/s]
  4%|█▌                                 | 3368/78125 [29:28<10:15:31,  2.02it/s]
  4%|█▌                                 | 3369/78125 [29:29<10:15:07,  2.03it/s]
  4%|█▌                                 | 3370/78125 [29:29<10:15:50,  2.02it/s]
  4%|█▌                                 | 3371/78125 [29:30<10:16:22,  2.02it/s]
  4%|█▌                                 | 3372/78125 [29:30<10:16:42,  2.02it/s]
  4%|█▌                                 | 3373/78125 [29:31<10:16:10,  2.02it/s]
  4%|█▌                                 | 3374/78125 [29:31<10:16:28,  2.02it/s]
  4%|█▌                                 | 3375/78125 [29:31<10:16:24,  2.02it/s]
  4%|█▌                                 | 3376/78125 [29:32<10:16:22,  2.02it/s]
  4%|█▌                                 | 3377/78125 [29:32<10:16:03,  2.02it/s]
  4%|█▌                                 | 3378/78125 [29:33<10:16:01,  2.02it/s]
  4%|█▌                                 | 3379/78125 [29:33<10:16:13,  2.02it/s]
  4%|█▌                                 | 3380/78125 [29:34<10:16:29,  2.02it/s]
  4%|█▌                                 | 3381/78125 [29:34<10:16:56,  2.02it/s]
  4%|█▌                                 | 3382/78125 [29:35<10:17:09,  2.02it/s]
  4%|█▌                                 | 3383/78125 [29:35<10:16:16,  2.02it/s]
  4%|█▌                                 | 3384/78125 [29:36<10:16:15,  2.02it/s]
  4%|█▌                                 | 3385/78125 [29:36<10:16:00,  2.02it/s]
  4%|█▌                                 | 3386/78125 [29:37<10:16:03,  2.02it/s]
  4%|█▌                                 | 3387/78125 [29:37<10:16:35,  2.02it/s]
  4%|█▌                                 | 3388/78125 [29:38<10:16:06,  2.02it/s]
  4%|█▌                                 | 3389/78125 [29:38<10:16:32,  2.02it/s]
  4%|█▌                                 | 3390/78125 [29:39<10:16:55,  2.02it/s]
  4%|█▌                                 | 3391/78125 [29:39<10:16:56,  2.02it/s]
  4%|█▌                                 | 3392/78125 [29:40<10:16:59,  2.02it/s]
  4%|█▌                                 | 3393/78125 [29:40<10:16:53,  2.02it/s]
  4%|█▌                                 | 3394/78125 [29:41<10:16:42,  2.02it/s]
  4%|█▌                                 | 3395/78125 [29:41<10:16:16,  2.02it/s]
  4%|█▌                                 | 3396/78125 [29:42<10:16:48,  2.02it/s]
  4%|█▌                                 | 3397/78125 [29:42<10:16:59,  2.02it/s]
  4%|█▌                                 | 3398/78125 [29:43<10:16:40,  2.02it/s]
  4%|█▌                                 | 3399/78125 [29:43<10:16:43,  2.02it/s]
  4%|█▌                                 | 3400/78125 [29:44<10:16:51,  2.02it/s]
  4%|█▌                                 | 3401/78125 [29:44<10:17:20,  2.02it/s]
  4%|█▌                                 | 3402/78125 [29:45<10:17:06,  2.02it/s]
  4%|█▌                                 | 3403/78125 [29:45<10:16:44,  2.02it/s]
  4%|█▌                                 | 3404/78125 [29:46<10:16:58,  2.02it/s]
  4%|█▌                                 | 3405/78125 [29:46<10:17:48,  2.02it/s]
  4%|█▌                                 | 3406/78125 [29:47<10:17:53,  2.02it/s]
  4%|█▌                                 | 3407/78125 [29:47<10:17:50,  2.02it/s]
  4%|█▌                                 | 3408/78125 [29:48<10:18:23,  2.01it/s]
  4%|█▌                                 | 3409/78125 [29:48<10:17:46,  2.02it/s]
  4%|█▌                                 | 3410/78125 [29:49<10:18:02,  2.01it/s]
  4%|█▌                                 | 3411/78125 [29:49<10:18:32,  2.01it/s]
  4%|█▌                                 | 3412/78125 [29:50<10:18:10,  2.01it/s]
  4%|█▌                                 | 3413/78125 [29:50<10:17:56,  2.02it/s]
  4%|█▌                                 | 3414/78125 [29:51<10:17:40,  2.02it/s]
  4%|█▌                                 | 3415/78125 [29:51<10:18:18,  2.01it/s]
  4%|█▌                                 | 3416/78125 [29:52<10:18:36,  2.01it/s]
  4%|█▌                                 | 3417/78125 [29:52<10:18:44,  2.01it/s]
  4%|█▌                                 | 3418/78125 [29:53<10:18:58,  2.01it/s]
  4%|█▌                                 | 3419/78125 [29:53<10:18:22,  2.01it/s]
  4%|█▌                                 | 3420/78125 [29:54<10:18:24,  2.01it/s]
  4%|█▌                                 | 3421/78125 [29:54<10:17:53,  2.02it/s]
  4%|█▌                                 | 3422/78125 [29:55<10:18:27,  2.01it/s]
  4%|█▌                                 | 3423/78125 [29:55<10:17:57,  2.01it/s]
  4%|█▌                                 | 3424/78125 [29:56<10:17:46,  2.02it/s]
  4%|█▌                                 | 3425/78125 [29:56<10:17:38,  2.02it/s]
  4%|█▌                                 | 3426/78125 [29:57<10:17:42,  2.02it/s]
  4%|█▌                                 | 3427/78125 [29:57<10:18:13,  2.01it/s]
  4%|█▌                                 | 3428/78125 [29:58<10:18:20,  2.01it/s]
  4%|█▌                                 | 3429/78125 [29:58<10:17:37,  2.02it/s]
  4%|█▌                                 | 3430/78125 [29:59<10:18:18,  2.01it/s]
  4%|█▌                                 | 3431/78125 [29:59<10:18:33,  2.01it/s]
  4%|█▌                                 | 3432/78125 [30:00<10:18:36,  2.01it/s]
  4%|█▌                                 | 3433/78125 [30:00<10:18:38,  2.01it/s]
  4%|█▌                                 | 3434/78125 [30:01<10:18:35,  2.01it/s]
  4%|█▌                                 | 3435/78125 [30:01<10:19:23,  2.01it/s]
  4%|█▌                                 | 3436/78125 [30:02<10:19:23,  2.01it/s]
  4%|█▌                                 | 3437/78125 [30:02<10:18:44,  2.01it/s]
  4%|█▌                                 | 3438/78125 [30:03<10:18:41,  2.01it/s]
  4%|█▌                                 | 3439/78125 [30:03<10:18:28,  2.01it/s]
  4%|█▌                                 | 3440/78125 [30:04<10:19:07,  2.01it/s]
  4%|█▌                                 | 3441/78125 [30:04<10:19:45,  2.01it/s]
  4%|█▌                                 | 3442/78125 [30:05<10:19:20,  2.01it/s]
  4%|█▌                                 | 3443/78125 [30:05<10:18:30,  2.01it/s]
  4%|█▌                                 | 3444/78125 [30:06<10:18:00,  2.01it/s]
  4%|█▌                                 | 3445/78125 [30:06<10:18:26,  2.01it/s]
  4%|█▌                                 | 3446/78125 [30:07<10:18:33,  2.01it/s]
  4%|█▌                                 | 3447/78125 [30:07<10:18:24,  2.01it/s]
  4%|█▌                                 | 3448/78125 [30:08<10:18:17,  2.01it/s]
  4%|█▌                                 | 3449/78125 [30:08<10:18:29,  2.01it/s]
  4%|█▌                                 | 3450/78125 [30:09<10:18:52,  2.01it/s]
  4%|█▌                                 | 3451/78125 [30:09<10:18:46,  2.01it/s]
  4%|█▌                                 | 3452/78125 [30:10<10:18:52,  2.01it/s]
  4%|█▌                                 | 3453/78125 [30:10<10:18:46,  2.01it/s]
  4%|█▌                                 | 3454/78125 [30:11<10:18:11,  2.01it/s]
  4%|█▌                                 | 3455/78125 [30:11<10:18:32,  2.01it/s]
  4%|█▌                                 | 3456/78125 [30:12<10:18:36,  2.01it/s]
  4%|█▌                                 | 3457/78125 [30:12<10:18:59,  2.01it/s]
  4%|█▌                                 | 3458/78125 [30:13<10:19:30,  2.01it/s]
  4%|█▌                                 | 3459/78125 [30:13<10:19:11,  2.01it/s]
  4%|█▌                                 | 3460/78125 [30:14<10:18:51,  2.01it/s]
  4%|█▌                                 | 3461/78125 [30:14<10:19:25,  2.01it/s]
  4%|█▌                                 | 3462/78125 [30:15<10:18:49,  2.01it/s]
  4%|█▌                                 | 3463/78125 [30:15<10:18:25,  2.01it/s]
  4%|█▌                                 | 3464/78125 [30:16<10:18:52,  2.01it/s]
  4%|█▌                                 | 3465/78125 [30:16<10:19:25,  2.01it/s]
  4%|█▌                                 | 3466/78125 [30:17<10:18:56,  2.01it/s]
  4%|█▌                                 | 3467/78125 [30:17<10:19:06,  2.01it/s]
  4%|█▌                                 | 3468/78125 [30:18<10:19:01,  2.01it/s]
  4%|█▌                                 | 3469/78125 [30:18<10:19:05,  2.01it/s]
  4%|█▌                                 | 3470/78125 [30:19<10:19:12,  2.01it/s]
  4%|█▌                                 | 3471/78125 [30:19<10:18:58,  2.01it/s]
  4%|█▌                                 | 3472/78125 [30:20<10:19:22,  2.01it/s]
  4%|█▌                                 | 3473/78125 [30:20<10:19:09,  2.01it/s]
  4%|█▌                                 | 3474/78125 [30:21<10:19:16,  2.01it/s]
  4%|█▌                                 | 3475/78125 [30:21<10:19:27,  2.01it/s]
  4%|█▌                                 | 3476/78125 [30:22<10:19:23,  2.01it/s]
  4%|█▌                                 | 3477/78125 [30:22<10:18:34,  2.01it/s]
  4%|█▌                                 | 3478/78125 [30:23<10:18:28,  2.01it/s]
  4%|█▌                                 | 3479/78125 [30:23<10:17:55,  2.01it/s]
  4%|█▌                                 | 3480/78125 [30:24<10:18:20,  2.01it/s]
  4%|█▌                                 | 3481/78125 [30:24<10:17:59,  2.01it/s]
  4%|█▌                                 | 3482/78125 [30:25<10:17:37,  2.01it/s]
  4%|█▌                                 | 3483/78125 [30:25<10:18:18,  2.01it/s]
  4%|█▌                                 | 3484/78125 [30:26<10:17:54,  2.01it/s]
  4%|█▌                                 | 3485/78125 [30:26<10:18:03,  2.01it/s]
  4%|█▌                                 | 3486/78125 [30:27<10:17:49,  2.01it/s]
  4%|█▌                                 | 3487/78125 [30:27<10:18:20,  2.01it/s]
  4%|█▌                                 | 3488/78125 [30:28<10:18:31,  2.01it/s]
  4%|█▌                                 | 3489/78125 [30:28<10:18:44,  2.01it/s]
  4%|█▌                                 | 3490/78125 [30:29<10:18:37,  2.01it/s]
  4%|█▌                                 | 3491/78125 [30:29<10:19:04,  2.01it/s]
  4%|█▌                                 | 3492/78125 [30:30<10:19:05,  2.01it/s]
  4%|█▌                                 | 3493/78125 [30:30<10:19:34,  2.01it/s]
  4%|█▌                                 | 3494/78125 [30:31<10:20:11,  2.01it/s]
  4%|█▌                                 | 3495/78125 [30:31<10:20:02,  2.01it/s]
  4%|█▌                                 | 3496/78125 [30:32<10:20:04,  2.01it/s]
  4%|█▌                                 | 3497/78125 [30:32<10:19:45,  2.01it/s]
  4%|█▌                                 | 3498/78125 [30:33<10:19:32,  2.01it/s]
  4%|█▌                                 | 3499/78125 [30:33<10:19:25,  2.01it/s]
  4%|█▌                                 | 3500/78125 [30:34<10:18:55,  2.01it/s]
  4%|█▌                                 | 3501/78125 [30:34<10:19:12,  2.01it/s]
  4%|█▌                                 | 3502/78125 [30:35<10:19:34,  2.01it/s]
  4%|█▌                                 | 3503/78125 [30:35<10:19:36,  2.01it/s]
  4%|█▌                                 | 3504/78125 [30:36<10:19:39,  2.01it/s]
  4%|█▌                                 | 3505/78125 [30:36<10:20:01,  2.01it/s]
  4%|█▌                                 | 3506/78125 [30:37<10:19:13,  2.01it/s]
  4%|█▌                                 | 3507/78125 [30:37<10:18:53,  2.01it/s]
  4%|█▌                                 | 3508/78125 [30:38<10:18:39,  2.01it/s]
  4%|█▌                                 | 3509/78125 [30:38<10:19:01,  2.01it/s]
  4%|█▌                                 | 3510/78125 [30:39<10:18:41,  2.01it/s]
  4%|█▌                                 | 3511/78125 [30:39<10:18:49,  2.01it/s]
  4%|█▌                                 | 3512/78125 [30:40<10:18:10,  2.01it/s]
  4%|█▌                                 | 3513/78125 [30:40<10:17:48,  2.01it/s]
  4%|█▌                                 | 3514/78125 [30:41<10:18:36,  2.01it/s]
  4%|█▌                                 | 3515/78125 [30:41<10:19:10,  2.01it/s]
  5%|█▌                                 | 3516/78125 [30:42<10:19:59,  2.01it/s]
  5%|█▌                                 | 3517/78125 [30:42<10:19:21,  2.01it/s]
  5%|█▌                                 | 3518/78125 [30:43<10:19:32,  2.01it/s]
  5%|█▌                                 | 3519/78125 [30:43<10:19:05,  2.01it/s]
  5%|█▌                                 | 3520/78125 [30:44<10:18:54,  2.01it/s]
  5%|█▌                                 | 3521/78125 [30:44<10:18:43,  2.01it/s]
  5%|█▌                                 | 3522/78125 [30:45<10:19:10,  2.01it/s]
  5%|█▌                                 | 3523/78125 [30:45<10:19:36,  2.01it/s]
  5%|█▌                                 | 3524/78125 [30:46<10:19:29,  2.01it/s]
  5%|█▌                                 | 3525/78125 [30:46<10:19:55,  2.01it/s]
  5%|█▌                                 | 3526/78125 [30:47<10:20:13,  2.00it/s]
  5%|█▌                                 | 3527/78125 [30:47<10:19:54,  2.01it/s]
  5%|█▌                                 | 3528/78125 [30:48<10:19:34,  2.01it/s]
  5%|█▌                                 | 3529/78125 [30:48<10:19:18,  2.01it/s]
  5%|█▌                                 | 3530/78125 [30:49<10:18:39,  2.01it/s]
  5%|█▌                                 | 3531/78125 [30:49<10:19:29,  2.01it/s]
  5%|█▌                                 | 3532/78125 [30:50<10:19:04,  2.01it/s]
  5%|█▌                                 | 3533/78125 [30:50<10:19:36,  2.01it/s]
  5%|█▌                                 | 3534/78125 [30:51<10:19:20,  2.01it/s]
  5%|█▌                                 | 3535/78125 [30:51<10:20:06,  2.00it/s]
  5%|█▌                                 | 3536/78125 [30:52<10:19:39,  2.01it/s]
  5%|█▌                                 | 3537/78125 [30:52<10:20:00,  2.01it/s]
  5%|█▌                                 | 3538/78125 [30:53<10:19:45,  2.01it/s]
  5%|█▌                                 | 3539/78125 [30:53<10:19:36,  2.01it/s]
  5%|█▌                                 | 3540/78125 [30:54<10:19:06,  2.01it/s]
  5%|█▌                                 | 3541/78125 [30:54<10:19:04,  2.01it/s]
  5%|█▌                                 | 3542/78125 [30:54<10:19:30,  2.01it/s]
  5%|█▌                                 | 3543/78125 [30:55<10:19:19,  2.01it/s]
  5%|█▌                                 | 3544/78125 [30:55<10:19:10,  2.01it/s]
  5%|█▌                                 | 3545/78125 [30:56<10:19:06,  2.01it/s]
  5%|█▌                                 | 3546/78125 [30:56<10:19:45,  2.01it/s]
  5%|█▌                                 | 3547/78125 [30:57<10:19:18,  2.01it/s]
  5%|█▌                                 | 3548/78125 [30:57<10:19:50,  2.01it/s]
  5%|█▌                                 | 3549/78125 [30:58<10:19:13,  2.01it/s]
  5%|█▌                                 | 3550/78125 [30:58<10:18:40,  2.01it/s]
  5%|█▌                                 | 3551/78125 [30:59<10:18:49,  2.01it/s]
  5%|█▌                                 | 3552/78125 [30:59<10:18:24,  2.01it/s]
  5%|█▌                                 | 3553/78125 [31:00<10:17:14,  2.01it/s]
  5%|█▌                                 | 3554/78125 [31:00<10:18:20,  2.01it/s]
  5%|█▌                                 | 3555/78125 [31:01<10:18:56,  2.01it/s]
  5%|█▌                                 | 3556/78125 [31:01<10:18:38,  2.01it/s]
  5%|█▌                                 | 3557/78125 [31:02<10:18:51,  2.01it/s]
  5%|█▌                                 | 3558/78125 [31:02<10:18:59,  2.01it/s]
  5%|█▌                                 | 3559/78125 [31:03<10:17:41,  2.01it/s]
  5%|█▌                                 | 3560/78125 [31:03<10:17:52,  2.01it/s]
  5%|█▌                                 | 3561/78125 [31:04<10:17:55,  2.01it/s]
  5%|█▌                                 | 3562/78125 [31:04<10:17:31,  2.01it/s]
  5%|█▌                                 | 3563/78125 [31:05<10:17:36,  2.01it/s]
  5%|█▌                                 | 3564/78125 [31:05<10:16:55,  2.01it/s]
  5%|█▌                                 | 3565/78125 [31:06<10:17:10,  2.01it/s]
  5%|█▌                                 | 3566/78125 [31:06<10:17:16,  2.01it/s]
  5%|█▌                                 | 3567/78125 [31:07<10:17:24,  2.01it/s]
  5%|█▌                                 | 3568/78125 [31:07<10:17:17,  2.01it/s]
  5%|█▌                                 | 3569/78125 [31:08<10:17:17,  2.01it/s]
  5%|█▌                                 | 3570/78125 [31:08<10:17:50,  2.01it/s]
  5%|█▌                                 | 3571/78125 [31:09<10:17:41,  2.01it/s]
  5%|█▌                                 | 3572/78125 [31:09<10:17:42,  2.01it/s]
  5%|█▌                                 | 3573/78125 [31:10<10:18:25,  2.01it/s]
  5%|█▌                                 | 3574/78125 [31:10<10:18:07,  2.01it/s]
  5%|█▌                                 | 3575/78125 [31:11<10:18:23,  2.01it/s]
  5%|█▌                                 | 3576/78125 [31:11<10:18:27,  2.01it/s]
  5%|█▌                                 | 3577/78125 [31:12<10:19:05,  2.01it/s]
  5%|█▌                                 | 3578/78125 [31:12<10:18:20,  2.01it/s]
  5%|█▌                                 | 3579/78125 [31:13<10:17:59,  2.01it/s]
  5%|█▌                                 | 3580/78125 [31:13<10:17:24,  2.01it/s]
  5%|█▌                                 | 3581/78125 [31:14<10:17:56,  2.01it/s]
  5%|█▌                                 | 3582/78125 [31:14<10:18:11,  2.01it/s]
  5%|█▌                                 | 3583/78125 [31:15<10:17:04,  2.01it/s]
  5%|█▌                                 | 3584/78125 [31:15<10:16:41,  2.01it/s]
  5%|█▌                                 | 3585/78125 [31:16<10:17:06,  2.01it/s]
  5%|█▌                                 | 3586/78125 [31:16<10:17:05,  2.01it/s]
  5%|█▌                                 | 3587/78125 [31:17<10:17:19,  2.01it/s]
  5%|█▌                                 | 3588/78125 [31:17<10:17:33,  2.01it/s]
  5%|█▌                                 | 3589/78125 [31:18<10:17:16,  2.01it/s]
  5%|█▌                                 | 3590/78125 [31:18<10:17:09,  2.01it/s]
  5%|█▌                                 | 3591/78125 [31:19<10:17:10,  2.01it/s]
  5%|█▌                                 | 3592/78125 [31:19<10:17:44,  2.01it/s]
  5%|█▌                                 | 3593/78125 [31:20<10:17:17,  2.01it/s]
  5%|█▌                                 | 3594/78125 [31:20<10:17:48,  2.01it/s]
  5%|█▌                                 | 3595/78125 [31:21<10:16:21,  2.02it/s]
  5%|█▌                                 | 3596/78125 [31:21<10:16:04,  2.02it/s]
  5%|█▌                                 | 3597/78125 [31:22<10:16:53,  2.01it/s]
  5%|█▌                                 | 3598/78125 [31:22<10:17:09,  2.01it/s]
  5%|█▌                                 | 3599/78125 [31:23<10:16:56,  2.01it/s]
  5%|█▌                                 | 3600/78125 [31:23<10:17:29,  2.01it/s]
  5%|█▌                                 | 3601/78125 [31:24<10:17:18,  2.01it/s]
  5%|█▌                                 | 3602/78125 [31:24<10:17:19,  2.01it/s]
  5%|█▌                                 | 3603/78125 [31:25<10:17:11,  2.01it/s]
  5%|█▌                                 | 3604/78125 [31:25<10:17:14,  2.01it/s]
  5%|█▌                                 | 3605/78125 [31:26<10:17:21,  2.01it/s]
  5%|█▌                                 | 3606/78125 [31:26<10:17:04,  2.01it/s]
  5%|█▌                                 | 3607/78125 [31:27<10:17:29,  2.01it/s]
  5%|█▌                                 | 3608/78125 [31:27<10:17:35,  2.01it/s]
  5%|█▌                                 | 3609/78125 [31:28<10:17:50,  2.01it/s]
  5%|█▌                                 | 3610/78125 [31:28<10:17:58,  2.01it/s]
  5%|█▌                                 | 3611/78125 [31:29<10:17:44,  2.01it/s]
  5%|█▌                                 | 3612/78125 [31:29<10:16:32,  2.01it/s]
  5%|█▌                                 | 3613/78125 [31:30<10:16:00,  2.02it/s]
  5%|█▌                                 | 3614/78125 [31:30<10:16:13,  2.02it/s]
  5%|█▌                                 | 3615/78125 [31:31<10:16:03,  2.02it/s]
  5%|█▌                                 | 3616/78125 [31:31<10:15:48,  2.02it/s]
  5%|█▌                                 | 3617/78125 [31:32<10:16:30,  2.01it/s]
  5%|█▌                                 | 3618/78125 [31:32<10:16:48,  2.01it/s]
  5%|█▌                                 | 3619/78125 [31:33<10:17:03,  2.01it/s]
  5%|█▌                                 | 3620/78125 [31:33<10:16:31,  2.01it/s]
  5%|█▌                                 | 3621/78125 [31:34<10:16:30,  2.01it/s]
  5%|█▌                                 | 3622/78125 [31:34<10:15:05,  2.02it/s]
  5%|█▌                                 | 3623/78125 [31:35<10:15:20,  2.02it/s]
  5%|█▌                                 | 3624/78125 [31:35<10:16:13,  2.02it/s]
  5%|█▌                                 | 3625/78125 [31:36<10:15:38,  2.02it/s]
  5%|█▌                                 | 3626/78125 [31:36<10:16:20,  2.01it/s]
  5%|█▌                                 | 3627/78125 [31:37<10:16:47,  2.01it/s]
  5%|█▋                                 | 3628/78125 [31:37<10:16:52,  2.01it/s]
  5%|█▋                                 | 3629/78125 [31:38<10:15:45,  2.02it/s]
  5%|█▋                                 | 3630/78125 [31:38<10:16:34,  2.01it/s]
  5%|█▋                                 | 3631/78125 [31:39<10:16:43,  2.01it/s]
  5%|█▋                                 | 3632/78125 [31:39<10:16:22,  2.01it/s]
  5%|█▋                                 | 3633/78125 [31:40<10:16:37,  2.01it/s]
  5%|█▋                                 | 3634/78125 [31:40<10:15:37,  2.02it/s]
  5%|█▋                                 | 3635/78125 [31:41<10:15:38,  2.02it/s]
  5%|█▋                                 | 3636/78125 [31:41<10:15:53,  2.02it/s]
  5%|█▋                                 | 3637/78125 [31:42<10:16:23,  2.01it/s]
  5%|█▋                                 | 3638/78125 [31:42<10:16:19,  2.01it/s]
  5%|█▋                                 | 3639/78125 [31:43<10:15:49,  2.02it/s]
  5%|█▋                                 | 3640/78125 [31:43<10:15:46,  2.02it/s]
  5%|█▋                                 | 3641/78125 [31:44<10:15:04,  2.02it/s]
  5%|█▋                                 | 3642/78125 [31:44<10:15:14,  2.02it/s]
  5%|█▋                                 | 3643/78125 [31:45<10:15:23,  2.02it/s]
  5%|█▋                                 | 3644/78125 [31:45<10:15:29,  2.02it/s]
  5%|█▋                                 | 3645/78125 [31:46<10:16:25,  2.01it/s]
  5%|█▋                                 | 3646/78125 [31:46<10:16:44,  2.01it/s]
  5%|█▋                                 | 3647/78125 [31:47<10:16:08,  2.01it/s]
  5%|█▋                                 | 3648/78125 [31:47<10:16:29,  2.01it/s]
  5%|█▋                                 | 3649/78125 [31:48<10:16:08,  2.01it/s]
  5%|█▋                                 | 3650/78125 [31:48<10:17:02,  2.01it/s]
  5%|█▋                                 | 3651/78125 [31:49<10:16:34,  2.01it/s]
  5%|█▋                                 | 3652/78125 [31:49<10:17:36,  2.01it/s]
  5%|█▋                                 | 3653/78125 [31:50<10:16:40,  2.01it/s]
  5%|█▋                                 | 3654/78125 [31:50<10:16:24,  2.01it/s]
  5%|█▋                                 | 3655/78125 [31:51<10:16:22,  2.01it/s]
  5%|█▋                                 | 3656/78125 [31:51<10:17:02,  2.01it/s]
  5%|█▋                                 | 3657/78125 [31:52<10:16:40,  2.01it/s]
  5%|█▋                                 | 3658/78125 [31:52<10:16:37,  2.01it/s]
  5%|█▋                                 | 3659/78125 [31:53<10:16:08,  2.01it/s]
  5%|█▋                                 | 3660/78125 [31:53<10:15:37,  2.02it/s]
  5%|█▋                                 | 3661/78125 [31:54<10:15:42,  2.02it/s]
  5%|█▋                                 | 3662/78125 [31:54<10:15:16,  2.02it/s]
  5%|█▋                                 | 3663/78125 [31:55<10:14:56,  2.02it/s]
  5%|█▋                                 | 3664/78125 [31:55<10:15:05,  2.02it/s]
  5%|█▋                                 | 3665/78125 [31:56<10:15:12,  2.02it/s]
  5%|█▋                                 | 3666/78125 [31:56<10:15:34,  2.02it/s]
  5%|█▋                                 | 3667/78125 [31:57<10:16:26,  2.01it/s]
  5%|█▋                                 | 3668/78125 [31:57<10:16:03,  2.01it/s]
  5%|█▋                                 | 3669/78125 [31:58<10:16:20,  2.01it/s]
  5%|█▋                                 | 3670/78125 [31:58<10:15:36,  2.02it/s]
  5%|█▋                                 | 3671/78125 [31:59<10:15:43,  2.02it/s]
  5%|█▋                                 | 3672/78125 [31:59<10:15:38,  2.02it/s]
  5%|█▋                                 | 3673/78125 [32:00<10:14:52,  2.02it/s]
  5%|█▋                                 | 3674/78125 [32:00<10:15:26,  2.02it/s]
  5%|█▋                                 | 3675/78125 [32:01<10:14:59,  2.02it/s]
  5%|█▋                                 | 3676/78125 [32:01<10:15:19,  2.02it/s]
  5%|█▋                                 | 3677/78125 [32:02<10:14:18,  2.02it/s]
  5%|█▋                                 | 3678/78125 [32:02<10:14:36,  2.02it/s]
  5%|█▋                                 | 3679/78125 [32:03<10:15:26,  2.02it/s]
  5%|█▋                                 | 36
760.3s 149 80/78125 [32:03<10:15:14,  2.02it/s]
  5%|█▋                                 | 3681/78125 [32:04<10:14:38,  2.02it/s]
  5%|█▋                                 | 3682/78125 [32:04<10:15:19,  2.02it/s]
  5%|█▋                                 | 3683/78125 [32:05<10:15:21,  2.02it/s]
  5%|█▋                                 | 3684/78125 [32:05<10:14:52,  2.02it/s]
  5%|█▋                                 | 3685/78125 [32:06<10:14:54,  2.02it/s]
  5%|█▋                                 | 3686/78125 [32:06<10:15:01,  2.02it/s]
  5%|█▋                                 | 3687/78125 [32:07<10:14:42,  2.02it/s]
  5%|█▋                                 | 3688/78125 [32:07<10:14:28,  2.02it/s]
  5%|█▋                                 | 3689/78125 [32:08<10:14:46,  2.02it/s]
  5%|█▋                                 | 3690/78125 [32:08<10:15:11,  2.02it/s]
  5%|█▋                                 | 3691/78125 [32:08<10:14:40,  2.02it/s]
  5%|█▋                                 | 3692/78125 [32:09<10:15:24,  2.02it/s]
  5%|█▋                                 | 3693/78125 [32:09<10:15:43,  2.01it/s]
  5%|█▋                                 | 3694/78125 [32:10<10:15:04,  2.02it/s]
  5%|█▋                                 | 3695/78125 [32:10<10:15:42,  2.01it/s]
  5%|█▋                                 | 3696/78125 [32:11<10:15:59,  2.01it/s]
  5%|█▋                                 | 3697/78125 [32:11<10:15:53,  2.01it/s]
  5%|█▋                                 | 3698/78125 [32:12<10:15:19,  2.02it/s]
  5%|█▋                                 | 3699/78125 [32:12<10:15:26,  2.02it/s]
  5%|█▋                                 | 3700/78125 [32:13<10:15:37,  2.01it/s]
  5%|█▋                                 | 3701/78125 [32:13<10:15:37,  2.01it/s]
  5%|█▋                                 | 3702/78125 [32:14<10:15:31,  2.02it/s]
  5%|█▋                                 | 3703/78125 [32:14<10:15:25,  2.02it/s]
  5%|█▋                                 | 3704/78125 [32:15<10:15:35,  2.01it/s]
  5%|█▋                                 | 3705/78125 [32:15<10:14:20,  2.02it/s]
  5%|█▋                                 | 3706/78125 [32:16<10:14:22,  2.02it/s]
  5%|█▋                                 | 3707/78125 [32:16<10:14:28,  2.02it/s]
  5%|█▋                                 | 3708/78125 [32:17<10:15:00,  2.02it/s]
  5%|█▋                                 | 3709/78125 [32:17<10:14:40,  2.02it/s]
  5%|█▋                                 | 3710/78125 [32:18<10:14:26,  2.02it/s]
  5%|█▋                                 | 3711/78125 [32:18<10:14:15,  2.02it/s]
  5%|█▋                                 | 3712/78125 [32:19<10:14:31,  2.02it/s]
  5%|█▋                                 | 3713/78125 [32:19<10:15:17,  2.02it/s]
  5%|█▋                                 | 3714/78125 [32:20<10:15:03,  2.02it/s]
  5%|█▋                                 | 3715/78125 [32:20<10:15:12,  2.02it/s]
  5%|█▋                                 | 3716/78125 [32:21<10:14:16,  2.02it/s]
  5%|█▋                                 | 3717/78125 [32:21<10:14:10,  2.02it/s]
  5%|█▋                                 | 3718/78125 [32:22<10:14:30,  2.02it/s]
  5%|█▋                                 | 3719/78125 [32:22<10:13:54,  2.02it/s]
  5%|█▋                                 | 3720/78125 [32:23<10:13:17,  2.02it/s]
  5%|█▋                                 | 3721/78125 [32:23<10:13:47,  2.02it/s]
  5%|█▋                                 | 3722/78125 [32:24<10:14:11,  2.02it/s]
  5%|█▋                                 | 3723/78125 [32:24<10:13:51,  2.02it/s]
  5%|█▋                                 | 3724/78125 [32:25<10:12:51,  2.02it/s]
  5%|█▋                                 | 3725/78125 [32:25<10:13:43,  2.02it/s]
  5%|█▋                                 | 3726/78125 [32:26<10:13:29,  2.02it/s]
  5%|█▋                                 | 3727/78125 [32:26<10:13:13,  2.02it/s]
  5%|█▋                                 | 3728/78125 [32:27<10:13:27,  2.02it/s]
  5%|█▋                                 | 3729/78125 [32:27<10:14:12,  2.02it/s]
  5%|█▋                                 | 3730/78125 [32:28<10:14:17,  2.02it/s]
  5%|█▋                                 | 3731/78125 [32:28<10:13:50,  2.02it/s]
  5%|█▋                                 | 3732/78125 [32:29<10:14:06,  2.02it/s]
  5%|█▋                                 | 3733/78125 [32:29<10:14:23,  2.02it/s]
  5%|█▋                                 | 3734/78125 [32:30<10:14:42,  2.02it/s]
  5%|█▋                                 | 3735/78125 [32:30<10:14:00,  2.02it/s]
  5%|█▋                                 | 3736/78125 [32:31<10:14:15,  2.02it/s]
  5%|█▋                                 | 3737/78125 [32:31<10:13:46,  2.02it/s]
  5%|█▋                                 | 3738/78125 [32:32<10:14:11,  2.02it/s]
  5%|█▋                                 | 3739/78125 [32:32<10:14:11,  2.02it/s]
  5%|█▋                                 | 3740/78125 [32:33<10:14:25,  2.02it/s]
  5%|█▋                                 | 3741/78125 [32:33<10:14:11,  2.02it/s]
  5%|█▋                                 | 3742/78125 [32:34<10:13:55,  2.02it/s]
  5%|█▋                                 | 3743/78125 [32:34<10:14:30,  2.02it/s]
  5%|█▋                                 | 3744/78125 [32:35<10:13:54,  2.02it/s]
  5%|█▋                                 | 3745/78125 [32:35<10:13:53,  2.02it/s]
  5%|█▋                                 | 3746/78125 [32:36<10:14:03,  2.02it/s]
  5%|█▋                                 | 3747/78125 [32:36<10:13:52,  2.02it/s]
  5%|█▋                                 | 3748/78125 [32:37<10:13:40,  2.02it/s]
  5%|█▋                                 | 3749/78125 [32:37<10:13:34,  2.02it/s]
  5%|█▋                                 | 3750/78125 [32:38<10:14:00,  2.02it/s]
  5%|█▋                                 | 3751/78125 [32:38<10:14:10,  2.02it/s]
  5%|█▋                                 | 3752/78125 [32:39<10:13:52,  2.02it/s]
  5%|█▋                                 | 3753/78125 [32:39<10:13:57,  2.02it/s]
  5%|█▋                                 | 3754/78125 [32:40<10:14:12,  2.02it/s]
  5%|█▋                                 | 3755/78125 [32:40<10:14:27,  2.02it/s]
  5%|█▋                                 | 3756/78125 [32:41<10:13:47,  2.02it/s]
  5%|█▋                                 | 3757/78125 [32:41<10:13:30,  2.02it/s]
  5%|█▋                                 | 3758/78125 [32:42<10:13:53,  2.02it/s]
  5%|█▋                                 | 3759/78125 [32:42<10:13:11,  2.02it/s]
  5%|█▋                                 | 3760/78125 [32:43<10:12:54,  2.02it/s]
  5%|█▋                                 | 3761/78125 [32:43<10:12:45,  2.02it/s]
  5%|█▋                                 | 3762/78125 [32:44<10:13:12,  2.02it/s]
  5%|█▋                                 | 3763/78125 [32:44<10:13:20,  2.02it/s]
  5%|█▋                                 | 3764/78125 [32:45<10:13:31,  2.02it/s]
  5%|█▋                                 | 3765/78125 [32:45<10:13:35,  2.02it/s]
  5%|█▋                                 | 3766/78125 [32:46<10:13:17,  2.02it/s]
  5%|█▋                                 | 3767/78125 [32:46<10:13:17,  2.02it/s]
  5%|█▋                                 | 3768/78125 [32:47<10:13:12,  2.02it/s]
  5%|█▋                                 | 3769/78125 [32:47<10:13:30,  2.02it/s]
  5%|█▋                                 | 3770/78125 [32:48<10:13:34,  2.02it/s]
  5%|█▋                                 | 3771/78125 [32:48<10:12:34,  2.02it/s]
  5%|█▋                                 | 3772/78125 [32:49<10:12:07,  2.02it/s]
  5%|█▋                                 | 3773/78125 [32:49<10:12:30,  2.02it/s]
  5%|█▋                                 | 3774/78125 [32:50<10:11:51,  2.03it/s]
  5%|█▋                                 | 3775/78125 [32:50<10:12:30,  2.02it/s]
  5%|█▋                                 | 3776/78125 [32:51<10:13:28,  2.02it/s]
  5%|█▋                                 | 3777/78125 [32:51<10:13:09,  2.02it/s]
  5%|█▋                                 | 3778/78125 [32:52<10:13:13,  2.02it/s]
  5%|█▋                                 | 3779/78125 [32:52<10:13:09,  2.02it/s]
  5%|█▋                                 | 3780/78125 [32:53<10:13:30,  2.02it/s]
  5%|█▋                                 | 3781/78125 [32:53<10:12:46,  2.02it/s]
  5%|█▋                                 | 3782/78125 [32:54<10:13:21,  2.02it/s]
  5%|█▋                                 | 3783/78125 [32:54<10:12:57,  2.02it/s]
  5%|█▋                                 | 3784/78125 [32:55<10:13:10,  2.02it/s]
  5%|█▋                                 | 3785/78125 [32:55<10:13:10,  2.02it/s]
  5%|█▋                                 | 3786/78125 [32:56<10:12:52,  2.02it/s]
  5%|█▋                                 | 3787/78125 [32:56<10:12:29,  2.02it/s]
  5%|█▋                                 | 3788/78125 [32:57<10:13:19,  2.02it/s]
  5%|█▋                                 | 3789/78125 [32:57<10:13:07,  2.02it/s]
  5%|█▋                                 | 3790/78125 [32:58<10:13:25,  2.02it/s]
  5%|█▋                                 | 3791/78125 [32:58<10:13:12,  2.02it/s]
  5%|█▋                                 | 3792/78125 [32:59<10:13:10,  2.02it/s]
  5%|█▋                                 | 3793/78125 [32:59<10:13:38,  2.02it/s]
  5%|█▋                                 | 3794/78125 [33:00<10:13:48,  2.02it/s]
  5%|█▋                                 | 3795/78125 [33:00<10:14:15,  2.02it/s]
  5%|█▋                                 | 3796/78125 [33:00<10:13:45,  2.02it/s]
  5%|█▋                                 | 3797/78125 [33:01<10:13:40,  2.02it/s]
  5%|█▋                                 | 3798/78125 [33:01<10:13:40,  2.02it/s]
  5%|█▋                                 | 3799/78125 [33:02<10:13:38,  2.02it/s]
  5%|█▋                                 | 3800/78125 [33:02<10:14:07,  2.02it/s]
  5%|█▋                                 | 3801/78125 [33:03<10:13:56,  2.02it/s]
  5%|█▋                                 | 3802/78125 [33:03<10:13:54,  2.02it/s]
  5%|█▋                                 | 3803/78125 [33:04<10:13:28,  2.02it/s]
  5%|█▋                                 | 3804/78125 [33:04<10:14:19,  2.02it/s]
  5%|█▋                                 | 3805/78125 [33:05<10:14:34,  2.02it/s]
  5%|█▋                                 | 3806/78125 [33:05<10:14:42,  2.02it/s]
  5%|█▋                                 | 3807/78125 [33:06<10:14:15,  2.02it/s]
  5%|█▋                                 | 3808/78125 [33:06<10:13:33,  2.02it/s]
  5%|█▋                                 | 3809/78125 [33:07<10:12:45,  2.02it/s]
  5%|█▋                                 | 3810/78125 [33:07<10:12:44,  2.02it/s]
  5%|█▋                                 | 3811/78125 [33:08<10:13:40,  2.02it/s]
  5%|█▋                                 | 3812/78125 [33:08<10:14:21,  2.02it/s]
  5%|█▋                                 | 3813/78125 [33:09<10:13:54,  2.02it/s]
  5%|█▋                                 | 3814/78125 [33:09<10:14:03,  2.02it/s]
  5%|█▋                                 | 3815/78125 [33:10<10:14:23,  2.02it/s]
  5%|█▋                                 | 3816/78125 [33:10<10:14:09,  2.02it/s]
  5%|█▋                                 | 3817/78125 [33:11<10:13:56,  2.02it/s]
  5%|█▋                                 | 3818/78125 [33:11<10:14:13,  2.02it/s]
  5%|█▋                                 | 3819/78125 [33:12<10:14:00,  2.02it/s]
  5%|█▋                                 | 3820/78125 [33:12<10:14:52,  2.01it/s]
  5%|█▋                                 | 3821/78125 [33:13<10:14:40,  2.01it/s]
  5%|█▋                                 | 3822/78125 [33:13<10:14:54,  2.01it/s]
  5%|█▋                                 | 3823/78125 [33:14<10:15:39,  2.01it/s]
  5%|█▋                                 | 3824/78125 [33:14<10:16:05,  2.01it/s]
  5%|█▋                                 | 3825/78125 [33:15<10:15:44,  2.01it/s]
  5%|█▋                                 | 3826/78125 [33:15<10:15:01,  2.01it/s]
  5%|█▋                                 | 3827/78125 [33:16<10:14:06,  2.02it/s]
  5%|█▋                                 | 3828/78125 [33:16<10:14:07,  2.02it/s]
  5%|█▋                                 | 3829/78125 [33:17<10:13:29,  2.02it/s]
  5%|█▋                                 | 3830/78125 [33:17<10:13:54,  2.02it/s]
  5%|█▋                                 | 3831/78125 [33:18<10:13:18,  2.02it/s]
  5%|█▋                                 | 3832/78125 [33:18<10:13:50,  2.02it/s]
  5%|█▋                                 | 3833/78125 [33:19<10:13:47,  2.02it/s]
  5%|█▋                                 | 3834/78125 [33:19<10:14:10,  2.02it/s]
  5%|█▋                                 | 3835/78125 [33:20<10:14:25,  2.02it/s]
  5%|█▋                                 | 3836/78125 [33:20<10:14:09,  2.02it/s]
  5%|█▋                                 | 3837/78125 [33:21<10:13:34,  2.02it/s]
  5%|█▋                                 | 3838/78125 [33:21<10:13:47,  2.02it/s]
  5%|█▋                                 | 3839/78125 [33:22<10:13:39,  2.02it/s]
  5%|█▋                                 | 3840/78125 [33:22<10:13:34,  2.02it/s]
  5%|█▋                                 | 3841/78125 [33:23<10:13:49,  2.02it/s]
  5%|█▋                                 | 3842/78125 [33:23<10:13:42,  2.02it/s]
  5%|█▋                                 | 3843/78125 [33:24<10:13:21,  2.02it/s]
  5%|█▋                                 | 3844/78125 [33:24<10:13:37,  2.02it/s]
  5%|█▋                                 | 3845/78125 [33:25<10:14:05,  2.02it/s]
  5%|█▋                                 | 3846/78125 [33:25<10:14:06,  2.02it/s]
  5%|█▋                                 | 3847/78125 [33:26<10:13:22,  2.02it/s]
  5%|█▋                                 | 3848/78125 [33:26<10:14:30,  2.01it/s]
  5%|█▋                                 | 3849/78125 [33:27<10:13:32,  2.02it/s]
  5%|█▋                                 | 3850/78125 [33:27<10:13:20,  2.02it/s]
  5%|█▋                                 | 3851/78125 [33:28<10:13:10,  2.02it/s]
  5%|█▋                                 | 3852/78125 [33:28<10:13:26,  2.02it/s]
  5%|█▋                                 | 3853/78125 [33:29<10:13:37,  2.02it/s]
  5%|█▋                                 | 3854/78125 [33:29<10:13:40,  2.02it/s]
  5%|█▋                                 | 3855/78125 [33:30<10:13:55,  2.02it/s]
  5%|█▋                                 | 3856/78125 [33:30<10:13:59,  2.02it/s]
  5%|█▋                                 | 3857/78125 [33:31<10:14:03,  2.02it/s]
  5%|█▋                                 | 3858/78125 [33:31<10:14:05,  2.02it/s]
  5%|█▋                                 | 3859/78125 [33:32<10:13:18,  2.02it/s]
  5%|█▋                                 | 3860/78125 [33:32<10:13:41,  2.02it/s]
  5%|█▋                                 | 3861/78125 [33:33<10:14:09,  2.02it/s]
  5%|█▋                                 | 3862/78125 [33:33<10:14:15,  2.01it/s]
  5%|█▋                                 | 3863/78125 [33:34<10:14:05,  2.02it/s]
  5%|█▋                                 | 3864/78125 [33:34<10:13:59,  2.02it/s]
  5%|█▋                                 | 3865/78125 [33:35<10:13:28,  2.02it/s]
  5%|█▋                                 | 3866/78125 [33:35<10:13:23,  2.02it/s]
  5%|█▋                                 | 3867/78125 [33:36<10:13:31,  2.02it/s]
  5%|█▋                                 | 3868/78125 [33:36<10:14:13,  2.01it/s]
  5%|█▋                                 | 3869/78125 [33:37<10:14:42,  2.01it/s]
  5%|█▋                                 | 3870/78125 [33:37<10:15:16,  2.01it/s]
  5%|█▋                                 | 3871/78125 [33:38<10:15:09,  2.01it/s]
  5%|█▋                                 | 3872/78125 [33:38<10:14:42,  2.01it/s]
  5%|█▋                                 | 3873/78125 [33:39<10:14:25,  2.01it/s]
  5%|█▋                                 | 3874/78125 [33:39<10:13:51,  2.02it/s]
  5%|█▋                                 | 3875/78125 [33:40<10:14:02,  2.02it/s]
  5%|█▋                                 | 3876/78125 [33:40<10:13:58,  2.02it/s]
  5%|█▋                                 | 3877/78125 [33:41<10:13:51,  2.02it/s]
  5%|█▋                                 | 3878/78125 [33:41<10:14:05,  2.02it/s]
  5%|█▋                                 | 3879/78125 [33:42<10:13:35,  2.02it/s]
  5%|█▋                                 | 3880/78125 [33:42<10:13:31,  2.02it/s]
  5%|█▋                                 | 3881/78125 [33:43<10:13:23,  2.02it/s]
  5%|█▋                                 | 3882/78125 [33:43<10:13:50,  2.02it/s]
  5%|█▋                                 | 3883/78125 [33:44<10:13:48,  2.02it/s]
  5%|█▋                                 | 3884/78125 [33:44<10:14:21,  2.01it/s]
  5%|█▋                                 | 3885/78125 [33:45<10:15:14,  2.01it/s]
  5%|█▋                                 | 3886/78125 [33:45<10:14:26,  2.01it/s]
  5%|█▋                                 | 3887/78125 [33:46<10:14:34,  2.01it/s]
  5%|█▋                                 | 3888/78125 [33:46<10:14:17,  2.01it/s]
  5%|█▋                                 | 3889/78125 [33:47<10:14:01,  2.02it/s]
  5%|█▋                                 | 3890/78125 [33:47<10:14:22,  2.01it/s]
  5%|█▋                                 | 3891/78125 [33:48<10:14:38,  2.01it/s]
  5%|█▋                                 | 3892/78125 [33:48<10:14:31,  2.01it/s]
  5%|█▋                                 | 3893/78125 [33:49<10:14:15,  2.01it/s]
  5%|█▋                                 | 3894/78125 [33:49<10:14:32,  2.01it/s]
  5%|█▋                                 | 3895/78125 [33:50<10:14:14,  2.01it/s]
  5%|█▋                                 | 3896/78125 [33:50<10:14:38,  2.01it/s]
  5%|█▋                                 | 3897/78125 [33:51<10:13:45,  2.02it/s]
  5%|█▋                                 | 3898/78125 [33:51<10:13:44,  2.02it/s]
  5%|█▋                                 | 3899/78125 [33:52<10:14:32,  2.01it/s]
  5%|█▋                                 | 3900/78125 [33:52<10:14:38,  2.01it/s]
  5%|█▋                                 | 3901/78125 [33:53<10:14:30,  2.01it/s]
  5%|█▋                                 | 3902/78125 [33:53<10:14:42,  2.01it/s]
  5%|█▋                                 | 3903/78125 [33:54<10:14:42,  2.01it/s]
  5%|█▋                                 | 3904/78125 [33:54<10:14:54,  2.01it/s]
  5%|█▋                                 | 3905/78125 [33:55<10:14:19,  2.01it/s]
  5%|█▋                                 | 3906/78125 [33:55<10:14:12,  2.01it/s]
  5%|█▊                                 | 3907/78125 [33:56<10:13:31,  2.02it/s]
  5%|█▊                                 | 3908/78125 [33:56<10:12:48,  2.02it/s]
  5%|█▊                                 | 3909/78125 [33:57<10:13:12,  2.02it/s]
  5%|█▊                                 | 3910/78125 [33:57<10:13:32,  2.02it/s]
  5%|█▊                                 | 3911/78125 [33:58<10:14:14,  2.01it/s]
  5%|█▊                                 | 3912/78125 [33:58<10:13:55,  2.01it/s]
  5%|█▊                                 | 3913/78125 [33:59<10:14:16,  2.01it/s]
  5%|█▊                                 | 3914/78125 [33:59<10:14:45,  2.01it/s]
  5%|█▊                                 | 3915/78125 [34:00<10:14:21,  2.01it/s]
  5%|█▊                                 | 3916/78125 [34:00<10:13:55,  2.01it/s]
  5%|█▊                                 | 3917/78125 [34:01<10:14:13,  2.01it/s]
  5%|█▊                                 | 3918/78125 [34:01<10:13:27,  2.02it/s]
  5%|█▊                                 | 3919/78125 [34:02<10:13:46,  2.02it/s]
  5%|█▊                                 | 3920/78125 [34:02<10:13:37,  2.02it/s]
  5%|█▊                                 | 3921/78125 [34:03<10:14:22,  2.01it/s]
  5%|█▊                                 | 3922/78125 [34:03<10:14:42,  2.01it/s]
  5%|█▊                                 | 3923/78125 [34:04<10:15:16,  2.01it/s]
  5%|█▊                                 | 3924/78125 [34:04<10:15:18,  2.01it/s]
  5%|█▊                                 | 3925/78125 [34:05<10:15:38,  2.01it/s]
  5%|█▊                                 | 3926/78125 [34:05<10:15:04,  2.01it/s]
  5%|█▊                                 | 3927/78125 [34:05<10:14:38,  2.01it/s]
  5%|█▊                                 | 3928/78125 [34:06<10:14:32,  2.01it/s]
  5%|█▊                                 | 3929/78125 [34:06<10:14:36,  2.01it/s]
  5%|█▊                                 | 3930/78125 [34:07<10:14:22,  2.01it/s]
  5%|█▊                                 | 3931/78125 [34:07<10:14:17,  2.01it/s]
  5%|█▊                                 | 3932/78125 [34:08<10:14:20,  2.01it/s]
  5%|█▊                                 | 3933/78125 [34:08<10:14:48,  2.01it/s]
  5%|█▊                                 | 3934/78125 [34:09<10:14:34,  2.01it/s]
  5%|█▊                                 | 3935/78125 [34:09<10:14:58,  2.01it/s]
  5%|█▊                                 | 3936/78125 [34:10<10:15:01,  2.01it/s]
  5%|█▊                                 | 3937/78125 [34:10<10:14:31,  2.01it/s]
  5%|█▊                                 | 3938/78125 [34:11<10:14:10,  2.01it/s]
  5%|█▊                                 | 3939/78125 [34:11<10:15:19,  2.01it/s]
  5%|█▊                                 | 3940/78125 [34:12<10:14:54,  2.01it/s]
  5%|█▊                                 | 3941/78125 [34:12<10:14:49,  2.01it/s]
  5%|█▊                                 | 3942/78125 [34:13<10:14:38,  2.01it/s]
  5%|█▊                                 | 3943/78125 [34:13<10:14:21,  2.01it/s]
  5%|█▊                                 | 3944/78125 [34:14<10:14:34,  2.01it/s]
  5%|█▊                                 | 3945/78125 [34:14<10:14:32,  2.01it/s]
  5%|█▊                                 | 3946/78125 [34:15<10:14:25,  2.01it/s]
  5%|█▊                                 | 3947/78125 [34:15<10:13:54,  2.01it/s]
  5%|█▊                                 | 3948/78125 [34:16<10:14:17,  2.01it/s]
  5%|█▊                                 | 3949/78125 [34:16<10:13:36,  2.01it/s]
  5%|█▊                                 | 3950/78125 [34:17<10:13:24,  2.02it/s]
  5%|█▊                                 | 3951/78125 [34:17<10:14:06,  2.01it/s]
  5%|█▊                                 | 3952/78125 [34:18<10:14:15,  2.01it/s]
  5%|█▊                                 | 3953/78125 [34:18<10:14:02,  2.01it/s]
  5%|█▊                                 | 3954/78125 [34:19<10:14:48,  2.01it/s]
  5%|█▊                                 | 3955/78125 [34:19<10:14:35,  2.01it/s]
  5%|█▊                                 | 3956/78125 [34:20<10:14:03,  2.01it/s]
  5%|█▊                                 | 3957/78125 [34:20<10:14:39,  2.01it/s]
  5%|█▊                                 | 3958/78125 [34:21<10:14:46,  2.01it/s]
  5%|█▊                                 | 3959/78125 [34:21<10:14:05,  2.01it/s]
  5%|█▊                                 | 3960/78125 [34:22<10:14:07,  2.01it/s]
  5%|█▊                                 | 3961/78125 [34:22<10:14:59,  2.01it/s]
  5%|█▊                                 | 3962/78125 [34:23<10:14:30,  2.01it/s]
  5%|█▊                                 | 3963/78125 [34:23<10:14:44,  2.01it/s]
  5%|█▊                                 | 3964/78125 [34:24<10:14:20,  2.01it/s]
  5%|█▊                                 | 3965/78125 [34:24<10:14:44,  2.01it/s]
  5%|█▊                                 | 3966/78125 [34:25<10:13:46,  2.01it/s]
  5%|█▊                                 | 3967/78125 [34:25<10:13:00,  2.02it/s]
  5%|█▊                                 | 3968/78125 [34:26<10:13:05,  2.02it/s]
  5%|█▊                                 | 3969/78125 [34:26<10:14:13,  2.01it/s]
  5%|█▊                                 | 3970/78125 [34:27<10:14:00,  2.01it/s]
  5%|█▊                                 | 3971/78125 [34:27<10:13:42,  2.01it/s]
  5%|█▊                                 | 3972/78125 [34:28<10:14:41,  2.01it/s]
  5%|█▊                                 | 3973/78125 [34:28<10:14:54,  2.01it/s]
  5%|█▊                                 | 3974/78125 [34:29<10:14:21,  2.01it/s]
  5%|█▊                                 | 3975/78125 [34:29<10:14:42,  2.01it/s]
  5%|█▊                                 | 3976/78125 [34:30<10:14:39,  2.01it/s]
  5%|█▊                                 | 3977/78125 [34:30<10:14:47,  2.01it/s]
  5%|█▊                                 | 3978/78125 [34:31<10:15:17,  2.01it/s]
  5%|█▊                                 | 3979/78125 [34:31<10:14:55,  2.01it/s]
  5%|█▊                                 | 3980/78125 [34:32<10:13:45,  2.01it/s]
  5%|█▊                                 | 3981/78125 [34:32<10:13:42,  2.01it/s]
  5%|█▊                                 | 3982/78125 [34:33<10:13:38,  2.01it/s]
  5%|█▊                                 | 3983/78125 [34:33<10:13:25,  2.01it/s]
  5%|█▊                                 | 3984/78125 [34:34<10:12:56,  2.02it/s]
  5%|█▊                                 | 3985/78125 [34:34<10:13:16,  2.01it/s]
  5%|█▊                                 | 3986/78125 [34:35<10:13:24,  2.01it/s]
  5%|█▊                                 | 3987/78125 [34:35<10:13:27,  2.01it/s]
  5%|█▊                                 | 3988/78125 [34:36<10:13:58,  2.01it/s]
  5%|█▊                                 | 3989/78125 [34:36<10:14:02,  2.01it/s]
  5%|█▊                                 | 3990/78125 [34:37<10:14:50,  2.01it/s]
  5%|█▊                                 | 3991/78125 [34:37<10:15:04,  2.01it/s]
  5%|█▊                                 | 3992/78125 [34:38<10:14:58,  2.01it/s]
  5%|█▊                                 | 3993/78125 [34:38<10:14:59,  2.01it/s]
  5%|█▊                                 | 3994/78125 [34:39<10:14:46,  2.01it/s]
  5%|█▊                                 | 3995/78125 [34:39<10:14:46,  2.01it/s]
  5%|█▊                                 | 3996/78125 [34:40<10:14:51,  2.01it/s]
  5%|█▊                                 | 3997/78125 [34:40<10:15:40,  2.01it/s]
  5%|█▊                                 | 3998/78125 [34:41<10:15:00,  2.01it/s]
  5%|█▊                                 | 3999/78125 [34:41<10:14:39,  2.01it/s]
  5%|█▊                                 | 4000/78125 [34:42<10:14:14,  2.01it/s]
  5%|█▊                                 | 4001/78125 [34:42<10:14:28,  2.01it/s]
  5%|█▊                                 | 4002/78125 [34:43<10:15:50,  2.01it/s]
  5%|█▊                                 | 4003/78125 [34:43<10:15:24,  2.01it/s]
  5%|█▊                                 | 4004/78125 [34:44<10:15:22,  2.01it/s]
  5%|█▊                                 | 4005/78125 [34:44<10:15:18,  2.01it/s]
  5%|█▊                                 | 4006/78125 [34:45<10:15:34,  2.01it/s]
  5%|█▊                                 | 4007/78125 [34:45<10:15:44,  2.01it/s]
  5%|█▊                                 | 4008/78125 [34:46<10:14:56,  2.01it/s]
  5%|█▊                                 | 4009/78125 [34:46<10:15:09,  2.01it/s]
  5%|█▊                                 | 4010/78125 [34:47<10:14:36,  2.01it/s]
  5%|█▊                                 | 4011/78125 [34:47<10:14:06,  2.01it/s]
  5%|█▊                                 | 4012/78125 [34:48<10:14:14,  2.01it/s]
  5%|█▊                                 | 4013/78125 [34:48<10:14:49,  2.01it/s]
  5%|█▊                                 | 4014/78125 [34:49<10:14:34,  2.01it/s]
  5%|█▊                                 | 4015/78125 [34:49<10:14:10,  2.01it/s]
  5%|█▊                                 | 4016/78125 [34:50<10:14:54,  2.01it/s]
  5%|█▊                                 | 4017/78125 [34:50<10:15:25,  2.01it/s]
  5%|█▊                                 | 4018/78125 [34:51<10:14:49,  2.01it/s]
  5%|█▊                                 | 4019/78125 [34:51<10:15:15,  2.01it/s]
  5%|█▊                                 | 4020/78125 [34:52<10:14:35,  2.01it/s]
  5%|█▊                                 | 4021/78125 [34:52<10:14:32,  2.01it/s]
  5%|█▊                                 | 4022/78125 [34:53<10:15:12,  2.01it/s]
  5%|█▊                                 | 4023/78125 [34:53<10:15:02,  2.01it/s]
  5%|█▊                                 | 4024/78125 [34:54<10:15:08,  2.01it/s]
  5%|█▊                                 | 4025/78125 [34:54<10:15:05,  2.01it/s]
  5%|█▊                                 | 4026/78125 [34:55<10:14:42,  2.01it/s]
  5%|█▊                                 | 4027/78125 [34:55<10:14:18,  2.01it/s]
  5%|█▊                                 | 4028/78125 [34:56<10:14:18,  2.01it/s]
  5%|█▊                                 | 4029/78125 [34:56<10:14:26,  2.01it/s]
  5%|█▊                                 | 4030/78125 [34:57<10:13:20,  2.01it/s]
  5%|█▊                                 | 4031/78125 [34:57<10:13:08,  2.01it/s]
  5%|█▊                                 | 4032/78125 [34:58<10:13:58,  2.01it/s]
  5%|█▊                                 | 4033/78125 [34:58<10:13:53,  2.01it/s]
  5%|█▊                                 | 4034/78125 [34:59<10:13:48,  2.01it/s]
  5%|█▊                                 | 4035/78125 [34:59<10:14:14,  2.01it/s]
  5%|█▊                                 | 4036/78125 [35:00<10:14:17,  2.01it/s]
  5%|█▊                                 | 4037/78125 [35:00<10:14:08,  2.01it/s]
  5%|█▊                                 | 4038/78125 [35:01<10:13:35,  2.01it/s]
  5%|█▊                                 | 4039/78125 [35:01<10:14:02,  2.01it/s]
  5%|█▊                                 | 4040/78125 [35:02<10:14:01,  2.01it/s]
  5%|█▊                                 | 4041/78125 [35:02<10:14:20,  2.01it/s]
  5%|█▊                                 | 4042/78125 [35:03<10:14:31,  2.01it/s]
  5%|█▊                                 | 4043/78125 [35:03<10:14:22,  2.01it/s]
  5%|█▊                                 | 4044/78125 [35:04<10:13:10,  2.01it/s]
  5%|█▊                                 | 4045/78125 [35:04<10:13:38,  2.01it/s]
  5%|█▊                                 | 4046/78125 [35:05<10:13:54,  2.01it/s]
  5%|█▊                                 | 4047/78125 [35:05<10:14:20,  2.01it/s]
  5%|█▊                                 | 4048/78125 [35:06<10:14:10,  2.01it/s]
  5%|█▊                                 | 4049/78125 [35:06<10:14:22,  2.01it/s]
  5%|█▊                                 | 4050/78125 [35:07<10:14:09,  2.01it/s]
  5%|█▊                                 | 4051/78125 [35:07<10:13:43,  2.01it/s]
  5%|█▊                                 | 4052/78125 [35:08<10:13:27,  2.01it/s]
  5%|█▊                                 | 4053/78125 [35:08<10:13:40,  2.01it/s]
  5%|█▊                                 | 4054/78125 [35:09<10:13:32,  2.01it/s]
  5%|█▊                                 | 4055/78125 [35:09<10:13:40,  2.01it/s]
  5%|█▊                                 | 4056/78125 [35:10<10:14:11,  2.01it/s]
  5%|█▊                                 | 4057/78125 [35:10<10:13:54,  2.01it/s]
  5%|█▊                                 | 4058/78125 [35:11<10:14:08,  2.01it/s]
  5%|█▊                                 | 4059/78125 [35:11<10:14:27,  2.01it/s]
  5%|█▊                                 | 4060/78125 [35:12<10:14:03,  2.01it/s]
  5%|█▊                                 | 4061/78125 [35:12<10:13:02,  2.01it/s]
  5%|█▊                                 | 4062/78125 [35:13<10:13:03,  2.01it/s]
  5%|█▊                                 | 4063/78125 [35:13<10:13:49,  2.01it/s]
  5%|█▊                                 | 4064/78125 [35:14<10:13:47,  2.01it/s]
  5%|█▊                                 | 4065/78125 [35:14<10:14:11,  2.01it/s]
  5%|█▊                                 | 4066/78125 [35:15<10:14:25,  2.01it/s]
  5%|█▊                                 | 4067/78125 [35:15<10:14:07,  2.01it/s]
  5%|█▊                                 | 4068/78125 [35:16<10:13:48,  2.01it/s]
  5%|█▊                                 | 4069/78125 [35:16<10:13:25,  2.01it/s]
  5%|█▊                                 | 4070/78125 [35:17<10:13:50,  2.01it/s]
  5%|█▊                                 | 4071/78125 [35:17<10:13:26,  2.01it/s]
  5%|█▊                                 | 4072/78125 [35:18<10:14:14,  2.01it/s]
  5%|█▊                                 | 4073/78125 [35:18<10:14:02,  2.01it/s]
  5%|█▊                                 | 4074/78125 [35:19<10:13:54,  2.01it/s]
  5%|█▊                                 | 4075/78125 [35:19<10:13:50,  2.01it/s]
  5%|█▊                                 | 4076/78125 [35:20<10:13:25,  2.01it/s]
  5%|█▊                                 | 4077/78125 [35:20<10:13:22,  2.01it/s]
  5%|█▊                                 | 4078/78125 [35:21<10:12:05,  2.02it/s]
  5%|█▊                                 | 4079/78125 [35:21<10:12:23,  2.02it/s]
  5%|█▊                                 | 4080/78125 [35:22<10:13:30,  2.01it/s]
  5%|█▊                                 | 4081/78125 [35:22<10:14:12,  2.01it/s]
  5%|█▊                                 | 4082/78125 [35:23<10:14:09,  2.01it/s]
  5%|█▊                                 | 4083/78125 [35:23<10:14:06,  2.01it/s]
  5%|█▊                                 | 4084/78125 [35:24<10:13:48,  2.01it/s]
  5%|█▊                                 | 4085/78125 [35:24<10:13:59,  2.01it/s]
  5%|█▊                                 | 4086/78125 [35:25<10:13:56,  2.01it/s]
  5%|█▊                                 | 4087/78125 [35:25<10:13:37,  2.01it/s]
  5%|█▊                                 | 4088/78125 [35:26<10:14:24,  2.01it/s]
  5%|█▊                                 | 4089/78125 [35:26<10:14:03,  2.01it/s]
  5%|█▊                                 | 4090/78125 [35:27<10:14:40,  2.01it/s]
  5%|█▊                                 | 4091/78125 [35:27<10:14:06,  2.01it/s]
  5%|█▊                                 | 4092/78125 [35:28<10:13:21,  2.01it/s]
  5%|█▊                                 | 4093/78125 [35:28<10:12:41,  2.01it/s]
  5%|█▊                                 | 4094/78125 [35:29<10:12:10,  2.02it/s]
  5%|█▊                                 | 4095/78125 [35:29<10:12:45,  2.01it/s]
  5%|█▊                                 | 4096/78125 [35:30<10:13:24,  2.01it/s]
  5%|█▊                                 | 4097/78125 [35:30<10:13:25,  2.01it/s]
  5%|█▊                                 | 4098/78125 [35:31<10:13:52,  2.01it/s]
  5%|█▊                                 | 4099/78125 [35:31<10:13:28,  2.01it/s]
  5%|█▊                                 | 4100/78125 [35:32<10:14:06,  2.01it/s]
  5%|█▊                                 | 4101/78125 [35:32<10:13:49,  2.01it/s]
  5%|█▊                                 | 4102/78125 [35:33<10:14:39,  2.01it/s]
  5%|█▊                                 | 4103/78125 [35:33<10:13:53,  2.01it/s]
  5%|█▊                                 | 4104/78125 [35:34<10:14:18,  2.01it/s]
  5%|█▊                                 | 4105/78125 [35:34<10:14:45,  2.01it/s]
  5%|█▊                                 | 4106/78125 [35:35<10:14:07,  2.01it/s]
  5%|█▊                                 | 4107/78125 [35:35<10:14:04,  2.01it/s]
  5%|█▊                                 | 4108/78125 [35:36<10:15:03,  2.01it/s]
  5%|█▊                                 | 4109/78125 [35:36<10:14:09,  2.01it/s]
  5%|█▊                                 | 4110/78125 [35:37<10:13:20,  2.01it/s]
  5%|█▊                                 | 4111/78125 [35:37<10:14:07,  2.01it/s]
  5%|█▊                                 | 4112/78125 [35:38<10:14:04,  2.01it/s]
  5%|█▊                                 | 4113/78125 [35:38<10:14:03,  2.01it/s]
  5%|█▊                                 | 4114/78125 [35:39<10:14:11,  2.01it/s]
  5%|█▊                                 | 4115/78125 [35:39<10:13:27,  2.01it/s]
  5%|█▊                                 | 4116/78125 [35:39<10:13:55,  2.01it/s]
  5%|█▊                                 | 4117/78125 [35:40<10:13:57,  2.01it/s]
  5%|█▊                                 | 4118/78125 [35:40<10:13:29,  2.01it/s]
  5%|█▊                                 | 4119/78125 [35:41<10:13:31,  2.01it/s]
  5%|█▊                                 | 4120/78125 [35:41<10:13:25,  2.01it/s]
  5%|█▊                                 | 4121/78125 [35:42<10:13:50,  2.01it/s]
  5%|█▊                                 | 4122/78125 [35:42<10:13:11,  2.01it/s]
  5%|█▊                                 | 4123/78125 [35:43<10:13:48,  2.01it/s]
  5%|█▊                                 | 4124/78125 [35:43<10:13:12,  2.01it/s]
  5%|█▊                                 | 4125/78125 [35:44<10:13:42,  2.01it/s]
  5%|█▊                                 | 4126/78125 [35:44<10:13:34,  2.01it/s]
  5%|█▊                                 | 4127/78125 [35:45<10:13:39,  2.01it/s]
  5%|█▊                                 | 4128/78125 [35:45<10:13:17,  2.01it/s]
  5%|█▊                                 | 4129/78125 [35:46<10:13:29,  2.01it/s]
  5%|█▊                                 | 4130/78125 [35:46<10:13:58,  2.01it/s]
  5%|█▊                                 | 4131/78125 [35:47<10:14:25,  2.01it/s]
  5%|█▊                                 | 4132/78125 [35:47<10:13:46,  2.01it/s]
  5%|█▊                                 | 4133/78125 [35:48<10:14:23,  2.01it/s]
  5%|█▊                                 | 4134/78125 [35:48<10:14:24,  2.01it/s]
  5%|█▊                                 | 4135/78125 [35:49<10:13:59,  2.01it/s]
  5%|█▊                                 | 4136/78125 [35:49<10:14:38,  2.01it/s]
  5%|█▊                                 | 4137/78125 [35:50<10:14:36,  2.01it/s]
  5%|█▊                                 | 4138/78125 [35:50<10:14:57,  2.01it/s]
  5%|█▊                                 | 4139/78125 [35:51<10:15:27,  2.00it/s]
  5%|█▊                                 | 4140/78125 [35:51<10:15:13,  2.00it/s]
  5%|█▊                                 | 4141/78125 [35:52<10:14:16,  2.01it/s]
  5%|█▊                                 | 4142/78125 [35:52<10:14:45,  2.01it/s]
  5%|█▊                                 | 4143/78125 [35:53<10:14:42,  2.01it/s]
  5%|█▊                                 | 4144/78125 [35:53<10:14:50,  2.01it/s]
  5%|█▊                                 | 4145/78125 [35:54<10:14:37,  2.01it/s]
  5%|█▊                                 | 4146/78125 [35:54<10:15:02,  2.00it/s]
  5%|█▊                                 | 4147/78125 [35:55<10:14:57,  2.00it/s]
  5%|█▊                                 | 4148/78125 [35:55<10:15:08,  2.00it/s]
  5%|█▊                                 | 4149/78125 [35:56<10:14:51,  2.01it/s]
  5%|█▊                                 | 4150/78125 [35:56<10:15:00,  2.00it/s]
  5%|█▊                                 | 4151/78125 [35:57<10:14:53,  2.01it/s]
  5%|█▊                                 | 4152/78125 [35:57<10:14:12,  2.01it/s]
  5%|█▊                                 | 4153/78125 [35:58<10:15:04,  2.00it/s]
  5%|█▊                                 | 4154/78125 [35:58<10:15:32,  2.00it/s]
  5%|█▊                                 | 4155/78125 [35:59<10:14:54,  2.00it/s]
  5%|█▊                                 | 4156/78125 [35:59<10:15:03,  2.00it/s]
  5%|█▊                                 | 4157/78125 [36:00<10:14:47,  2.01it/s]
  5%|█▊                                 | 4158/78125 [36:00<10:14:47,  2.01it/s]
  5%|█▊                                 | 4159/78125 [36:01<10:15:08,  2.00it/s]
  5%|█▊                                 | 4160/78125 [36:01<10:15:15,  2.00it/s]
  5%|█▊                                 | 4161/78125 [36:02<10:15:20,  2.00it/s]
  5%|█▊                                 | 4162/78125 [36:02<10:14:45,  2.01it/s]
  5%|█▊                                 | 4163/78125 [36:03<10:13:31,  2.01it/s]
  5%|█▊                                 | 4164/78125 [36:03<10:13:55,  2.01it/s]
  5%|█▊                                 | 4165/78125 [36:04<10:13:33,  2.01it/s]
  5%|█▊                                 | 4166/78125 [36:04<10:13:49,  2.01it/s]
  5%|█▊                                 | 4167/78125 [36:05<10:13:38,  2.01it/s]
  5%|█▊                                 | 4168/78125 [36:05<10:14:02,  2.01it/s]
  5%|█▊                                 | 4169/78125 [36:06<10:13:43,  2.01it/s]
  5%|█▊                                 | 4170/78125 [36:06<10:13:29,  2.01it/s]
  5%|█▊                                 | 4171/78125 [36:07<10:13:48,  2.01it/s]
  5%|█▊                                 | 4172/78125 [36:07<10:14:10,  2.01it/s]
  5%|█▊                                 | 4173/78125 [36:08<10:13:17,  2.01it/s]
  5%|█▊                                 | 4174/78125 [36:08<10:13:09,  2.01it/s]
  5%|█▊                                 | 4175/78125 [36:09<10:13:45,  2.01it/s]
  5%|█▊                                 | 4176/78125 [36:09<10:12:55,  2.01it/s]
  5%|█▊                                 | 4177/78125 [36:10<10:13:20,  2.01it/s]
  5%|█▊                                 | 4178/78125 [36:10<10:13:51,  2.01it/s]
  5%|█▊                                 | 4179/78125 [36:11<10:14:12,  2.01it/s]
  5%|█▊                                 | 4180/78125 [36:11<10:14:17,  2.01it/s]
  5%|█▊                                 | 4181/78125 [36:12<10:13:22,  2.01it/s]
  5%|█▊                                 | 4182/78125 [36:12<10:13:30,  2.01it/s]
  5%|█▊                                 | 4183/78125 [36:13<10:12:55,  2.01it/s]
  5%|█▊                                 | 4184/78125 [36:13<10:12:25,  2.01it/s]
  5%|█▊                                 | 4185/78125 [36:14<10:12:52,  2.01it/s]
  5%|█▉                                 | 4186/78125 [36:14<10:13:30,  2.01it/s]
  5%|█▉                                 | 4187/78125 [36:15<10:12:48,  2.01it/s]
  5%|█▉                                 | 4188/78125 [36:15<10:13:10,  2.01it/s]
  5%|█▉                                 | 4189/78125 [36:16<10:12:54,  2.01it/s]
  5%|█▉                                 | 4190/78125 [36:16<10:13:03,  2.01it/s]
  5%|█▉                                 | 4191/78125 [36:17<10:12:48,  2.01it/s]
  5%|█▉                                 | 4192/78125 [36:17<10:13:35,  2.01it/s]
  5%|█▉                                 | 4193/78125 [36:18<10:13:24,  2.01it/s]
  5%|█▉                                 | 4194/78125 [36:18<10:13:10,  2.01it/s]
  5%|█▉                                 | 4195/78125 [36:19<10:13:30,  2.01it/s]
  5%|█▉                                 | 4196/78125 [36:19<10:12:44,  2.01it/s]
  5%|█▉                                 | 4197/78125 [36:20<10:13:11,  2.01it/s]
  5%|█▉                                 | 4198/78125 [36:20<10:14:10,  2.01it/s]
  5%|█▉                                 | 4199/78125 [36:21<10:13:43,  2.01it/s]
  5%|█▉                                 | 4200/78125 [36:21<10:13:16,  2.01it/s]
  5%|█▉                                 | 4201/78125 [36:22<10:14:01,  2.01it/s]
  5%|█▉                                 | 4202/78125 [36:22<10:14:09,  2.01it/s]
  5%|█▉                                 | 4203/78125 [36:23<10:14:30,  2.00it/s]
  5%|█▉                                 | 4204/78125 [36:23<10:14:43,  2.00it/s]
  5%|█▉                                 | 4205/78125 [36:24<10:14:13,  2.01it/s]
  5%|█▉                                 | 4206/78125 [36:24<10:14:43,  2.00it/s]
  5%|█▉                                 | 4207/78125 [36:25<10:14:30,  2.00it/s]
  5%|█▉                                 | 4208/78125 [36:25<10:15:41,  2.00it/s]
  5%|█▉                                 | 4209/78125 [36:26<10:15:12,  2.00it/s]
  5%|█▉                                 | 4210/78125 [36:26<10:14:20,  2.01it/s]
  5%|█▉                                 | 4211/78125 [36:27<10:13:52,  2.01it/s]
  5%|█▉                                 | 4212/78125 [36:27<10:13:22,  2.01it/s]
  5%|█▉                                 | 4213/78125 [36:28<10:12:54,  2.01it/s]
  5%|█▉                                 | 4214/78125 [36:28<10:12:55,  2.01it/s]
  5%|█▉                                 | 4215/78125 [36:29<10:12:41,  2.01it/s]
  5%|█▉                                 | 4216/78125 [36:29<10:13:55,  2.01it/s]
  5%|█▉                                 | 4217/78125 [36:30<10:13:33,  2.01it/s]
  5%|█▉                                 | 4218/78125 [36:30<10:13:25,  2.01it/s]
  5%|█▉                                 | 4219/78125 [36:31<10:13:33,  2.01it/s]
  5%|█▉                                 | 4220/78125 [36:31<10:13:46,  2.01it/s]
  5%|█▉                                 | 4221/78125 [36:32<10:13:57,  2.01it/s]
  5%|█▉                                 | 4222/78125 [36:32<10:13:39,  2.01it/s]
  5%|█▉                                 | 4223/78125 [36:33<10:13:56,  2.01it/s]
  5%|█▉                                 | 4224/78125 [36:33<10:13:18,  2.01it/s]
  5%|█▉                                 | 4225/78125 [36:34<10:13:29,  2.01it/s]
  5%|█▉                                 | 4226/78125 [36:34<10:13:20,  2.01it/s]
  5%|█▉                                 | 4227/78125 [36:35<10:12:37,  2.01it/s]
  5%|█▉                                 | 4228/78125 [36:35<10:12:52,  2.01it/s]
  5%|█▉                                 | 4229/78125 [36:36<10:13:25,  2.01it/s]
  5%|█▉                                 | 4230/78125 [36:36<10:13:41,  2.01it/s]
  5%|█▉                                 | 4231/78125 [36:37<10:13:34,  2.01it/s]
  5%|█▉                                 | 4232/78125 [36:37<10:13:36,  2.01it/s]
  5%|█▉                                 | 4233/78125 [36:38<10:13:42,  2.01it/s]
  5%|█▉                                 | 4234/78125 [36:38<10:13:59,  2.01it/s]
  5%|█▉                                 | 4235/78125 [36:39<10:13:15,  2.01it/s]
  5%|█▉                                 | 4236/78125 [36:39<10:14:11,  2.01it/s]
  5%|█▉                                 | 4237/78125 [36:40<10:13:52,  2.01it/s]
  5%|█▉                                 | 4238/78125 [36:40<10:14:17,  2.00it/s]
  5%|█▉                                 | 4239/78125 [36:41<10:13:15,  2.01it/s]
  5%|█▉                                 | 4240/78125 [36:41<10:13:27,  2.01it/s]
  5%|█▉                                 | 4241/78125 [36:42<10:13:29,  2.01it/s]
  5%|█▉                                 | 4242/78125 [36:42<10:13:51,  2.01it/s]
  5%|█▉                                 | 4243/78125 [36:43<10:13:29,  2.01it/s]
  5%|█▉                                 | 4244/78125 [36:43<10:13:18,  2.01it/s]
  5%|█▉                                 | 4245/78125 [36:44<10:13:16,  2.01it/s]
  5%|█▉                                 | 4246/78125 [36:44<10:13:20,  2.01it/s]
  5%|█▉                                 | 4247/78125 [36:45<10:13:32,  2.01it/s]
  5%|█▉                                 | 4248/78125 [36:45<10:13:15,  2.01it/s]
  5%|█▉                                 | 4249/78125 [36:46<10:12:53,  2.01it/s]
  5%|█▉                                 | 4250/78125 [36:46<10:13:13,  2.01it/s]
  5%|█▉                                 | 4251/78125 [36:47<10:12:24,  2.01it/s]
  5%|█▉                                 | 4252/78125 [36:47<10:13:14,  2.01it/s]
  5%|█▉                                 | 4253/78125 [36:48<10:13:04,  2.01it/s]
  5%|█▉                                 | 4254/78125 [36:48<10:12:29,  2.01it/s]
  5%|█▉                                 | 4255/78125 [36:49<10:11:38,  2.01it/s]
  5%|█▉                                 | 4256/78125 [36:49<10:12:15,  2.01it/s]
  5%|█▉                                 | 4257/78125 [36:50<10:12:17,  2.01it/s]
  5%|█▉                                 | 4258/78125 [36:50<10:12:43,  2.01it/s]
  5%|█▉                                 | 4259/78125 [36:51<10:12:16,  2.01it/s]
  5%|█▉                                 | 4260/78125 [36:51<10:13:19,  2.01it/s]
  5%|█▉                                 | 4261/78125 [36:52<10:12:15,  2.01it/s]
  5%|█▉                                 | 4262/78125 [36:52<10:12:13,  2.01it/s]
  5%|█▉                                 | 4263/78125 [36:53<10:12:15,  2.01it/s]
  5%|█▉                                 | 4264/78125 [36:53<10:12:47,  2.01it/s]
  5%|█▉                                 | 4265/78125 [36:54<10:13:26,  2.01it/s]
  5%|█▉                                 | 4266/78125 [36:54<10:14:01,  2.00it/s]
  5%|█▉                                 | 4267/78125 [36:55<10:13:38,  2.01it/s]
  5%|█▉                                 | 4268/78125 [36:55<10:13:42,  2.01it/s]
  5%|█▉                                 | 4269/78125 [36:56<10:12:27,  2.01it/s]
  5%|█▉                                 | 4270/78125 [36:56<10:11:45,  2.01it/s]
  5%|█▉                                 | 4271/78125 [36:57<10:12:16,  2.01it/s]
  5%|█▉                                 | 4272/78125 [36:57<10:12:37,  2.01it/s]
  5%|█▉                                 | 4273/78125 [36:58<10:13:00,  2.01it/s]
  5%|█▉                                 | 4274/78125 [36:58<10:12:56,  2.01it/s]
  5%|█▉                                 | 4275/78125 [36:59<10:13:06,  2.01it/s]
  5%|█▉                                 | 4276/78125 [36:59<10:12:49,  2.01it/s]
  5%|█▉                                 | 4277/78125 [37:00<10:13:11,  2.01it/s]
  5%|█▉                                 | 4278/78125 [37:00<10:12:55,  2.01it/s]
  5%|█▉                                 | 4279/78125 [37:01<10:13:45,  2.01it/s]
  5%|█▉                                 | 4280/78125 [37:01<10:13:48,  2.01it/s]
  5%|█▉                                 | 4281/78125 [37:02<10:13:29,  2.01it/s]
  5%|█▉                                 | 4282/78125 [37:02<10:13:30,  2.01it/s]
  5%|█▉                                 | 4283/78125 [37:03<10:13:47,  2.01it/s]
  5%|█▉                                 | 4284/78125 [37:03<10:13:04,  2.01it/s]
  5%|█▉                                 | 4285/78125 [37:04<10:13:06,  2.01it/s]
  5%|█▉                                 | 4286/78125 [37:04<10:12:49,  2.01it/s]
  5%|█▉                                 | 4287/78125 [37:05<10:13:01,  2.01it/s]
  5%|█▉                                 | 4288/78125 [37:05<10:13:07,  2.01it/s]
  5%|█▉                                 | 4289/78125 [37:06<10:13:16,  2.01it/s]
  5%|█▉                                 | 4290/78125 [37:06<10:12:34,  2.01it/s]
  5%|█▉                                 | 4291/78125 [37:07<10:12:55,  2.01it/s]
  5%|█▉                                 | 4292/78125 [37:07<10:13:32,  2.01it/s]
  5%|█▉                                 | 4293/78125 [37:08<10:13:48,  2.00it/s]
  5%|█▉                                 | 4294/78125 [37:08<10:14:22,  2.00it/s]
  5%|█▉                                 | 4295/78125 [37:09<10:14:06,  2.00it/s]
  5%|█▉                                 | 4296/78125 [37:09<10:13:46,  2.00it/s]
  6%|█▉                                 | 4297/78125 [37:10<10:13:56,  2.00it/s]
  6%|█▉                                 | 4298/78125 [37:10<10:14:19,  2.00it/s]
  6%|█▉                                 | 4299/78125 [37:11<10:13:52,  2.00it/s]
  6%|█▉                                 | 4300/78125 [37:11<10:13:21,  2.01it/s]
  6%|█▉                                 | 4301/78125 [37:12<10:13:34,  2.01it/s]
  6%|█▉                                 | 4302/78125 [37:12<10:13:50,  2.00it/s]
  6%|█▉                                 | 4303/78125 [37:13<10:13:39,  2.00it/s]
  6%|█▉                                 | 4304/78125 [37:13<10:13:32,  2.01it/s]
  6%|█▉                                 | 4305/78125 [37:14<10:13:31,  2.01it/s]
  6%|█▉                                 | 4306/78125 [37:14<10:13:10,  2.01it/s]
  6%|█▉                                 | 4307/78125 [37:15<10:13:34,  2.01it/s]
  6%|█▉                                 | 4308/78125 [37:15<10:13:50,  2.00it/s]
  6%|█▉                                 | 4309/78125 [37:16<10:13:00,  2.01it/s]
  6%|█▉                                 | 4310/78125 [37:16<10:13:09,  2.01it/s]
  6%|█▉                                 | 4311/78125 [37:17<10:12:59,  2.01it/s]
  6%|█▉                                 | 4312/78125 [37:17<10:13:29,  2.01it/s]
  6%|█▉                                 | 4313/78125 [37:18<10:13:23,  2.01it/s]
  6%|█▉                                 | 4314/78125 [37:18<10:12:51,  2.01it/s]
  6%|█▉                                 | 4315/78125 [37:19<10:12:55,  2.01it/s]
  6%|█▉                                 | 4316/78125 [37:19<10:12:56,  2.01it/s]
  6%|█▉                                 | 4317/78125 [37:20<10:13:05,  2.01it/s]
  6%|█▉                                 | 4318/78125 [37:20<10:13:28,  2.01it/s]
  6%|█▉                                 | 4319/78125 [37:21<10:13:26,  2.01it/s]
  6%|█▉                                 | 4320/78125 [37:21<10:13:35,  2.00it/s]
  6%|█▉                                 | 4321/78125 [37:22<10:12:51,  2.01it/s]
  6%|█▉                                 | 4322/78125 [37:22<10:12:29,  2.01it/s]
  6%|█▉                                 | 4323/78125 [37:23<10:12:00,  2.01it/s]
  6%|█▉                                 | 4324/78125 [37:23<10:11:31,  2.01it/s]
  6%|█▉                                 | 4325/78125 [37:24<10:11:42,  2.01it/s]
  6%|█▉                                 | 4326/78125 [37:24<10:12:19,  2.01it/s]
  6%|█▉                                 | 4327/78125 [37:25<10:11:41,  2.01it/s]
  6%|█▉                                 | 4328/78125 [37:25<10:12:49,  2.01it/s]
  6%|█▉                                 | 4329/78125 [37:26<10:12:23,  2.01it/s]
  6%|█▉                                 | 4330/78125 [37:26<10:12:29,  2.01it/s]
  6%|█▉                                 | 4331/78125 [37:27<10:13:24,  2.01it/s]
  6%|█▉                                 | 4332/78125 [37:27<10:13:02,  2.01it/s]
  6%|█▉                                 | 4333/78125 [37:28<10:12:43,  2.01it/s]
  6%|█▉                                 | 4334/78125 [37:28<10:12:13,  2.01it/s]
  6%|█▉                                 | 4335/78125 [37:29<10:12:09,  2.01it/s]
  6%|█▉                                 | 4336/78125 [37:29<10:12:24,  2.01it/s]
  6%|█▉                                 | 4337/78125 [37:30<10:12:15,  2.01it/s]
  6%|█▉                                 | 4338/78125 [37:30<10:12:25,  2.01it/s]
  6%|█▉                                 | 4339/78125 [37:31<10:12:13,  2.01it/s]
  6%|█▉                                 | 4340/78125 [37:31<10:12:26,  2.01it/s]
  6%|█▉                                 | 4341/78125 [37:32<10:12:19,  2.01it/s]
  6%|█▉                                 | 4342/78125 [37:32<10:12:01,  2.01it/s]
  6%|█▉                                 | 4343/78125 [37:33<10:11:17,  2.01it/s]
  6%|█▉                                 | 4344/78125 [37:33<10:11:39,  2.01it/s]
  6%|█▉                                 | 4345/78125 [37:34<10:11:03,  2.01it/s]
  6%|█▉                                 | 4346/78125 [37:34<10:11:28,  2.01it/s]
  6%|█▉                                 | 4347/78125 [37:35<10:12:17,  2.01it/s]
  6%|█▉                                 | 4348/78125 [37:35<10:12:19,  2.01it/s]
  6%|█▉                                 | 4349/78125 [37:36<10:11:14,  2.01it/s]
  6%|█▉                                 | 4350/78125 [37:36<10:11:13,  2.01it/s]
  6%|█▉                                 | 4351/78125 [37:37<10:11:25,  2.01it/s]
  6%|█▉                                 | 4352/78125 [37:37<10:11:26,  2.01it/s]
  6%|█▉                                 | 4353/78125 [37:38<10:11:43,  2.01it/s]
  6%|█▉                                 | 4354/78125 [37:38<10:12:32,  2.01it/s]
  6%|█▉                                 | 4355/78125 [37:39<10:11:50,  2.01it/s]
  6%|█▉                                 | 4356/78125 [37:39<10:11:25,  2.01it/s]
  6%|█▉                                 | 4357/78125 [37:40<10:11:45,  2.01it/s]
  6%|█▉                                 | 4358/78125 [37:40<10:12:23,  2.01it/s]
  6%|█▉                                 | 4359/78125 [37:41<10:11:39,  2.01it/s]
  6%|█▉                                 | 4360/78125 [37:41<10:12:32,  2.01it/s]
  6%|█▉                                 | 4361/78125 [37:42<10:11:51,  2.01it/s]
  6%|█▉                                 | 4362/78125 [37:42<10:12:02,  2.01it/s]
  6%|█▉                                 | 4363/78125 [37:43<10:11:42,  2.01it/s]
  6%|█▉                                 | 4364/78125 [37:43<10:11:50,  2.01it/s]
  6%|█▉                                 | 4365/78125 [37:44<10:10:58,  2.01it/s]
  6%|█▉                                 | 4366/78125 [37:44<10:11:26,  2.01it/s]
  6%|█▉                                 | 4367/78125 [37:45<10:10:49,  2.01it/s]
  6%|█▉                                 | 4368/78125 [37:45<10:11:08,  2.01it/s]
  6%|█▉                                 | 4369/78125 [37:46<10:10:19,  2.01it/s]
  6%|█▉                                 | 4370/78125 [37:46<10:10:21,  2.01it/s]
  6%|█▉                                 | 4371/78125 [37:47<10:11:15,  2.01it/s]
  6%|█▉                                 | 4372/78125 [37:47<10:10:45,  2.01it/s]
  6%|█▉                                 | 4373/78125 [37:47<10:11:20,  2.01it/s]
  6%|█▉                                 | 4374/78125 [37:48<10:10:38,  2.01it/s]
  6%|█▉                                 | 4375/78125 [37:48<10:10:46,  2.01it/s]
  6%|█▉                                 | 4376/78125 [37:49<10:11:07,  2.01it/s]
  6%|█▉                                 | 4377/78125 [37:49<10:11:22,  2.01it/s]
  6%|█▉                                 | 4378/78125 [37:50<10:11:38,  2.01it/s]
  6%|█▉                                 | 4379/78125 [37:50<10:11:22,  2.01it/s]
  6%|█▉                                 | 4380/78125 [37:51<10:12:01,  2.01it/s]
  6%|█▉                                 | 4381/78125 [37:51<10:11:46,  2.01it/s]
  6%|█▉                                 | 4382/78125 [37:52<10:11:25,  2.01it/s]
  6%|█▉                                 | 4383/78125 [37:52<10:11:24,  2.01it/s]
  6%|█▉                                 | 4384/78125 [37:53<10:11:09,  2.01it/s]
  6%|█▉                                 | 4385/78125 [37:53<10:10:32,  2.01it/s]
  6%|█▉                                 | 4386/78125 [37:54<10:10:35,  2.01it/s]
  6%|█▉                                 | 4387/78125 [37:54<10:10:45,  2.01it/s]
  6%|█▉                                 | 4388/78125 [37:55<10:09:53,  2.02it/s]
  6%|█▉                                 | 4389/78125 [37:55<10:09:46,  2.02it/s]
  6%|█▉                                 | 4390/78125 [37:56<10:09:34,  2.02it/s]
  6%|█▉                                 | 4391/78125 [37:56<10:09:53,  2.01it/s]
  6%|█▉                                 | 4392/78125 [37:57<10:09:42,  2.02it/s]
  6%|█▉                                 | 4393/78125 [37:57<10:09:15,  2.02it/s]
  6%|█▉                                 | 4394/78125 [37:58<10:09:44,  2.02it/s]
  6%|█▉                                 | 4395/78125 [37:58<10:09:43,  2.02it/s]
  6%|█▉                                 | 4396/78125 [37:59<10:10:48,  2.01it/s]
  6%|█▉                                 | 4397/78125 [37:59<10:10:49,  2.01it/s]
  6%|█▉                                 | 4398/78125 [38:00<10:10:49,  2.01it/s]
  6%|█▉                                 | 4399/78125 [38:00<10:10:41,  2.01it/s]
  6%|█▉                                 | 4400/78125 [38:01<10:11:08,  2.01it/s]
  6%|█▉                                 | 4401/78125 [38:01<10:10:40,  2.01it/s]
  6%|█▉                                 | 4402/78125 [38:02<10:10:10,  2.01it/s]
  6%|█▉                                 | 4403/78125 [38:02<10:10:26,  2.01it/s]
  6%|█▉                                 | 4404/78125 [38:03<10:09:50,  2.01it/s]
  6%|█▉                                 | 4405/78125 [38:03<10:09:31,  2.02it/s]
  6%|█▉                                 | 4406/78125 [38:04<10:09:33,  2.02it/s]
  6%|█▉                                 | 4407/78125 [38:04<10:10:03,  2.01it/s]
  6%|█▉                                 | 4408/78125 [38:05<10:10:02,  2.01it/s]
  6%|█▉                                 | 4409/78125 [38:05<10:09:34,  2.02it/s]
  6%|█▉                                 | 4410/78125 [38:06<10:10:43,  2.01it/s]
  6%|█▉                                 | 4411/78125 [38:06<10:09:37,  2.02it/s]
  6%|█▉                                 | 4412/78125 [38:07<10:09:07,  2.02it/s]
  6%|█▉                                 | 4413/78125 [38:07<10:08:20,  2.02it/s]
  6%|█▉                                 | 4414/78125 [38:08<10:08:50,  2.02it/s]
  6%|█▉                                 | 4415/78125 [38:08<10:09:07,  2.02it/s]
  6%|█▉                                 | 4416/78125 [38:09<10:09:31,  2.02it/s]
  6%|█▉                                 | 4417/78125 [38:09<10:09:39,  2.02it/s]
  6%|█▉                                 | 4418/78125 [38:10<10:09:00,  2.02it/s]
  6%|█▉                                 | 4419/78125 [38:10<10:09:12,  2.02it/s]
  6%|█▉                                 | 4420/78125 [38:11<10:09:06,  2.02it/s]
  6%|█▉                                 | 4421/78125 [38:11<10:08:41,  2.02it/s]
  6%|█▉                                 | 4422/78125 [38:12<10:09:14,  2.02it/s]
  6%|█▉                                 | 4423/78125 [38:12<10:09:02,  2.02it/s]
  6%|█▉                                 | 4424/78125 [38:13<10:08:49,  2.02it/s]
  6%|█▉                                 | 4425/78125 [38:13<10:08:34,  2.02it/s]
  6%|█▉                                 | 4426/78125 [38:14<10:08:31,  2.02it/s]
  6%|█▉                                 | 4427/78125 [38:14<10:08:06,  2.02it/s]
  6%|█▉                                 | 4428/78125 [38:15<10:08:34,  2.02it/s]
  6%|█▉                                 | 4429/78125 [38:15<10:08:46,  2.02it/s]
  6%|█▉                                 | 4430/78125 [38:16<10:09:22,  2.02it/s]
  6%|█▉                                 | 4431/78125 [38:16<10:09:11,  2.02it/s]
  6%|█▉                                 | 4432/78125 [38:17<10:09:06,  2.02it/s]
  6%|█▉                                 | 4433/78125 [38:17<10:09:50,  2.01it/s]
  6%|█▉                                 | 4434/78125 [38:18<10:09:16,  2.02it/s]
  6%|█▉                                 | 4435/78125 [38:18<10:09:17,  2.02it/s]
  6%|█▉                                 | 4436/78125 [38:19<10:08:55,  2.02it/s]
  6%|█▉                                 | 4437/78125 [38:19<10:08:39,  2.02it/s]
  6%|█▉                                 | 4438/78125 [38:20<10:09:19,  2.02it/s]
  6%|█▉                                 | 4439/78125 [38:20<10:08:35,  2.02it/s]
  6%|█▉                                 | 4440/78125 [38:21<10:09:29,  2.01it/s]
  6%|█▉                                 | 4441/78125 [38:21<10:09:28,  2.01it/s]
  6%|█▉                                 | 4442/78125 [38:22<10:09:20,  2.02it/s]
  6%|█▉                                 | 4443/78125 [38:22<10:08:54,  2.02it/s]
  6%|█▉                                 | 4444/78125 [38:23<10:08:56,  2.02it/s]
  6%|█▉                                 | 4445/78125 [38:23<10:09:00,  2.02it/s]
  6%|█▉                                 | 4446/78125 [38:24<10:09:08,  2.02it/s]
  6%|█▉                                 | 4447/78125 [38:24<10:09:40,  2.01it/s]
  6%|█▉                                 | 4448/78125 [38:25<10:10:06,  2.01it/s]
  6%|█▉                                 | 4449/78125 [38:25<10:09:41,  2.01it/s]
  6%|█▉                                 | 4450/78125 [38:26<10:08:03,  2.02it/s]
  6%|█▉                                 | 4451/78125 [38:26<10:08:52,  2.02it/s]
  6%|█▉                                 | 4452/78125 [38:27<10:08:31,  2.02it/s]
  6%|█▉                                 | 4453/78125 [38:27<10:08:18,  2.02it/s]
  6%|█▉                                 | 4454/78125 [38:28<10:08:44,  2.02it/s]
  6%|█▉                                 | 4455/78125 [38:28<10:08:24,  2.02it/s]
  6%|█▉                                 | 4456/78125 [38:29<10:08:43,  2.02it/s]
  6%|█▉                                 | 4457/78125 [38:29<10:08:00,  2.02it/s]
  6%|█▉                                 | 4458/78125 [38:30<10:08:08,  2.02it/s]
  6%|█▉                                 | 4459/78125 [38:30<10:08:18,  2.02it/s]
  6%|█▉                                 | 4460/78125 [38:31<10:09:07,  2.02it/s]
  6%|█▉                                 | 4461/78125 [38:31<10:08:47,  2.02it/s]
  6%|█▉                                 | 4462/78125 [38:32<10:08:48,  2.02it/s]
  6%|█▉                                 | 4463/78125 [38:32<10:09:01,  2.02it/s]
  6%|█▉                                 | 4464/78125 [38:33<10:09:41,  2.01it/s]
  6%|██                                 | 4465/78125 [38:33<10:10:06,  2.01it/s]
  6%|██                                 | 4466/78125 [38:34<10:10:16,  2.01it/s]
  6%|██                                 | 4467/78125 [38:34<10:09:08,  2.02it/s]
  6%|██                                 | 4468/78125 [38:35<10:09:19,  2.01it/s]
  6%|██                                 | 4469/78125 [38:35<10:09:25,  2.01it/s]
  6%|██                                 | 4470/78125 [38:36<10:09:55,  2.01it/s]
  6%|██                                 | 4471/78125 [38:36<10:09:28,  2.01it/s]
  6%|██                                 | 4472/78125 [38:37<10:09:09,  2.02it/s]
  6%|██                                 | 4473/78125 [38:37<10:09:12,  2.01it/s]
  6%|██                                 | 4474/78125 [38:38<10:09:05,  2.02it/s]
  6%|██                                 | 4475/78125 [38:38<10:08:26,  2.02it/s]
  6%|██                                 | 4476/78125 [38:39<10:08:03,  2.02it/s]
  6%|██                                 | 4477/78125 [38:39<10:08:56,  2.02it/s]
  6%|██                                 | 4478/78125 [38:40<10:09:08,  2.02it/s]
  6%|██                                 | 4479/78125 [38:40<10:09:10,  2.01it/s]
  6%|██                                 | 4480/78125 [38:41<10:09:13,  2.01it/s]
  6%|██                                 | 4481/78125 [38:41<10:09:02,  2.02it/s]
  6%|██                                 | 4482/78125 [38:42<10:08:50,  2.02it/s]
  6%|██                                 | 4483/78125 [38:42<10:08:56,  2.02it/s]
  6%|██                                 | 4484/78125 [38:43<10:09:05,  2.02it/s]
  6%|██                                 | 4485/78125 [38:43<10:08:45,  2.02it/s]
  6%|██                                 | 4486/78125 [38:44<10:08:42,  2.02it/s]
  6%|██                                 | 4487/78125 [38:44<10:08:24,  2.02it/s]
  6%|██                                 | 4488/78125 [38:45<10:07:40,  2.02it/s]
  6%|██                                 | 4489/78125 [38:45<10:08:58,  2.02it/s]
  6%|██                                 | 4490/78125 [38:46<10:09:05,  2.01it/s]
  6%|██                                 | 4491/78125 [38:46<10:08:08,  2.02it/s]
  6%|██                                 | 4492/78125 [38:47<10:07:26,  2.02it/s]
  6%|██                                 | 4493/78125 [38:47<10:07:34,  2.02it/s]
  6%|██                                 | 4494/78125 [38:48<10:07:44,  2.02it/s]
  6%|██                                 | 4495/78125 [38:48<10:08:19,  2.02it/s]
  6%|██                                 | 4496/78125 [38:49<10:08:29,  2.02it/s]
  6%|██                                 | 4497/78125 [38:49<10:08:14,  2.02it/s]
  6%|██                                 | 4498/78125 [38:50<10:07:35,  2.02it/s]
  6%|██                                 | 4499/78125 [38:50<10:07:18,  2.02it/s]
  6%|██                                 | 4500/78125 [38:51<10:07:59,  2.02it/s]
  6%|██                                 | 4501/78125 [38:51<10:07:30,  2.02it/s]
  6%|██                                 | 4502/78125 [38:51<10:07:27,  2.02it/s]
  6%|██                                 | 4503/78125 [38:52<10:08:33,  2.02it/s]
  6%|██                                 | 4504/78125 [38:52<10:08:01,  2.02it/s]
  6%|██                                 | 4505/78125 [38:53<10:08:05,  2.02it/s]
  6%|██                                 | 4506/78125 [38:53<10:08:02,  2.02it/s]
  6%|██                                 | 4507/78125 [38:54<10:08:09,  2.02it/s]
  6%|██                                 | 4508/78125 [38:54<10:08:00,  2.02it/s]
  6%|██                                 | 4509/78125 [38:55<10:07:16,  2.02it/s]
  6%|██                                 | 4510/78125 [38:55<10:07:14,  2.02it/s]
  6%|██                                 | 4511/78125 [38:56<10:07:59,  2.02it/s]
  6%|██                                 | 4512/78125 [38:56<10:08:11,  2.02it/s]
  6%|██                                 | 4513/78125 [38:57<10:08:25,  2.02it/s]
  6%|██                                 | 4514/78125 [38:57<10:07:50,  2.02it/s]
  6%|██                                 | 4515/78125 [38:58<10:07:49,  2.02it/s]
  6%|██                                 | 4516/78125 [38:58<10:08:20,  2.02it/s]
  6%|██                                 | 4517/78125 [38:59<10:08:04,  2.02it/s]
  6%|██                                 | 4518/78125 [38:59<10:07:21,  2.02it/s]
  6%|██                                 | 4519/78125 [39:00<10:07:21,  2.02it/s]
  6%|██                                 | 4520/78125 [39:00<10:07:13,  2.02it/s]
  6%|██                                 | 4521/78125 [39:01<10:07:18,  2.02it/s]
  6%|██                                 | 4522/78125 [39:01<10:06:34,  2.02it/s]
  6%|██                                 | 4523/78125 [39:02<10:06:30,  2.02it/s]
  6%|██                                 | 4524/78125 [39:02<10:07:23,  2.02it/s]
  6%|██                                 | 4525/78125 [39:03<10:07:36,  2.02it/s]
  6%|██                                 | 4526/78125 [39:03<10:07:33,  2.02it/s]
  6%|██                                 | 4527/78125 [39:04<10:08:14,  2.02it/s]
  6%|██                                 | 4528/78125 [39:04<10:08:15,  2.02it/s]
  6%|██                                 | 4529/78125 [39:05<10:07:19,  2.02it/s]
  6%|██                                 | 4530/78125 [39:05<10:06:27,  2.02it/s]
  6%|██                                 | 4531/78125 [39:06<10:06:51,  2.02it/s]
  6%|██                                 | 4532/78125 [39:06<10:06:20,  2.02it/s]
  6%|██                                 | 4533/78125 [39:07<10:06:21,  2.02it/s]
  6%|██                                 | 4534/78125 [39:07<10:07:02,  2.02it/s]
  6%|██                                 | 4535/78125 [39:08<10:07:19,  2.02it/s]
  6%|██                                 | 4536/78125 [39:08<10:06:58,  2.02it/s]
  6%|██                                 | 4537/78125 [39:09<10:06:09,  2.02it/s]
  6%|██                                 | 4538/78125 [39:09<10:05:55,  2.02it/s]
  6%|██                                 | 4539/78125 [39:10<10:06:39,  2.02it/s]
  6%|██                                 | 4540/78125 [39:10<10:06:47,  2.02it/s]
  6%|██                                 | 4541/78125 [39:11<10:06:43,  2.02it/s]
  6%|██                                 | 4542/78125 [39:11<10:07:12,  2.02it/s]
  6%|██                                 | 4543/78125 [39:12<10:07:29,  2.02it/s]
  6%|██                                 | 4544/78125 [39:12<10:07:38,  2.02it/s]
  6%|██                                 | 4545/78125 [39:13<10:07:53,  2.02it/s]
  6%|██                                 | 4546/78125 [39:13<10:07:39,  2.02it/s]
  6%|██                                 | 4547/78125 [39:14<10:07:52,  2.02it/s]
  6%|██                                 | 4548/78125 [39:14<10:08:02,  2.02it/s]
  6%|██                                 | 4549/78125 [39:15<10:08:18,  2.02it/s]
  6%|██                                 | 4550/78125 [39:15<10:08:16,  2.02it/s]
  6%|██                                 | 4551/78125 [39:16<10:07:27,  2.02it/s]
  6%|██                                 | 4552/78125 [39:16<10:07:58,  2.02it/s]
  6%|██                                 | 4553/78125 [39:17<10:08:07,  2.02it/s]
  6%|██                                 | 4554/78125 [39:17<10:08:11,  2.02it/s]
  6%|██                                 | 4555/78125 [39:18<10:08:07,  2.02it/s]
  6%|██                                 | 4556/78125 [39:18<10:08:17,  2.02it/s]
  6%|██                                 | 4557/78125 [39:19<10:08:01,  2.02it/s]
  6%|██                                 | 4558/78125 [39:19<10:07:46,  2.02it/s]
  6%|██                                 | 4559/78125 [39:20<10:06:58,  2.02it/s]
  6%|██                                 | 4560/78125 [39:20<10:06:57,  2.02it/s]
  6%|██                                 | 4561/78125 [39:21<10:07:20,  2.02it/s]
  6%|██                                 | 4562/78125 [39:21<10:07:55,  2.02it/s]
  6%|██                                 | 4563/78125 [39:22<10:07:38,  2.02it/s]
  6%|██                                 | 4564/78125 [39:22<10:07:48,  2.02it/s]
  6%|██                                 | 4565/78125 [39:23<10:07:58,  2.02it/s]
  6%|██                                 | 4566/78125 [39:23<10:07:53,  2.02it/s]
  6%|██                                 | 4567/78125 [39:24<10:07:32,  2.02it/s]
  6%|██                                 | 4568/78125 [39:24<10:07:57,  2.02it/s]
  6%|██                                 | 4569/78125 [39:25<10:07:54,  2.02it/s]
  6%|██                                 | 4570/78125 [39:25<10:06:56,  2.02it/s]
  6%|██                                 | 4571/78125 [39:26<10:06:55,  2.02it/s]
  6%|██                                 | 4572/78125 [39:26<10:07:16,  2.02it/s]
  6%|██                                 | 4573/78125 [39:27<10:07:31,  2.02it/s]
  6%|██                                 | 4574/78125 [39:27<10:07:29,  2.02it/s]
  6%|██                                 | 4575/78125 [39:28<10:07:56,  2.02it/s]
  6%|██                                 | 4576/78125 [39:28<10:07:23,  2.02it/s]
  6%|██                                 | 4577/78125 [39:29<10:07:51,  2.02it/s]
  6%|██                                 | 4578/78125 [39:29<10:08:03,  2.02it/s]
  6%|██                                 | 4579/78125 [39:30<10:07:34,  2.02it/s]
  6%|██                                 | 4580/78125 [39:30<10:07:11,  2.02it/s]
  6%|██                                 | 4581/78125 [39:31<10:07:13,  2.02it/s]
  6%|██                                 | 4582/78125 [39:31<10:07:32,  2.02it/s]
  6%|██                                 | 4583/78125 [39:32<10:07:24,  2.02it/s]
  6%|██                                 | 4584/78125 [39:32<10:07:26,  2.02it/s]
  6%|██                                 | 4585/78125 [39:33<10:07:18,  2.02it/s]
  6%|██                                 | 4586/78125 [39:33<10:07:45,  2.02it/s]
  6%|██                                 | 4587/78125 [39:34<10:07:38,  2.02it/s]
  6%|██                                 | 4588/78125 [39:34<10:07:21,  2.02it/s]
  6%|██                                 | 4589/78125 [39:35<10:07:22,  2.02it/s]
  6%|██                                 | 4590/78125 [39:35<10:07:49,  2.02it/s]
  6%|██                                 | 4591/78125 [39:36<10:07:50,  2.02it/s]
  6%|██                                 | 4592/78125 [39:36<10:07:58,  2.02it/s]
  6%|██                                 | 4593/78125 [39:37<10:07:45,  2.02it/s]
  6%|██                                 | 4594/78125 [39:37<10:07:23,  2.02it/s]
  6%|██                                 | 4595/78125 [39:38<10:07:19,  2.02it/s]
  6%|██                                 | 4596/78125 [39:38<10:07:44,  2.02it/s]
  6%|██                                 | 4597/78125 [39:39<10:07:03,  2.02it/s]
  6%|██                                 | 4598/78125 [39:39<10:07:40,  2.02it/s]
  6%|██                                 | 4599/78125 [39:40<10:07:04,  2.02it/s]
  6%|██                                 | 4600/78125 [39:40<10:07:28,  2.02it/s]
  6%|██                                 | 4601/78125 [39:41<10:07:24,  2.02it/s]
  6%|██                                 | 4602/78125 [39:41<10:07:58,  2.02it/s]
  6%|██                                 | 4603/78125 [39:42<10:08:05,  2.02it/s]
  6%|██                                 | 4604/78125 [39:42<10:08:12,  2.01it/s]
  6%|██                                 | 4605/78125 [39:43<10:08:46,  2.01it/s]
  6%|██                                 | 4606/78125 [39:43<10:07:48,  2.02it/s]
  6%|██                                 | 4607/78125 [39:44<10:07:38,  2.02it/s]
  6%|██                                 | 4608/78125 [39:44<10:07:37,  2.02it/s]
  6%|██                                 | 4609/78125 [39:45<10:08:08,  2.01it/s]
  6%|██                                 | 4610/78125 [39:45<10:07:54,  2.02it/s]
  6%|██                                 | 4611/78125 [39:46<10:07:42,  2.02it/s]
  6%|██                                 | 4612/78125 [39:46<10:07:23,  2.02it/s]
  6%|██                                 | 4613/78125 [39:46<10:07:57,  2.02it/s]
  6%|██                                 | 4614/78125 [39:47<10:07:21,  2.02it/s]
  6%|██                                 | 4615/78125 [39:47<10:07:52,  2.02it/s]
  6%|██                                 | 4616/78125 [39:48<10:07:49,  2.02it/s]
  6%|██                                 | 4617/78125 [39:48<10:08:07,  2.01it/s]
  6%|██                                 | 4618/78125 [39:49<10:08:07,  2.01it/s]
  6%|██                                 | 4619/78125 [39:49<10:07:44,  2.02it/s]
  6%|██                                 | 4620/78125 [39:50<10:07:42,  2.02it/s]
  6%|██                                 | 4621/78125 [39:50<10:06:52,  2.02it/s]
  6%|██                                 | 4622/78125 [39:51<10:08:16,  2.01it/s]
  6%|██                                 | 4623/78125 [39:51<10:08:06,  2.01it/s]
  6%|██                                 | 4624/78125 [39:52<10:08:49,  2.01it/s]
  6%|██                                 | 4625/78125 [39:52<10:08:21,  2.01it/s]
  6%|██                                 | 4626/78125 [39:53<10:07:50,  2.02it/s]
  6%|██                                 | 4627/78125 [39:53<10:08:14,  2.01it/s]
  6%|██                                 | 4628/78125 [39:54<10:07:05,  2.02it/s]
  6%|██                                 | 4629/78125 [39:54<10:06:30,  2.02it/s]
  6%|██                                 | 4630/78125 [39:55<10:07:00,  2.02it/s]
  6%|██                                 | 4631/78125 [39:55<10:06:41,  2.02it/s]
  6%|██                                 | 4632/78125 [39:56<10:06:59,  2.02it/s]
  6%|██                                 | 4633/78125 [39:56<10:06:32,  2.02it/s]
  6%|██                                 | 4634/78125 [39:57<10:07:13,  2.02it/s]
  6%|██                                 | 4635/78125 [39:57<10:07:48,  2.02it/s]
  6%|██                                 | 4636/78125 [39:58<10:07:14,  2.02it/s]
  6%|██                                 | 4637/78125 [39:58<10:07:03,  2.02it/s]
  6%|██                                 | 4638/78125 [39:59<10:07:30,  2.02it/s]
  6%|██                                 | 4639/78125 [39:59<10:07:08,  2.02it/s]
  6%|██                                 | 4640/78125 [40:00<10:07:07,  2.02it/s]
  6%|██                                 | 4641/78125 [40:00<10:07:10,  2.02it/s]
  6%|██                                 | 4642/78125 [40:01<10:07:42,  2.02it/s]
  6%|██                                 | 4643/78125 [40:01<10:07:23,  2.02it/s]
  6%|██                                 | 4644/78125 [40:02<10:07:27,  2.02it/s]
  6%|██                                 | 4645/78125 [40:02<10:08:10,  2.01it/s]
  6%|██                                 | 4646/78125 [40:03<10:07:49,  2.01it/s]
  6%|██                                 | 4647/78125 [40:03<10:07:46,  2.01it/s]
  6%|██                                 | 4648/78125 [40:04<10:08:17,  2.01it/s]
  6%|██                                 | 4649/78125 [40:04<10:08:20,  2.01it/s]
  6%|██                                 | 4650/78125 [40:05<10:08:44,  2.01it/s]
  6%|██                                 | 4651/78125 [40:05<10:09:15,  2.01it/s]
  6%|██                                 | 4652/78125 [40:06<10:09:26,  2.01it/s]
  6%|██                                 | 4653/78125 [40:06<10:09:06,  2.01it/s]
  6%|██                                 | 4654/78125 [40:07<10:07:55,  2.01it/s]
  6%|██                                 | 4655/78125 [40:07<10:07:37,  2.02it/s]
  6%|██                                 | 4656/78125 [40:08<10:07:11,  2.02it/s]
  6%|██                                 | 4657/78125 [40:08<10:07:43,  2.01it/s]
  6%|██                                 | 4658/78125 [40:09<10:07:28,  2.02it/s]
  6%|██                                 | 4659/78125 [40:09<10:07:58,  2.01it/s]
  6%|██                                 | 4660/78125 [40:10<10:08:27,  2.01it/s]
  6%|██                                 | 4661/78125 [40:10<10:08:46,  2.01it/s]
  6%|██                                 | 4662/78125 [40:11<10:09:28,  2.01it/s]
  6%|██                                 | 4663/78125 [40:11<10:09:46,  2.01it/s]
  6%|██                                 | 4664/78125 [40:12<10:09:48,  2.01it/s]
  6%|██                                 | 4665/78125 [40:12<10:10:12,  2.01it/s]
  6%|██                                 | 4666/78125 [40:13<10:09:30,  2.01it/s]
  6%|██                                 | 4667/78125 [40:13<10:10:14,  2.01it/s]
  6%|██                                 | 4668/78125 [40:14<10:10:40,  2.00it/s]
  6%|██                                 | 4669/78125 [40:14<10:10:59,  2.00it/s]
  6%|██                                 | 4670/78125 [40:15<10:11:05,  2.00it/s]
  6%|██                                 | 4671/78125 [40:15<10:11:47,  2.00it/s]
  6%|██                                 | 4672/78125 [40:16<10:11:33,  2.00it/s]
  6%|██                                 | 4673/78125 [40:16<10:11:22,  2.00it/s]
  6%|██                                 | 4674/78125 [40:17<10:12:00,  2.00it/s]
  6%|██                                 | 4675/78125 [40:17<10:11:56,  2.00it/s]
  6%|██                                 | 4676/78125 [40:18<10:12:31,  2.00it/s]
  6%|██                                 | 4677/78125 [40:18<10:12:17,  2.00it/s]
  6%|██                                 | 4678/78125 [40:19<10:11:58,  2.00it/s]
  6%|██                                 | 4679/78125 [40:19<10:12:11,  2.00it/s]
  6%|██                                 | 4680/78125 [40:20<10:11:58,  2.00it/s]
  6%|██                                 | 4681/78125 [40:20<10:12:20,  2.00it/s]
  6%|██                                 | 4682/78125 [40:21<10:11:33,  2.00it/s]
  6%|██                                 | 4683/78125 [40:21<10:11:19,  2.00it/s]
  6%|██                                 | 4684/78125 [40:22<10:11:04,  2.00it/s]
  6%|██                                 | 4685/78125 [40:22<10:10:54,  2.00it/s]
  6%|██                                 | 4686/78125 [40:23<10:10:39,  2.00it/s]
  6%|██                                 | 4687/78125 [40:23<10:10:34,  2.00it/s]
  6%|██                                 | 4688/78125 [40:24<10:10:28,  2.00it/s]
  6%|██                                 | 4689/78125 [40:24<10:09:54,  2.01it/s]
  6%|██                                 | 4690/78125 [40:25<10:10:09,  2.01it/s]
  6%|██                                 | 4691/78125 [40:25<10:10:24,  2.01it/s]
  6%|██                                 | 4692/78125 [40:26<10:10:50,  2.00it/s]
  6%|██                                 | 4693/78125 [40:26<10:12:04,  2.00it/s]
  6%|██                                 | 4694/78125 [40:27<10:11:22,  2.00it/s]
  6%|██                                 | 4695/78125 [40:27<10:10:46,  2.00it/s]
  6%|██                                 | 4696/78125 [40:28<10:10:52,  2.00it/s]
  6%|██                                 | 4697/78125 [40:28<10:11:18,  2.00it/s]
  6%|██                                 | 4698/78125 [40:29<10:10:54,  2.00it/s]
  6%|██                                 | 4699/78125 [40:29<10:11:23,  2.00it/s]
  6%|██                                 | 4700/78125 [40:30<10:11:21,  2.00it/s]
  6%|██                                 | 4701/78125 [40:30<10:11:19,  2.00it/s]
  6%|██                                 | 4702/78125 [40:31<10:11:37,  2.00it/s]
  6%|██                                 | 4703/78125 [40:31<10:11:22,  2.00it/s]
  6%|██                                 | 4704/78125 [40:32<10:10:32,  2.00it/s]
  6%|██                                 | 4705/78125 [40:32<10:09:58,  2.01it/s]
  6%|██                                 | 4706/78125 [40:33<10:10:23,  2.00it/s]
  6%|██                                 | 4707/78125 [40:33<10:10:00,  2.01it/s]
  6%|██                                 | 4708/78125 [40:34<10:09:59,  2.01it/s]
  6%|██                                 | 4709/78125 [40:34<10:10:10,  2.01it/s]
  6%|██                                 | 4710/78125 [40:35<10:09:53,  2.01it/s]
  6%|██                                 | 4711/78125 [40:35<10:10:05,  2.01it/s]
  6%|██                                 | 4712/78125 [40:36<10:10:23,  2.00it/s]
  6%|██                                 | 4713/78125 [40:36<10:10:01,  2.01it/s]
  6%|██                                 | 4714/78125 [40:37<10:10:56,  2.00it/s]
  6%|██                                 | 4715/78125 [40:37<10:10:54,  2.00it/s]
  6%|██                                 | 4716/78125 [40:38<10:10:34,  2.00it/s]
  6%|██                                 | 4717/78125 [40:38<10:09:50,  2.01it/s]
  6%|██                                 | 4718/78125 [40:39<10:09:32,  2.01it/s]
  6%|██                                 | 4719/78125 [40:39<10:08:57,  2.01it/s]
  6%|██                                 | 4720/78125 [40:40<10:08:55,  2.01it/s]
  6%|██                                 | 4721/78125 [40:40<10:09:29,  2.01it/s]
  6%|██                                 | 4722/78125 [40:41<10:09:40,  2.01it/s]
  6%|██                                 | 4723/78125 [40:41<10:09:34,  2.01it/s]
  6%|██                                 | 4724/78125 [40:42<10:09:34,  2.01it/s]
  6%|██                                 | 4725/78125 [40:42<10:10:01,  2.01it/s]
  6%|██                                 | 4726/78125 [40:43<10:09:24,  2.01it/s]
  6%|██                                 | 4727/78125 [40:43<10:09:45,  2.01it/s]
  6%|██                                 | 4728/78125 [40:44<10:08:59,  2.01it/s]
  6%|██                                 | 4729/78125 [40:44<10:10:20,  2.00it/s]
  6%|██                                 | 4730/78125 [40:45<10:09:57,  2.01it/s]
  6%|██                                 | 4731/78125 [40:45<10:10:01,  2.01it/s]
  6%|██                                 | 4732/78125 [40:46<10:10:04,  2.01it/s]
  6%|██                                 | 4733/78125 [40:46<10:09:42,  2.01it/s]
  6%|██                                 | 4734/78125 [40:47<10:09:26,  2.01it/s]
  6%|██                                 | 4735/78125 [40:47<10:09:09,  2.01it/s]
  6%|██                                 | 4736/78125 [40:48<10:07:53,  2.01it/s]
  6%|██                                 | 4737/78125 [40:48<10:08:15,  2.01it/s]
  6%|██                                 | 4738/78125 [40:49<10:08:34,  2.01it/s]
  6%|██                                 | 4739/78125 [40:49<10:08:15,  2.01it/s]
  6%|██                                 | 4740/78125 [40:50<10:07:40,  2.01it/s]
  6%|██                                 | 4741/78125 [40:50<10:07:30,  2.01it/s]
  6%|██                                 | 4742/78125 [40:51<10:07:55,  2.01it/s]
  6%|██                                 | 4743/78125 [40:51<10:08:12,  2.01it/s]
  6%|██▏                                | 4744/78125 [40:52<10:08:08,  2.01it/s]
  6%|██▏                                | 4745/78125 [40:52<10:08:36,  2.01it/s]
  6%|██▏                                | 4746/78125 [40:53<10:08:04,  2.01it/s]
  6%|██▏                                | 4747/78125 [40:53<10:07:47,  2.01it/s]
  6%|██▏                                | 4748/78125 [40:54<10:07:10,  2.01it/s]
  6%|██▏                                | 4749/78125 [40:54<10:07:44,  2.01it/s]
  6%|██▏                                | 4750/78125 [40:55<10:07:04,  2.01it/s]
  6%|██▏                                | 4751/78125 [40:55<10:07:49,  2.01it/s]
  6%|██▏                                | 4752/78125 [40:56<10:07:40,  2.01it/s]
  6%|██▏                                | 4753/78125 [40:56<10:07:52,  2.01it/s]
  6%|██▏                                | 4754/78125 [40:57<10:08:19,  2.01it/s]
  6%|██▏                                | 4755/78125 [40:57<10:07:58,  2.01it/s]
  6%|██▏                                | 4756/78125 [40:58<10:07:20,  2.01it/s]
  6%|██▏                                | 4757/78125 [40:58<10:07:42,  2.01it/s]
  6%|██▏                                | 4758/78125 [40:59<10:07:34,  2.01it/s]
  6%|██▏                                | 4759/78125 [40:59<10:07:13,  2.01it/s]
  6%|██▏                                | 4760/78125 [41:00<10:06:54,  2.01it/s]
  6%|██▏                                | 4761/78125 [41:00<10:06:51,  2.01it/s]
  6%|██▏                                | 4762/78125 [41:01<10:07:38,  2.01it/s]
  6%|██▏                                | 4763/78125 [41:01<10:07:38,  2.01it/s]
  6%|██▏                                | 4764/78125 [41:02<10:07:26,  2.01it/s]
  6%|██▏                                | 4765/78125 [41:02<10:07:46,  2.01it/s]
  6%|██▏                                | 4766/78125 [41:03<10:06:58,  2.01it/s]
  6%|██▏                                | 4767/78125 [41:03<10:07:47,  2.01it/s]
  6%|██▏                                | 4768/78125 [41:04<10:08:00,  2.01it/s]
  6%|██▏                                | 4769/78125 [41:04<10:07:29,  2.01it/s]
  6%|██▏                                | 4770/78125 [41:05<10:06:53,  2.01it/s]
  6%|██▏                                | 4771/78125 [41:05<10:06:22,  2.02it/s]
  6%|██▏                                | 4772/78125 [41:06<10:05:47,  2.02it/s]
  6%|██▏                                | 4773/78125 [41:06<10:05:50,  2.02it/s]
  6%|██▏                                | 4774/78125 [41:07<10:06:34,  2.02it/s]
  6%|██▏                                | 4775/78125 [41:07<10:07:05,  2.01it/s]
  6%|██▏                                | 4776/78125 [41:08<10:07:24,  2.01it/s]
  6%|██▏                                | 4777/78125 [41:08<10:07:43,  2.01it/s]
  6%|██▏                                | 4778/78125 [41:09<10:07:35,  2.01it/s]
  6%|██▏                                | 4779/78125 [41:09<10:06:34,  2.02it/s]
  6%|██▏                                | 4780/78125 [41:10<10:06:07,  2.02it/s]
  6%|██▏                                | 4781/78125 [41:10<10:05:42,  2.02it/s]
  6%|██▏                                | 4782/78125 [41:11<10:05:55,  2.02it/s]
  6%|██▏                                | 4783/78125 [41:11<10:05:49,  2.02it/s]
  6%|██▏                                | 4784/78125 [41:12<10:06:17,  2.02it/s]
  6%|██▏                                | 4785/78125 [41:12<10:06:33,  2.02it/s]
  6%|██▏                                | 4786/78125 [41:13<10:06:55,  2.01it/s]
  6%|██▏                                | 4787/78125 [41:13<10:06:35,  2.02it/s]
  6%|██▏                                | 4788/78125 [41:14<10:06:19,  2.02it/s]
  6%|██▏                                | 4789/78125 [41:14<10:06:04,  2.02it/s]
  6%|██▏                                | 4790/78125 [41:15<10:05:31,  2.02it/s]
  6%|██▏                                | 4791/78125 [41:15<10:06:37,  2.01it/s]
  6%|██▏                                | 4792/78125 [41:16<10:06:12,  2.02it/s]
  6%|██▏                                | 4793/78125 [41:16<10:06:10,  2.02it/s]
  6%|██▏                                | 4794/78125 [41:17<10:06:44,  2.01it/s]
  6%|██▏                                | 4795/78125 [41:17<10:06:27,  2.02it/s]
  6%|██▏                                | 4796/78125 [41:18<10:05:59,  2.02it/s]
  6%|██▏                                | 4797/78125 [41:18<10:05:38,  2.02it/s]
  6%|██▏                                | 4798/78125 [41:19<10:05:27,  2.02it/s]
  6%|██▏                                | 4799/78125 [41:19<10:05:19,  2.02it/s]
  6%|██▏                                | 4800/78125 [41:20<10:05:53,  2.02it/s]
  6%|██▏                                | 4801/78125 [41:20<10:06:01,  2.02it/s]
  6%|██▏                                | 4802/78125 [41:21<10:06:25,  2.02it/s]
  6%|██▏                                | 4803/78125 [41:21<10:06:18,  2.02it/s]
  6%|██▏                                | 4804/78125 [41:21<10:06:11,  2.02it/s]
  6%|██▏                                | 4805/78125 [41:22<10:05:27,  2.02it/s]
  6%|██▏                                | 4806/78125 [41:22<10:05:41,  2.02it/s]
  6%|██▏                                | 4807/78125 [41:23<10:05:31,  2.02it/s]
  6%|██▏                                | 4808/78125 [41:23<10:05:59,  2.02it/s]
  6%|██▏                                | 4809/78125 [41:24<10:05:46,  2.02it/s]
  6%|██▏                                | 4810/78125 [41:24<10:04:43,  2.02it/s]
  6%|██▏                                | 4811/78125 [41:25<10:04:08,  2.02it/s]
  6%|██▏                                | 4812/78125 [41:25<10:04:43,  2.02it/s]
  6%|██▏                                | 4813/78125 [41:26<10:04:44,  2.02it/s]
  6%|██▏                                | 4814/78125 [41:26<10:04:52,  2.02it/s]
  6%|██▏                                | 4815/78125 [41:27<10:05:31,  2.02it/s]
  6%|██▏                                | 4816/78125 [41:27<10:06:10,  2.02it/s]
  6%|██▏                                | 4817/78125 [41:28<10:05:57,  2.02it/s]
  6%|██▏                                | 4818/78125 [41:28<10:05:48,  2.02it/s]
  6%|██▏                                | 4819/78125 [41:29<10:05:12,  2.02it/s]
  6%|██▏                                | 4820/78125 [41:29<10:05:05,  2.02it/s]
  6%|██▏                                | 4821/78125 [41:30<10:04:19,  2.02it/s]
  6%|██▏                                | 4822/78125 [41:30<10:04:18,  2.02it/s]
  6%|██▏                                | 4823/78125 [41:31<10:04:44,  2.02it/s]
  6%|██▏                                | 4824/78125 [41:31<10:04:24,  2.02it/s]
  6%|██▏                                | 4825/78125 [41:32<10:04:12,  2.02it/s]
  6%|██▏                                | 4826/78125 [41:32<10:04:54,  2.02it/s]
  6%|██▏                                | 4827/78125 [41:33<10:05:07,  2.02it/s]
  6%|██▏                                | 4828/78125 [41:33<10:04:54,  2.02it/s]
  6%|██▏                                | 4829/78125 [41:34<10:04:56,  2.02it/s]
  6%|██▏                                | 4830/78125 [41:34<10:05:47,  2.02it/s]
  6%|██▏                                | 4831/78125 [41:35<10:05:18,  2.02it/s]
  6%|██▏                                | 4832/78125 [41:35<10:05:02,  2.02it/s]
  6%|██▏                                | 4833/78125 [41:36<10:05:09,  2.02it/s]
  6%|██▏                                | 4834/78125 [41:36<10:05:31,  2.02it/s]
  6%|██▏                                | 4835/78125 [41:37<10:05:29,  2.02it/s]
  6%|██▏                                | 4836/78125 [41:37<10:05:25,  2.02it/s]
  6%|██▏                                | 4837/78125 [41:38<10:05:32,  2.02it/s]
  6%|██▏                                | 4838/78125 [41:38<10:06:04,  2.02it/s]
  6%|██▏                                | 4839/78125 [41:39<10:05:31,  2.02it/s]
  6%|██▏                                | 4840/78125 [41:39<10:06:04,  2.02it/s]
  6%|██▏                                | 4841/78125 [41:40<10:06:33,  2.01it/s]
  6%|██▏                                | 4842/78125 [41:40<10:06:36,  2.01it/s]
  6%|██▏                                | 4843/78125 [41:41<10:06:25,  2.01it/s]
  6%|██▏                                | 4844/78125 [41:41<10:05:01,  2.02it/s]
  6%|██▏                                | 4845/78125 [41:42<10:05:32,  2.02it/s]
  6%|██▏                                | 4846/78125 [41:42<10:06:18,  2.01it/s]
  6%|██▏                                | 4847/78125 [41:43<10:05:43,  2.02it/s]
  6%|██▏                                | 4848/78125 [41:43<10:05:50,  2.02it/s]
  6%|██▏                                | 4849/78125 [41:44<10:05:01,  2.02it/s]
  6%|██▏                                | 4850/78125 [41:44<10:05:45,  2.02it/s]
  6%|██▏                                | 4851/78125 [41:45<10:06:14,  2.01it/s]
  6%|██▏                                | 4852/78125 [41:45<10:05:55,  2.02it/s]
  6%|██▏                                | 4853/78125 [41:46<10:05:46,  2.02it/s]
  6%|██▏                                | 4854/78125 [41:46<10:06:15,  2.01it/s]
  6%|██▏                                | 4855/78125 [41:47<10:05:36,  2.02it/s]
  6%|██▏                                | 4856/78125 [41:47<10:05:18,  2.02it/s]
  6%|██▏                                | 4857/78125 [41:48<10:06:01,  2.01it/s]
  6%|██▏                                | 4858/78125 [41:48<10:05:26,  2.02it/s]
  6%|██▏                                | 4859/78125 [41:49<10:05:19,  2.02it/s]
  6%|██▏                                | 4860/78125 [41:49<10:05:19,  2.02it/s]
  6%|██▏                                | 4861/78125 [41:50<10:04:45,  2.02it/s]
  6%|██▏                                | 4862/78125 [41:50<10:04:43,  2.02it/s]
  6%|██▏                                | 4863/78125 [41:51<10:04:10,  2.02it/s]
  6%|██▏                                | 4864/78125 [41:51<10:04:27,  2.02it/s]
  6%|██▏                                | 4865/78125 [41:52<10:04:57,  2.02it/s]
  6%|██▏                                | 4866/78125 [41:52<10:04:51,  2.02it/s]
  6%|██▏                                | 4867/78125 [41:53<10:05:16,  2.02it/s]
  6%|██▏                                | 4868/78125 [41:53<10:05:32,  2.02it/s]
  6%|██▏                                | 4869/78125 [41:54<10:05:31,  2.02it/s]
  6%|██▏                                | 4870/78125 [41:54<10:05:39,  2.02it/s]
  6%|██▏                                | 4871/78125 [41:55<10:06:50,  2.01it/s]
  6%|██▏                                | 4872/78125 [41:55<10:06:07,  2.01it/s]
  6%|██▏                                | 4873/78125 [41:56<10:06:09,  2.01it/s]
  6%|██▏                                | 4874/78125 [41:56<10:06:02,  2.01it/s]
  6%|██▏                                | 4875/78125 [41:57<10:05:46,  2.02it/s]
  6%|██▏                                | 4876/78125 [41:57<10:05:50,  2.02it/s]
  6%|██▏                                | 4877/78125 [41:58<10:05:57,  2.01it/s]
  6%|██▏                                | 4878/78125 [41:58<10:05:24,  2.02it/s]
  6%|██▏                                | 4879/78125 [41:59<10:05:37,  2.02it/s]
  6%|██▏                                | 4880/78125 [41:59<10:05:16,  2.02it/s]
  6%|██▏                                | 4881/78125 [42:00<10:05:27,  2.02it/s]
760.3s 150 6%|██▏                                | 4882/78125 [42:00<10:05:40,  2.02it/s]
  6%|██▏                                | 4883/78125 [42:01<10:05:39,  2.02it/s]
  6%|██▏                                | 4884/78125 [42:01<10:05:54,  2.01it/s]
  6%|██▏                                | 4885/78125 [42:02<10:05:31,  2.02it/s]
  6%|██▏                                | 4886/78125 [42:02<10:05:24,  2.02it/s]
  6%|██▏                                | 4887/78125 [42:03<10:05:34,  2.02it/s]
  6%|██▏                                | 4888/78125 [42:03<10:05:50,  2.01it/s]
  6%|██▏                                | 4889/78125 [42:04<10:06:05,  2.01it/s]
  6%|██▏                                | 4890/78125 [42:04<10:06:05,  2.01it/s]
  6%|██▏                                | 4891/78125 [42:05<10:05:41,  2.02it/s]
  6%|██▏                                | 4892/78125 [42:05<10:05:34,  2.02it/s]
  6%|██▏                                | 4893/78125 [42:06<10:05:53,  2.01it/s]
  6%|██▏                                | 4894/78125 [42:06<10:05:29,  2.02it/s]
  6%|██▏                                | 4895/78125 [42:07<10:05:35,  2.02it/s]
  6%|██▏                                | 4896/78125 [42:07<10:05:42,  2.01it/s]
  6%|██▏                                | 4897/78125 [42:08<10:05:44,  2.01it/s]
  6%|██▏                                | 4898/78125 [42:08<10:05:32,  2.02it/s]
  6%|██▏                                | 4899/78125 [42:09<10:05:12,  2.02it/s]
  6%|██▏                                | 4900/78125 [42:09<10:05:30,  2.02it/s]
  6%|██▏                                | 4901/78125 [42:10<10:05:06,  2.02it/s]
  6%|██▏                                | 4902/78125 [42:10<10:05:43,  2.01it/s]
  6%|██▏                                | 4903/78125 [42:11<10:05:05,  2.02it/s]
  6%|██▏                                | 4904/78125 [42:11<10:05:40,  2.01it/s]
  6%|██▏                                | 4905/78125 [42:12<10:05:39,  2.01it/s]
  6%|██▏                                | 4906/78125 [42:12<10:05:28,  2.02it/s]
  6%|██▏                                | 4907/78125 [42:13<10:04:56,  2.02it/s]
  6%|██▏                                | 4908/78125 [42:13<10:05:50,  2.01it/s]
  6%|██▏                                | 4909/78125 [42:14<10:05:35,  2.02it/s]
  6%|██▏                                | 4910/78125 [42:14<10:06:29,  2.01it/s]
  6%|██▏                                | 4911/78125 [42:15<10:06:11,  2.01it/s]
  6%|██▏                                | 4912/78125 [42:15<10:06:42,  2.01it/s]
  6%|██▏                                | 4913/78125 [42:16<10:06:43,  2.01it/s]
  6%|██▏                                | 4914/78125 [42:16<10:06:05,  2.01it/s]
  6%|██▏                                | 4915/78125 [42:17<10:05:32,  2.02it/s]
  6%|██▏                                | 4916/78125 [42:17<10:05:22,  2.02it/s]
  6%|██▏                                | 4917/78125 [42:18<10:05:09,  2.02it/s]
  6%|██▏                                | 4918/78125 [42:18<10:04:52,  2.02it/s]
  6%|██▏                                | 4919/78125 [42:19<10:05:00,  2.02it/s]
  6%|██▏                                | 4920/78125 [42:19<10:05:20,  2.02it/s]
  6%|██▏                                | 4921/78125 [42:20<10:06:09,  2.01it/s]
  6%|██▏                                | 4922/78125 [42:20<10:06:02,  2.01it/s]
  6%|██▏                                | 4923/78125 [42:21<10:06:15,  2.01it/s]
  6%|██▏                                | 4924/78125 [42:21<10:06:30,  2.01it/s]
  6%|██▏                                | 4925/78125 [42:22<10:06:39,  2.01it/s]
  6%|██▏                                | 4926/78125 [42:22<10:07:01,  2.01it/s]
  6%|██▏                                | 4927/78125 [42:23<10:07:04,  2.01it/s]
  6%|██▏                                | 4928/78125 [42:23<10:07:31,  2.01it/s]
  6%|██▏                                | 4929/78125 [42:24<10:07:14,  2.01it/s]
  6%|██▏                                | 4930/78125 [42:24<10:07:12,  2.01it/s]
  6%|██▏                                | 4931/78125 [42:24<10:06:34,  2.01it/s]
  6%|██▏                                | 4932/78125 [42:25<10:05:53,  2.01it/s]
  6%|██▏                                | 4933/78125 [42:25<10:04:40,  2.02it/s]
  6%|██▏                                | 4934/78125 [42:26<10:04:18,  2.02it/s]
  6%|██▏                                | 4935/78125 [42:26<10:04:46,  2.02it/s]
  6%|██▏                                | 4936/78125 [42:27<10:05:03,  2.02it/s]
  6%|██▏                                | 4937/78125 [42:27<10:05:04,  2.02it/s]
  6%|██▏                                | 4938/78125 [42:28<10:05:45,  2.01it/s]
  6%|██▏                                | 4939/78125 [42:28<10:06:15,  2.01it/s]
  6%|██▏                                | 4940/78125 [42:29<10:06:04,  2.01it/s]
  6%|██▏                                | 4941/78125 [42:29<10:06:22,  2.01it/s]
  6%|██▏                                | 4942/78125 [42:30<10:06:07,  2.01it/s]
  6%|██▏                                | 4943/78125 [42:30<10:06:14,  2.01it/s]
  6%|██▏                                | 4944/78125 [42:31<10:07:10,  2.01it/s]
  6%|██▏                                | 4945/78125 [42:31<10:06:32,  2.01it/s]
  6%|██▏                                | 4946/78125 [42:32<10:06:34,  2.01it/s]
  6%|██▏                                | 4947/78125 [42:32<10:06:33,  2.01it/s]
  6%|██▏                                | 4948/78125 [42:33<10:06:08,  2.01it/s]
  6%|██▏                                | 4949/78125 [42:33<10:05:38,  2.01it/s]
  6%|██▏                                | 4950/78125 [42:34<10:04:59,  2.02it/s]
  6%|██▏                                | 4951/78125 [42:34<10:04:57,  2.02it/s]
  6%|██▏                                | 4952/78125 [42:35<10:05:27,  2.01it/s]
  6%|██▏                                | 4953/78125 [42:35<10:06:13,  2.01it/s]
  6%|██▏                                | 4954/78125 [42:36<10:06:21,  2.01it/s]
  6%|██▏                                | 4955/78125 [42:36<10:06:24,  2.01it/s]
  6%|██▏                                | 4956/78125 [42:37<10:07:10,  2.01it/s]
  6%|██▏                                | 4957/78125 [42:37<10:06:55,  2.01it/s]
  6%|██▏                                | 4958/78125 [42:38<10:06:41,  2.01it/s]
  6%|██▏                                | 4959/78125 [42:38<10:06:29,  2.01it/s]
  6%|██▏                                | 4960/78125 [42:39<10:06:25,  2.01it/s]
  6%|██▏                                | 4961/78125 [42:39<10:06:44,  2.01it/s]
  6%|██▏                                | 4962/78125 [42:40<10:06:34,  2.01it/s]
  6%|██▏                                | 4963/78125 [42:40<10:07:16,  2.01it/s]
  6%|██▏                                | 4964/78125 [42:41<10:06:29,  2.01it/s]
  6%|██▏                                | 4965/78125 [42:41<10:05:50,  2.01it/s]
  6%|██▏                                | 4966/78125 [42:42<10:06:18,  2.01it/s]
  6%|██▏                                | 4967/78125 [42:42<10:06:35,  2.01it/s]
  6%|██▏                                | 4968/78125 [42:43<10:05:50,  2.01it/s]
  6%|██▏                                | 4969/78125 [42:43<10:06:47,  2.01it/s]
  6%|██▏                                | 4970/78125 [42:44<10:06:22,  2.01it/s]
  6%|██▏                                | 4971/78125 [42:44<10:05:41,  2.01it/s]
  6%|██▏                                | 4972/78125 [42:45<10:06:13,  2.01it/s]
  6%|██▏                                | 4973/78125 [42:45<10:06:31,  2.01it/s]
  6%|██▏                                | 4974/78125 [42:46<10:06:53,  2.01it/s]
  6%|██▏                                | 4975/78125 [42:46<10:06:39,  2.01it/s]
  6%|██▏                                | 4976/78125 [42:47<10:06:06,  2.01it/s]
  6%|██▏                                | 4977/78125 [42:47<10:06:51,  2.01it/s]
  6%|██▏                                | 4978/78125 [42:48<10:07:13,  2.01it/s]
  6%|██▏                                | 4979/78125 [42:48<10:06:55,  2.01it/s]
  6%|██▏                                | 4980/78125 [42:49<10:07:18,  2.01it/s]
  6%|██▏                                | 4981/78125 [42:49<10:07:07,  2.01it/s]
  6%|██▏                                | 4982/78125 [42:50<10:06:59,  2.01it/s]
  6%|██▏                                | 4983/78125 [42:50<10:06:44,  2.01it/s]
  6%|██▏                                | 4984/78125 [42:51<10:07:03,  2.01it/s]
  6%|██▏                                | 4985/78125 [42:51<10:06:15,  2.01it/s]
  6%|██▏                                | 4986/78125 [42:52<10:06:43,  2.01it/s]
  6%|██▏                                | 4987/78125 [42:52<10:06:35,  2.01it/s]
  6%|██▏                                | 4988/78125 [42:53<10:07:22,  2.01it/s]
  6%|██▏                                | 4989/78125 [42:53<10:07:28,  2.01it/s]
  6%|██▏                                | 4990/78125 [42:54<10:07:40,  2.01it/s]
  6%|██▏                                | 4991/78125 [42:54<10:07:13,  2.01it/s]
  6%|██▏                                | 4992/78125 [42:55<10:07:05,  2.01it/s]
  6%|██▏                                | 4993/78125 [42:55<10:06:11,  2.01it/s]
  6%|██▏                                | 4994/78125 [42:56<10:06:33,  2.01it/s]
  6%|██▏                                | 4995/78125 [42:56<10:06:15,  2.01it/s]
  6%|██▏                                | 4996/78125 [42:57<10:07:16,  2.01it/s]
  6%|██▏                                | 4997/78125 [42:57<10:07:35,  2.01it/s]
  6%|██▏                                | 4998/78125 [42:58<10:06:37,  2.01it/s]
  6%|██▏                                | 4999/78125 [42:58<10:07:15,  2.01it/s]
  6%|██▏                                | 5000/78125 [42:59<10:07:39,  2.01it/s]
  6%|██▏                                | 5001/78125 [42:59<10:07:32,  2.01it/s]
  6%|██▏                                | 5002/78125 [43:00<10:08:06,  2.00it/s]
  6%|██▏                                | 5003/78125 [43:00<10:08:07,  2.00it/s]
  6%|██▏                                | 5004/78125 [43:01<10:07:17,  2.01it/s]
  6%|██▏                                | 5005/78125 [43:01<10:07:40,  2.01it/s]
  6%|██▏                                | 5006/78125 [43:02<10:07:34,  2.01it/s]
  6%|██▏                                | 5007/78125 [43:02<10:07:45,  2.01it/s]
  6%|██▏                                | 5008/78125 [43:03<10:07:51,  2.00it/s]
  6%|██▏                                | 5009/78125 [43:03<10:08:10,  2.00it/s]
  6%|██▏                                | 5010/78125 [43:04<10:08:27,  2.00it/s]
  6%|██▏                                | 5011/78125 [43:04<10:08:15,  2.00it/s]
  6%|██▏                                | 5012/78125 [43:05<10:08:30,  2.00it/s]
  6%|██▏                                | 5013/78125 [43:05<10:08:01,  2.00it/s]
  6%|██▏                                | 5014/78125 [43:06<10:07:54,  2.00it/s]
  6%|██▏                                | 5015/78125 [43:06<10:08:15,  2.00it/s]
  6%|██▏                                | 5016/78125 [43:07<10:08:07,  2.00it/s]
  6%|██▏                                | 5017/78125 [43:07<10:07:12,  2.01it/s]
  6%|██▏                                | 5018/78125 [43:08<10:06:48,  2.01it/s]
  6%|██▏                                | 5019/78125 [43:08<10:07:04,  2.01it/s]
  6%|██▏                                | 5020/78125 [43:09<10:06:56,  2.01it/s]
  6%|██▏                                | 5021/78125 [43:09<10:06:23,  2.01it/s]
  6%|██▏                                | 5022/78125 [43:10<10:06:38,  2.01it/s]
  6%|██▎                                | 5023/78125 [43:10<10:07:04,  2.01it/s]
  6%|██▎                                | 5024/78125 [43:11<10:07:14,  2.01it/s]
  6%|██▎                                | 5025/78125 [43:11<10:06:54,  2.01it/s]
  6%|██▎                                | 5026/78125 [43:12<10:06:58,  2.01it/s]
  6%|██▎                                | 5027/78125 [43:12<10:07:19,  2.01it/s]
  6%|██▎                                | 5028/78125 [43:13<10:07:00,  2.01it/s]
  6%|██▎                                | 5029/78125 [43:13<10:07:28,  2.01it/s]
  6%|██▎                                | 5030/78125 [43:14<10:06:55,  2.01it/s]
  6%|██▎                                | 5031/78125 [43:14<10:06:40,  2.01it/s]
  6%|██▎                                | 5032/78125 [43:15<10:07:11,  2.01it/s]
  6%|██▎                                | 5033/78125 [43:15<10:07:31,  2.01it/s]
  6%|██▎                                | 5034/78125 [43:16<10:07:29,  2.01it/s]
  6%|██▎                                | 5035/78125 [43:16<10:07:06,  2.01it/s]
  6%|██▎                                | 5036/78125 [43:17<10:07:51,  2.00it/s]
  6%|██▎                                | 5037/78125 [43:17<10:07:52,  2.00it/s]
  6%|██▎                                | 5038/78125 [43:18<10:07:52,  2.00it/s]
  6%|██▎                                | 5039/78125 [43:18<10:08:12,  2.00it/s]
  6%|██▎                                | 5040/78125 [43:19<10:08:06,  2.00it/s]
  6%|██▎                                | 5041/78125 [43:19<10:07:54,  2.00it/s]
  6%|██▎                                | 5042/78125 [43:20<10:07:12,  2.01it/s]
  6%|██▎                                | 5043/78125 [43:20<10:07:29,  2.01it/s]
  6%|██▎                                | 5044/78125 [43:21<10:07:34,  2.00it/s]
  6%|██▎                                | 5045/78125 [43:21<10:07:29,  2.00it/s]
  6%|██▎                                | 5046/78125 [43:22<10:07:08,  2.01it/s]
  6%|██▎                                | 5047/78125 [43:22<10:07:05,  2.01it/s]
  6%|██▎                                | 5048/78125 [43:23<10:07:22,  2.01it/s]
  6%|██▎                                | 5049/78125 [43:23<10:07:25,  2.01it/s]
  6%|██▎                                | 5050/78125 [43:24<10:07:32,  2.00it/s]
  6%|██▎                                | 5051/78125 [43:24<10:08:05,  2.00it/s]
  6%|██▎                                | 5052/78125 [43:25<10:07:46,  2.00it/s]
  6%|██▎                                | 5053/78125 [43:25<10:08:11,  2.00it/s]
  6%|██▎                                | 5054/78125 [43:26<10:08:03,  2.00it/s]
  6%|██▎                                | 5055/78125 [43:26<10:08:05,  2.00it/s]
  6%|██▎                                | 5056/78125 [43:27<10:07:12,  2.01it/s]
  6%|██▎                                | 5057/78125 [43:27<10:07:59,  2.00it/s]
  6%|██▎                                | 5058/78125 [43:28<10:07:39,  2.00it/s]
  6%|██▎                                | 5059/78125 [43:28<10:06:51,  2.01it/s]
  6%|██▎                                | 5060/78125 [43:29<10:06:27,  2.01it/s]
  6%|██▎                                | 5061/78125 [43:29<10:07:02,  2.01it/s]
  6%|██▎                                | 5062/78125 [43:30<10:07:11,  2.01it/s]
  6%|██▎                                | 5063/78125 [43:30<10:07:09,  2.01it/s]
  6%|██▎                                | 5064/78125 [43:31<10:07:21,  2.00it/s]
  6%|██▎                                | 5065/78125 [43:31<10:06:54,  2.01it/s]
  6%|██▎                                | 5066/78125 [43:32<10:05:47,  2.01it/s]
  6%|██▎                                | 5067/78125 [43:32<10:05:58,  2.01it/s]
  6%|██▎                                | 5068/78125 [43:33<10:06:27,  2.01it/s]
  6%|██▎                                | 5069/78125 [43:33<10:06:15,  2.01it/s]
  6%|██▎                                | 5070/78125 [43:34<10:06:28,  2.01it/s]
  6%|██▎                                | 5071/78125 [43:34<10:07:06,  2.01it/s]
  6%|██▎                                | 5072/78125 [43:35<10:06:43,  2.01it/s]
  6%|██▎                                | 5073/78125 [43:35<10:07:03,  2.01it/s]
  6%|██▎                                | 5074/78125 [43:36<10:07:05,  2.01it/s]
  6%|██▎                                | 5075/78125 [43:36<10:06:56,  2.01it/s]
  6%|██▎                                | 5076/78125 [43:37<10:07:21,  2.00it/s]
  6%|██▎                                | 5077/78125 [43:37<10:07:21,  2.00it/s]
  6%|██▎                                | 5078/78125 [43:38<10:07:07,  2.01it/s]
  7%|██▎                                | 5079/78125 [43:38<10:07:09,  2.01it/s]
  7%|██▎                                | 5080/78125 [43:39<10:07:34,  2.00it/s]
  7%|██▎                                | 5081/78125 [43:39<10:08:20,  2.00it/s]
  7%|██▎                                | 5082/78125 [43:40<10:07:33,  2.00it/s]
  7%|██▎                                | 5083/78125 [43:40<10:07:17,  2.00it/s]
  7%|██▎                                | 5084/78125 [43:41<10:07:12,  2.00it/s]
  7%|██▎                                | 5085/78125 [43:41<10:07:10,  2.00it/s]
  7%|██▎                                | 5086/78125 [43:42<10:06:38,  2.01it/s]
  7%|██▎                                | 5087/78125 [43:42<10:07:31,  2.00it/s]
  7%|██▎                                | 5088/78125 [43:43<10:08:18,  2.00it/s]
  7%|██▎                                | 5089/78125 [43:43<10:07:53,  2.00it/s]
  7%|██▎                                | 5090/78125 [43:44<10:07:34,  2.00it/s]
  7%|██▎                                | 5091/78125 [43:44<10:08:42,  2.00it/s]
  7%|██▎                                | 5092/78125 [43:45<10:09:07,  2.00it/s]
  7%|██▎                                | 5093/78125 [43:45<10:09:26,  2.00it/s]
  7%|██▎                                | 5094/78125 [43:46<10:09:09,  2.00it/s]
  7%|██▎                                | 5095/78125 [43:46<10:08:22,  2.00it/s]
  7%|██▎                                | 5096/78125 [43:47<10:07:33,  2.00it/s]
  7%|██▎                                | 5097/78125 [43:47<10:08:10,  2.00it/s]
  7%|██▎                                | 5098/78125 [43:48<10:07:28,  2.00it/s]
  7%|██▎                                | 5099/78125 [43:48<10:06:52,  2.01it/s]
  7%|██▎                                | 5100/78125 [43:49<10:06:45,  2.01it/s]
  7%|██▎                                | 5101/78125 [43:49<10:06:45,  2.01it/s]
  7%|██▎                                | 5102/78125 [43:50<10:06:43,  2.01it/s]
  7%|██▎                                | 5103/78125 [43:50<10:07:21,  2.00it/s]
  7%|██▎                                | 5104/78125 [43:51<10:07:14,  2.00it/s]
  7%|██▎                                | 5105/78125 [43:51<10:07:40,  2.00it/s]
  7%|██▎                                | 5106/78125 [43:52<10:07:43,  2.00it/s]
  7%|██▎                                | 5107/78125 [43:52<10:06:49,  2.01it/s]
  7%|██▎                                | 5108/78125 [43:53<10:07:15,  2.00it/s]
  7%|██▎                                | 5109/78125 [43:53<10:07:26,  2.00it/s]
  7%|██▎                                | 5110/78125 [43:54<10:06:52,  2.01it/s]
  7%|██▎                                | 5111/78125 [43:54<10:07:25,  2.00it/s]
  7%|██▎                                | 5112/78125 [43:55<10:07:17,  2.00it/s]
  7%|██▎                                | 5113/78125 [43:55<10:07:03,  2.00it/s]
  7%|██▎                                | 5114/78125 [43:56<10:07:33,  2.00it/s]
  7%|██▎                                | 5115/78125 [43:56<10:08:13,  2.00it/s]
  7%|██▎                                | 5116/78125 [43:57<10:07:17,  2.00it/s]
  7%|██▎                                | 5117/78125 [43:57<10:06:53,  2.00it/s]
  7%|██▎                                | 5118/78125 [43:58<10:07:31,  2.00it/s]
  7%|██▎                                | 5119/78125 [43:58<10:07:15,  2.00it/s]
  7%|██▎                                | 5120/78125 [43:59<10:06:34,  2.01it/s]
  7%|██▎                                | 5121/78125 [43:59<10:06:53,  2.00it/s]
  7%|██▎                                | 5122/78125 [44:00<10:06:06,  2.01it/s]
  7%|██▎                                | 5123/78125 [44:00<10:06:37,  2.01it/s]
  7%|██▎                                | 5124/78125 [44:01<10:07:21,  2.00it/s]
  7%|██▎                                | 5125/78125 [44:01<10:07:18,  2.00it/s]
  7%|██▎                                | 5126/78125 [44:02<10:07:11,  2.00it/s]
  7%|██▎                                | 5127/78125 [44:02<10:07:16,  2.00it/s]
  7%|██▎                                | 5128/78125 [44:03<10:06:34,  2.01it/s]
  7%|██▎                                | 5129/78125 [44:03<10:07:17,  2.00it/s]
  7%|██▎                                | 5130/78125 [44:04<10:06:42,  2.01it/s]
  7%|██▎                                | 5131/78125 [44:04<10:07:23,  2.00it/s]
  7%|██▎                                | 5132/78125 [44:05<10:07:14,  2.00it/s]
  7%|██▎                                | 5133/78125 [44:05<10:06:56,  2.00it/s]
  7%|██▎                                | 5134/78125 [44:06<10:06:39,  2.01it/s]
  7%|██▎                                | 5135/78125 [44:06<10:06:34,  2.01it/s]
  7%|██▎                                | 5136/78125 [44:07<10:07:44,  2.00it/s]
  7%|██▎                                | 5137/78125 [44:07<10:07:29,  2.00it/s]
  7%|██▎                                | 5138/78125 [44:08<10:07:04,  2.00it/s]
  7%|██▎                                | 5139/78125 [44:08<10:06:31,  2.01it/s]
  7%|██▎                                | 5140/78125 [44:09<10:06:45,  2.00it/s]
  7%|██▎                                | 5141/78125 [44:09<10:06:32,  2.01it/s]
  7%|██▎                                | 5142/78125 [44:10<10:06:02,  2.01it/s]
  7%|██▎                                | 5143/78125 [44:10<10:06:35,  2.01it/s]
  7%|██▎                                | 5144/78125 [44:11<10:06:37,  2.01it/s]
  7%|██▎                                | 5145/78125 [44:11<10:06:49,  2.00it/s]
  7%|██▎                                | 5146/78125 [44:12<10:07:12,  2.00it/s]
  7%|██▎                                | 5147/78125 [44:12<10:06:42,  2.00it/s]
  7%|██▎                                | 5148/78125 [44:13<10:06:12,  2.01it/s]
  7%|██▎                                | 5149/78125 [44:13<10:06:22,  2.01it/s]
  7%|██▎                                | 5150/78125 [44:14<10:06:59,  2.00it/s]
  7%|██▎                                | 5151/78125 [44:14<10:06:42,  2.00it/s]
  7%|██▎                                | 5152/78125 [44:15<10:06:21,  2.01it/s]
  7%|██▎                                | 5153/78125 [44:15<10:06:25,  2.01it/s]
  7%|██▎                                | 5154/78125 [44:16<10:06:49,  2.00it/s]
  7%|██▎                                | 5155/78125 [44:16<10:05:52,  2.01it/s]
  7%|██▎                                | 5156/78125 [44:17<10:05:37,  2.01it/s]
  7%|██▎                                | 5157/78125 [44:17<10:06:11,  2.01it/s]
  7%|██▎                                | 5158/78125 [44:18<10:06:14,  2.01it/s]
  7%|██▎                                | 5159/78125 [44:18<10:06:07,  2.01it/s]
  7%|██▎                                | 5160/78125 [44:19<10:06:42,  2.00it/s]
  7%|██▎                                | 5161/78125 [44:19<10:06:17,  2.01it/s]
  7%|██▎                                | 5162/78125 [44:20<10:05:49,  2.01it/s]
  7%|██▎                                | 5163/78125 [44:20<10:05:41,  2.01it/s]
  7%|██▎                                | 5164/78125 [44:21<10:05:48,  2.01it/s]
  7%|██▎                                | 5165/78125 [44:21<10:05:38,  2.01it/s]
  7%|██▎                                | 5166/78125 [44:22<10:04:41,  2.01it/s]
  7%|██▎                                | 5167/78125 [44:22<10:05:12,  2.01it/s]
  7%|██▎                                | 5168/78125 [44:23<10:05:31,  2.01it/s]
  7%|██▎                                | 5169/78125 [44:23<10:05:25,  2.01it/s]
  7%|██▎                                | 5170/78125 [44:24<10:05:41,  2.01it/s]
  7%|██▎                                | 5171/78125 [44:24<10:05:37,  2.01it/s]
  7%|██▎                                | 5172/78125 [44:25<10:06:09,  2.01it/s]
  7%|██▎                                | 5173/78125 [44:25<10:06:15,  2.01it/s]
  7%|██▎                                | 5174/78125 [44:26<10:06:31,  2.00it/s]
  7%|██▎                                | 5175/78125 [44:26<10:06:38,  2.00it/s]
  7%|██▎                                | 5176/78125 [44:27<10:06:39,  2.00it/s]
  7%|██▎                                | 5177/78125 [44:27<10:06:44,  2.00it/s]
  7%|██▎                                | 5178/78125 [44:28<10:06:44,  2.00it/s]
  7%|██▎                                | 5179/78125 [44:28<10:06:35,  2.00it/s]
  7%|██▎                                | 5180/78125 [44:29<10:06:27,  2.00it/s]
  7%|██▎                                | 5181/78125 [44:29<10:05:54,  2.01it/s]
  7%|██▎                                | 5182/78125 [44:30<10:06:16,  2.01it/s]
  7%|██▎                                | 5183/78125 [44:30<10:06:26,  2.00it/s]
  7%|██▎                                | 5184/78125 [44:31<10:05:48,  2.01it/s]
  7%|██▎                                | 5185/78125 [44:31<10:05:36,  2.01it/s]
  7%|██▎                                | 5186/78125 [44:32<10:05:52,  2.01it/s]
  7%|██▎                                | 5187/78125 [44:32<10:05:11,  2.01it/s]
  7%|██▎                                | 5188/78125 [44:33<10:05:28,  2.01it/s]
  7%|██▎                                | 5189/78125 [44:33<10:05:25,  2.01it/s]
  7%|██▎                                | 5190/78125 [44:34<10:05:46,  2.01it/s]
  7%|██▎                                | 5191/78125 [44:34<10:06:19,  2.00it/s]
  7%|██▎                                | 5192/78125 [44:35<10:06:04,  2.01it/s]
  7%|██▎                                | 5193/78125 [44:35<10:05:18,  2.01it/s]
  7%|██▎                                | 5194/78125 [44:36<10:04:51,  2.01it/s]
  7%|██▎                                | 5195/78125 [44:36<10:04:55,  2.01it/s]
  7%|██▎                                | 5196/78125 [44:37<10:04:21,  2.01it/s]
  7%|██▎                                | 5197/78125 [44:37<10:04:10,  2.01it/s]
  7%|██▎                                | 5198/78125 [44:38<10:04:17,  2.01it/s]
  7%|██▎                                | 5199/78125 [44:38<10:03:54,  2.01it/s]
  7%|██▎                                | 5200/78125 [44:39<10:04:42,  2.01it/s]
  7%|██▎                                | 5201/78125 [44:39<10:04:14,  2.01it/s]
  7%|██▎                                | 5202/78125 [44:40<10:04:41,  2.01it/s]
  7%|██▎                                | 5203/78125 [44:40<10:05:01,  2.01it/s]
  7%|██▎                                | 5204/78125 [44:41<10:05:29,  2.01it/s]
  7%|██▎                                | 5205/78125 [44:41<10:05:09,  2.01it/s]
  7%|██▎                                | 5206/78125 [44:42<10:05:28,  2.01it/s]
  7%|██▎                                | 5207/78125 [44:42<10:06:00,  2.01it/s]
  7%|██▎                                | 5208/78125 [44:43<10:06:01,  2.01it/s]
  7%|██▎                                | 5209/78125 [44:43<10:05:17,  2.01it/s]
  7%|██▎                                | 5210/78125 [44:44<10:05:45,  2.01it/s]
  7%|██▎                                | 5211/78125 [44:44<10:05:29,  2.01it/s]
  7%|██▎                                | 5212/78125 [44:45<10:05:08,  2.01it/s]
  7%|██▎                                | 5213/78125 [44:45<10:05:09,  2.01it/s]
  7%|██▎                                | 5214/78125 [44:46<10:05:13,  2.01it/s]
  7%|██▎                                | 5215/78125 [44:46<10:04:39,  2.01it/s]
  7%|██▎                                | 5216/78125 [44:47<10:04:14,  2.01it/s]
  7%|██▎                                | 5217/78125 [44:47<10:03:40,  2.01it/s]
  7%|██▎                                | 5218/78125 [44:48<10:03:37,  2.01it/s]
  7%|██▎                                | 5219/78125 [44:48<10:03:49,  2.01it/s]
  7%|██▎                                | 5220/78125 [44:49<10:04:15,  2.01it/s]
  7%|██▎                                | 5221/78125 [44:49<10:03:27,  2.01it/s]
  7%|██▎                                | 5222/78125 [44:49<10:03:15,  2.01it/s]
  7%|██▎                                | 5223/78125 [44:50<10:03:59,  2.01it/s]
  7%|██▎                                | 5224/78125 [44:50<10:04:09,  2.01it/s]
  7%|██▎                                | 5225/78125 [44:51<10:04:19,  2.01it/s]
  7%|██▎                                | 5226/78125 [44:51<10:04:24,  2.01it/s]
  7%|██▎                                | 5227/78125 [44:52<10:04:20,  2.01it/s]
  7%|██▎                                | 5228/78125 [44:52<10:03:23,  2.01it/s]
  7%|██▎                                | 5229/78125 [44:53<10:03:04,  2.01it/s]
  7%|██▎                                | 5230/78125 [44:53<10:02:32,  2.02it/s]
  7%|██▎                                | 5231/78125 [44:54<10:02:14,  2.02it/s]
  7%|██▎                                | 5232/78125 [44:54<10:02:45,  2.02it/s]
  7%|██▎                                | 5233/78125 [44:55<10:02:42,  2.02it/s]
  7%|██▎                                | 5234/78125 [44:55<10:02:36,  2.02it/s]
  7%|██▎                                | 5235/78125 [44:56<10:02:18,  2.02it/s]
  7%|██▎                                | 5236/78125 [44:56<10:01:44,  2.02it/s]
  7%|██▎                                | 5237/78125 [44:57<10:01:50,  2.02it/s]
  7%|██▎                                | 5238/78125 [44:57<10:02:04,  2.02it/s]
  7%|██▎                                | 5239/78125 [44:58<10:03:07,  2.01it/s]
  7%|██▎                                | 5240/78125 [44:58<10:02:16,  2.02it/s]
  7%|██▎                                | 5241/78125 [44:59<10:02:36,  2.02it/s]
  7%|██▎                                | 5242/78125 [44:59<10:02:25,  2.02it/s]
  7%|██▎                                | 5243/78125 [45:00<10:02:03,  2.02it/s]
  7%|██▎                                | 5244/78125 [45:00<10:02:14,  2.02it/s]
  7%|██▎                                | 5245/78125 [45:01<10:01:43,  2.02it/s]
  7%|██▎                                | 5246/78125 [45:01<10:02:05,  2.02it/s]
  7%|██▎                                | 5247/78125 [45:02<10:02:05,  2.02it/s]
  7%|██▎                                | 5248/78125 [45:02<10:01:38,  2.02it/s]
  7%|██▎                                | 5249/78125 [45:03<10:00:47,  2.02it/s]
  7%|██▎                                | 5250/78125 [45:03<10:01:05,  2.02it/s]
  7%|██▎                                | 5251/78125 [45:04<10:00:47,  2.02it/s]
  7%|██▎                                | 5252/78125 [45:04<10:00:34,  2.02it/s]
  7%|██▎                                | 5253/78125 [45:05<10:00:54,  2.02it/s]
  7%|██▎                                | 5254/78125 [45:05<10:00:56,  2.02it/s]
  7%|██▎                                | 5255/78125 [45:06<10:01:02,  2.02it/s]
  7%|██▎                                | 5256/78125 [45:06<10:00:43,  2.02it/s]
  7%|██▎                                | 5257/78125 [45:07<10:01:02,  2.02it/s]
  7%|██▎                                | 5258/78125 [45:07<10:01:06,  2.02it/s]
  7%|██▎                                | 5259/78125 [45:08<10:00:51,  2.02it/s]
  7%|██▎                                | 5260/78125 [45:08<10:00:28,  2.02it/s]
  7%|██▎                                | 5261/78125 [45:09<10:01:18,  2.02it/s]
  7%|██▎                                | 5262/78125 [45:09<10:01:45,  2.02it/s]
  7%|██▎                                | 5263/78125 [45:10<10:01:11,  2.02it/s]
  7%|██▎                                | 5264/78125 [45:10<10:01:46,  2.02it/s]
  7%|██▎                                | 5265/78125 [45:11<10:02:17,  2.02it/s]
  7%|██▎                                | 5266/78125 [45:11<10:02:14,  2.02it/s]
  7%|██▎                                | 5267/78125 [45:12<10:01:46,  2.02it/s]
  7%|██▎                                | 5268/78125 [45:12<10:02:34,  2.02it/s]
  7%|██▎                                | 5269/78125 [45:13<10:03:06,  2.01it/s]
  7%|██▎                                | 5270/78125 [45:13<10:01:56,  2.02it/s]
  7%|██▎                                | 5271/78125 [45:14<10:01:14,  2.02it/s]
  7%|██▎                                | 5272/78125 [45:14<10:01:05,  2.02it/s]
  7%|██▎                                | 5273/78125 [45:15<10:01:36,  2.02it/s]
  7%|██▎                                | 5274/78125 [45:15<10:02:14,  2.02it/s]
  7%|██▎                                | 5275/78125 [45:16<10:02:09,  2.02it/s]
  7%|██▎                                | 5276/78125 [45:16<10:02:42,  2.01it/s]
  7%|██▎                                | 5277/78125 [45:17<10:03:09,  2.01it/s]
  7%|██▎                                | 5278/78125 [45:17<10:02:46,  2.01it/s]
  7%|██▎                                | 5279/78125 [45:18<10:02:55,  2.01it/s]
  7%|██▎                                | 5280/78125 [45:18<10:03:10,  2.01it/s]
  7%|██▎                                | 5281/78125 [45:19<10:03:01,  2.01it/s]
  7%|██▎                                | 5282/78125 [45:19<10:02:56,  2.01it/s]
  7%|██▎                                | 5283/78125 [45:20<10:02:59,  2.01it/s]
  7%|██▎                                | 5284/78125 [45:20<10:02:58,  2.01it/s]
  7%|██▎                                | 5285/78125 [45:21<10:02:29,  2.01it/s]
  7%|██▎                                | 5286/78125 [45:21<10:03:19,  2.01it/s]
  7%|██▎                                | 5287/78125 [45:22<10:02:21,  2.02it/s]
  7%|██▎                                | 5288/78125 [45:22<10:02:39,  2.01it/s]
  7%|██▎                                | 5289/78125 [45:23<10:01:57,  2.02it/s]
  7%|██▎                                | 5290/78125 [45:23<10:02:10,  2.02it/s]
  7%|██▎                                | 5291/78125 [45:24<10:02:40,  2.01it/s]
  7%|██▎                                | 5292/78125 [45:24<10:02:47,  2.01it/s]
  7%|██▎                                | 5293/78125 [45:25<10:02:42,  2.01it/s]
  7%|██▎                                | 5294/78125 [45:25<10:02:52,  2.01it/s]
  7%|██▎                                | 5295/78125 [45:26<10:02:10,  2.02it/s]
  7%|██▎                                | 5296/78125 [45:26<10:02:05,  2.02it/s]
  7%|██▎                                | 5297/78125 [45:27<10:01:49,  2.02it/s]
  7%|██▎                                | 5298/78125 [45:27<10:01:55,  2.02it/s]
  7%|██▎                                | 5299/78125 [45:28<10:02:03,  2.02it/s]
  7%|██▎                                | 5300/78125 [45:28<10:02:46,  2.01it/s]
  7%|██▎                                | 5301/78125 [45:29<10:02:01,  2.02it/s]
  7%|██▍                                | 5302/78125 [45:29<10:02:54,  2.01it/s]
  7%|██▍                                | 5303/78125 [45:30<10:02:12,  2.02it/s]
  7%|██▍                                | 5304/78125 [45:30<10:02:49,  2.01it/s]
  7%|██▍                                | 5305/78125 [45:31<10:03:06,  2.01it/s]
  7%|██▍                                | 5306/78125 [45:31<10:03:44,  2.01it/s]
  7%|██▍                                | 5307/78125 [45:32<10:03:35,  2.01it/s]
  7%|██▍                                | 5308/78125 [45:32<10:02:49,  2.01it/s]
  7%|██▍                                | 5309/78125 [45:33<10:03:09,  2.01it/s]
  7%|██▍                                | 5310/78125 [45:33<10:03:29,  2.01it/s]
  7%|██▍                                | 5311/78125 [45:34<10:03:17,  2.01it/s]
  7%|██▍                                | 5312/78125 [45:34<10:03:42,  2.01it/s]
  7%|██▍                                | 5313/78125 [45:35<10:03:12,  2.01it/s]
  7%|██▍                                | 5314/78125 [45:35<10:03:15,  2.01it/s]
  7%|██▍                                | 5315/78125 [45:36<10:03:37,  2.01it/s]
  7%|██▍                                | 5316/78125 [45:36<10:03:41,  2.01it/s]
  7%|██▍                                | 5317/78125 [45:37<10:03:05,  2.01it/s]
  7%|██▍                                | 5318/78125 [45:37<10:03:23,  2.01it/s]
  7%|██▍                                | 5319/78125 [45:38<10:03:37,  2.01it/s]
  7%|██▍                                | 5320/78125 [45:38<10:03:09,  2.01it/s]
  7%|██▍                                | 5321/78125 [45:39<10:02:39,  2.01it/s]
  7%|██▍                                | 5322/78125 [45:39<10:03:06,  2.01it/s]
  7%|██▍                                | 5323/78125 [45:40<10:03:00,  2.01it/s]
  7%|██▍                                | 5324/78125 [45:40<10:03:35,  2.01it/s]
  7%|██▍                                | 5325/78125 [45:41<10:02:45,  2.01it/s]
  7%|██▍                                | 5326/78125 [45:41<10:03:03,  2.01it/s]
  7%|██▍                                | 5327/78125 [45:42<10:02:50,  2.01it/s]
  7%|██▍                                | 5328/78125 [45:42<10:03:15,  2.01it/s]
  7%|██▍                                | 5329/78125 [45:43<10:03:37,  2.01it/s]
  7%|██▍                                | 5330/78125 [45:43<10:03:22,  2.01it/s]
  7%|██▍                                | 5331/78125 [45:44<10:03:08,  2.01it/s]
  7%|██▍                                | 5332/78125 [45:44<10:03:31,  2.01it/s]
  7%|██▍                                | 5333/78125 [45:45<10:03:39,  2.01it/s]
  7%|██▍                                | 5334/78125 [45:45<10:03:53,  2.01it/s]
  7%|██▍                                | 5335/78125 [45:46<10:04:26,  2.01it/s]
  7%|██▍                                | 5336/78125 [45:46<10:04:19,  2.01it/s]
  7%|██▍                                | 5337/78125 [45:47<10:04:27,  2.01it/s]
  7%|██▍                                | 5338/78125 [45:47<10:04:33,  2.01it/s]
  7%|██▍                                | 5339/78125 [45:48<10:04:17,  2.01it/s]
  7%|██▍                                | 5340/78125 [45:48<10:03:56,  2.01it/s]
  7%|██▍                                | 5341/78125 [45:49<10:04:12,  2.01it/s]
  7%|██▍                                | 5342/78125 [45:49<10:04:28,  2.01it/s]
  7%|██▍                                | 5343/78125 [45:50<10:04:17,  2.01it/s]
  7%|██▍                                | 5344/78125 [45:50<10:05:14,  2.00it/s]
  7%|██▍                                | 5345/78125 [45:51<10:05:13,  2.00it/s]
  7%|██▍                                | 5346/78125 [45:51<10:05:23,  2.00it/s]
  7%|██▍                                | 5347/78125 [45:52<10:05:51,  2.00it/s]
  7%|██▍                                | 5348/78125 [45:52<10:05:12,  2.00it/s]
  7%|██▍                                | 5349/78125 [45:53<10:05:06,  2.00it/s]
  7%|██▍                                | 5350/78125 [45:53<10:05:15,  2.00it/s]
  7%|██▍                                | 5351/78125 [45:54<10:05:01,  2.00it/s]
  7%|██▍                                | 5352/78125 [45:54<10:04:19,  2.01it/s]
  7%|██▍                                | 5353/78125 [45:55<10:04:51,  2.01it/s]
  7%|██▍                                | 5354/78125 [45:55<10:05:11,  2.00it/s]
  7%|██▍                                | 5355/78125 [45:56<10:04:55,  2.00it/s]
  7%|██▍                                | 5356/78125 [45:56<10:05:26,  2.00it/s]
  7%|██▍                                | 5357/78125 [45:57<10:05:43,  2.00it/s]
  7%|██▍                                | 5358/78125 [45:57<10:05:55,  2.00it/s]
  7%|██▍                                | 5359/78125 [45:58<10:05:50,  2.00it/s]
  7%|██▍                                | 5360/78125 [45:58<10:05:40,  2.00it/s]
  7%|██▍                                | 5361/78125 [45:59<10:05:43,  2.00it/s]
  7%|██▍                                | 5362/78125 [45:59<10:05:58,  2.00it/s]
  7%|██▍                                | 5363/78125 [46:00<10:06:03,  2.00it/s]
  7%|██▍                                | 5364/78125 [46:00<10:06:20,  2.00it/s]
  7%|██▍                                | 5365/78125 [46:01<10:06:13,  2.00it/s]
  7%|██▍                                | 5366/78125 [46:01<10:06:24,  2.00it/s]
  7%|██▍                                | 5367/78125 [46:02<10:05:38,  2.00it/s]
  7%|██▍                                | 5368/78125 [46:02<10:04:56,  2.00it/s]
  7%|██▍                                | 5369/78125 [46:03<10:04:41,  2.01it/s]
  7%|██▍                                | 5370/78125 [46:03<10:04:34,  2.01it/s]
  7%|██▍                                | 5371/78125 [46:04<10:05:12,  2.00it/s]
  7%|██▍                                | 5372/78125 [46:04<10:05:30,  2.00it/s]
  7%|██▍                                | 5373/78125 [46:05<10:06:05,  2.00it/s]
  7%|██▍                                | 5374/78125 [46:05<10:06:18,  2.00it/s]
  7%|██▍                                | 5375/78125 [46:06<10:06:02,  2.00it/s]
  7%|██▍                                | 5376/78125 [46:06<10:06:09,  2.00it/s]
  7%|██▍                                | 5377/78125 [46:07<10:06:05,  2.00it/s]
  7%|██▍                                | 5378/78125 [46:07<10:05:33,  2.00it/s]
  7%|██▍                                | 5379/78125 [46:08<10:05:41,  2.00it/s]
  7%|██▍                                | 5380/78125 [46:08<10:05:20,  2.00it/s]
  7%|██▍                                | 5381/78125 [46:09<10:04:39,  2.01it/s]
  7%|██▍                                | 5382/78125 [46:09<10:05:09,  2.00it/s]
  7%|██▍                                | 5383/78125 [46:10<10:05:16,  2.00it/s]
  7%|██▍                                | 5384/78125 [46:10<10:04:47,  2.00it/s]
  7%|██▍                                | 5385/78125 [46:11<10:04:54,  2.00it/s]
  7%|██▍                                | 5386/78125 [46:11<10:04:30,  2.01it/s]
  7%|██▍                                | 5387/78125 [46:12<10:03:11,  2.01it/s]
  7%|██▍                                | 5388/78125 [46:12<10:02:45,  2.01it/s]
  7%|██▍                                | 5389/78125 [46:13<10:04:05,  2.01it/s]
  7%|██▍                                | 5390/78125 [46:13<10:04:36,  2.01it/s]
  7%|██▍                                | 5391/78125 [46:14<10:04:19,  2.01it/s]
  7%|██▍                                | 5392/78125 [46:14<10:04:20,  2.01it/s]
  7%|██▍                                | 5393/78125 [46:15<10:03:37,  2.01it/s]
  7%|██▍                                | 5394/78125 [46:15<10:04:05,  2.01it/s]
  7%|██▍                                | 5395/78125 [46:16<10:04:23,  2.01it/s]
  7%|██▍                                | 5396/78125 [46:16<10:04:16,  2.01it/s]
  7%|██▍                                | 5397/78125 [46:17<10:04:42,  2.00it/s]
  7%|██▍                                | 5398/78125 [46:17<10:04:29,  2.01it/s]
  7%|██▍                                | 5399/78125 [46:18<10:04:22,  2.01it/s]
  7%|██▍                                | 5400/78125 [46:18<10:04:52,  2.00it/s]
  7%|██▍                                | 5401/78125 [46:19<10:03:56,  2.01it/s]
  7%|██▍                                | 5402/78125 [46:19<10:03:31,  2.01it/s]
  7%|██▍                                | 5403/78125 [46:20<10:03:31,  2.01it/s]
  7%|██▍                                | 5404/78125 [46:20<10:03:29,  2.01it/s]
  7%|██▍                                | 5405/78125 [46:21<10:03:29,  2.01it/s]
  7%|██▍                                | 5406/78125 [46:21<10:03:17,  2.01it/s]
  7%|██▍                                | 5407/78125 [46:21<10:02:46,  2.01it/s]
  7%|██▍                                | 5408/78125 [46:22<10:03:00,  2.01it/s]
  7%|██▍                                | 5409/78125 [46:22<10:03:04,  2.01it/s]
  7%|██▍                                | 5410/78125 [46:23<10:03:16,  2.01it/s]
  7%|██▍                                | 5411/78125 [46:23<10:02:33,  2.01it/s]
  7%|██▍                                | 5412/78125 [46:24<10:02:11,  2.01it/s]
  7%|██▍                                | 5413/78125 [46:24<10:02:22,  2.01it/s]
  7%|██▍                                | 5414/78125 [46:25<10:02:05,  2.01it/s]
  7%|██▍                                | 5415/78125 [46:25<10:02:14,  2.01it/s]
  7%|██▍                                | 5416/78125 [46:26<10:01:58,  2.01it/s]
  7%|██▍                                | 5417/78125 [46:26<10:02:34,  2.01it/s]
  7%|██▍                                | 5418/78125 [46:27<10:02:24,  2.01it/s]
  7%|██▍                                | 5419/78125 [46:27<10:01:33,  2.01it/s]
  7%|██▍                                | 5420/78125 [46:28<10:01:08,  2.02it/s]
  7%|██▍                                | 5421/78125 [46:28<10:01:51,  2.01it/s]
  7%|██▍                                | 5422/78125 [46:29<10:01:48,  2.01it/s]
  7%|██▍                                | 5423/78125 [46:29<10:02:09,  2.01it/s]
  7%|██▍                                | 5424/78125 [46:30<10:01:54,  2.01it/s]
  7%|██▍                                | 5425/78125 [46:30<10:02:38,  2.01it/s]
  7%|██▍                                | 5426/78125 [46:31<10:01:56,  2.01it/s]
  7%|██▍                                | 5427/78125 [46:31<10:01:27,  2.01it/s]
  7%|██▍                                | 5428/78125 [46:32<10:01:03,  2.02it/s]
  7%|██▍                                | 5429/78125 [46:32<10:01:39,  2.01it/s]
  7%|██▍                                | 5430/78125 [46:33<10:02:03,  2.01it/s]
  7%|██▍                                | 5431/78125 [46:33<10:01:43,  2.01it/s]
  7%|██▍                                | 5432/78125 [46:34<10:02:26,  2.01it/s]
  7%|██▍                                | 5433/78125 [46:34<10:01:41,  2.01it/s]
  7%|██▍                                | 5434/78125 [46:35<10:01:12,  2.02it/s]
  7%|██▍                                | 5435/78125 [46:35<10:00:55,  2.02it/s]
  7%|██▍                                | 5436/78125 [46:36<10:01:34,  2.01it/s]
  7%|██▍                                | 5437/78125 [46:36<10:01:29,  2.01it/s]
  7%|██▍                                | 5438/78125 [46:37<10:01:14,  2.01it/s]
  7%|██▍                                | 5439/78125 [46:37<10:01:18,  2.01it/s]
  7%|██▍                                | 5440/78125 [46:38<10:01:39,  2.01it/s]
  7%|██▍                                | 5441/78125 [46:38<10:01:32,  2.01it/s]
  7%|██▍                                | 5442/78125 [46:39<10:01:26,  2.01it/s]
  7%|██▍                                | 5443/78125 [46:39<10:01:26,  2.01it/s]
  7%|██▍                                | 5444/78125 [46:40<10:01:04,  2.02it/s]
  7%|██▍                                | 5445/78125 [46:40<10:01:07,  2.02it/s]
  7%|██▍                                | 5446/78125 [46:41<10:01:30,  2.01it/s]
  7%|██▍                                | 5447/78125 [46:41<10:01:20,  2.01it/s]
  7%|██▍                                | 5448/78125 [46:42<10:01:00,  2.02it/s]
  7%|██▍                                | 5449/78125 [46:42<10:00:29,  2.02it/s]
  7%|██▍                                | 5450/78125 [46:43<10:00:46,  2.02it/s]
  7%|██▍                                | 5451/78125 [46:43<10:01:07,  2.01it/s]
  7%|██▍                                | 5452/78125 [46:44<10:01:13,  2.01it/s]
  7%|██▍                                | 5453/78125 [46:44<10:00:06,  2.02it/s]
  7%|██▍                                | 5454/78125 [46:45<10:01:12,  2.01it/s]
  7%|██▍                                | 5455/78125 [46:45<10:01:05,  2.01it/s]
  7%|██▍                                | 5456/78125 [46:46<10:00:16,  2.02it/s]
  7%|██▍                                | 5457/78125 [46:46<10:00:10,  2.02it/s]
  7%|██▍                                | 5458/78125 [46:47<10:00:26,  2.02it/s]
  7%|██▍                                | 5459/78125 [46:47<10:00:41,  2.02it/s]
  7%|██▌                                 | 5460/78125 [46:48<9:59:59,  2.02it/s]
  7%|██▍                                | 5461/78125 [46:48<10:00:17,  2.02it/s]
  7%|██▍                                | 5462/78125 [46:49<10:00:28,  2.02it/s]
  7%|██▍                                | 5463/78125 [46:49<10:00:59,  2.02it/s]
  7%|██▍                                | 5464/78125 [46:50<10:01:16,  2.01it/s]
  7%|██▍                                | 5465/78125 [46:50<10:01:06,  2.01it/s]
  7%|██▍                                | 5466/78125 [46:51<10:00:51,  2.02it/s]
  7%|██▍                                | 5467/78125 [46:51<10:01:16,  2.01it/s]
  7%|██▍                                | 5468/78125 [46:52<10:01:17,  2.01it/s]
  7%|██▍                                | 5469/78125 [46:52<10:00:51,  2.02it/s]
  7%|██▍                                | 5470/78125 [46:53<10:00:23,  2.02it/s]
  7%|██▍                                | 5471/78125 [46:53<10:01:17,  2.01it/s]
  7%|██▍                                | 5472/78125 [46:54<10:00:35,  2.02it/s]
  7%|██▍                                | 5473/78125 [46:54<10:01:08,  2.01it/s]
  7%|██▍                                | 5474/78125 [46:55<10:00:51,  2.02it/s]
  7%|██▍                                | 5475/78125 [46:55<10:01:24,  2.01it/s]
  7%|██▍                                | 5476/78125 [46:56<10:01:07,  2.01it/s]
  7%|██▍                                | 5477/78125 [46:56<10:00:24,  2.02it/s]
  7%|██▍                                | 5478/78125 [46:57<10:00:34,  2.02it/s]
  7%|██▍                                | 5479/78125 [46:57<10:01:08,  2.01it/s]
  7%|██▍                                | 5480/78125 [46:58<10:01:02,  2.01it/s]
  7%|██▍                                | 5481/78125 [46:58<10:00:48,  2.02it/s]
  7%|██▍                                | 5482/78125 [46:59<10:00:47,  2.02it/s]
  7%|██▍                                | 5483/78125 [46:59<10:01:22,  2.01it/s]
  7%|██▍                                | 5484/78125 [47:00<10:01:47,  2.01it/s]
  7%|██▍                                | 5485/78125 [47:00<10:01:24,  2.01it/s]
  7%|██▍                                | 5486/78125 [47:01<10:00:52,  2.01it/s]
  7%|██▍                                | 5487/78125 [47:01<10:00:29,  2.02it/s]
  7%|██▍                                | 5488/78125 [47:02<10:01:09,  2.01it/s]
  7%|██▍                                | 5489/78125 [47:02<10:01:30,  2.01it/s]
  7%|██▍                                | 5490/78125 [47:03<10:01:53,  2.01it/s]
  7%|██▍                                | 5491/78125 [47:03<10:02:06,  2.01it/s]
  7%|██▍                                | 5492/78125 [47:04<10:02:18,  2.01it/s]
  7%|██▍                                | 5493/78125 [47:04<10:02:02,  2.01it/s]
  7%|██▍                                | 5494/78125 [47:05<10:01:32,  2.01it/s]
  7%|██▍                                | 5495/78125 [47:05<10:00:54,  2.01it/s]
  7%|██▍                                | 5496/78125 [47:06<10:00:59,  2.01it/s]
  7%|██▍                                | 5497/78125 [47:06<10:01:36,  2.01it/s]
  7%|██▍                                | 5498/78125 [47:07<10:00:53,  2.01it/s]
  7%|██▍                                | 5499/78125 [47:07<10:00:43,  2.01it/s]
  7%|██▍                                | 5500/78125 [47:08<10:00:36,  2.02it/s]
  7%|██▍                                | 5501/78125 [47:08<10:01:28,  2.01it/s]
  7%|██▍                                | 5502/78125 [47:09<10:01:33,  2.01it/s]
  7%|██▍                                | 5503/78125 [47:09<10:01:40,  2.01it/s]
  7%|██▍                                | 5504/78125 [47:10<10:01:57,  2.01it/s]
  7%|██▍                                | 5505/78125 [47:10<10:01:53,  2.01it/s]
  7%|██▍                                | 5506/78125 [47:11<10:01:08,  2.01it/s]
  7%|██▍                                | 5507/78125 [47:11<10:00:59,  2.01it/s]
  7%|██▍                                | 5508/78125 [47:12<10:00:31,  2.02it/s]
  7%|██▍                                | 5509/78125 [47:12<10:00:12,  2.02it/s]
  7%|██▍                                | 5510/78125 [47:13<10:00:46,  2.01it/s]
  7%|██▍                                | 5511/78125 [47:13<10:00:40,  2.01it/s]
  7%|██▌                                 | 5512/78125 [47:14<9:59:57,  2.02it/s]
  7%|██▌                                 | 5513/78125 [47:14<9:59:51,  2.02it/s]
  7%|██▍                                | 5514/78125 [47:15<10:00:12,  2.02it/s]
  7%|██▍                                | 5515/78125 [47:15<10:00:19,  2.02it/s]
  7%|██▍                                | 5516/78125 [47:16<10:00:33,  2.02it/s]
  7%|██▌                                 | 5517/78125 [47:16<9:59:45,  2.02it/s]
  7%|██▌                                 | 5518/78125 [47:17<9:59:33,  2.02it/s]
  7%|██▌                                 | 5519/78125 [47:17<9:59:58,  2.02it/s]
  7%|██▍                                | 5520/78125 [47:18<10:00:09,  2.02it/s]
  7%|██▍                                | 5521/78125 [47:18<10:00:39,  2.01it/s]
  7%|██▍                                | 5522/78125 [47:19<10:00:28,  2.02it/s]
  7%|██▍                                | 5523/78125 [47:19<10:00:50,  2.01it/s]
  7%|██▍                                | 5524/78125 [47:20<10:00:30,  2.02it/s]
  7%|██▍                                | 5525/78125 [47:20<10:00:32,  2.01it/s]
  7%|██▍                                | 5526/78125 [47:21<10:00:42,  2.01it/s]
  7%|██▍                                | 5527/78125 [47:21<10:00:13,  2.02it/s]
  7%|██▍                                | 5528/78125 [47:22<10:00:10,  2.02it/s]
  7%|██▍                                | 5529/78125 [47:22<10:00:21,  2.02it/s]
  7%|██▍                                | 5530/78125 [47:23<10:00:00,  2.02it/s]
  7%|██▍                                | 5531/78125 [47:23<10:00:10,  2.02it/s]
  7%|██▍                                | 5532/78125 [47:24<10:00:27,  2.01it/s]
  7%|██▍                                | 5533/78125 [47:24<10:00:53,  2.01it/s]
  7%|██▍                                | 5534/78125 [47:25<10:00:31,  2.01it/s]
  7%|██▍                                | 5535/78125 [47:25<10:00:37,  2.01it/s]
  7%|██▍                                | 5536/78125 [47:26<10:00:30,  2.01it/s]
  7%|██▍                                | 5537/78125 [47:26<10:00:25,  2.01it/s]
  7%|██▌                                 | 5538/78125 [47:27<9:59:45,  2.02it/s]
  7%|██▌                                 | 5539/78125 [47:27<9:59:37,  2.02it/s]
  7%|██▌                                 | 5540/78125 [47:28<9:59:51,  2.02it/s]
  7%|██▌                                 | 5541/78125 [47:28<9:59:25,  2.02it/s]
  7%|██▌                                 | 5542/78125 [47:29<9:59:38,  2.02it/s]
  7%|██▍                                | 5543/78125 [47:29<10:00:28,  2.01it/s]
  7%|██▌                                 | 5544/78125 [47:30<9:59:57,  2.02it/s]
  7%|██▌                                 | 5545/78125 [47:30<9:59:55,  2.02it/s]
  7%|██▍                                | 5546/78125 [47:30<10:00:10,  2.02it/s]
  7%|██▌                                 | 5547/78125 [47:31<9:59:29,  2.02it/s]
  7%|██▌                                 | 5548/78125 [47:31<9:59:27,  2.02it/s]
  7%|██▌                                 | 5549/78125 [47:32<9:59:37,  2.02it/s]
  7%|██▌                                 | 5550/78125 [47:32<9:59:32,  2.02it/s]
  7%|██▌                                 | 5551/78125 [47:33<9:59:29,  2.02it/s]
  7%|██▌                                 | 5552/78125 [47:33<9:59:13,  2.02it/s]
  7%|██▌                                 | 5553/78125 [47:34<9:58:53,  2.02it/s]
  7%|██▌                                 | 5554/78125 [47:34<9:59:03,  2.02it/s]
  7%|██▌                                 | 5555/78125 [47:35<9:58:42,  2.02it/s]
  7%|██▌                                 | 5556/78125 [47:35<9:58:41,  2.02it/s]
  7%|██▌                                 | 5557/78125 [47:36<9:59:39,  2.02it/s]
  7%|██▍                                | 5558/78125 [47:36<10:00:01,  2.02it/s]
  7%|██▍                                | 5559/78125 [47:37<10:00:07,  2.02it/s]
  7%|██▍                                | 5560/78125 [47:37<10:00:10,  2.02it/s]
  7%|██▍                                | 5561/78125 [47:38<10:00:11,  2.02it/s]
  7%|██▌                                 | 5562/78125 [47:38<9:59:55,  2.02it/s]
  7%|██▌                                 | 5563/78125 [47:39<9:59:49,  2.02it/s]
  7%|██▌                                 | 5564/78125 [47:39<9:59:59,  2.02it/s]
  7%|██▌                                 | 5565/78125 [47:40<9:59:16,  2.02it/s]
  7%|██▌                                 | 5566/78125 [47:40<9:59:38,  2.02it/s]
  7%|██▌                                 | 5567/78125 [47:41<9:59:44,  2.02it/s]
  7%|██▌                                 | 5568/78125 [47:41<9:59:35,  2.02it/s]
  7%|██▌                                 | 5569/78125 [47:42<9:59:45,  2.02it/s]
  7%|██▍                                | 5570/78125 [47:42<10:00:25,  2.01it/s]
  7%|██▍                                | 5571/78125 [47:43<10:00:03,  2.02it/s]
  7%|██▌                                 | 5572/78125 [47:43<9:58:38,  2.02it/s]
  7%|██▌                                 | 5573/78125 [47:44<9:59:03,  2.02it/s]
  7%|██▌                                 | 5574/78125 [47:44<9:59:59,  2.02it/s]
  7%|██▍                                | 5575/78125 [47:45<10:00:10,  2.01it/s]
  7%|██▌                                 | 5576/78125 [47:45<9:59:53,  2.02it/s]
  7%|██▌                                 | 5577/78125 [47:46<9:59:36,  2.02it/s]
  7%|██▌                                 | 5578/78125 [47:46<9:59:32,  2.02it/s]
  7%|██▌                                 | 5579/78125 [47:47<9:59:35,  2.02it/s]
  7%|██▌                                 | 5580/78125 [47:47<9:59:34,  2.02it/s]
  7%|██▌                                 | 5581/78125 [47:48<9:59:54,  2.02it/s]
  7%|██▌                                | 5582/78125 [47:48<10:00:37,  2.01it/s]
  7%|██▌                                | 5583/78125 [47:49<10:00:52,  2.01it/s]
  7%|██▌                                | 5584/78125 [47:49<10:00:01,  2.01it/s]
  7%|██▌                                | 5585/78125 [47:50<10:00:46,  2.01it/s]
  7%|██▌                                | 5586/78125 [47:50<10:00:38,  2.01it/s]
  7%|██▌                                | 5587/78125 [47:51<10:00:19,  2.01it/s]
  7%|██▌                                | 5588/78125 [47:51<10:00:12,  2.01it/s]
  7%|██▌                                | 5589/78125 [47:52<10:00:19,  2.01it/s]
  7%|██▌                                 | 5590/78125 [47:52<9:59:59,  2.01it/s]
  7%|██▌                                | 5591/78125 [47:53<10:00:01,  2.01it/s]
  7%|██▌                                | 5592/78125 [47:53<10:00:50,  2.01it/s]
  7%|██▌                                | 5593/78125 [47:54<10:00:41,  2.01it/s]
  7%|██▌                                | 5594/78125 [47:54<10:00:32,  2.01it/s]
  7%|██▌                                | 5595/78125 [47:55<10:00:43,  2.01it/s]
  7%|██▌                                | 5596/78125 [47:55<10:00:22,  2.01it/s]
  7%|██▌                                | 5597/78125 [47:56<10:00:25,  2.01it/s]
  7%|██▌                                | 5598/78125 [47:56<10:01:30,  2.01it/s]
  7%|██▌                                | 5599/78125 [47:57<10:00:50,  2.01it/s]
  7%|██▌                                | 5600/78125 [47:57<10:00:39,  2.01it/s]
  7%|██▌                                | 5601/78125 [47:58<10:01:18,  2.01it/s]
  7%|██▌                                | 5602/78125 [47:58<10:00:49,  2.01it/s]
  7%|██▌                                | 5603/78125 [47:59<10:01:42,  2.01it/s]
  7%|██▌                                | 5604/78125 [47:59<10:01:06,  2.01it/s]
  7%|██▌                                | 5605/78125 [48:00<10:01:26,  2.01it/s]
  7%|██▌                                | 5606/78125 [48:00<10:00:44,  2.01it/s]
  7%|██▌                                | 5607/78125 [48:01<10:00:52,  2.01it/s]
  7%|██▌                                | 5608/78125 [48:01<10:01:24,  2.01it/s]
  7%|██▌                                | 5609/78125 [48:02<10:00:58,  2.01it/s]
  7%|██▌                                | 5610/78125 [48:02<10:01:00,  2.01it/s]
  7%|██▌                                | 5611/78125 [48:03<10:01:02,  2.01it/s]
  7%|██▌                                | 5612/78125 [48:03<10:01:56,  2.01it/s]
  7%|██▌                                | 5613/78125 [48:04<10:01:28,  2.01it/s]
  7%|██▌                                | 5614/78125 [48:04<10:02:34,  2.01it/s]
  7%|██▌                                | 5615/78125 [48:05<10:02:32,  2.01it/s]
  7%|██▌                                | 5616/78125 [48:05<10:02:37,  2.01it/s]
  7%|██▌                                | 5617/78125 [48:06<10:02:23,  2.01it/s]
  7%|██▌                                | 5618/78125 [48:06<10:03:33,  2.00it/s]
  7%|██▌                                | 5619/78125 [48:07<10:03:43,  2.00it/s]
  7%|██▌                                | 5620/78125 [48:07<10:03:19,  2.00it/s]
  7%|██▌                                | 5621/78125 [48:08<10:03:12,  2.00it/s]
  7%|██▌                                | 5622/78125 [48:08<10:02:43,  2.00it/s]
  7%|██▌                                | 5623/78125 [48:09<10:02:44,  2.00it/s]
  7%|██▌                                | 5624/78125 [48:09<10:02:09,  2.01it/s]
  7%|██▌                                | 5625/78125 [48:10<10:02:15,  2.01it/s]
  7%|██▌                                | 5626/78125 [48:10<10:01:50,  2.01it/s]
  7%|██▌                                | 5627/78125 [48:11<10:01:15,  2.01it/s]
  7%|██▌                                | 5628/78125 [48:11<10:00:48,  2.01it/s]
  7%|██▌                                | 5629/78125 [48:12<10:00:57,  2.01it/s]
  7%|██▌                                | 5630/78125 [48:12<10:01:48,  2.01it/s]
  7%|██▌                                | 5631/78125 [48:13<10:01:36,  2.01it/s]
  7%|██▌                                | 5632/78125 [48:13<10:01:37,  2.01it/s]
  7%|██▌                                | 5633/78125 [48:14<10:01:58,  2.01it/s]
  7%|██▌                                | 5634/78125 [48:14<10:02:22,  2.01it/s]
  7%|██▌                                | 5635/78125 [48:15<10:02:17,  2.01it/s]
  7%|██▌                                | 5636/78125 [48:15<10:01:53,  2.01it/s]
  7%|██▌                                | 5637/78125 [48:16<10:02:12,  2.01it/s]
  7%|██▌                                | 5638/78125 [48:16<10:02:19,  2.01it/s]
  7%|██▌                                | 5639/78125 [48:17<10:02:40,  2.00it/s]
  7%|██▌                                | 5640/78125 [48:17<10:03:16,  2.00it/s]
  7%|██▌                                | 5641/78125 [48:18<10:03:13,  2.00it/s]
  7%|██▌                                | 5642/78125 [48:18<10:03:08,  2.00it/s]
  7%|██▌                                | 5643/78125 [48:19<10:02:13,  2.01it/s]
  7%|██▌                                | 5644/78125 [48:19<10:02:24,  2.01it/s]
  7%|██▌                                | 5645/78125 [48:20<10:02:14,  2.01it/s]
  7%|██▌                                | 5646/78125 [48:20<10:02:11,  2.01it/s]
  7%|██▌                                | 5647/78125 [48:21<10:02:13,  2.01it/s]
  7%|██▌                                | 5648/78125 [48:21<10:02:47,  2.00it/s]
  7%|██▌                                | 5649/78125 [48:22<10:02:32,  2.00it/s]
  7%|██▌                                | 5650/78125 [48:22<10:02:29,  2.00it/s]
  7%|██▌                                | 5651/78125 [48:23<10:02:49,  2.00it/s]
  7%|██▌                                | 5652/78125 [48:23<10:02:43,  2.00it/s]
  7%|██▌                                | 5653/78125 [48:24<10:02:15,  2.01it/s]
  7%|██▌                                | 5654/78125 [48:24<10:03:25,  2.00it/s]
  7%|██▌                                | 5655/78125 [48:25<10:03:02,  2.00it/s]
  7%|██▌                                | 5656/78125 [48:25<10:02:32,  2.00it/s]
  7%|██▌                                | 5657/78125 [48:26<10:02:14,  2.01it/s]
  7%|██▌                                | 5658/78125 [48:26<10:03:13,  2.00it/s]
  7%|██▌                                | 5659/78125 [48:27<10:02:27,  2.00it/s]
  7%|██▌                                | 5660/78125 [48:27<10:02:33,  2.00it/s]
  7%|██▌                                | 5661/78125 [48:28<10:02:50,  2.00it/s]
  7%|██▌                                | 5662/78125 [48:28<10:02:39,  2.00it/s]
  7%|██▌                                | 5663/78125 [48:29<10:02:00,  2.01it/s]
  7%|██▌                                | 5664/78125 [48:29<10:01:35,  2.01it/s]
  7%|██▌                                | 5665/78125 [48:30<10:01:20,  2.01it/s]
  7%|██▌                                | 5666/78125 [48:30<10:02:02,  2.01it/s]
  7%|██▌                                | 5667/78125 [48:31<10:02:10,  2.01it/s]
  7%|██▌                                | 5668/78125 [48:31<10:02:38,  2.00it/s]
  7%|██▌                                | 5669/78125 [48:32<10:02:21,  2.00it/s]
  7%|██▌                                | 5670/78125 [48:32<10:03:00,  2.00it/s]
  7%|██▌                                | 5671/78125 [48:33<10:02:37,  2.00it/s]
  7%|██▌                                | 5672/78125 [48:33<10:02:30,  2.00it/s]
  7%|██▌                                | 5673/78125 [48:34<10:02:32,  2.00it/s]
  7%|██▌                                | 5674/78125 [48:34<10:03:23,  2.00it/s]
  7%|██▌                                | 5675/78125 [48:35<10:03:44,  2.00it/s]
  7%|██▌                                | 5676/78125 [48:35<10:03:18,  2.00it/s]
  7%|██▌                                | 5677/78125 [48:36<10:03:48,  2.00it/s]
  7%|██▌                                | 5678/78125 [48:36<10:02:54,  2.00it/s]
  7%|██▌                                | 5679/78125 [48:37<10:03:15,  2.00it/s]
  7%|██▌                                | 5680/78125 [48:37<10:03:14,  2.00it/s]
  7%|██▌                                | 5681/78125 [48:38<10:03:21,  2.00it/s]
  7%|██▌                                | 5682/78125 [48:38<10:03:02,  2.00it/s]
  7%|██▌                                | 5683/78125 [48:39<10:03:08,  2.00it/s]
  7%|██▌                                | 5684/78125 [48:39<10:03:17,  2.00it/s]
  7%|██▌                                | 5685/78125 [48:40<10:03:53,  2.00it/s]
  7%|██▌                                | 5686/78125 [48:40<10:03:20,  2.00it/s]
  7%|██▌                                | 5687/78125 [48:41<10:03:00,  2.00it/s]
  7%|██▌                                | 5688/78125 [48:41<10:03:08,  2.00it/s]
  7%|██▌                                | 5689/78125 [48:42<10:03:18,  2.00it/s]
  7%|██▌                                | 5690/78125 [48:42<10:02:44,  2.00it/s]
  7%|██▌                                | 5691/78125 [48:43<10:02:33,  2.00it/s]
  7%|██▌                                | 5692/78125 [48:43<10:03:16,  2.00it/s]
  7%|██▌                                | 5693/78125 [48:44<10:03:08,  2.00it/s]
  7%|██▌                                | 5694/78125 [48:44<10:04:19,  2.00it/s]
  7%|██▌                                | 5695/78125 [48:45<10:04:18,  2.00it/s]
  7%|██▌                                | 5696/78125 [48:45<10:03:25,  2.00it/s]
  7%|██▌                                | 5697/78125 [48:46<10:03:16,  2.00it/s]
  7%|██▌                                | 5698/78125 [48:46<10:03:48,  2.00it/s]
  7%|██▌                                | 5699/78125 [48:47<10:03:41,  2.00it/s]
  7%|██▌                                | 5700/78125 [48:47<10:03:57,  2.00it/s]
  7%|██▌                                | 5701/78125 [48:48<10:04:05,  2.00it/s]
  7%|██▌                                | 5702/78125 [48:48<10:03:35,  2.00it/s]
  7%|██▌                                | 5703/78125 [48:49<10:03:26,  2.00it/s]
  7%|██▌                                | 5704/78125 [48:49<10:03:01,  2.00it/s]
  7%|██▌                                | 5705/78125 [48:50<10:02:39,  2.00it/s]
  7%|██▌                                | 5706/78125 [48:50<10:02:40,  2.00it/s]
  7%|██▌                                | 5707/78125 [48:51<10:02:01,  2.00it/s]
  7%|██▌                                | 5708/78125 [48:51<10:01:42,  2.01it/s]
  7%|██▌                                | 5709/78125 [48:52<10:02:32,  2.00it/s]
  7%|██▌                                | 5710/78125 [48:52<10:02:12,  2.00it/s]
  7%|██▌                                | 5711/78125 [48:53<10:02:17,  2.00it/s]
  7%|██▌                                | 5712/78125 [48:53<10:02:30,  2.00it/s]
  7%|██▌                                | 5713/78125 [48:54<10:02:55,  2.00it/s]
  7%|██▌                                | 5714/78125 [48:54<10:02:39,  2.00it/s]
  7%|██▌                                | 5715/78125 [48:55<10:03:01,  2.00it/s]
  7%|██▌                                | 5716/78125 [48:55<10:03:31,  2.00it/s]
  7%|██▌                                | 5717/78125 [48:56<10:03:19,  2.00it/s]
  7%|██▌                                | 5718/78125 [48:56<10:02:47,  2.00it/s]
  7%|██▌                                | 5719/78125 [48:57<10:02:33,  2.00it/s]
  7%|██▌                                | 5720/78125 [48:57<10:01:48,  2.01it/s]
  7%|██▌                                | 5721/78125 [48:58<10:01:42,  2.01it/s]
  7%|██▌                                | 5722/78125 [48:58<10:02:06,  2.00it/s]
  7%|██▌                                | 5723/78125 [48:59<10:02:27,  2.00it/s]
  7%|██▌                                | 5724/78125 [48:59<10:02:52,  2.00it/s]
  7%|██▌                                | 5725/78125 [49:00<10:02:35,  2.00it/s]
  7%|██▌                                | 5726/78125 [49:00<10:03:10,  2.00it/s]
  7%|██▌                                | 5727/78125 [49:01<10:03:16,  2.00it/s]
  7%|██▌                                | 5728/78125 [49:01<10:02:25,  2.00it/s]
  7%|██▌                                | 5729/78125 [49:02<10:01:29,  2.01it/s]
  7%|██▌                                | 5730/78125 [49:02<10:01:50,  2.00it/s]
  7%|██▌                                | 5731/78125 [49:03<10:02:22,  2.00it/s]
  7%|██▌                                | 5732/78125 [49:03<10:03:02,  2.00it/s]
  7%|██▌                                | 5733/78125 [49:04<10:03:04,  2.00it/s]
  7%|██▌                                | 5734/78125 [49:04<10:03:17,  2.00it/s]
  7%|██▌                                | 5735/78125 [49:05<10:02:54,  2.00it/s]
  7%|██▌                                | 5736/78125 [49:05<10:02:11,  2.00it/s]
  7%|██▌                                | 5737/78125 [49:06<10:02:50,  2.00it/s]
  7%|██▌                                | 5738/78125 [49:06<10:02:36,  2.00it/s]
  7%|██▌                                | 5739/78125 [49:07<10:02:00,  2.00it/s]
  7%|██▌                                | 5740/78125 [49:07<10:01:12,  2.01it/s]
  7%|██▌                                | 5741/78125 [49:08<10:01:25,  2.01it/s]
  7%|██▌                                | 5742/78125 [49:08<10:01:41,  2.01it/s]
  7%|██▌                                | 5743/78125 [49:09<10:02:00,  2.00it/s]
  7%|██▌                                | 5744/78125 [49:09<10:01:24,  2.01it/s]
  7%|██▌                                | 5745/78125 [49:10<10:02:39,  2.00it/s]
  7%|██▌                                | 5746/78125 [49:10<10:02:21,  2.00it/s]
  7%|██▌                                | 5747/78125 [49:11<10:01:48,  2.00it/s]
  7%|██▌                                | 5748/78125 [49:11<10:02:11,  2.00it/s]
  7%|██▌                                | 5749/78125 [49:12<10:02:22,  2.00it/s]
  7%|██▌                                | 5750/78125 [49:12<10:02:10,  2.00it/s]
  7%|██▌                                | 5751/78125 [49:13<10:02:39,  2.00it/s]
  7%|██▌                                | 5752/78125 [49:13<10:02:08,  2.00it/s]
  7%|██▌                                | 5753/78125 [49:14<10:02:44,  2.00it/s]
  7%|██▌                                | 5754/78125 [49:14<10:02:52,  2.00it/s]
  7%|██▌                                | 5755/78125 [49:15<10:03:27,  2.00it/s]
  7%|██▌                                | 5756/78125 [49:15<10:02:34,  2.00it/s]
  7%|██▌                                | 5757/78125 [49:16<10:02:08,  2.00it/s]
  7%|██▌                                | 5758/78125 [49:16<10:01:51,  2.00it/s]
  7%|██▌                                | 5759/78125 [49:17<10:01:50,  2.00it/s]
  7%|██▌                                | 5760/78125 [49:17<10:00:50,  2.01it/s]
  7%|██▌                                | 5761/78125 [49:18<10:01:50,  2.00it/s]
  7%|██▌                                | 5762/78125 [49:18<10:01:27,  2.01it/s]
  7%|██▌                                | 5763/78125 [49:19<10:01:20,  2.01it/s]
  7%|██▌                                | 5764/78125 [49:19<10:01:39,  2.00it/s]
  7%|██▌                                | 5765/78125 [49:20<10:02:05,  2.00it/s]
  7%|██▌                                | 5766/78125 [49:20<10:01:30,  2.00it/s]
  7%|██▌                                | 5767/78125 [49:21<10:01:37,  2.00it/s]
  7%|██▌                                | 5768/78125 [49:21<10:01:54,  2.00it/s]
  7%|██▌                                | 5769/78125 [49:22<10:01:36,  2.00it/s]
  7%|██▌                                | 5770/78125 [49:22<10:01:36,  2.00it/s]
  7%|██▌                                | 5771/78125 [49:23<10:02:02,  2.00it/s]
  7%|██▌                                | 5772/78125 [49:23<10:02:08,  2.00it/s]
  7%|██▌                                | 5773/78125 [49:24<10:01:44,  2.00it/s]
  7%|██▌                                | 5774/78125 [49:24<10:01:58,  2.00it/s]
  7%|██▌                                | 5775/78125 [49:25<10:02:27,  2.00it/s]
  7%|██▌                                | 5776/78125 [49:25<10:03:16,  2.00it/s]
  7%|██▌                                | 5777/78125 [49:26<10:03:20,  2.00it/s]
  7%|██▌                                | 5778/78125 [49:26<10:02:33,  2.00it/s]
  7%|██▌                                | 5779/78125 [49:27<10:03:23,  2.00it/s]
  7%|██▌                                | 5780/78125 [49:27<10:03:51,  2.00it/s]
  7%|██▌                                | 5781/78125 [49:28<10:03:29,  2.00it/s]
  7%|██▌                                | 5782/78125 [49:28<10:03:03,  2.00it/s]
  7%|██▌                                | 5783/78125 [49:29<10:02:30,  2.00it/s]
  7%|██▌                                | 5784/78125 [49:29<10:02:28,  2.00it/s]
  7%|██▌                                | 5785/78125 [49:30<10:01:48,  2.00it/s]
  7%|██▌                                | 5786/78125 [49:30<10:02:41,  2.00it/s]
  7%|██▌                                | 5787/78125 [49:31<10:02:14,  2.00it/s]
  7%|██▌                                | 5788/78125 [49:31<10:01:51,  2.00it/s]
  7%|██▌                                | 5789/78125 [49:32<10:02:15,  2.00it/s]
  7%|██▌                                | 5790/78125 [49:32<10:02:45,  2.00it/s]
  7%|██▌                                | 5791/78125 [49:33<10:02:23,  2.00it/s]
  7%|██▌                                | 5792/78125 [49:33<10:02:27,  2.00it/s]
  7%|██▌                                | 5793/78125 [49:34<10:03:07,  2.00it/s]
  7%|██▌                                | 5794/78125 [49:34<10:03:40,  2.00it/s]
  7%|██▌                                | 5795/78125 [49:35<10:03:17,  2.00it/s]
  7%|██▌                                | 5796/78125 [49:35<10:03:08,  2.00it/s]
  7%|██▌                                | 5797/78125 [49:36<10:02:40,  2.00it/s]
  7%|██▌                                | 5798/78125 [49:36<10:02:44,  2.00it/s]
  7%|██▌                                | 5799/78125 [49:37<10:02:06,  2.00it/s]
  7%|██▌                                | 5800/78125 [49:37<10:02:05,  2.00it/s]
  7%|██▌                                | 5801/78125 [49:38<10:01:34,  2.00it/s]
  7%|██▌                                | 5802/78125 [49:38<10:01:34,  2.00it/s]
  7%|██▌                                | 5803/78125 [49:39<10:01:27,  2.00it/s]
  7%|██▌                                | 5804/78125 [49:39<10:02:10,  2.00it/s]
  7%|██▌                                | 5805/78125 [49:40<10:01:55,  2.00it/s]
  7%|██▌                                | 5806/78125 [49:40<10:02:23,  2.00it/s]
  7%|██▌                                | 5807/78125 [49:41<10:01:34,  2.00it/s]
  7%|██▌                                | 5808/78125 [49:41<10:01:52,  2.00it/s]
  7%|██▌                                | 5809/78125 [49:42<10:01:28,  2.00it/s]
  7%|██▌                                | 5810/78125 [49:42<10:01:51,  2.00it/s]
  7%|██▌                                | 5811/78125 [49:43<10:01:45,  2.00it/s]
  7%|██▌                                | 5812/78125 [49:43<10:01:08,  2.00it/s]
  7%|██▌                                | 5813/78125 [49:44<10:01:15,  2.00it/s]
  7%|██▌                                | 5814/78125 [49:44<10:01:35,  2.00it/s]
  7%|██▌                                | 5815/78125 [49:45<10:01:45,  2.00it/s]
  7%|██▌                                | 5816/78125 [49:45<10:01:43,  2.00it/s]
  7%|██▌                                | 5817/78125 [49:46<10:01:54,  2.00it/s]
  7%|██▌                                | 5818/78125 [49:46<10:02:22,  2.00it/s]
  7%|██▌                                | 5819/78125 [49:47<10:02:11,  2.00it/s]
  7%|██▌                                | 5820/78125 [49:47<10:02:05,  2.00it/s]
  7%|██▌                                | 5821/78125 [49:48<10:01:19,  2.00it/s]
  7%|██▌                                | 5822/78125 [49:48<10:01:37,  2.00it/s]
  7%|██▌                                | 5823/78125 [49:49<10:01:48,  2.00it/s]
  7%|██▌                                | 5824/78125 [49:49<10:01:44,  2.00it/s]
  7%|██▌                                | 5825/78125 [49:50<10:01:48,  2.00it/s]
  7%|██▌                                | 5826/78125 [49:50<10:01:43,  2.00it/s]
  7%|██▌                                | 5827/78125 [49:51<10:01:51,  2.00it/s]
  7%|██▌                                | 5828/78125 [49:51<10:01:47,  2.00it/s]
  7%|██▌                                | 5829/78125 [49:52<10:01:27,  2.00it/s]
  7%|██▌                                | 5830/78125 [49:52<10:01:18,  2.00it/s]
  7%|██▌                                | 5831/78125 [49:53<10:01:25,  2.00it/s]
  7%|██▌                                | 5832/78125 [49:53<10:01:57,  2.00it/s]
  7%|██▌                                | 5833/78125 [49:54<10:01:28,  2.00it/s]
  7%|██▌                                | 5834/78125 [49:54<10:01:18,  2.00it/s]
  7%|██▌                                | 5835/78125 [49:55<10:01:13,  2.00it/s]
  7%|██▌                                | 5836/78125 [49:55<10:01:20,  2.00it/s]
  7%|██▌                                | 5837/78125 [49:56<10:00:41,  2.01it/s]
  7%|██▌                                | 5838/78125 [49:56<10:01:09,  2.00it/s]
  7%|██▌                                | 5839/78125 [49:57<10:00:46,  2.01it/s]
  7%|██▌                                | 5840/78125 [49:57<10:01:10,  2.00it/s]
  7%|██▌                                | 5841/78125 [49:58<10:01:16,  2.00it/s]
  7%|██▌                                | 5842/78125 [49:58<10:00:31,  2.01it/s]
  7%|██▌                                | 5843/78125 [49:59<10:00:49,  2.01it/s]
  7%|██▌                                | 5844/78125 [49:59<10:00:55,  2.00it/s]
  7%|██▌                                | 5845/78125 [50:00<10:02:04,  2.00it/s]
  7%|██▌                                | 5846/78125 [50:00<10:02:13,  2.00it/s]
  7%|██▌                                | 5847/78125 [50:01<10:01:59,  2.00it/s]
  7%|██▌                                | 5848/78125 [50:01<10:01:58,  2.00it/s]
  7%|██▌                                | 5849/78125 [50:02<10:01:07,  2.00it/s]
  7%|██▌                                | 5850/78125 [50:02<10:01:35,  2.00it/s]
  7%|██▌                                | 5851/78125 [50:03<10:01:20,  2.00it/s]
  7%|██▌                                | 5852/78125 [50:03<10:01:43,  2.00it/s]
  7%|██▌                                | 5853/78125 [50:04<10:01:39,  2.00it/s]
  7%|██▌                                | 5854/78125 [50:04<10:01:53,  2.00it/s]
  7%|██▌                                | 5855/78125 [50:05<10:01:56,  2.00it/s]
  7%|██▌                                | 5856/78125 [50:05<10:02:04,  2.00it/s]
  7%|██▌                                | 5857/78125 [50:06<10:02:18,  2.00it/s]
  7%|██▌                                | 5858/78125 [50:06<10:02:08,  2.00it/s]
  7%|██▌                                | 5859/78125 [50:07<10:02:33,  2.00it/s]
  8%|██▋                                | 5860/78125 [50:07<10:01:59,  2.00it/s]
  8%|██▋                                | 5861/78125 [50:08<10:01:23,  2.00it/s]
  8%|██▋                                | 5862/78125 [50:08<10:01:38,  2.00it/s]
  8%|██▋                                | 5863/78125 [50:09<10:00:16,  2.01it/s]
  8%|██▋                                 | 5864/78125 [50:09<9:59:26,  2.01it/s]
  8%|██▋                                 | 5865/78125 [50:10<9:59:55,  2.01it/s]
  8%|██▋                                | 5866/78125 [50:10<10:00:31,  2.01it/s]
  8%|██▋                                | 5867/78125 [50:11<10:00:26,  2.01it/s]
  8%|██▋                                | 5868/78125 [50:11<10:00:26,  2.01it/s]
  8%|██▋                                 | 5869/78125 [50:12<9:59:49,  2.01it/s]
  8%|██▋                                 | 5870/78125 [50:12<9:59:48,  2.01it/s]
  8%|██▋                                 | 5871/78125 [50:13<9:59:57,  2.01it/s]
  8%|██▋                                | 5872/78125 [50:13<10:00:59,  2.00it/s]
  8%|██▋                                | 5873/78125 [50:14<10:01:32,  2.00it/s]
  8%|██▋                                | 5874/78125 [50:14<10:01:08,  2.00it/s]
  8%|██▋                                | 5875/78125 [50:15<10:01:15,  2.00it/s]
  8%|██▋                                | 5876/78125 [50:15<10:01:04,  2.00it/s]
  8%|██▋                                | 5877/78125 [50:16<10:01:34,  2.00it/s]
  8%|██▋                                | 5878/78125 [50:16<10:00:55,  2.00it/s]
  8%|██▋                                | 5879/78125 [50:17<10:01:02,  2.00it/s]
  8%|██▋                                | 5880/78125 [50:17<10:01:05,  2.00it/s]
  8%|██▋                                | 5881/78125 [50:18<10:01:00,  2.00it/s]
  8%|██▋                                | 5882/78125 [50:18<10:00:49,  2.00it/s]
  8%|██▋                                | 5883/78125 [50:19<10:00:09,  2.01it/s]
  8%|██▋                                | 5884/78125 [50:19<10:00:22,  2.01it/s]
  8%|██▋                                | 5885/78125 [50:20<10:00:03,  2.01it/s]
  8%|██▋                                 | 5886/78125 [50:20<9:59:33,  2.01it/s]
  8%|██▋                                | 5887/78125 [50:21<10:00:05,  2.01it/s]
  8%|██▋                                 | 5888/78125 [50:21<9:59:55,  2.01it/s]
  8%|██▋                                | 5889/78125 [50:22<10:00:03,  2.01it/s]
  8%|██▋                                 | 5890/78125 [50:22<9:59:31,  2.01it/s]
  8%|██▋                                 | 5891/78125 [50:23<9:59:58,  2.01it/s]
  8%|██▋                                | 5892/78125 [50:23<10:00:22,  2.01it/s]
  8%|██▋                                 | 5893/78125 [50:24<9:59:50,  2.01it/s]
  8%|██▋                                | 5894/78125 [50:24<10:00:20,  2.01it/s]
  8%|██▋                                | 5895/78125 [50:25<10:00:05,  2.01it/s]
  8%|██▋                                | 5896/78125 [50:25<10:00:44,  2.00it/s]
  8%|██▋                                | 5897/78125 [50:26<10:00:25,  2.00it/s]
  8%|██▋                                | 5898/78125 [50:26<10:00:09,  2.01it/s]
  8%|██▋                                | 5899/78125 [50:27<10:00:53,  2.00it/s]
  8%|██▋                                | 5900/78125 [50:27<10:00:35,  2.00it/s]
  8%|██▋                                 | 5901/78125 [50:28<9:59:54,  2.01it/s]
  8%|██▋                                 | 5902/78125 [50:28<9:58:41,  2.01it/s]
  8%|██▋                                 | 5903/78125 [50:29<9:59:34,  2.01it/s]
  8%|██▋                                 | 5904/78125 [50:29<9:59:38,  2.01it/s]
  8%|██▋                                 | 5905/78125 [50:30<9:59:09,  2.01it/s]
  8%|██▋                                 | 5906/78125 [50:30<9:59:23,  2.01it/s]
  8%|██▋                                 | 5907/78125 [50:31<9:59:55,  2.01it/s]
  8%|██▋                                 | 5908/78125 [50:31<9:59:51,  2.01it/s]
  8%|██▋                                | 5909/78125 [50:32<10:01:04,  2.00it/s]
  8%|██▋                                | 5910/78125 [50:32<10:00:49,  2.00it/s]
  8%|██▋                                | 5911/78125 [50:33<10:00:10,  2.01it/s]
  8%|██▋                                 | 5912/78125 [50:33<9:59:13,  2.01it/s]
  8%|██▋                                 | 5913/78125 [50:34<9:59:54,  2.01it/s]
  8%|██▋                                | 5914/78125 [50:34<10:00:23,  2.00it/s]
  8%|██▋                                | 5915/78125 [50:35<10:00:04,  2.01it/s]
  8%|██▋                                | 5916/78125 [50:35<10:00:09,  2.01it/s]
  8%|██▋                                | 5917/78125 [50:36<10:00:09,  2.01it/s]
  8%|██▋                                 | 5918/78125 [50:36<9:59:40,  2.01it/s]
  8%|██▋                                 | 5919/78125 [50:36<9:59:24,  2.01it/s]
  8%|██▋                                 | 5920/78125 [50:37<9:59:32,  2.01it/s]
  8%|██▋                                 | 5921/78125 [50:37<9:59:28,  2.01it/s]
  8%|██▋                                 | 5922/78125 [50:38<9:59:54,  2.01it/s]
  8%|██▋                                 | 5923/78125 [50:38<9:59:52,  2.01it/s]
  8%|██▋                                | 5924/78125 [50:39<10:00:32,  2.00it/s]
  8%|██▋                                | 5925/78125 [50:39<10:01:03,  2.00it/s]
  8%|██▋                                | 5926/78125 [50:40<10:01:11,  2.00it/s]
  8%|██▋                                | 5927/78125 [50:40<10:00:48,  2.00it/s]
  8%|██▋                                | 5928/78125 [50:41<10:00:42,  2.00it/s]
  8%|██▋                                 | 5929/78125 [50:41<9:59:57,  2.01it/s]
  8%|██▋                                | 5930/78125 [50:42<10:00:09,  2.00it/s]
  8%|██▋                                | 5931/78125 [50:42<10:00:10,  2.00it/s]
  8%|██▋                                | 5932/78125 [50:43<10:00:01,  2.01it/s]
  8%|██▋                                | 5933/78125 [50:43<10:00:33,  2.00it/s]
  8%|██▋                                 | 5934/78125 [50:44<9:59:34,  2.01it/s]
  8%|██▋                                 | 5935/78125 [50:44<9:59:18,  2.01it/s]
  8%|██▋                                 | 5936/78125 [50:45<9:59:18,  2.01it/s]
  8%|██▋                                 | 5937/78125 [50:45<9:59:48,  2.01it/s]
  8%|██▋                                 | 5938/78125 [50:46<9:59:55,  2.01it/s]
  8%|██▋                                | 5939/78125 [50:46<10:00:11,  2.00it/s]
  8%|██▋                                 | 5940/78125 [50:47<9:59:51,  2.01it/s]
  8%|██▋                                 | 5941/78125 [50:47<9:59:47,  2.01it/s]
  8%|██▋                                 | 5942/78125 [50:48<9:59:22,  2.01it/s]
  8%|██▋                                 | 5943/78125 [50:48<9:59:54,  2.01it/s]
  8%|██▋                                 | 5944/78125 [50:49<9:59:38,  2.01it/s]
  8%|██▋                                 | 5945/78125 [50:49<9:59:29,  2.01it/s]
  8%|██▋                                | 5946/78125 [50:50<10:00:07,  2.00it/s]
  8%|██▋                                | 5947/78125 [50:50<10:00:12,  2.00it/s]
  8%|██▋                                 | 5948/78125 [50:51<9:59:52,  2.01it/s]
  8%|██▋                                 | 5949/78125 [50:51<9:59:32,  2.01it/s]
  8%|██▋                                 | 5950/78125 [50:52<9:59:44,  2.01it/s]
  8%|██▋                                 | 5951/78125 [50:52<9:59:50,  2.01it/s]
  8%|██▋                                | 5952/78125 [50:53<10:00:18,  2.00it/s]
  8%|██▋                                | 5953/78125 [50:53<10:00:09,  2.00it/s]
  8%|██▋                                 | 5954/78125 [50:54<9:59:51,  2.01it/s]
  8%|██▋                                 | 5955/78125 [50:54<9:59:52,  2.01it/s]
  8%|██▋                                 | 5956/78125 [50:55<9:59:56,  2.00it/s]
  8%|██▋                                | 5957/78125 [50:55<10:00:24,  2.00it/s]
  8%|██▋                                 | 5958/78125 [50:56<9:59:44,  2.01it/s]
  8%|██▋                                 | 5959/78125 [50:56<9:59:38,  2.01it/s]
  8%|██▋                                 | 5960/78125 [50:57<9:58:59,  2.01it/s]
  8%|██▋                                 | 5961/78125 [50:57<9:59:20,  2.01it/s]
  8%|██▋                                 | 5962/78125 [50:58<9:59:29,  2.01it/s]
  8%|██▋                                 | 5963/78125 [50:58<9:59:48,  2.01it/s]
  8%|██▋                                 | 5964/78125 [50:59<9:59:44,  2.01it/s]
  8%|██▋                                 | 5965/78125 [50:59<9:59:19,  2.01it/s]
  8%|██▋                                 | 5966/78125 [51:00<9:58:57,  2.01it/s]
  8%|██▋                                 | 5967/78125 [51:00<9:58:52,  2.01it/s]
  8%|██▊                                 | 5968/78125 [51:01<9:59:13,  2.01it/s]
  8%|██▊                                 | 5969/78125 [51:01<9:58:54,  2.01it/s]
  8%|██▊                                 | 5970/78125 [51:02<9:59:30,  2.01it/s]
  8%|██▊                                 | 5971/78125 [51:02<9:59:50,  2.00it/s]
  8%|██▊                                 | 5972/78125 [51:03<9:59:03,  2.01it/s]
  8%|██▊                                 | 5973/78125 [51:03<9:59:05,  2.01it/s]
  8%|██▊                                 | 5974/78125 [51:04<9:58:47,  2.01it/s]
  8%|██▊                                 | 5975/78125 [51:04<9:59:16,  2.01it/s]
  8%|██▊                                 | 5976/78125 [51:05<9:59:22,  2.01it/s]
  8%|██▊                                 | 5977/78125 [51:05<9:59:42,  2.01it/s]
  8%|██▋                                | 5978/78125 [51:06<10:00:22,  2.00it/s]
  8%|██▋                                | 5979/78125 [51:06<10:00:04,  2.00it/s]
  8%|██▊                                 | 5980/78125 [51:07<9:59:53,  2.00it/s]
  8%|██▋                                | 5981/78125 [51:07<10:00:24,  2.00it/s]
  8%|██▊                                 | 5982/78125 [51:08<9:59:54,  2.00it/s]
  8%|██▊                                 | 5983/78125 [51:08<9:59:47,  2.00it/s]
  8%|██▊                                 | 5984/78125 [51:09<9:59:16,  2.01it/s]
  8%|██▊                                 | 5985/78125 [51:09<9:59:41,  2.00it/s]
  8%|██▊                                 | 5986/78125 [51:10<9:59:21,  2.01it/s]
  8%|██▊                                 | 5987/78125 [51:10<9:59:14,  2.01it/s]
  8%|██▊                                 | 5988/78125 [51:11<9:59:35,  2.01it/s]
  8%|██▊                                 | 5989/78125 [51:11<9:58:53,  2.01it/s]
  8%|██▊                                 | 5990/78125 [51:12<9:58:09,  2.01it/s]
  8%|██▊                                 | 5991/78125 [51:12<9:58:22,  2.01it/s]
  8%|██▊                                 | 5992/78125 [51:13<9:59:11,  2.01it/s]
  8%|██▊                                 | 5993/78125 [51:13<9:59:05,  2.01it/s]
  8%|██▊                                 | 5994/78125 [51:14<9:59:29,  2.01it/s]
  8%|██▊                                 | 5995/78125 [51:14<9:59:30,  2.01it/s]
  8%|██▊                                 | 5996/78125 [51:15<9:59:06,  2.01it/s]
  8%|██▊                                 | 5997/78125 [51:15<9:59:30,  2.01it/s]
  8%|██▊                                 | 5998/78125 [51:16<9:59:22,  2.01it/s]
  8%|██▊                                 | 5999/78125 [51:16<9:59:19,  2.01it/s]
  8%|██▊                                 | 6000/78125 [51:17<9:58:45,  2.01it/s]
  8%|██▊                                 | 6001/78125 [51:17<9:59:04,  2.01it/s]
  8%|██▊                                 | 6002/78125 [51:18<9:59:30,  2.01it/s]
  8%|██▊                                 | 6003/78125 [51:18<9:59:33,  2.00it/s]
  8%|██▋                                | 6004/78125 [51:19<10:00:43,  2.00it/s]
  8%|██▋                                | 6005/78125 [51:19<10:00:26,  2.00it/s]
  8%|██▋                                | 6006/78125 [51:20<10:00:09,  2.00it/s]
  8%|██▊                                 | 6007/78125 [51:20<9:59:38,  2.00it/s]
  8%|██▋                                | 6008/78125 [51:21<10:00:02,  2.00it/s]
  8%|██▊                                 | 6009/78125 [51:21<9:59:48,  2.00it/s]
  8%|██▊                                 | 6010/78125 [51:22<9:59:25,  2.01it/s]
  8%|██▊                                 | 6011/78125 [51:22<9:59:31,  2.00it/s]
  8%|██▊                                 | 6012/78125 [51:23<9:59:28,  2.00it/s]
  8%|██▋                                | 6013/78125 [51:23<10:00:00,  2.00it/s]
  8%|██▋                                | 6014/78125 [51:24<10:00:27,  2.00it/s]
  8%|██▋                                | 6015/78125 [51:24<10:00:30,  2.00it/s]
  8%|██▋                                | 6016/78125 [51:25<10:00:05,  2.00it/s]
  8%|██▋                                | 6017/78125 [51:25<10:00:21,  2.00it/s]
  8%|██▋                                | 6018/78125 [51:26<10:00:05,  2.00it/s]
  8%|██▋                                | 6019/78125 [51:26<10:00:16,  2.00it/s]
  8%|██▊                                 | 6020/78125 [51:27<9:59:39,  2.00it/s]
  8%|██▊                                 | 6021/78125 [51:27<9:58:19,  2.01it/s]
  8%|██▊                                 | 6022/78125 [51:28<9:57:49,  2.01it/s]
  8%|██▊                                 | 6023/78125 [51:28<9:58:35,  2.01it/s]
  8%|██▊                                 | 6024/78125 [51:29<9:58:20,  2.01it/s]
  8%|██▊                                 | 6025/78125 [51:29<9:58:00,  2.01it/s]
  8%|██▊                                 | 6026/78125 [51:30<9:58:22,  2.01it/s]
  8%|██▊                                 | 6027/78125 [51:30<9:59:22,  2.00it/s]
  8%|██▊                                 | 6028/78125 [51:31<9:59:20,  2.00it/s]
  8%|██▊                                 | 6029/78125 [51:31<9:59:13,  2.01it/s]
  8%|██▊                                 | 6030/78125 [51:32<9:58:54,  2.01it/s]
  8%|██▊                                 | 6031/78125 [51:32<9:58:52,  2.01it/s]
  8%|██▊                                 | 6032/78125 [51:33<9:58:17,  2.01it/s]
  8%|██▊                                 | 6033/78125 [51:33<9:58:03,  2.01it/s]
  8%|██▊                                 | 6034/78125 [51:34<9:57:50,  2.01it/s]
  8%|██▊                                 | 6035/78125 [51:34<9:58:28,  2.01it/s]
  8%|██▊                                 | 6036/78125 [51:35<9:58:10,  2.01it/s]
  8%|██▊                                 | 6037/78125 [51:35<9:57:48,  2.01it/s]
  8%|██▊                                 | 6038/78125 [51:36<9:58:26,  2.01it/s]
  8%|██▊                                 | 6039/78125 [51:36<9:58:10,  2.01it/s]
  8%|██▊                                 | 6040/78125 [51:37<9:58:15,  2.01it/s]
  8%|██▊                                 | 6041/78125 [51:37<9:59:31,  2.00it/s]
  8%|██▊                                 | 6042/78125 [51:38<9:59:32,  2.00it/s]
  8%|██▊                                 | 6043/78125 [51:38<9:58:43,  2.01it/s]
  8%|██▊                                 | 6044/78125 [51:39<9:57:58,  2.01it/s]
  8%|██▊                                 | 6045/78125 [51:39<9:57:42,  2.01it/s]
  8%|██▊                                 | 6046/78125 [51:40<9:58:08,  2.01it/s]
  8%|██▊                                 | 6047/78125 [51:40<9:58:19,  2.01it/s]
  8%|██▊                                 | 6048/78125 [51:41<9:57:56,  2.01it/s]
  8%|██▊                                 | 6049/78125 [51:41<9:58:16,  2.01it/s]
  8%|██▊                                 | 6050/78125 [51:42<9:58:11,  2.01it/s]
  8%|██▊                                 | 6051/78125 [51:42<9:58:32,  2.01it/s]
  8%|██▊                                 | 6052/78125 [51:43<9:58:47,  2.01it/s]
  8%|██▊                                 | 6053/78125 [51:43<9:58:25,  2.01it/s]
  8%|██▊                                 | 6054/78125 [51:44<9:58:33,  2.01it/s]
  8%|██▊                                 | 6055/78125 [51:44<9:57:50,  2.01it/s]
  8%|██▊                                 | 6056/78125 [51:45<9:58:47,  2.01it/s]
  8%|██▊                                 | 6057/78125 [51:45<9:57:57,  2.01it/s]
  8%|██▊                                 | 6058/78125 [51:46<9:58:07,  2.01it/s]
  8
760.3s 151 %|██▊                                 | 6059/78125 [51:46<9:58:28,  2.01it/s]
  8%|██▊                                 | 6060/78125 [51:47<9:58:45,  2.01it/s]
  8%|██▊                                 | 6061/78125 [51:47<9:58:45,  2.01it/s]
  8%|██▊                                 | 6062/78125 [51:48<9:59:03,  2.00it/s]
  8%|██▊                                 | 6063/78125 [51:48<9:58:52,  2.01it/s]
  8%|██▊                                 | 6064/78125 [51:49<9:58:43,  2.01it/s]
  8%|██▊                                 | 6065/78125 [51:49<9:59:05,  2.00it/s]
  8%|██▊                                 | 6066/78125 [51:50<9:58:21,  2.01it/s]
  8%|██▊                                 | 6067/78125 [51:50<9:57:56,  2.01it/s]
  8%|██▊                                 | 6068/78125 [51:51<9:57:22,  2.01it/s]
  8%|██▊                                 | 6069/78125 [51:51<9:57:53,  2.01it/s]
  8%|██▊                                 | 6070/78125 [51:52<9:57:45,  2.01it/s]
  8%|██▊                                 | 6071/78125 [51:52<9:58:15,  2.01it/s]
  8%|██▊                                 | 6072/78125 [51:53<9:57:44,  2.01it/s]
  8%|██▊                                 | 6073/78125 [51:53<9:58:22,  2.01it/s]
  8%|██▊                                 | 6074/78125 [51:54<9:58:41,  2.01it/s]
  8%|██▊                                 | 6075/78125 [51:54<9:58:15,  2.01it/s]
  8%|██▊                                 | 6076/78125 [51:55<9:58:46,  2.01it/s]
  8%|██▊                                 | 6077/78125 [51:55<9:59:05,  2.00it/s]
  8%|██▊                                 | 6078/78125 [51:56<9:58:46,  2.01it/s]
  8%|██▊                                 | 6079/78125 [51:56<9:59:10,  2.00it/s]
  8%|██▊                                 | 6080/78125 [51:57<9:59:04,  2.00it/s]
  8%|██▊                                 | 6081/78125 [51:57<9:58:53,  2.00it/s]
  8%|██▊                                 | 6082/78125 [51:58<9:58:41,  2.01it/s]
  8%|██▊                                 | 6083/78125 [51:58<9:58:56,  2.00it/s]
  8%|██▊                                 | 6084/78125 [51:59<9:58:50,  2.01it/s]
  8%|██▊                                 | 6085/78125 [51:59<9:59:19,  2.00it/s]
  8%|██▊                                 | 6086/78125 [52:00<9:58:10,  2.01it/s]
  8%|██▊                                 | 6087/78125 [52:00<9:59:00,  2.00it/s]
  8%|██▊                                 | 6088/78125 [52:01<9:59:45,  2.00it/s]
  8%|██▊                                 | 6089/78125 [52:01<9:58:58,  2.00it/s]
  8%|██▊                                 | 6090/78125 [52:02<9:58:48,  2.00it/s]
  8%|██▊                                 | 6091/78125 [52:02<9:59:03,  2.00it/s]
  8%|██▊                                 | 6092/78125 [52:03<9:58:32,  2.01it/s]
  8%|██▊                                 | 6093/78125 [52:03<9:58:49,  2.00it/s]
  8%|██▊                                 | 6094/78125 [52:04<9:58:56,  2.00it/s]
  8%|██▊                                 | 6095/78125 [52:04<9:58:16,  2.01it/s]
  8%|██▊                                 | 6096/78125 [52:05<9:58:25,  2.01it/s]
  8%|██▊                                 | 6097/78125 [52:05<9:58:01,  2.01it/s]
  8%|██▊                                 | 6098/78125 [52:06<9:57:57,  2.01it/s]
  8%|██▊                                 | 6099/78125 [52:06<9:58:30,  2.01it/s]
  8%|██▊                                 | 6100/78125 [52:07<9:57:48,  2.01it/s]
  8%|██▊                                 | 6101/78125 [52:07<9:57:15,  2.01it/s]
  8%|██▊                                 | 6102/78125 [52:08<9:57:20,  2.01it/s]
  8%|██▊                                 | 6103/78125 [52:08<9:57:34,  2.01it/s]
  8%|██▊                                 | 6104/78125 [52:09<9:57:48,  2.01it/s]
  8%|██▊                                 | 6105/78125 [52:09<9:58:00,  2.01it/s]
  8%|██▊                                 | 6106/78125 [52:10<9:57:41,  2.01it/s]
  8%|██▊                                 | 6107/78125 [52:10<9:57:36,  2.01it/s]
  8%|██▊                                 | 6108/78125 [52:11<9:57:20,  2.01it/s]
  8%|██▊                                 | 6109/78125 [52:11<9:57:40,  2.01it/s]
  8%|██▊                                 | 6110/78125 [52:12<9:57:53,  2.01it/s]
  8%|██▊                                 | 6111/78125 [52:12<9:57:32,  2.01it/s]
  8%|██▊                                 | 6112/78125 [52:13<9:57:33,  2.01it/s]
  8%|██▊                                 | 6113/78125 [52:13<9:57:39,  2.01it/s]
  8%|██▊                                 | 6114/78125 [52:14<9:57:51,  2.01it/s]
  8%|██▊                                 | 6115/78125 [52:14<9:57:46,  2.01it/s]
  8%|██▊                                 | 6116/78125 [52:15<9:57:42,  2.01it/s]
  8%|██▊                                 | 6117/78125 [52:15<9:58:08,  2.01it/s]
  8%|██▊                                 | 6118/78125 [52:16<9:57:21,  2.01it/s]
  8%|██▊                                 | 6119/78125 [52:16<9:56:37,  2.01it/s]
  8%|██▊                                 | 6120/78125 [52:17<9:57:09,  2.01it/s]
  8%|██▊                                 | 6121/78125 [52:17<9:57:11,  2.01it/s]
  8%|██▊                                 | 6122/78125 [52:18<9:57:19,  2.01it/s]
  8%|██▊                                 | 6123/78125 [52:18<9:57:35,  2.01it/s]
  8%|██▊                                 | 6124/78125 [52:19<9:58:17,  2.01it/s]
  8%|██▊                                 | 6125/78125 [52:19<9:58:10,  2.01it/s]
  8%|██▊                                 | 6126/78125 [52:20<9:57:28,  2.01it/s]
  8%|██▊                                 | 6127/78125 [52:20<9:57:31,  2.01it/s]
  8%|██▊                                 | 6128/78125 [52:21<9:57:17,  2.01it/s]
  8%|██▊                                 | 6129/78125 [52:21<9:57:45,  2.01it/s]
  8%|██▊                                 | 6130/78125 [52:22<9:57:27,  2.01it/s]
  8%|██▊                                 | 6131/78125 [52:22<9:56:55,  2.01it/s]
  8%|██▊                                 | 6132/78125 [52:23<9:57:06,  2.01it/s]
  8%|██▊                                 | 6133/78125 [52:23<9:57:10,  2.01it/s]
  8%|██▊                                 | 6134/78125 [52:24<9:57:17,  2.01it/s]
  8%|██▊                                 | 6135/78125 [52:24<9:56:49,  2.01it/s]
  8%|██▊                                 | 6136/78125 [52:25<9:56:50,  2.01it/s]
  8%|██▊                                 | 6137/78125 [52:25<9:57:20,  2.01it/s]
  8%|██▊                                 | 6138/78125 [52:26<9:56:52,  2.01it/s]
  8%|██▊                                 | 6139/78125 [52:26<9:56:47,  2.01it/s]
  8%|██▊                                 | 6140/78125 [52:27<9:57:40,  2.01it/s]
  8%|██▊                                 | 6141/78125 [52:27<9:56:43,  2.01it/s]
  8%|██▊                                 | 6142/78125 [52:28<9:56:47,  2.01it/s]
  8%|██▊                                 | 6143/78125 [52:28<9:57:25,  2.01it/s]
  8%|██▊                                 | 6144/78125 [52:29<9:56:36,  2.01it/s]
  8%|██▊                                 | 6145/78125 [52:29<9:56:56,  2.01it/s]
  8%|██▊                                 | 6146/78125 [52:30<9:57:10,  2.01it/s]
  8%|██▊                                 | 6147/78125 [52:30<9:57:33,  2.01it/s]
  8%|██▊                                 | 6148/78125 [52:31<9:57:01,  2.01it/s]
  8%|██▊                                 | 6149/78125 [52:31<9:56:45,  2.01it/s]
  8%|██▊                                 | 6150/78125 [52:32<9:56:36,  2.01it/s]
  8%|██▊                                 | 6151/78125 [52:32<9:56:14,  2.01it/s]
  8%|██▊                                 | 6152/78125 [52:33<9:56:30,  2.01it/s]
  8%|██▊                                 | 6153/78125 [52:33<9:56:29,  2.01it/s]
  8%|██▊                                 | 6154/78125 [52:34<9:56:56,  2.01it/s]
  8%|██▊                                 | 6155/78125 [52:34<9:57:16,  2.01it/s]
  8%|██▊                                 | 6156/78125 [52:35<9:56:43,  2.01it/s]
  8%|██▊                                 | 6157/78125 [52:35<9:56:45,  2.01it/s]
  8%|██▊                                 | 6158/78125 [52:36<9:57:10,  2.01it/s]
  8%|██▊                                 | 6159/78125 [52:36<9:56:52,  2.01it/s]
  8%|██▊                                 | 6160/78125 [52:37<9:56:40,  2.01it/s]
  8%|██▊                                 | 6161/78125 [52:37<9:57:07,  2.01it/s]
  8%|██▊                                 | 6162/78125 [52:38<9:57:08,  2.01it/s]
  8%|██▊                                 | 6163/78125 [52:38<9:56:47,  2.01it/s]
  8%|██▊                                 | 6164/78125 [52:39<9:56:28,  2.01it/s]
  8%|██▊                                 | 6165/78125 [52:39<9:57:47,  2.01it/s]
  8%|██▊                                 | 6166/78125 [52:40<9:58:04,  2.01it/s]
  8%|██▊                                 | 6167/78125 [52:40<9:57:43,  2.01it/s]
  8%|██▊                                 | 6168/78125 [52:41<9:58:00,  2.01it/s]
  8%|██▊                                 | 6169/78125 [52:41<9:58:16,  2.00it/s]
  8%|██▊                                 | 6170/78125 [52:42<9:58:09,  2.00it/s]
  8%|██▊                                 | 6171/78125 [52:42<9:58:42,  2.00it/s]
  8%|██▊                                 | 6172/78125 [52:43<9:58:28,  2.00it/s]
  8%|██▊                                 | 6173/78125 [52:43<9:58:28,  2.00it/s]
  8%|██▊                                 | 6174/78125 [52:44<9:57:52,  2.01it/s]
  8%|██▊                                 | 6175/78125 [52:44<9:57:07,  2.01it/s]
  8%|██▊                                 | 6176/78125 [52:45<9:56:38,  2.01it/s]
  8%|██▊                                 | 6177/78125 [52:45<9:56:45,  2.01it/s]
  8%|██▊                                 | 6178/78125 [52:46<9:56:04,  2.01it/s]
  8%|██▊                                 | 6179/78125 [52:46<9:56:05,  2.01it/s]
  8%|██▊                                 | 6180/78125 [52:47<9:56:14,  2.01it/s]
  8%|██▊                                 | 6181/78125 [52:47<9:56:50,  2.01it/s]
  8%|██▊                                 | 6182/78125 [52:48<9:57:39,  2.01it/s]
  8%|██▊                                 | 6183/78125 [52:48<9:57:04,  2.01it/s]
  8%|██▊                                 | 6184/78125 [52:49<9:56:46,  2.01it/s]
  8%|██▊                                 | 6185/78125 [52:49<9:57:13,  2.01it/s]
  8%|██▊                                 | 6186/78125 [52:50<9:57:37,  2.01it/s]
  8%|██▊                                 | 6187/78125 [52:50<9:57:02,  2.01it/s]
  8%|██▊                                 | 6188/78125 [52:51<9:57:29,  2.01it/s]
  8%|██▊                                 | 6189/78125 [52:51<9:57:24,  2.01it/s]
  8%|██▊                                 | 6190/78125 [52:52<9:57:29,  2.01it/s]
  8%|██▊                                 | 6191/78125 [52:52<9:56:40,  2.01it/s]
  8%|██▊                                 | 6192/78125 [52:53<9:56:30,  2.01it/s]
  8%|██▊                                 | 6193/78125 [52:53<9:57:00,  2.01it/s]
  8%|██▊                                 | 6194/78125 [52:54<9:57:31,  2.01it/s]
  8%|██▊                                 | 6195/78125 [52:54<9:56:50,  2.01it/s]
  8%|██▊                                 | 6196/78125 [52:55<9:56:21,  2.01it/s]
  8%|██▊                                 | 6197/78125 [52:55<9:56:18,  2.01it/s]
  8%|██▊                                 | 6198/78125 [52:56<9:55:56,  2.01it/s]
  8%|██▊                                 | 6199/78125 [52:56<9:55:16,  2.01it/s]
  8%|██▊                                 | 6200/78125 [52:57<9:56:12,  2.01it/s]
  8%|██▊                                 | 6201/78125 [52:57<9:56:48,  2.01it/s]
  8%|██▊                                 | 6202/78125 [52:58<9:56:47,  2.01it/s]
  8%|██▊                                 | 6203/78125 [52:58<9:56:30,  2.01it/s]
  8%|██▊                                 | 6204/78125 [52:59<9:57:34,  2.01it/s]
  8%|██▊                                 | 6205/78125 [52:59<9:58:08,  2.00it/s]
  8%|██▊                                 | 6206/78125 [53:00<9:57:32,  2.01it/s]
  8%|██▊                                 | 6207/78125 [53:00<9:57:30,  2.01it/s]
  8%|██▊                                 | 6208/78125 [53:01<9:57:41,  2.01it/s]
  8%|██▊                                 | 6209/78125 [53:01<9:57:17,  2.01it/s]
  8%|██▊                                 | 6210/78125 [53:02<9:57:18,  2.01it/s]
  8%|██▊                                 | 6211/78125 [53:02<9:57:28,  2.01it/s]
  8%|██▊                                 | 6212/78125 [53:03<9:57:40,  2.01it/s]
  8%|██▊                                 | 6213/78125 [53:03<9:57:24,  2.01it/s]
  8%|██▊                                 | 6214/78125 [53:03<9:56:44,  2.01it/s]
  8%|██▊                                 | 6215/78125 [53:04<9:56:56,  2.01it/s]
  8%|██▊                                 | 6216/78125 [53:04<9:57:10,  2.01it/s]
  8%|██▊                                 | 6217/78125 [53:05<9:57:51,  2.00it/s]
  8%|██▊                                 | 6218/78125 [53:05<9:57:28,  2.01it/s]
  8%|██▊                                 | 6219/78125 [53:06<9:58:20,  2.00it/s]
  8%|██▊                                 | 6220/78125 [53:06<9:57:38,  2.01it/s]
  8%|██▊                                 | 6221/78125 [53:07<9:56:51,  2.01it/s]
  8%|██▊                                 | 6222/78125 [53:07<9:56:50,  2.01it/s]
  8%|██▊                                 | 6223/78125 [53:08<9:56:20,  2.01it/s]
  8%|██▊                                 | 6224/78125 [53:08<9:56:06,  2.01it/s]
  8%|██▊                                 | 6225/78125 [53:09<9:55:41,  2.01it/s]
  8%|██▊                                 | 6226/78125 [53:09<9:55:39,  2.01it/s]
  8%|██▊                                 | 6227/78125 [53:10<9:56:01,  2.01it/s]
  8%|██▊                                 | 6228/78125 [53:10<9:56:42,  2.01it/s]
  8%|██▊                                 | 6229/78125 [53:11<9:56:05,  2.01it/s]
  8%|██▊                                 | 6230/78125 [53:11<9:56:06,  2.01it/s]
  8%|██▊                                 | 6231/78125 [53:12<9:56:32,  2.01it/s]
  8%|██▊                                 | 6232/78125 [53:12<9:57:14,  2.01it/s]
  8%|██▊                                 | 6233/78125 [53:13<9:57:00,  2.01it/s]
  8%|██▊                                 | 6234/78125 [53:13<9:57:17,  2.01it/s]
  8%|██▊                                 | 6235/78125 [53:14<9:56:17,  2.01it/s]
  8%|██▊                                 | 6236/78125 [53:14<9:56:42,  2.01it/s]
  8%|██▊                                 | 6237/78125 [53:15<9:56:09,  2.01it/s]
  8%|██▊                                 | 6238/78125 [53:15<9:56:41,  2.01it/s]
  8%|██▊                                 | 6239/78125 [53:16<9:57:05,  2.01it/s]
  8%|██▉                                 | 6240/78125 [53:16<9:57:19,  2.01it/s]
  8%|██▉                                 | 6241/78125 [53:17<9:56:59,  2.01it/s]
  8%|██▉                                 | 6242/78125 [53:17<9:56:05,  2.01it/s]
  8%|██▉                                 | 6243/78125 [53:18<9:55:40,  2.01it/s]
  8%|██▉                                 | 6244/78125 [53:18<9:56:11,  2.01it/s]
  8%|██▉                                 | 6245/78125 [53:19<9:55:13,  2.01it/s]
  8%|██▉                                 | 6246/78125 [53:19<9:55:10,  2.01it/s]
  8%|██▉                                 | 6247/78125 [53:20<9:55:36,  2.01it/s]
  8%|██▉                                 | 6248/78125 [53:20<9:55:59,  2.01it/s]
  8%|██▉                                 | 6249/78125 [53:21<9:56:26,  2.01it/s]
  8%|██▉                                 | 6250/78125 [53:21<9:56:15,  2.01it/s]
  8%|██▉                                 | 6251/78125 [53:22<9:56:42,  2.01it/s]
  8%|██▉                                 | 6252/78125 [53:22<9:56:09,  2.01it/s]
  8%|██▉                                 | 6253/78125 [53:23<9:55:57,  2.01it/s]
  8%|██▉                                 | 6254/78125 [53:23<9:57:12,  2.01it/s]
  8%|██▉                                 | 6255/78125 [53:24<9:57:14,  2.01it/s]
  8%|██▉                                 | 6256/78125 [53:24<9:57:01,  2.01it/s]
  8%|██▉                                 | 6257/78125 [53:25<9:56:52,  2.01it/s]
  8%|██▉                                 | 6258/78125 [53:25<9:56:55,  2.01it/s]
  8%|██▉                                 | 6259/78125 [53:26<9:56:14,  2.01it/s]
  8%|██▉                                 | 6260/78125 [53:26<9:56:07,  2.01it/s]
  8%|██▉                                 | 6261/78125 [53:27<9:56:46,  2.01it/s]
  8%|██▉                                 | 6262/78125 [53:27<9:56:44,  2.01it/s]
  8%|██▉                                 | 6263/78125 [53:28<9:57:05,  2.01it/s]
  8%|██▉                                 | 6264/78125 [53:28<9:57:02,  2.01it/s]
  8%|██▉                                 | 6265/78125 [53:29<9:56:46,  2.01it/s]
  8%|██▉                                 | 6266/78125 [53:29<9:56:40,  2.01it/s]
  8%|██▉                                 | 6267/78125 [53:30<9:56:24,  2.01it/s]
  8%|██▉                                 | 6268/78125 [53:30<9:56:16,  2.01it/s]
  8%|██▉                                 | 6269/78125 [53:31<9:56:14,  2.01it/s]
  8%|██▉                                 | 6270/78125 [53:31<9:56:40,  2.01it/s]
  8%|██▉                                 | 6271/78125 [53:32<9:56:50,  2.01it/s]
  8%|██▉                                 | 6272/78125 [53:32<9:55:57,  2.01it/s]
  8%|██▉                                 | 6273/78125 [53:33<9:56:21,  2.01it/s]
  8%|██▉                                 | 6274/78125 [53:33<9:56:15,  2.01it/s]
  8%|██▉                                 | 6275/78125 [53:34<9:56:10,  2.01it/s]
  8%|██▉                                 | 6276/78125 [53:34<9:56:48,  2.01it/s]
  8%|██▉                                 | 6277/78125 [53:35<9:56:13,  2.01it/s]
  8%|██▉                                 | 6278/78125 [53:35<9:55:59,  2.01it/s]
  8%|██▉                                 | 6279/78125 [53:36<9:56:30,  2.01it/s]
  8%|██▉                                 | 6280/78125 [53:36<9:56:36,  2.01it/s]
  8%|██▉                                 | 6281/78125 [53:37<9:56:51,  2.01it/s]
  8%|██▉                                 | 6282/78125 [53:37<9:56:14,  2.01it/s]
  8%|██▉                                 | 6283/78125 [53:38<9:56:12,  2.01it/s]
  8%|██▉                                 | 6284/78125 [53:38<9:56:18,  2.01it/s]
  8%|██▉                                 | 6285/78125 [53:39<9:56:24,  2.01it/s]
  8%|██▉                                 | 6286/78125 [53:39<9:56:05,  2.01it/s]
  8%|██▉                                 | 6287/78125 [53:40<9:56:16,  2.01it/s]
  8%|██▉                                 | 6288/78125 [53:40<9:55:28,  2.01it/s]
  8%|██▉                                 | 6289/78125 [53:41<9:55:16,  2.01it/s]
  8%|██▉                                 | 6290/78125 [53:41<9:55:17,  2.01it/s]
  8%|██▉                                 | 6291/78125 [53:42<9:55:41,  2.01it/s]
  8%|██▉                                 | 6292/78125 [53:42<9:54:58,  2.01it/s]
  8%|██▉                                 | 6293/78125 [53:43<9:55:04,  2.01it/s]
  8%|██▉                                 | 6294/78125 [53:43<9:55:40,  2.01it/s]
  8%|██▉                                 | 6295/78125 [53:44<9:55:18,  2.01it/s]
  8%|██▉                                 | 6296/78125 [53:44<9:54:27,  2.01it/s]
  8%|██▉                                 | 6297/78125 [53:45<9:55:03,  2.01it/s]
  8%|██▉                                 | 6298/78125 [53:45<9:55:51,  2.01it/s]
  8%|██▉                                 | 6299/78125 [53:46<9:54:55,  2.01it/s]
  8%|██▉                                 | 6300/78125 [53:46<9:54:55,  2.01it/s]
  8%|██▉                                 | 6301/78125 [53:47<9:54:51,  2.01it/s]
  8%|██▉                                 | 6302/78125 [53:47<9:54:48,  2.01it/s]
  8%|██▉                                 | 6303/78125 [53:48<9:55:40,  2.01it/s]
  8%|██▉                                 | 6304/78125 [53:48<9:56:49,  2.01it/s]
  8%|██▉                                 | 6305/78125 [53:49<9:55:50,  2.01it/s]
  8%|██▉                                 | 6306/78125 [53:49<9:55:24,  2.01it/s]
  8%|██▉                                 | 6307/78125 [53:50<9:55:02,  2.01it/s]
  8%|██▉                                 | 6308/78125 [53:50<9:55:26,  2.01it/s]
  8%|██▉                                 | 6309/78125 [53:51<9:55:56,  2.01it/s]
  8%|██▉                                 | 6310/78125 [53:51<9:56:27,  2.01it/s]
  8%|██▉                                 | 6311/78125 [53:52<9:56:34,  2.01it/s]
  8%|██▉                                 | 6312/78125 [53:52<9:55:46,  2.01it/s]
  8%|██▉                                 | 6313/78125 [53:53<9:55:54,  2.01it/s]
  8%|██▉                                 | 6314/78125 [53:53<9:55:38,  2.01it/s]
  8%|██▉                                 | 6315/78125 [53:54<9:56:01,  2.01it/s]
  8%|██▉                                 | 6316/78125 [53:54<9:56:09,  2.01it/s]
  8%|██▉                                 | 6317/78125 [53:55<9:56:08,  2.01it/s]
  8%|██▉                                 | 6318/78125 [53:55<9:54:51,  2.01it/s]
  8%|██▉                                 | 6319/78125 [53:56<9:55:16,  2.01it/s]
  8%|██▉                                 | 6320/78125 [53:56<9:55:09,  2.01it/s]
  8%|██▉                                 | 6321/78125 [53:57<9:55:10,  2.01it/s]
  8%|██▉                                 | 6322/78125 [53:57<9:55:38,  2.01it/s]
  8%|██▉                                 | 6323/78125 [53:58<9:55:02,  2.01it/s]
  8%|██▉                                 | 6324/78125 [53:58<9:54:20,  2.01it/s]
  8%|██▉                                 | 6325/78125 [53:59<9:54:59,  2.01it/s]
  8%|██▉                                 | 6326/78125 [53:59<9:55:19,  2.01it/s]
  8%|██▉                                 | 6327/78125 [54:00<9:55:23,  2.01it/s]
  8%|██▉                                 | 6328/78125 [54:00<9:55:10,  2.01it/s]
  8%|██▉                                 | 6329/78125 [54:01<9:55:58,  2.01it/s]
  8%|██▉                                 | 6330/78125 [54:01<9:56:07,  2.01it/s]
  8%|██▉                                 | 6331/78125 [54:02<9:55:59,  2.01it/s]
  8%|██▉                                 | 6332/78125 [54:02<9:55:41,  2.01it/s]
  8%|██▉                                 | 6333/78125 [54:03<9:55:48,  2.01it/s]
  8%|██▉                                 | 6334/78125 [54:03<9:56:09,  2.01it/s]
  8%|██▉                                 | 6335/78125 [54:04<9:55:55,  2.01it/s]
  8%|██▉                                 | 6336/78125 [54:04<9:56:15,  2.01it/s]
  8%|██▉                                 | 6337/78125 [54:05<9:56:32,  2.01it/s]
  8%|██▉                                 | 6338/78125 [54:05<9:56:49,  2.00it/s]
  8%|██▉                                 | 6339/78125 [54:06<9:56:16,  2.01it/s]
  8%|██▉                                 | 6340/78125 [54:06<9:55:56,  2.01it/s]
  8%|██▉                                 | 6341/78125 [54:07<9:55:39,  2.01it/s]
  8%|██▉                                 | 6342/78125 [54:07<9:54:50,  2.01it/s]
  8%|██▉                                 | 6343/78125 [54:08<9:54:45,  2.01it/s]
  8%|██▉                                 | 6344/78125 [54:08<9:54:26,  2.01it/s]
  8%|██▉                                 | 6345/78125 [54:09<9:54:13,  2.01it/s]
  8%|██▉                                 | 6346/78125 [54:09<9:53:50,  2.01it/s]
  8%|██▉                                 | 6347/78125 [54:10<9:54:07,  2.01it/s]
  8%|██▉                                 | 6348/78125 [54:10<9:53:52,  2.01it/s]
  8%|██▉                                 | 6349/78125 [54:11<9:54:12,  2.01it/s]
  8%|██▉                                 | 6350/78125 [54:11<9:54:53,  2.01it/s]
  8%|██▉                                 | 6351/78125 [54:12<9:55:39,  2.01it/s]
  8%|██▉                                 | 6352/78125 [54:12<9:55:28,  2.01it/s]
  8%|██▉                                 | 6353/78125 [54:13<9:55:44,  2.01it/s]
  8%|██▉                                 | 6354/78125 [54:13<9:55:18,  2.01it/s]
  8%|██▉                                 | 6355/78125 [54:14<9:55:21,  2.01it/s]
  8%|██▉                                 | 6356/78125 [54:14<9:55:54,  2.01it/s]
  8%|██▉                                 | 6357/78125 [54:15<9:55:56,  2.01it/s]
  8%|██▉                                 | 6358/78125 [54:15<9:55:35,  2.01it/s]
  8%|██▉                                 | 6359/78125 [54:16<9:54:50,  2.01it/s]
  8%|██▉                                 | 6360/78125 [54:16<9:54:43,  2.01it/s]
  8%|██▉                                 | 6361/78125 [54:17<9:54:04,  2.01it/s]
  8%|██▉                                 | 6362/78125 [54:17<9:55:03,  2.01it/s]
  8%|██▉                                 | 6363/78125 [54:18<9:54:51,  2.01it/s]
  8%|██▉                                 | 6364/78125 [54:18<9:55:06,  2.01it/s]
  8%|██▉                                 | 6365/78125 [54:19<9:55:29,  2.01it/s]
  8%|██▉                                 | 6366/78125 [54:19<9:56:09,  2.01it/s]
  8%|██▉                                 | 6367/78125 [54:20<9:55:40,  2.01it/s]
  8%|██▉                                 | 6368/78125 [54:20<9:56:07,  2.01it/s]
  8%|██▉                                 | 6369/78125 [54:21<9:55:22,  2.01it/s]
  8%|██▉                                 | 6370/78125 [54:21<9:55:26,  2.01it/s]
  8%|██▉                                 | 6371/78125 [54:22<9:55:10,  2.01it/s]
  8%|██▉                                 | 6372/78125 [54:22<9:55:20,  2.01it/s]
  8%|██▉                                 | 6373/78125 [54:23<9:55:22,  2.01it/s]
  8%|██▉                                 | 6374/78125 [54:23<9:55:11,  2.01it/s]
  8%|██▉                                 | 6375/78125 [54:24<9:55:16,  2.01it/s]
  8%|██▉                                 | 6376/78125 [54:24<9:54:39,  2.01it/s]
  8%|██▉                                 | 6377/78125 [54:25<9:54:55,  2.01it/s]
  8%|██▉                                 | 6378/78125 [54:25<9:55:09,  2.01it/s]
  8%|██▉                                 | 6379/78125 [54:26<9:55:03,  2.01it/s]
  8%|██▉                                 | 6380/78125 [54:26<9:55:45,  2.01it/s]
  8%|██▉                                 | 6381/78125 [54:27<9:55:26,  2.01it/s]
  8%|██▉                                 | 6382/78125 [54:27<9:55:38,  2.01it/s]
  8%|██▉                                 | 6383/78125 [54:28<9:56:09,  2.01it/s]
  8%|██▉                                 | 6384/78125 [54:28<9:55:32,  2.01it/s]
  8%|██▉                                 | 6385/78125 [54:29<9:55:30,  2.01it/s]
  8%|██▉                                 | 6386/78125 [54:29<9:54:58,  2.01it/s]
  8%|██▉                                 | 6387/78125 [54:30<9:53:52,  2.01it/s]
  8%|██▉                                 | 6388/78125 [54:30<9:54:29,  2.01it/s]
  8%|██▉                                 | 6389/78125 [54:31<9:53:51,  2.01it/s]
  8%|██▉                                 | 6390/78125 [54:31<9:54:31,  2.01it/s]
  8%|██▉                                 | 6391/78125 [54:32<9:54:18,  2.01it/s]
  8%|██▉                                 | 6392/78125 [54:32<9:55:05,  2.01it/s]
  8%|██▉                                 | 6393/78125 [54:33<9:55:05,  2.01it/s]
  8%|██▉                                 | 6394/78125 [54:33<9:55:17,  2.01it/s]
  8%|██▉                                 | 6395/78125 [54:34<9:56:09,  2.01it/s]
  8%|██▉                                 | 6396/78125 [54:34<9:56:12,  2.01it/s]
  8%|██▉                                 | 6397/78125 [54:35<9:55:05,  2.01it/s]
  8%|██▉                                 | 6398/78125 [54:35<9:55:44,  2.01it/s]
  8%|██▉                                 | 6399/78125 [54:36<9:55:46,  2.01it/s]
  8%|██▉                                 | 6400/78125 [54:36<9:55:35,  2.01it/s]
  8%|██▉                                 | 6401/78125 [54:37<9:55:36,  2.01it/s]
  8%|██▉                                 | 6402/78125 [54:37<9:55:20,  2.01it/s]
  8%|██▉                                 | 6403/78125 [54:38<9:54:50,  2.01it/s]
  8%|██▉                                 | 6404/78125 [54:38<9:54:46,  2.01it/s]
  8%|██▉                                 | 6405/78125 [54:39<9:55:00,  2.01it/s]
  8%|██▉                                 | 6406/78125 [54:39<9:54:22,  2.01it/s]
  8%|██▉                                 | 6407/78125 [54:40<9:54:02,  2.01it/s]
  8%|██▉                                 | 6408/78125 [54:40<9:54:27,  2.01it/s]
  8%|██▉                                 | 6409/78125 [54:41<9:55:14,  2.01it/s]
  8%|██▉                                 | 6410/78125 [54:41<9:55:31,  2.01it/s]
  8%|██▉                                 | 6411/78125 [54:42<9:55:33,  2.01it/s]
  8%|██▉                                 | 6412/78125 [54:42<9:55:37,  2.01it/s]
  8%|██▉                                 | 6413/78125 [54:43<9:55:29,  2.01it/s]
  8%|██▉                                 | 6414/78125 [54:43<9:55:44,  2.01it/s]
  8%|██▉                                 | 6415/78125 [54:44<9:54:59,  2.01it/s]
  8%|██▉                                 | 6416/78125 [54:44<9:54:23,  2.01it/s]
  8%|██▉                                 | 6417/78125 [54:45<9:54:25,  2.01it/s]
  8%|██▉                                 | 6418/78125 [54:45<9:54:20,  2.01it/s]
  8%|██▉                                 | 6419/78125 [54:46<9:54:50,  2.01it/s]
  8%|██▉                                 | 6420/78125 [54:46<9:55:55,  2.01it/s]
  8%|██▉                                 | 6421/78125 [54:47<9:55:06,  2.01it/s]
  8%|██▉                                 | 6422/78125 [54:47<9:54:36,  2.01it/s]
  8%|██▉                                 | 6423/78125 [54:48<9:55:37,  2.01it/s]
  8%|██▉                                 | 6424/78125 [54:48<9:55:33,  2.01it/s]
  8%|██▉                                 | 6425/78125 [54:49<9:54:25,  2.01it/s]
  8%|██▉                                 | 6426/78125 [54:49<9:54:08,  2.01it/s]
  8%|██▉                                 | 6427/78125 [54:50<9:53:57,  2.01it/s]
  8%|██▉                                 | 6428/78125 [54:50<9:53:57,  2.01it/s]
  8%|██▉                                 | 6429/78125 [54:51<9:54:01,  2.01it/s]
  8%|██▉                                 | 6430/78125 [54:51<9:54:24,  2.01it/s]
  8%|██▉                                 | 6431/78125 [54:52<9:53:47,  2.01it/s]
  8%|██▉                                 | 6432/78125 [54:52<9:53:24,  2.01it/s]
  8%|██▉                                 | 6433/78125 [54:53<9:52:53,  2.02it/s]
  8%|██▉                                 | 6434/78125 [54:53<9:53:22,  2.01it/s]
  8%|██▉                                 | 6435/78125 [54:53<9:53:58,  2.01it/s]
  8%|██▉                                 | 6436/78125 [54:54<9:53:58,  2.01it/s]
  8%|██▉                                 | 6437/78125 [54:54<9:55:03,  2.01it/s]
  8%|██▉                                 | 6438/78125 [54:55<9:54:32,  2.01it/s]
  8%|██▉                                 | 6439/78125 [54:55<9:54:22,  2.01it/s]
  8%|██▉                                 | 6440/78125 [54:56<9:54:21,  2.01it/s]
  8%|██▉                                 | 6441/78125 [54:56<9:54:24,  2.01it/s]
  8%|██▉                                 | 6442/78125 [54:57<9:54:29,  2.01it/s]
  8%|██▉                                 | 6443/78125 [54:57<9:54:41,  2.01it/s]
  8%|██▉                                 | 6444/78125 [54:58<9:54:11,  2.01it/s]
  8%|██▉                                 | 6445/78125 [54:58<9:53:20,  2.01it/s]
  8%|██▉                                 | 6446/78125 [54:59<9:54:26,  2.01it/s]
  8%|██▉                                 | 6447/78125 [54:59<9:54:54,  2.01it/s]
  8%|██▉                                 | 6448/78125 [55:00<9:54:37,  2.01it/s]
  8%|██▉                                 | 6449/78125 [55:00<9:54:10,  2.01it/s]
  8%|██▉                                 | 6450/78125 [55:01<9:54:17,  2.01it/s]
  8%|██▉                                 | 6451/78125 [55:01<9:54:09,  2.01it/s]
  8%|██▉                                 | 6452/78125 [55:02<9:54:27,  2.01it/s]
  8%|██▉                                 | 6453/78125 [55:02<9:54:39,  2.01it/s]
  8%|██▉                                 | 6454/78125 [55:03<9:55:07,  2.01it/s]
  8%|██▉                                 | 6455/78125 [55:03<9:55:24,  2.01it/s]
  8%|██▉                                 | 6456/78125 [55:04<9:54:29,  2.01it/s]
  8%|██▉                                 | 6457/78125 [55:04<9:53:55,  2.01it/s]
  8%|██▉                                 | 6458/78125 [55:05<9:53:44,  2.01it/s]
  8%|██▉                                 | 6459/78125 [55:05<9:54:13,  2.01it/s]
  8%|██▉                                 | 6460/78125 [55:06<9:54:44,  2.01it/s]
  8%|██▉                                 | 6461/78125 [55:06<9:54:03,  2.01it/s]
  8%|██▉                                 | 6462/78125 [55:07<9:54:22,  2.01it/s]
  8%|██▉                                 | 6463/78125 [55:07<9:54:01,  2.01it/s]
  8%|██▉                                 | 6464/78125 [55:08<9:54:02,  2.01it/s]
  8%|██▉                                 | 6465/78125 [55:08<9:54:11,  2.01it/s]
  8%|██▉                                 | 6466/78125 [55:09<9:55:05,  2.01it/s]
  8%|██▉                                 | 6467/78125 [55:09<9:54:32,  2.01it/s]
  8%|██▉                                 | 6468/78125 [55:10<9:54:45,  2.01it/s]
  8%|██▉                                 | 6469/78125 [55:10<9:55:24,  2.01it/s]
  8%|██▉                                 | 6470/78125 [55:11<9:54:53,  2.01it/s]
  8%|██▉                                 | 6471/78125 [55:11<9:55:10,  2.01it/s]
  8%|██▉                                 | 6472/78125 [55:12<9:55:59,  2.00it/s]
  8%|██▉                                 | 6473/78125 [55:12<9:56:43,  2.00it/s]
  8%|██▉                                 | 6474/78125 [55:13<9:55:21,  2.01it/s]
  8%|██▉                                 | 6475/78125 [55:13<9:55:14,  2.01it/s]
  8%|██▉                                 | 6476/78125 [55:14<9:55:13,  2.01it/s]
  8%|██▉                                 | 6477/78125 [55:14<9:54:45,  2.01it/s]
  8%|██▉                                 | 6478/78125 [55:15<9:55:07,  2.01it/s]
  8%|██▉                                 | 6479/78125 [55:15<9:55:17,  2.01it/s]
  8%|██▉                                 | 6480/78125 [55:16<9:54:58,  2.01it/s]
  8%|██▉                                 | 6481/78125 [55:16<9:54:49,  2.01it/s]
  8%|██▉                                 | 6482/78125 [55:17<9:55:20,  2.01it/s]
  8%|██▉                                 | 6483/78125 [55:17<9:55:35,  2.00it/s]
  8%|██▉                                 | 6484/78125 [55:18<9:55:29,  2.01it/s]
  8%|██▉                                 | 6485/78125 [55:18<9:55:00,  2.01it/s]
  8%|██▉                                 | 6486/78125 [55:19<9:54:44,  2.01it/s]
  8%|██▉                                 | 6487/78125 [55:19<9:53:45,  2.01it/s]
  8%|██▉                                 | 6488/78125 [55:20<9:53:43,  2.01it/s]
  8%|██▉                                 | 6489/78125 [55:20<9:53:43,  2.01it/s]
  8%|██▉                                 | 6490/78125 [55:21<9:53:22,  2.01it/s]
  8%|██▉                                 | 6491/78125 [55:21<9:53:52,  2.01it/s]
  8%|██▉                                 | 6492/78125 [55:22<9:53:26,  2.01it/s]
  8%|██▉                                 | 6493/78125 [55:22<9:53:17,  2.01it/s]
  8%|██▉                                 | 6494/78125 [55:23<9:53:35,  2.01it/s]
  8%|██▉                                 | 6495/78125 [55:23<9:53:43,  2.01it/s]
  8%|██▉                                 | 6496/78125 [55:24<9:54:03,  2.01it/s]
  8%|██▉                                 | 6497/78125 [55:24<9:54:25,  2.01it/s]
  8%|██▉                                 | 6498/78125 [55:25<9:55:18,  2.01it/s]
  8%|██▉                                 | 6499/78125 [55:25<9:54:34,  2.01it/s]
  8%|██▉                                 | 6500/78125 [55:26<9:54:33,  2.01it/s]
  8%|██▉                                 | 6501/78125 [55:26<9:54:12,  2.01it/s]
  8%|██▉                                 | 6502/78125 [55:27<9:54:29,  2.01it/s]
  8%|██▉                                 | 6503/78125 [55:27<9:54:09,  2.01it/s]
  8%|██▉                                 | 6504/78125 [55:28<9:54:08,  2.01it/s]
  8%|██▉                                 | 6505/78125 [55:28<9:53:23,  2.01it/s]
  8%|██▉                                 | 6506/78125 [55:29<9:53:33,  2.01it/s]
  8%|██▉                                 | 6507/78125 [55:29<9:53:35,  2.01it/s]
  8%|██▉                                 | 6508/78125 [55:30<9:54:08,  2.01it/s]
  8%|██▉                                 | 6509/78125 [55:30<9:53:50,  2.01it/s]
  8%|██▉                                 | 6510/78125 [55:31<9:54:10,  2.01it/s]
  8%|███                                 | 6511/78125 [55:31<9:54:26,  2.01it/s]
  8%|███                                 | 6512/78125 [55:32<9:53:52,  2.01it/s]
  8%|███                                 | 6513/78125 [55:32<9:53:58,  2.01it/s]
  8%|███                                 | 6514/78125 [55:33<9:54:13,  2.01it/s]
  8%|███                                 | 6515/78125 [55:33<9:54:16,  2.01it/s]
  8%|███                                 | 6516/78125 [55:34<9:54:02,  2.01it/s]
  8%|███                                 | 6517/78125 [55:34<9:54:02,  2.01it/s]
  8%|███                                 | 6518/78125 [55:35<9:53:28,  2.01it/s]
  8%|███                                 | 6519/78125 [55:35<9:52:37,  2.01it/s]
  8%|███                                 | 6520/78125 [55:36<9:52:48,  2.01it/s]
  8%|███                                 | 6521/78125 [55:36<9:52:52,  2.01it/s]
  8%|███                                 | 6522/78125 [55:37<9:53:01,  2.01it/s]
  8%|███                                 | 6523/78125 [55:37<9:53:05,  2.01it/s]
  8%|███                                 | 6524/78125 [55:38<9:53:09,  2.01it/s]
  8%|███                                 | 6525/78125 [55:38<9:53:13,  2.01it/s]
  8%|███                                 | 6526/78125 [55:39<9:53:00,  2.01it/s]
  8%|███                                 | 6527/78125 [55:39<9:53:30,  2.01it/s]
  8%|███                                 | 6528/78125 [55:40<9:52:39,  2.01it/s]
  8%|███                                 | 6529/78125 [55:40<9:52:47,  2.01it/s]
  8%|███                                 | 6530/78125 [55:41<9:52:38,  2.01it/s]
  8%|███                                 | 6531/78125 [55:41<9:52:50,  2.01it/s]
  8%|███                                 | 6532/78125 [55:42<9:52:58,  2.01it/s]
  8%|███                                 | 6533/78125 [55:42<9:53:04,  2.01it/s]
  8%|███                                 | 6534/78125 [55:43<9:52:59,  2.01it/s]
  8%|███                                 | 6535/78125 [55:43<9:53:43,  2.01it/s]
  8%|███                                 | 6536/78125 [55:44<9:54:05,  2.01it/s]
  8%|███                                 | 6537/78125 [55:44<9:54:37,  2.01it/s]
  8%|███                                 | 6538/78125 [55:45<9:54:07,  2.01it/s]
  8%|███                                 | 6539/78125 [55:45<9:54:42,  2.01it/s]
  8%|███                                 | 6540/78125 [55:46<9:54:23,  2.01it/s]
  8%|███                                 | 6541/78125 [55:46<9:54:00,  2.01it/s]
  8%|███                                 | 6542/78125 [55:47<9:54:14,  2.01it/s]
  8%|███                                 | 6543/78125 [55:47<9:54:37,  2.01it/s]
  8%|███                                 | 6544/78125 [55:48<9:54:06,  2.01it/s]
  8%|███                                 | 6545/78125 [55:48<9:53:32,  2.01it/s]
  8%|███                                 | 6546/78125 [55:49<9:53:52,  2.01it/s]
  8%|███                                 | 6547/78125 [55:49<9:53:53,  2.01it/s]
  8%|███                                 | 6548/78125 [55:50<9:53:55,  2.01it/s]
  8%|███                                 | 6549/78125 [55:50<9:54:00,  2.01it/s]
  8%|███                                 | 6550/78125 [55:51<9:53:40,  2.01it/s]
  8%|███                                 | 6551/78125 [55:51<9:54:25,  2.01it/s]
  8%|███                                 | 6552/78125 [55:52<9:55:01,  2.00it/s]
  8%|███                                 | 6553/78125 [55:52<9:55:17,  2.00it/s]
  8%|███                                 | 6554/78125 [55:53<9:54:27,  2.01it/s]
  8%|███                                 | 6555/78125 [55:53<9:53:56,  2.01it/s]
  8%|███                                 | 6556/78125 [55:54<9:53:48,  2.01it/s]
  8%|███                                 | 6557/78125 [55:54<9:54:35,  2.01it/s]
  8%|███                                 | 6558/78125 [55:55<9:54:46,  2.01it/s]
  8%|███                                 | 6559/78125 [55:55<9:54:33,  2.01it/s]
  8%|███                                 | 6560/78125 [55:56<9:54:32,  2.01it/s]
  8%|███                                 | 6561/78125 [55:56<9:54:52,  2.01it/s]
  8%|███                                 | 6562/78125 [55:57<9:54:38,  2.01it/s]
  8%|███                                 | 6563/78125 [55:57<9:54:12,  2.01it/s]
  8%|███                                 | 6564/78125 [55:58<9:54:03,  2.01it/s]
  8%|███                                 | 6565/78125 [55:58<9:53:51,  2.01it/s]
  8%|███                                 | 6566/78125 [55:59<9:54:10,  2.01it/s]
  8%|███                                 | 6567/78125 [55:59<9:53:19,  2.01it/s]
  8%|███                                 | 6568/78125 [56:00<9:53:13,  2.01it/s]
  8%|███                                 | 6569/78125 [56:00<9:53:37,  2.01it/s]
  8%|███                                 | 6570/78125 [56:01<9:52:48,  2.01it/s]
  8%|███                                 | 6571/78125 [56:01<9:53:21,  2.01it/s]
  8%|███                                 | 6572/78125 [56:02<9:53:53,  2.01it/s]
  8%|███                                 | 6573/78125 [56:02<9:53:27,  2.01it/s]
  8%|███                                 | 6574/78125 [56:03<9:52:34,  2.01it/s]
  8%|███                                 | 6575/78125 [56:03<9:53:06,  2.01it/s]
  8%|███                                 | 6576/78125 [56:04<9:53:35,  2.01it/s]
  8%|███                                 | 6577/78125 [56:04<9:54:11,  2.01it/s]
  8%|███                                 | 6578/78125 [56:05<9:54:29,  2.01it/s]
  8%|███                                 | 6579/78125 [56:05<9:54:01,  2.01it/s]
  8%|███                                 | 6580/78125 [56:06<9:53:57,  2.01it/s]
  8%|███                                 | 6581/78125 [56:06<9:54:10,  2.01it/s]
  8%|███                                 | 6582/78125 [56:07<9:53:26,  2.01it/s]
  8%|███                                 | 6583/78125 [56:07<9:53:38,  2.01it/s]
  8%|███                                 | 6584/78125 [56:08<9:53:22,  2.01it/s]
  8%|███                                 | 6585/78125 [56:08<9:52:39,  2.01it/s]
  8%|███                                 | 6586/78125 [56:09<9:52:51,  2.01it/s]
  8%|███                                 | 6587/78125 [56:09<9:53:24,  2.01it/s]
  8%|███                                 | 6588/78125 [56:10<9:53:10,  2.01it/s]
  8%|███                                 | 6589/78125 [56:10<9:53:24,  2.01it/s]
  8%|███                                 | 6590/78125 [56:11<9:53:21,  2.01it/s]
  8%|███                                 | 6591/78125 [56:11<9:53:11,  2.01it/s]
  8%|███                                 | 6592/78125 [56:12<9:53:30,  2.01it/s]
  8%|███                                 | 6593/78125 [56:12<9:53:58,  2.01it/s]
  8%|███                                 | 6594/78125 [56:13<9:55:13,  2.00it/s]
  8%|███                                 | 6595/78125 [56:13<9:54:22,  2.01it/s]
  8%|███                                 | 6596/78125 [56:14<9:54:39,  2.00it/s]
  8%|███                                 | 6597/78125 [56:14<9:53:46,  2.01it/s]
  8%|███                                 | 6598/78125 [56:15<9:53:51,  2.01it/s]
  8%|███                                 | 6599/78125 [56:15<9:53:41,  2.01it/s]
  8%|███                                 | 6600/78125 [56:16<9:52:56,  2.01it/s]
  8%|███                                 | 6601/78125 [56:16<9:53:32,  2.01it/s]
  8%|███                                 | 6602/78125 [56:17<9:53:49,  2.01it/s]
  8%|███                                 | 6603/78125 [56:17<9:53:40,  2.01it/s]
  8%|███                                 | 6604/78125 [56:18<9:53:51,  2.01it/s]
  8%|███                                 | 6605/78125 [56:18<9:53:14,  2.01it/s]
  8%|███                                 | 6606/78125 [56:19<9:53:21,  2.01it/s]
  8%|███                                 | 6607/78125 [56:19<9:52:49,  2.01it/s]
  8%|███                                 | 6608/78125 [56:20<9:53:05,  2.01it/s]
  8%|███                                 | 6609/78125 [56:20<9:53:33,  2.01it/s]
  8%|███                                 | 6610/78125 [56:21<9:53:40,  2.01it/s]
  8%|███                                 | 6611/78125 [56:21<9:52:56,  2.01it/s]
  8%|███                                 | 6612/78125 [56:22<9:53:10,  2.01it/s]
  8%|███                                 | 6613/78125 [56:22<9:53:55,  2.01it/s]
  8%|███                                 | 6614/78125 [56:23<9:53:10,  2.01it/s]
  8%|███                                 | 6615/78125 [56:23<9:52:46,  2.01it/s]
  8%|███                                 | 6616/78125 [56:24<9:52:51,  2.01it/s]
  8%|███                                 | 6617/78125 [56:24<9:53:00,  2.01it/s]
  8%|███                                 | 6618/78125 [56:25<9:52:39,  2.01it/s]
  8%|███                                 | 6619/78125 [56:25<9:53:09,  2.01it/s]
  8%|███                                 | 6620/78125 [56:26<9:52:56,  2.01it/s]
  8%|███                                 | 6621/78125 [56:26<9:53:32,  2.01it/s]
  8%|███                                 | 6622/78125 [56:27<9:53:47,  2.01it/s]
  8%|███                                 | 6623/78125 [56:27<9:53:40,  2.01it/s]
  8%|███                                 | 6624/78125 [56:28<9:53:15,  2.01it/s]
  8%|███                                 | 6625/78125 [56:28<9:53:15,  2.01it/s]
  8%|███                                 | 6626/78125 [56:29<9:53:45,  2.01it/s]
  8%|███                                 | 6627/78125 [56:29<9:52:58,  2.01it/s]
  8%|███                                 | 6628/78125 [56:30<9:53:16,  2.01it/s]
  8%|███                                 | 6629/78125 [56:30<9:53:00,  2.01it/s]
  8%|███                                 | 6630/78125 [56:31<9:53:10,  2.01it/s]
  8%|███                                 | 6631/78125 [56:31<9:53:32,  2.01it/s]
  8%|███                                 | 6632/78125 [56:32<9:52:28,  2.01it/s]
  8%|███                                 | 6633/78125 [56:32<9:52:21,  2.01it/s]
  8%|███                                 | 6634/78125 [56:33<9:52:31,  2.01it/s]
  8%|███                                 | 6635/78125 [56:33<9:52:44,  2.01it/s]
  8%|███                                 | 6636/78125 [56:34<9:52:20,  2.01it/s]
  8%|███                                 | 6637/78125 [56:34<9:52:14,  2.01it/s]
  8%|███                                 | 6638/78125 [56:35<9:51:53,  2.01it/s]
  8%|███                                 | 6639/78125 [56:35<9:51:54,  2.01it/s]
  8%|███                                 | 6640/78125 [56:36<9:52:56,  2.01it/s]
  9%|███                                 | 6641/78125 [56:36<9:54:11,  2.01it/s]
  9%|███                                 | 6642/78125 [56:37<9:53:42,  2.01it/s]
  9%|███                                 | 6643/78125 [56:37<9:53:08,  2.01it/s]
  9%|███                                 | 6644/78125 [56:38<9:53:48,  2.01it/s]
  9%|███                                 | 6645/78125 [56:38<9:53:15,  2.01it/s]
  9%|███                                 | 6646/78125 [56:39<9:52:37,  2.01it/s]
  9%|███                                 | 6647/78125 [56:39<9:53:12,  2.01it/s]
  9%|███                                 | 6648/78125 [56:40<9:53:02,  2.01it/s]
  9%|███                                 | 6649/78125 [56:40<9:52:33,  2.01it/s]
  9%|███                                 | 6650/78125 [56:41<9:52:09,  2.01it/s]
  9%|███                                 | 6651/78125 [56:41<9:52:42,  2.01it/s]
  9%|███                                 | 6652/78125 [56:42<9:52:51,  2.01it/s]
  9%|███                                 | 6653/78125 [56:42<9:52:40,  2.01it/s]
  9%|███                                 | 6654/78125 [56:43<9:52:51,  2.01it/s]
  9%|███                                 | 6655/78125 [56:43<9:52:31,  2.01it/s]
  9%|███                                 | 6656/78125 [56:44<9:51:59,  2.01it/s]
  9%|███                                 | 6657/78125 [56:44<9:51:11,  2.01it/s]
  9%|███                                 | 6658/78125 [56:44<9:52:13,  2.01it/s]
  9%|███                                 | 6659/78125 [56:45<9:52:17,  2.01it/s]
  9%|███                                 | 6660/78125 [56:45<9:52:14,  2.01it/s]
  9%|███                                 | 6661/78125 [56:46<9:51:56,  2.01it/s]
  9%|███                                 | 6662/78125 [56:46<9:51:07,  2.01it/s]
  9%|███                                 | 6663/78125 [56:47<9:50:51,  2.02it/s]
  9%|███                                 | 6664/78125 [56:47<9:51:04,  2.01it/s]
  9%|███                                 | 6665/78125 [56:48<9:51:04,  2.01it/s]
  9%|███                                 | 6666/78125 [56:48<9:51:43,  2.01it/s]
  9%|███                                 | 6667/78125 [56:49<9:52:08,  2.01it/s]
  9%|███                                 | 6668/78125 [56:49<9:52:32,  2.01it/s]
  9%|███                                 | 6669/78125 [56:50<9:52:39,  2.01it/s]
  9%|███                                 | 6670/78125 [56:50<9:52:30,  2.01it/s]
  9%|███                                 | 6671/78125 [56:51<9:52:14,  2.01it/s]
  9%|███                                 | 6672/78125 [56:51<9:52:42,  2.01it/s]
  9%|███                                 | 6673/78125 [56:52<9:52:12,  2.01it/s]
  9%|███                                 | 6674/78125 [56:52<9:52:40,  2.01it/s]
  9%|███                                 | 6675/78125 [56:53<9:52:29,  2.01it/s]
  9%|███                                 | 6676/78125 [56:53<9:52:27,  2.01it/s]
  9%|███                                 | 6677/78125 [56:54<9:53:01,  2.01it/s]
  9%|███                                 | 6678/78125 [56:54<9:53:16,  2.01it/s]
  9%|███                                 | 6679/78125 [56:55<9:52:41,  2.01it/s]
  9%|███                                 | 6680/78125 [56:55<9:53:29,  2.01it/s]
  9%|███                                 | 6681/78125 [56:56<9:53:06,  2.01it/s]
  9%|███                                 | 6682/78125 [56:56<9:52:36,  2.01it/s]
  9%|███                                 | 6683/78125 [56:57<9:53:21,  2.01it/s]
  9%|███                                 | 6684/78125 [56:57<9:52:57,  2.01it/s]
  9%|███                                 | 6685/78125 [56:58<9:52:33,  2.01it/s]
  9%|███                                 | 6686/78125 [56:58<9:52:35,  2.01it/s]
  9%|███                                 | 6687/78125 [56:59<9:52:24,  2.01it/s]
  9%|███                                 | 6688/78125 [56:59<9:52:28,  2.01it/s]
  9%|███                                 | 6689/78125 [57:00<9:51:46,  2.01it/s]
  9%|███                                 | 6690/78125 [57:00<9:51:44,  2.01it/s]
  9%|███                                 | 6691/78125 [57:01<9:51:46,  2.01it/s]
  9%|███                                 | 6692/78125 [57:01<9:50:55,  2.01it/s]
  9%|███                                 | 6693/78125 [57:02<9:50:59,  2.01it/s]
  9%|███                                 | 6694/78125 [57:02<9:51:29,  2.01it/s]
  9%|███                                 | 6695/78125 [57:03<9:51:34,  2.01it/s]
  9%|███                                 | 6696/78125 [57:03<9:52:14,  2.01it/s]
  9%|███                                 | 6697/78125 [57:04<9:52:44,  2.01it/s]
  9%|███                                 | 6698/78125 [57:04<9:52:36,  2.01it/s]
  9%|███                                 | 6699/78125 [57:05<9:51:43,  2.01it/s]
  9%|███                                 | 6700/78125 [57:05<9:51:59,  2.01it/s]
  9%|███                                 | 6701/78125 [57:06<9:52:12,  2.01it/s]
  9%|███                                 | 6702/78125 [57:06<9:52:26,  2.01it/s]
  9%|███                                 | 6703/78125 [57:07<9:51:31,  2.01it/s]
  9%|███                                 | 6704/78125 [57:07<9:51:53,  2.01it/s]
  9%|███                                 | 6705/78125 [57:08<9:51:53,  2.01it/s]
  9%|███                                 | 6706/78125 [57:08<9:51:52,  2.01it/s]
  9%|███                                 | 6707/78125 [57:09<9:51:20,  2.01it/s]
  9%|███                                 | 6708/78125 [57:09<9:51:11,  2.01it/s]
  9%|███                                 | 6709/78125 [57:10<9:52:06,  2.01it/s]
  9%|███                                 | 6710/78125 [57:10<9:51:49,  2.01it/s]
  9%|███                                 | 6711/78125 [57:11<9:51:42,  2.01it/s]
  9%|███                                 | 6712/78125 [57:11<9:52:21,  2.01it/s]
  9%|███                                 | 6713/78125 [57:12<9:51:38,  2.01it/s]
  9%|███                                 | 6714/78125 [57:12<9:51:29,  2.01it/s]
  9%|███                                 | 6715/78125 [57:13<9:51:45,  2.01it/s]
  9%|███                                 | 6716/78125 [57:13<9:52:49,  2.01it/s]
  9%|███                                 | 6717/78125 [57:14<9:52:36,  2.01it/s]
  9%|███                                 | 6718/78125 [57:14<9:52:12,  2.01it/s]
  9%|███                                 | 6719/78125 [57:15<9:52:20,  2.01it/s]
  9%|███                                 | 6720/78125 [57:15<9:52:22,  2.01it/s]
  9%|███                                 | 6721/78125 [57:16<9:52:14,  2.01it/s]
  9%|███                                 | 6722/78125 [57:16<9:53:03,  2.01it/s]
  9%|███                                 | 6723/78125 [57:17<9:53:02,  2.01it/s]
  9%|███                                 | 6724/78125 [57:17<9:52:50,  2.01it/s]
  9%|███                                 | 6725/78125 [57:18<9:52:17,  2.01it/s]
  9%|███                                 | 6726/78125 [57:18<9:52:42,  2.01it/s]
  9%|███                                 | 6727/78125 [57:19<9:52:53,  2.01it/s]
  9%|███                                 | 6728/78125 [57:19<9:52:45,  2.01it/s]
  9%|███                                 | 6729/78125 [57:20<9:53:10,  2.01it/s]
  9%|███                                 | 6730/78125 [57:20<9:52:38,  2.01it/s]
  9%|███                                 | 6731/78125 [57:21<9:51:51,  2.01it/s]
  9%|███                                 | 6732/78125 [57:21<9:52:14,  2.01it/s]
  9%|███                                 | 6733/78125 [57:22<9:52:30,  2.01it/s]
  9%|███                                 | 6734/78125 [57:22<9:52:34,  2.01it/s]
  9%|███                                 | 6735/78125 [57:23<9:52:40,  2.01it/s]
  9%|███                                 | 6736/78125 [57:23<9:52:38,  2.01it/s]
  9%|███                                 | 6737/78125 [57:24<9:52:43,  2.01it/s]
  9%|███                                 | 6738/78125 [57:24<9:52:39,  2.01it/s]
  9%|███                                 | 6739/78125 [57:25<9:53:09,  2.01it/s]
  9%|███                                 | 6740/78125 [57:25<9:53:05,  2.01it/s]
  9%|███                                 | 6741/78125 [57:26<9:53:07,  2.01it/s]
  9%|███                                 | 6742/78125 [57:26<9:52:49,  2.01it/s]
  9%|███                                 | 6743/78125 [57:27<9:52:48,  2.01it/s]
  9%|███                                 | 6744/78125 [57:27<9:53:01,  2.01it/s]
  9%|███                                 | 6745/78125 [57:28<9:53:20,  2.01it/s]
  9%|███                                 | 6746/78125 [57:28<9:53:01,  2.01it/s]
  9%|███                                 | 6747/78125 [57:29<9:53:22,  2.00it/s]
  9%|███                                 | 6748/78125 [57:29<9:53:23,  2.00it/s]
  9%|███                                 | 6749/78125 [57:30<9:53:41,  2.00it/s]
  9%|███                                 | 6750/78125 [57:30<9:53:31,  2.00it/s]
  9%|███                                 | 6751/78125 [57:31<9:52:28,  2.01it/s]
  9%|███                                 | 6752/78125 [57:31<9:52:30,  2.01it/s]
  9%|███                                 | 6753/78125 [57:32<9:52:14,  2.01it/s]
  9%|███                                 | 6754/78125 [57:32<9:51:52,  2.01it/s]
  9%|███                                 | 6755/78125 [57:33<9:51:58,  2.01it/s]
  9%|███                                 | 6756/78125 [57:33<9:52:09,  2.01it/s]
  9%|███                                 | 6757/78125 [57:34<9:51:23,  2.01it/s]
  9%|███                                 | 6758/78125 [57:34<9:51:45,  2.01it/s]
  9%|███                                 | 6759/78125 [57:35<9:51:38,  2.01it/s]
  9%|███                                 | 6760/78125 [57:35<9:51:31,  2.01it/s]
  9%|███                                 | 6761/78125 [57:36<9:51:42,  2.01it/s]
  9%|███                                 | 6762/78125 [57:36<9:52:02,  2.01it/s]
  9%|███                                 | 6763/78125 [57:37<9:52:10,  2.01it/s]
  9%|███                                 | 6764/78125 [57:37<9:51:34,  2.01it/s]
  9%|███                                 | 6765/78125 [57:38<9:50:57,  2.01it/s]
  9%|███                                 | 6766/78125 [57:38<9:51:34,  2.01it/s]
  9%|███                                 | 6767/78125 [57:39<9:51:13,  2.01it/s]
  9%|███                                 | 6768/78125 [57:39<9:51:13,  2.01it/s]
  9%|███                                 | 6769/78125 [57:40<9:51:43,  2.01it/s]
  9%|███                                 | 6770/78125 [57:40<9:52:08,  2.01it/s]
  9%|███                                 | 6771/78125 [57:41<9:51:50,  2.01it/s]
  9%|███                                 | 6772/78125 [57:41<9:52:44,  2.01it/s]
  9%|███                                 | 6773/78125 [57:42<9:51:36,  2.01it/s]
  9%|███                                 | 6774/78125 [57:42<9:51:52,  2.01it/s]
  9%|███                                 | 6775/78125 [57:43<9:51:11,  2.01it/s]
  9%|███                                 | 6776/78125 [57:43<9:51:57,  2.01it/s]
  9%|███                                 | 6777/78125 [57:44<9:51:16,  2.01it/s]
  9%|███                                 | 6778/78125 [57:44<9:51:59,  2.01it/s]
  9%|███                                 | 6779/78125 [57:45<9:51:21,  2.01it/s]
  9%|███                                 | 6780/78125 [57:45<9:51:35,  2.01it/s]
  9%|███                                 | 6781/78125 [57:46<9:52:02,  2.01it/s]
  9%|███▏                                | 6782/78125 [57:46<9:52:29,  2.01it/s]
  9%|███▏                                | 6783/78125 [57:47<9:52:15,  2.01it/s]
  9%|███▏                                | 6784/78125 [57:47<9:52:04,  2.01it/s]
  9%|███▏                                | 6785/78125 [57:48<9:52:16,  2.01it/s]
  9%|███▏                                | 6786/78125 [57:48<9:52:32,  2.01it/s]
  9%|███▏                                | 6787/78125 [57:49<9:52:22,  2.01it/s]
  9%|███▏                                | 6788/78125 [57:49<9:52:00,  2.01it/s]
  9%|███▏                                | 6789/78125 [57:50<9:52:25,  2.01it/s]
  9%|███▏                                | 6790/78125 [57:50<9:51:26,  2.01it/s]
  9%|███▏                                | 6791/78125 [57:51<9:51:27,  2.01it/s]
  9%|███▏                                | 6792/78125 [57:51<9:51:45,  2.01it/s]
  9%|███▏                                | 6793/78125 [57:52<9:51:49,  2.01it/s]
  9%|███▏                                | 6794/78125 [57:52<9:51:38,  2.01it/s]
  9%|███▏                                | 6795/78125 [57:53<9:51:57,  2.01it/s]
  9%|███▏                                | 6796/78125 [57:53<9:51:33,  2.01it/s]
  9%|███▏                                | 6797/78125 [57:54<9:51:27,  2.01it/s]
  9%|███▏                                | 6798/78125 [57:54<9:51:21,  2.01it/s]
  9%|███▏                                | 6799/78125 [57:55<9:51:48,  2.01it/s]
  9%|███▏                                | 6800/78125 [57:55<9:51:07,  2.01it/s]
  9%|███▏                                | 6801/78125 [57:56<9:51:46,  2.01it/s]
  9%|███▏                                | 6802/78125 [57:56<9:52:09,  2.01it/s]
  9%|███▏                                | 6803/78125 [57:57<9:52:18,  2.01it/s]
  9%|███▏                                | 6804/78125 [57:57<9:51:51,  2.01it/s]
  9%|███▏                                | 6805/78125 [57:58<9:51:21,  2.01it/s]
  9%|███▏                                | 6806/78125 [57:58<9:51:19,  2.01it/s]
  9%|███▏                                | 6807/78125 [57:59<9:51:24,  2.01it/s]
  9%|███▏                                | 6808/78125 [57:59<9:51:42,  2.01it/s]
  9%|███▏                                | 6809/78125 [58:00<9:52:08,  2.01it/s]
  9%|███▏                                | 6810/78125 [58:00<9:51:59,  2.01it/s]
  9%|███▏                                | 6811/78125 [58:01<9:51:06,  2.01it/s]
  9%|███▏                                | 6812/78125 [58:01<9:51:47,  2.01it/s]
  9%|███▏                                | 6813/78125 [58:02<9:51:00,  2.01it/s]
  9%|███▏                                | 6814/78125 [58:02<9:51:41,  2.01it/s]
  9%|███▏                                | 6815/78125 [58:03<9:50:52,  2.01it/s]
  9%|███▏                                | 6816/78125 [58:03<9:51:38,  2.01it/s]
  9%|███▏                                | 6817/78125 [58:04<9:51:48,  2.01it/s]
  9%|███▏                                | 6818/78125 [58:04<9:51:48,  2.01it/s]
  9%|███▏                                | 6819/78125 [58:05<9:51:59,  2.01it/s]
  9%|███▏                                | 6820/78125 [58:05<9:51:53,  2.01it/s]
  9%|███▏                                | 6821/78125 [58:06<9:51:24,  2.01it/s]
  9%|███▏                                | 6822/78125 [58:06<9:51:01,  2.01it/s]
  9%|███▏                                | 6823/78125 [58:07<9:50:48,  2.01it/s]
  9%|███▏                                | 6824/78125 [58:07<9:50:25,  2.01it/s]
  9%|███▏                                | 6825/78125 [58:08<9:50:18,  2.01it/s]
  9%|███▏                                | 6826/78125 [58:08<9:51:44,  2.01it/s]
  9%|███▏                                | 6827/78125 [58:09<9:51:27,  2.01it/s]
  9%|███▏                                | 6828/78125 [58:09<9:51:30,  2.01it/s]
  9%|███▏                                | 6829/78125 [58:10<9:51:47,  2.01it/s]
  9%|███▏                                | 6830/78125 [58:10<9:52:10,  2.01it/s]
  9%|███▏                                | 6831/78125 [58:11<9:51:12,  2.01it/s]
  9%|███▏                                | 6832/78125 [58:11<9:51:30,  2.01it/s]
  9%|███▏                                | 6833/78125 [58:12<9:51:22,  2.01it/s]
  9%|███▏                                | 6834/78125 [58:12<9:51:31,  2.01it/s]
  9%|███▏                                | 6835/78125 [58:13<9:51:13,  2.01it/s]
  9%|███▏                                | 6836/78125 [58:13<9:50:32,  2.01it/s]
  9%|███▏                                | 6837/78125 [58:14<9:51:04,  2.01it/s]
  9%|███▏                                | 6838/78125 [58:14<9:51:05,  2.01it/s]
  9%|███▏                                | 6839/78125 [58:15<9:51:16,  2.01it/s]
  9%|███▏                                | 6840/78125 [58:15<9:51:13,  2.01it/s]
  9%|███▏                                | 6841/78125 [58:16<9:50:58,  2.01it/s]
  9%|███▏                                | 6842/78125 [58:16<9:51:04,  2.01it/s]
  9%|███▏                                | 6843/78125 [58:17<9:50:12,  2.01it/s]
  9%|███▏                                | 6844/78125 [58:17<9:50:14,  2.01it/s]
  9%|███▏                                | 6845/78125 [58:18<9:50:39,  2.01it/s]
  9%|███▏                                | 6846/78125 [58:18<9:50:02,  2.01it/s]
  9%|███▏                                | 6847/78125 [58:19<9:50:52,  2.01it/s]
  9%|███▏                                | 6848/78125 [58:19<9:50:54,  2.01it/s]
  9%|███▏                                | 6849/78125 [58:20<9:50:51,  2.01it/s]
  9%|███▏                                | 6850/78125 [58:20<9:50:35,  2.01it/s]
  9%|███▏                                | 6851/78125 [58:21<9:51:01,  2.01it/s]
  9%|███▏                                | 6852/78125 [58:21<9:51:34,  2.01it/s]
  9%|███▏                                | 6853/78125 [58:22<9:51:04,  2.01it/s]
  9%|███▏                                | 6854/78125 [58:22<9:50:57,  2.01it/s]
  9%|███▏                                | 6855/78125 [58:23<9:50:16,  2.01it/s]
  9%|███▏                                | 6856/78125 [58:23<9:49:33,  2.01it/s]
  9%|███▏                                | 6857/78125 [58:24<9:50:16,  2.01it/s]
  9%|███▏                                | 6858/78125 [58:24<9:49:41,  2.01it/s]
  9%|███▏                                | 6859/78125 [58:25<9:50:00,  2.01it/s]
  9%|███▏                                | 6860/78125 [58:25<9:49:53,  2.01it/s]
  9%|███▏                                | 6861/78125 [58:26<9:50:25,  2.01it/s]
  9%|███▏                                | 6862/78125 [58:26<9:50:30,  2.01it/s]
  9%|███▏                                | 6863/78125 [58:27<9:50:27,  2.01it/s]
  9%|███▏                                | 6864/78125 [58:27<9:49:54,  2.01it/s]
  9%|███▏                                | 6865/78125 [58:27<9:49:51,  2.01it/s]
  9%|███▏                                | 6866/78125 [58:28<9:50:03,  2.01it/s]
  9%|███▏                                | 6867/78125 [58:28<9:50:19,  2.01it/s]
  9%|███▏                                | 6868/78125 [58:29<9:50:24,  2.01it/s]
  9%|███▏                                | 6869/78125 [58:29<9:50:58,  2.01it/s]
  9%|███▏                                | 6870/78125 [58:30<9:50:41,  2.01it/s]
  9%|███▏                                | 6871/78125 [58:30<9:51:37,  2.01it/s]
  9%|███▏                                | 6872/78125 [58:31<9:50:59,  2.01it/s]
  9%|███▏                                | 6873/78125 [58:31<9:51:00,  2.01it/s]
  9%|███▏                                | 6874/78125 [58:32<9:50:23,  2.01it/s]
  9%|███▏                                | 6875/78125 [58:32<9:51:10,  2.01it/s]
  9%|███▏                                | 6876/78125 [58:33<9:51:13,  2.01it/s]
  9%|███▏                                | 6877/78125 [58:33<9:51:30,  2.01it/s]
  9%|███▏                                | 6878/78125 [58:34<9:50:31,  2.01it/s]
  9%|███▏                                | 6879/78125 [58:34<9:50:30,  2.01it/s]
  9%|███▏                                | 6880/78125 [58:35<9:50:09,  2.01it/s]
  9%|███▏                                | 6881/78125 [58:35<9:50:10,  2.01it/s]
  9%|███▏                                | 6882/78125 [58:36<9:50:26,  2.01it/s]
  9%|███▏                                | 6883/78125 [58:36<9:50:14,  2.01it/s]
  9%|███▏                                | 6884/78125 [58:37<9:50:19,  2.01it/s]
  9%|███▏                                | 6885/78125 [58:37<9:51:04,  2.01it/s]
  9%|███▏                                | 6886/78125 [58:38<9:51:02,  2.01it/s]
  9%|███▏                                | 6887/78125 [58:38<9:50:27,  2.01it/s]
  9%|███▏                                | 6888/78125 [58:39<9:50:05,  2.01it/s]
  9%|███▏                                | 6889/78125 [58:39<9:50:41,  2.01it/s]
  9%|███▏                                | 6890/78125 [58:40<9:50:18,  2.01it/s]
  9%|███▏                                | 6891/78125 [58:40<9:50:11,  2.01it/s]
  9%|███▏                                | 6892/78125 [58:41<9:50:28,  2.01it/s]
  9%|███▏                                | 6893/78125 [58:41<9:50:45,  2.01it/s]
  9%|███▏                                | 6894/78125 [58:42<9:51:08,  2.01it/s]
  9%|███▏                                | 6895/78125 [58:42<9:51:04,  2.01it/s]
  9%|███▏                                | 6896/78125 [58:43<9:50:16,  2.01it/s]
  9%|███▏                                | 6897/78125 [58:43<9:49:35,  2.01it/s]
  9%|███▏                                | 6898/78125 [58:44<9:49:12,  2.01it/s]
  9%|███▏                                | 6899/78125 [58:44<9:49:49,  2.01it/s]
  9%|███▏                                | 6900/78125 [58:45<9:50:18,  2.01it/s]
  9%|███▏                                | 6901/78125 [58:45<9:50:13,  2.01it/s]
  9%|███▏                                | 6902/78125 [58:46<9:50:18,  2.01it/s]
  9%|███▏                                | 6903/78125 [58:46<9:49:38,  2.01it/s]
  9%|███▏                                | 6904/78125 [58:47<9:50:11,  2.01it/s]
  9%|███▏                                | 6905/78125 [58:47<9:50:25,  2.01it/s]
  9%|███▏                                | 6906/78125 [58:48<9:51:14,  2.01it/s]
  9%|███▏                                | 6907/78125 [58:48<9:50:49,  2.01it/s]
  9%|███▏                                | 6908/78125 [58:49<9:50:03,  2.01it/s]
  9%|███▏                                | 6909/78125 [58:49<9:50:14,  2.01it/s]
  9%|███▏                                | 6910/78125 [58:50<9:49:53,  2.01it/s]
  9%|███▏                                | 6911/78125 [58:50<9:49:44,  2.01it/s]
  9%|███▏                                | 6912/78125 [58:51<9:50:28,  2.01it/s]
  9%|███▏                                | 6913/78125 [58:51<9:50:34,  2.01it/s]
  9%|███▏                                | 6914/78125 [58:52<9:50:48,  2.01it/s]
  9%|███▏                                | 6915/78125 [58:52<9:50:22,  2.01it/s]
  9%|███▏                                | 6916/78125 [58:53<9:50:36,  2.01it/s]
  9%|███▏                                | 6917/78125 [58:53<9:50:38,  2.01it/s]
  9%|███▏                                | 6918/78125 [58:54<9:50:13,  2.01it/s]
  9%|███▏                                | 6919/78125 [58:54<9:49:48,  2.01it/s]
  9%|███▏                                | 6920/78125 [58:55<9:50:30,  2.01it/s]
  9%|███▏                                | 6921/78125 [58:55<9:50:02,  2.01it/s]
  9%|███▏                                | 6922/78125 [58:56<9:49:34,  2.01it/s]
  9%|███▏                                | 6923/78125 [58:56<9:50:00,  2.01it/s]
  9%|███▏                                | 6924/78125 [58:57<9:50:50,  2.01it/s]
  9%|███▏                                | 6925/78125 [58:57<9:49:23,  2.01it/s]
  9%|███▏                                | 6926/78125 [58:58<9:50:12,  2.01it/s]
  9%|███▏                                | 6927/78125 [58:58<9:50:51,  2.01it/s]
  9%|███▏                                | 6928/78125 [58:59<9:50:28,  2.01it/s]
  9%|███▏                                | 6929/78125 [58:59<9:49:57,  2.01it/s]
  9%|███▏                                | 6930/78125 [59:00<9:49:17,  2.01it/s]
  9%|███▏                                | 6931/78125 [59:00<9:49:58,  2.01it/s]
  9%|███▏                                | 6932/78125 [59:01<9:49:29,  2.01it/s]
  9%|███▏                                | 6933/78125 [59:01<9:49:32,  2.01it/s]
  9%|███▏                                | 6934/78125 [59:02<9:50:04,  2.01it/s]
  9%|███▏                                | 6935/78125 [59:02<9:50:40,  2.01it/s]
  9%|███▏                                | 6936/78125 [59:03<9:50:14,  2.01it/s]
  9%|███▏                                | 6937/78125 [59:03<9:50:49,  2.01it/s]
  9%|███▏                                | 6938/78125 [59:04<9:50:58,  2.01it/s]
  9%|███▏                                | 6939/78125 [59:04<9:50:24,  2.01it/s]
  9%|███▏                                | 6940/78125 [59:05<9:50:23,  2.01it/s]
  9%|███▏                                | 6941/78125 [59:05<9:50:09,  2.01it/s]
  9%|███▏                                | 6942/78125 [59:06<9:50:11,  2.01it/s]
  9%|███▏                                | 6943/78125 [59:06<9:50:24,  2.01it/s]
  9%|███▏                                | 6944/78125 [59:07<9:50:22,  2.01it/s]
  9%|███▏                                | 6945/78125 [59:07<9:49:58,  2.01it/s]
  9%|███▏                                | 6946/78125 [59:08<9:50:01,  2.01it/s]
  9%|███▏                                | 6947/78125 [59:08<9:49:55,  2.01it/s]
  9%|███▏                                | 6948/78125 [59:09<9:49:54,  2.01it/s]
  9%|███▏                                | 6949/78125 [59:09<9:50:10,  2.01it/s]
  9%|███▏                                | 6950/78125 [59:10<9:50:08,  2.01it/s]
  9%|███▏                                | 6951/78125 [59:10<9:50:13,  2.01it/s]
  9%|███▏                                | 6952/78125 [59:11<9:50:39,  2.01it/s]
  9%|███▏                                | 6953/78125 [59:11<9:50:24,  2.01it/s]
  9%|███▏                                | 6954/78125 [59:12<9:50:08,  2.01it/s]
  9%|███▏                                | 6955/78125 [59:12<9:50:21,  2.01it/s]
  9%|███▏                                | 6956/78125 [59:13<9:50:54,  2.01it/s]
  9%|███▏                                | 6957/78125 [59:13<9:50:46,  2.01it/s]
  9%|███▏                                | 6958/78125 [59:14<9:50:37,  2.01it/s]
  9%|███▏                                | 6959/78125 [59:14<9:50:04,  2.01it/s]
  9%|███▏                                | 6960/78125 [59:15<9:50:46,  2.01it/s]
  9%|███▏                                | 6961/78125 [59:15<9:50:23,  2.01it/s]
  9%|███▏                                | 6962/78125 [59:16<9:50:43,  2.01it/s]
  9%|███▏                                | 6963/78125 [59:16<9:51:10,  2.01it/s]
  9%|███▏                                | 6964/78125 [59:17<9:50:58,  2.01it/s]
  9%|███▏                                | 6965/78125 [59:17<9:50:01,  2.01it/s]
  9%|███▏                                | 6966/78125 [59:18<9:49:36,  2.01it/s]
  9%|███▏                                | 6967/78125 [59:18<9:50:33,  2.01it/s]
  9%|███▏                                | 6968/78125 [59:19<9:50:46,  2.01it/s]
  9%|███▏                                | 6969/78125 [59:19<9:50:24,  2.01it/s]
  9%|███▏                                | 6970/78125 [59:20<9:50:15,  2.01it/s]
  9%|███▏                                | 6971/78125 [59:20<9:49:59,  2.01it/s]
  9%|███▏                                | 6972/78125 [59:21<9:50:10,  2.01it/s]
  9%|███▏                                | 6973/78125 [59:21<9:50:59,  2.01it/s]
  9%|███▏                                | 6974/78125 [59:22<9:51:26,  2.00it/s]
  9%|███▏                                | 6975/78125 [59:22<9:51:51,  2.00it/s]
  9%|███▏                                | 6976/78125 [59:23<9:51:33,  2.00it/s]
  9%|███▏                                | 6977/78125 [59:23<9:50:44,  2.01it/s]
  9%|███▏                                | 6978/78125 [59:24<9:50:33,  2.01it/s]
  9%|███▏                                | 6979/78125 [59:24<9:51:01,  2.01it/s]
  9%|███▏                                | 6980/78125 [59:25<9:50:04,  2.01it/s]
  9%|███▏                                | 6981/78125 [59:25<9:49:24,  2.01it/s]
  9%|███▏                                | 6982/78125 [59:26<9:50:03,  2.01it/s]
  9%|███▏                                | 6983/78125 [59:26<9:50:08,  2.01it/s]
  9%|███▏                                | 6984/78125 [59:27<9:50:34,  2.01it/s]
  9%|███▏                                | 6985/78125 [59:27<9:51:15,  2.01it/s]
  9%|███▏                                | 6986/78125 [59:28<9:50:07,  2.01it/s]
  9%|███▏                                | 6987/78125 [59:28<9:49:56,  2.01it/s]
  9%|███▏                                | 6988/78125 [59:29<9:49:49,  2.01it/s]
  9%|███▏                                | 6989/78125 [59:29<9:50:17,  2.01it/s]
  9%|███▏                                | 6990/78125 [59:30<9:50:35,  2.01it/s]
  9%|███▏                                | 6991/78125 [59:30<9:50:21,  2.01it/s]
  9%|███▏                                | 6992/78125 [59:31<9:49:41,  2.01it/s]
  9%|███▏                                | 6993/78125 [59:31<9:49:57,  2.01it/s]
  9%|███▏                                | 6994/78125 [59:32<9:49:55,  2.01it/s]
  9%|███▏                                | 6995/78125 [59:32<9:50:15,  2.01it/s]
  9%|███▏                                | 6996/78125 [59:33<9:50:58,  2.01it/s]
  9%|███▏                                | 6997/78125 [59:33<9:50:07,  2.01it/s]
  9%|███▏                                | 6998/78125 [59:34<9:50:14,  2.01it/s]
  9%|███▏                                | 6999/78125 [59:34<9:50:18,  2.01it/s]
  9%|███▏                                | 7000/78125 [59:35<9:49:06,  2.01it/s]
  9%|███▏                                | 7001/78125 [59:35<9:49:35,  2.01it/s]
  9%|███▏                                | 7002/78125 [59:36<9:49:22,  2.01it/s]
  9%|███▏                                | 7003/78125 [59:36<9:50:00,  2.01it/s]
  9%|███▏                                | 7004/78125 [59:37<9:49:41,  2.01it/s]
  9%|███▏                                | 7005/78125 [59:37<9:49:53,  2.01it/s]
  9%|███▏                                | 7006/78125 [59:38<9:49:27,  2.01it/s]
  9%|███▏                                | 7007/78125 [59:38<9:49:42,  2.01it/s]
  9%|███▏                                | 7008/78125 [59:39<9:49:55,  2.01it/s]
  9%|███▏                                | 7009/78125 [59:39<9:50:13,  2.01it/s]
  9%|███▏                                | 7010/78125 [59:40<9:49:57,  2.01it/s]
  9%|███▏                                | 7011/78125 [59:40<9:49:52,  2.01it/s]
  9%|███▏                                | 7012/78125 [59:41<9:49:28,  2.01it/s]
  9%|███▏                                | 7013/78125 [59:41<9:49:24,  2.01it/s]
  9%|███▏                                | 7014/78125 [59:42<9:48:55,  2.01it/s]
  9%|███▏                                | 7015/78125 [59:42<9:48:47,  2.01it/s]
  9%|███▏                                | 7016/78125 [59:43<9:48:40,  2.01it/s]
  9%|███▏                                | 7017/78125 [59:43<9:49:07,  2.01it/s]
  9%|███▏                                | 7018/78125 [59:44<9:49:34,  2.01it/s]
  9%|███▏                                | 7019/78125 [59:44<9:49:58,  2.01it/s]
  9%|███▏                                | 7020/78125 [59:45<9:49:24,  2.01it/s]
  9%|███▏                                | 7021/78125 [59:45<9:49:36,  2.01it/s]
  9%|███▏                                | 7022/78125 [59:46<9:49:12,  2.01it/s]
  9%|███▏                                | 7023/78125 [59:46<9:48:39,  2.01it/s]
  9%|███▏                                | 7024/78125 [59:47<9:49:03,  2.01it/s]
  9%|███▏                                | 7025/78125 [59:47<9:49:20,  2.01it/s]
  9%|███▏                                | 7026/78125 [59:48<9:49:44,  2.01it/s]
  9%|███▏                                | 7027/78125 [59:48<9:49:19,  2.01it/s]
  9%|███▏                                | 7028/78125 [59:49<9:48:18,  2.01it/s]
  9%|███▏                                | 7029/78125 [59:49<9:49:05,  2.01it/s]
  9%|███▏                                | 7030/78125 [59:50<9:49:07,  2.01it/s]
  9%|███▏                                | 7031/78125 [59:50<9:48:27,  2.01it/s]
  9%|███▏                                | 7032/78125 [59:51<9:48:54,  2.01it/s]
  9%|███▏                                | 7033/78125 [59:51<9:48:29,  2.01it/s]
  9%|███▏                                | 7034/78125 [59:52<9:47:49,  2.02it/s]
  9%|███▏                                | 7035/78125 [59:52<9:48:00,  2.01it/s]
  9%|███▏                                | 7036/78125 [59:53<9:47:54,  2.02it/s]
  9%|███▏                                | 7037/78125 [59:53<9:48:02,  2.01it/s]
  9%|███▏                                | 7038/78125 [59:54<9:47:47,  2.02it/s]
  9%|███▏                                | 7039/78125 [59:54<9:49:00,  2.01it/s]
  9%|███▏                                | 7040/78125 [59:55<9:48:44,  2.01it/s]
  9%|███▏                                | 7041/78125 [59:55<9:48:40,  2.01it/s]
  9%|███▏                                | 7042/78125 [59:56<9:49:26,  2.01it/s]
  9%|███▏                                | 7043/78125 [59:56<9:49:21,  2.01it/s]
  9%|███▏                                | 7044/78125 [59:57<9:48:22,  2.01it/s]
  9%|███▏                                | 7045/78125 [59:57<9:48:52,  2.01it/s]
  9%|███▏                                | 7046/78125 [59:58<9:49:01,  2.01it/s]
  9%|███▏                                | 7047/78125 [59:58<9:49:02,  2.01it/s]
  9%|███▏                                | 7048/78125 [59:59<9:49:26,  2.01it/s]
  9%|███▏                                | 7049/78125 [59:59<9:49:30,  2.01it/s]
  9%|███                               | 7050/78125 [1:00:00<9:50:08,  2.01it/s]
  9%|███                               | 7051/78125 [1:00:00<9:49:58,  2.01it/s]
  9%|███                               | 7052/78125 [1:00:01<9:48:58,  2.01it/s]
  9%|███                               | 7053/78125 [1:00:01<9:49:10,  2.01it/s]
  9%|███                               | 7054/78125 [1:00:02<9:49:56,  2.01it/s]
  9%|███                               | 7055/78125 [1:00:02<9:48:56,  2.01it/s]
  9%|███                               | 7056/78125 [1:00:03<9:48:41,  2.01it/s]
  9%|███                               | 7057/78125 [1:00:03<9:48:21,  2.01it/s]
  9%|███                               | 7058/78125 [1:00:04<9:48:40,  2.01it/s]
  9%|███                               | 7059/78125 [1:00:04<9:49:11,  2.01it/s]
  9%|███                               | 7060/78125 [1:00:05<9:49:07,  2.01it/s]
  9%|███                               | 7061/78125 [1:00:05<9:48:51,  2.01it/s]
  9%|███                               | 7062/78125 [1:00:05<9:48:21,  2.01it/s]
  9%|███                               | 7063/78125 [1:00:06<9:48:55,  2.01it/s]
  9%|███                               | 7064/78125 [1:00:06<9:48:57,  2.01it/s]
  9%|███                               | 7065/78125 [1:00:07<9:49:43,  2.01it/s]
  9%|███                               | 7066/78125 [1:00:07<9:49:10,  2.01it/s]
  9%|███                               | 7067/78125 [1:00:08<9:48:58,  2.01it/s]
  9%|███                               | 7068/78125 [1:00:08<9:48:58,  2.01it/s]
  9%|███                               | 7069/78125 [1:00:09<9:49:06,  2.01it/s]
  9%|███                               | 7070/78125 [1:00:09<9:49:36,  2.01it/s]
  9%|███                               | 7071/78125 [1:00:10<9:49:42,  2.01it/s]
  9%|███                               | 7072/78125 [1:00:10<9:48:57,  2.01it/s]
  9%|███                               | 7073/78125 [1:00:11<9:48:51,  2.01it/s]
  9%|███                               | 7074/78125 [1:00:11<9:49:19,  2.01it/s]
  9%|███                               | 7075/78125 [1:00:12<9:48:38,  2.01it/s]
  9%|███                               | 7076/78125 [1:00:12<9:48:50,  2.01it/s]
  9%|███                               | 7077/78125 [1:00:13<9:49:26,  2.01it/s]
  9%|███                               | 7078/78125 [1:00:13<9:48:54,  2.01it/s]
  9%|███                               | 7079/78125 [1:00:14<9:48:15,  2.01it/s]
  9%|███                               | 7080/78125 [1:00:14<9:48:40,  2.01it/s]
  9%|███                               | 7081/78125 [1:00:15<9:48:38,  2.01it/s]
  9%|███                               | 7082/78125 [1:00:15<9:48:40,  2.01it/s]
  9%|███                               | 7083/78125 [1:00:16<9:48:01,  2.01it/s]
  9%|███                               | 7084/78125 [1:00:16<9:48:09,  2.01it/s]
  9%|███                               | 7085/78125 [1:00:17<9:48:13,  2.01it/s]
  9%|███                               | 7086/78125 [1:00:17<9:49:02,  2.01it/s]
  9%|███                               | 7087/78125 [1:00:18<9:49:14,  2.01it/s]
  9%|███                               | 7088/78125 [1:00:18<9:48:31,  2.01it/s]
  9%|███                               | 7089/78125 [1:00:19<9:49:10,  2.01it/s]
  9%|███                               | 7090/78125 [1:00:19<9:49:17,  2.01it/s]
  9%|███                               | 7091/78125 [1:00:20<9:48:30,  2.01it/s]
  9%|███                               | 7092/78125 [1:00:20<9:49:17,  2.01it/s]
  9%|███                               | 7093/78125 [1:00:21<9:48:52,  2.01it/s]
  9%|███                               | 7094/78125 [1:00:21<9:48:41,  2.01it/s]
  9%|███                               | 7095/78125 [1:00:22<9:49:04,  2.01it/s]
  9%|███                               | 7096/78125 [1:00:22<9:48:59,  2.01it/s]
  9%|███                               | 7097/78125 [1:00:23<9:49:29,  2.01it/s]
  9%|███                               | 7098/78125 [1:00:23<9:49:00,  2.01it/s]
  9%|███                               | 7099/78125 [1:00:24<9:48:29,  2.01it/s]
  9%|███                               | 7100/78125 [1:00:24<9:49:00,  2.01it/s]
  9%|███                               | 7101/78125 [1:00:25<9:49:04,  2.01it/s]
  9%|███                               | 7102/78125 [1:00:25<9:48:33,  2.01it/s]
  9%|███                               | 7103/78125 [1:00:26<9:48:32,  2.01it/s]
  9%|███                               | 7104/78125 [1:00:26<9:49:33,  2.01it/s]
  9%|███                               | 7105/78125 [1:00:27<9:49:33,  2.01it/s]
  9%|███                               | 7106/78125 [1:00:27<9:49:19,  2.01it/s]
  9%|███                               | 7107/78125 [1:00:28<9:49:12,  2.01it/s]
  9%|███                               | 7108/78125 [1:00:28<9:48:49,  2.01it/s]
  9%|███                               | 7109/78125 [1:00:29<9:49:13,  2.01it/s]
  9%|███                               | 7110/78125 [1:00:29<9:48:19,  2.01it/s]
  9%|███                               | 7111/78125 [1:00:30<9:48:39,  2.01it/s]
  9%|███                               | 7112/78125 [1:00:30<9:48:12,  2.01it/s]
  9%|███                               | 7113/78125 [1:00:31<9:49:09,  2.01it/s]
  9%|███                               | 7114/78125 [1:00:31<9:49:12,  2.01it/s]
  9%|███                               | 7115/78125 [1:00:32<9:49:15,  2.01it/s]
  9%|███                               | 7116/78125 [1:00:32<9:49:24,  2.01it/s]
  9%|███                               | 7117/78125 [1:00:33<9:49:00,  2.01it/s]
  9%|███                               | 7118/78125 [1:00:33<9:48:18,  2.01it/s]
  9%|███                               | 7119/78125 [1:00:34<9:48:48,  2.01it/s]
  9%|███                               | 7120/78125 [1:00:34<9:48:27,  2.01it/s]
  9%|███                               | 7121/78125 [1:00:35<9:48:42,  2.01it/s]
  9%|███                               | 7122/78125 [1:00:35<9:48:20,  2.01it/s]
  9%|███                               | 7123/78125 [1:00:36<9:48:08,  2.01it/s]
  9%|███                               | 7124/78125 [1:00:36<9:48:28,  2.01it/s]
  9%|███                               | 7125/78125 [1:00:37<9:48:39,  2.01it/s]
  9%|███                               | 7126/78125 [1:00:37<9:48:35,  2.01it/s]
  9%|███                               | 7127/78125 [1:00:38<9:48:27,  2.01it/s]
  9%|███                               | 7128/78125 [1:00:38<9:48:38,  2.01it/s]
  9%|███                               | 7129/78125 [1:00:39<9:47:59,  2.01it/s]
  9%|███                               | 7130/78125 [1:00:39<9:47:19,  2.01it/s]
  9%|███                               | 7131/78125 [1:00:40<9:47:03,  2.02it/s]
  9%|███                               | 7132/78125 [1:00:40<9:47:55,  2.01it/s]
  9%|███                               | 7133/78125 [1:00:41<9:47:55,  2.01it/s]
  9%|███                               | 7134/78125 [1:00:41<9:47:43,  2.01it/s]
  9%|███                               | 7135/78125 [1:00:42<9:47:52,  2.01it/s]
  9%|███                               | 7136/78125 [1:00:42<9:48:06,  2.01it/s]
  9%|███                               | 7137/78125 [1:00:43<9:48:20,  2.01it/s]
  9%|███                               | 7138/78125 [1:00:43<9:48:44,  2.01it/s]
  9%|███                               | 7139/78125 [1:00:44<9:48:44,  2.01it/s]
  9%|███                               | 7140/78125 [1:00:44<9:48:37,  2.01it/s]
  9%|███                               | 7141/78125 [1:00:45<9:49:01,  2.01it/s]
  9%|███                               | 7142/78125 [1:00:45<9:48:58,  2.01it/s]
  9%|███                               | 7143/78125 [1:00:46<9:48:46,  2.01it/s]
  9%|███                               | 7144/78125 [1:00:46<9:48:02,  2.01it/s]
  9%|███                               | 7145/78125 [1:00:47<9:48:20,  2.01it/s]
  9%|███                               | 7146/78125 [1:00:47<9:47:45,  2.01it/s]
  9%|███                               | 7147/78125 [1:00:48<9:47:39,  2.01it/s]
  9%|███                               | 7148/78125 [1:00:48<9:47:57,  2.01it/s]
  9%|███                               | 7149/78125 [1:00:49<9:48:35,  2.01it/s]
  9%|███                               | 7150/78125 [1:00:49<9:48:39,  2.01it/s]
  9%|███                               | 7151/78125 [1:00:50<9:48:23,  2.01it/s]
  9%|███                               | 7152/78125 [1:00:50<9:48:20,  2.01it/s]
  9%|███                               | 7153/78125 [1:00:51<9:48:06,  2.01it/s]
  9%|███                               | 7154/78125 [1:00:51<9:47:34,  2.01it/s]
  9%|███                               | 7155/78125 [1:00:52<9:47:49,  2.01it/s]
  9%|███                               | 7156/78125 [1:00:52<9:48:07,  2.01it/s]
  9%|███                               | 7157/78125 [1:00:53<9:48:25,  2.01it/s]
  9%|███                               | 7158/78125 [1:00:53<9:47:59,  2.01it/s]
  9%|███                               | 7159/78125 [1:00:54<9:48:05,  2.01it/s]
  9%|███                               | 7160/78125 [1:00:54<9:49:05,  2.01it/s]
  9%|███                               | 7161/78125 [1:00:55<9:49:03,  2.01it/s]
  9%|███                               | 7162/78125 [1:00:55<9:49:17,  2.01it/s]
  9%|███                               | 7163/78125 [1:00:56<9:49:02,  2.01it/s]
  9%|███                               | 7164/78125 [1:00:56<9:48:44,  2.01it/s]
  9%|███                               | 7165/78125 [1:00:57<9:49:07,  2.01it/s]
  9%|███                               | 7166/78125 [1:00:57<9:48:42,  2.01it/s]
  9%|███                               | 7167/78125 [1:00:58<9:48:45,  2.01it/s]
  9%|███                               | 7168/78125 [1:00:58<9:48:38,  2.01it/s]
  9%|███                               | 7169/78125 [1:00:59<9:48:09,  2.01it/s]
  9%|███                               | 7170/78125 [1:00:59<9:48:36,  2.01it/s]
  9%|███                               | 7171/78125 [1:01:00<9:48:50,  2.01it/s]
  9%|███                               | 7172/78125 [1:01:00<9:48:55,  2.01it/s]
  9%|███                               | 7173/78125 [1:01:01<9:48:39,  2.01it/s]
  9%|███                               | 7174/78125 [1:01:01<9:48:23,  2.01it/s]
  9%|███                               | 7175/78125 [1:01:02<9:49:02,  2.01it/s]
  9%|███                               | 7176/78125 [1:01:02<9:48:07,  2.01it/s]
  9%|███                               | 7177/78125 [1:01:03<9:48:21,  2.01it/s]
  9%|███                               | 7178/78125 [1:01:03<9:48:30,  2.01it/s]
  9%|███                               | 7179/78125 [1:01:04<9:49:05,  2.01it/s]
  9%|███                               | 7180/78125 [1:01:04<9:49:28,  2.01it/s]
  9%|███▏                              | 7181/78125 [1:01:05<9:50:06,  2.00it/s]
  9%|███▏                              | 7182/78125 [1:01:05<9:50:09,  2.00it/s]
  9%|███▏                              | 7183/78125 [1:01:06<9:49:25,  2.01it/s]
  9%|███▏                              | 7184/78125 [1:01:06<9:48:43,  2.01it/s]
  9%|███▏                              | 7185/78125 [1:01:07<9:48:01,  2.01it/s]
  9%|███▏                              | 7186/78125 [1:01:07<9:48:55,  2.01it/s]
  9%|███▏                              | 7187/78125 [1:01:08<9:48:38,  2.01it/s]
  9%|███▏                              | 7188/78125 [1:01:08<9:48:18,  2.01it/s]
  9%|███▏                              | 7189/78125 [1:01:09<9:47:40,  2.01it/s]
  9%|███▏                              | 7190/78125 [1:01:09<9:47:53,  2.01it/s]
  9%|███▏                              | 7191/78125 [1:01:10<9:47:34,  2.01it/s]
  9%|███▏                              | 7192/78125 [1:01:10<9:48:16,  2.01it/s]
  9%|███▏                              | 7193/78125 [1:01:11<9:47:48,  2.01it/s]
  9%|███▏                              | 7194/78125 [1:01:11<9:47:21,  2.01it/s]
  9%|███▏                              | 7195/78125 [1:01:12<9:46:42,  2.01it/s]
  9%|███▏                              | 7196/78125 [1:01:12<9:47:59,  2.01it/s]
  9%|███▏                              | 7197/78125 [1:01:13<9:47:44,  2.01it/s]
  9%|███▏                              | 7198/78125 [1:01:13<9:47:32,  2.01it/s]
  9%|███▏                              | 7199/78125 [1:01:14<9:47:55,  2.01it/s]
  9%|███▏                              | 7200/78125 [1:01:14<9:47:02,  2.01it/s]
  9%|███▏                              | 7201/78125 [1:01:15<9:47:20,  2.01it/s]
  9%|███▏                              | 7202/78125 [1:01:15<9:47:27,  2.01it/s]
  9%|███▏                              | 7203/78125 [1:01:16<9:47:27,  2.01it/s]
  9%|███▏                              | 7204/78125 [1:01:16<9:47:16,  2.01it/s]
  9%|███▏                              | 7205/78125 [1:01:17<9:47:15,  2.01it/s]
  9%|███▏                              | 7206/78125 [1:01:17<9:47:05,  2.01it/s]
  9%|███▏                              | 7207/78125 [1:01:18<9:46:57,  2.01it/s]
  9%|███▏                              | 7208/78125 [1:01:18<9:46:58,  2.01it/s]
  9%|███▏                              | 7209/78125 [1:01:19<9:47:13,  2.01it/s]
  9%|███▏                              | 7210/78125 [1:01:19<9:47:36,  2.01it/s]
  9%|███▏                              | 7211/78125 [1:01:20<9:47:32,  2.01it/s]
  9%|███▏                              | 7212/78125 [1:01:20<9:48:23,  2.01it/s]
  9%|███▏                              | 7213/78125 [1:01:21<9:48:22,  2.01it/s]
  9%|███▏                              | 7214/78125 [1:01:21<9:49:18,  2.01it/s]
  9%|███▏                              | 7215/78125 [1:01:22<9:47:51,  2.01it/s]
  9%|███▏                              | 7216/78125 [1:01:22<9:47:10,  2.01it/s]
  9%|███▏                              | 7217/78125 [1:01:23<9:47:15,  2.01it/s]
  9%|███▏                              | 7218/78125 [1:01:23<9:47:21,  2.01it/s]
  9%|███▏                              | 7219/78125 [1:01:24<9:47:03,  2.01it/s]
  9%|███▏                              | 7220/78125 [1:01:24<9:47:20,  2.01it/s]
  9%|███▏                              | 7221/78125 [1:01:25<9:47:05,  2.01it/s]
  9%|███▏                              | 7222/78125 [1:01:25<9:46:37,  2.01it/s]
  9%|███▏                              | 7223/78125 [1:01:26<9:46:56,  2.01it/s]
  9%|███▏                              | 7224/78125 [1:01:26<9:47:09,  2.01it/s]
  9%|███▏                              | 7225/78125 [1:01:27<9:47:21,  2.01it/s]
  9%|███▏                              | 7226/78125 [1:01:27<9:47:20,  2.01it/s]
  9%|███▏                              | 7227/78125 [1:01:28<9:47:28,  2.01it/s]
  9%|███▏                              | 7228/78125 [1:01:28<9
760.3s 152 :47:06,  2.01it/s]
  9%|███▏                              | 7229/78125 [1:01:29<9:47:12,  2.01it/s]
  9%|███▏                              | 7230/78125 [1:01:29<9:47:37,  2.01it/s]
  9%|███▏                              | 7231/78125 [1:01:30<9:47:29,  2.01it/s]
  9%|███▏                              | 7232/78125 [1:01:30<9:46:59,  2.01it/s]
  9%|███▏                              | 7233/78125 [1:01:31<9:47:08,  2.01it/s]
  9%|███▏                              | 7234/78125 [1:01:31<9:47:04,  2.01it/s]
  9%|███▏                              | 7235/78125 [1:01:32<9:47:44,  2.01it/s]
  9%|███▏                              | 7236/78125 [1:01:32<9:47:15,  2.01it/s]
  9%|███▏                              | 7237/78125 [1:01:33<9:47:41,  2.01it/s]
  9%|███▏                              | 7238/78125 [1:01:33<9:48:03,  2.01it/s]
  9%|███▏                              | 7239/78125 [1:01:34<9:48:17,  2.01it/s]
  9%|███▏                              | 7240/78125 [1:01:34<9:47:44,  2.01it/s]
  9%|███▏                              | 7241/78125 [1:01:35<9:48:08,  2.01it/s]
  9%|███▏                              | 7242/78125 [1:01:35<9:47:37,  2.01it/s]
  9%|███▏                              | 7243/78125 [1:01:36<9:48:02,  2.01it/s]
  9%|███▏                              | 7244/78125 [1:01:36<9:47:56,  2.01it/s]
  9%|███▏                              | 7245/78125 [1:01:37<9:47:57,  2.01it/s]
  9%|███▏                              | 7246/78125 [1:01:37<9:47:46,  2.01it/s]
  9%|███▏                              | 7247/78125 [1:01:38<9:46:43,  2.01it/s]
  9%|███▏                              | 7248/78125 [1:01:38<9:46:24,  2.01it/s]
  9%|███▏                              | 7249/78125 [1:01:39<9:46:30,  2.01it/s]
  9%|███▏                              | 7250/78125 [1:01:39<9:46:23,  2.01it/s]
  9%|███▏                              | 7251/78125 [1:01:40<9:46:38,  2.01it/s]
  9%|███▏                              | 7252/78125 [1:01:40<9:47:33,  2.01it/s]
  9%|███▏                              | 7253/78125 [1:01:40<9:47:39,  2.01it/s]
  9%|███▏                              | 7254/78125 [1:01:41<9:47:47,  2.01it/s]
  9%|███▏                              | 7255/78125 [1:01:41<9:47:40,  2.01it/s]
  9%|███▏                              | 7256/78125 [1:01:42<9:47:09,  2.01it/s]
  9%|███▏                              | 7257/78125 [1:01:42<9:46:25,  2.01it/s]
  9%|███▏                              | 7258/78125 [1:01:43<9:46:39,  2.01it/s]
  9%|███▏                              | 7259/78125 [1:01:43<9:47:13,  2.01it/s]
  9%|███▏                              | 7260/78125 [1:01:44<9:47:35,  2.01it/s]
  9%|███▏                              | 7261/78125 [1:01:44<9:46:28,  2.01it/s]
  9%|███▏                              | 7262/78125 [1:01:45<9:47:06,  2.01it/s]
  9%|███▏                              | 7263/78125 [1:01:45<9:47:23,  2.01it/s]
  9%|███▏                              | 7264/78125 [1:01:46<9:47:01,  2.01it/s]
  9%|███▏                              | 7265/78125 [1:01:46<9:45:44,  2.02it/s]
  9%|███▏                              | 7266/78125 [1:01:47<9:45:53,  2.02it/s]
  9%|███▏                              | 7267/78125 [1:01:47<9:46:59,  2.01it/s]
  9%|███▏                              | 7268/78125 [1:01:48<9:47:06,  2.01it/s]
  9%|███▏                              | 7269/78125 [1:01:48<9:47:14,  2.01it/s]
  9%|███▏                              | 7270/78125 [1:01:49<9:47:39,  2.01it/s]
  9%|███▏                              | 7271/78125 [1:01:49<9:47:48,  2.01it/s]
  9%|███▏                              | 7272/78125 [1:01:50<9:46:50,  2.01it/s]
  9%|███▏                              | 7273/78125 [1:01:50<9:47:10,  2.01it/s]
  9%|███▏                              | 7274/78125 [1:01:51<9:47:01,  2.01it/s]
  9%|███▏                              | 7275/78125 [1:01:51<9:47:11,  2.01it/s]
  9%|███▏                              | 7276/78125 [1:01:52<9:47:20,  2.01it/s]
  9%|███▏                              | 7277/78125 [1:01:52<9:46:42,  2.01it/s]
  9%|███▏                              | 7278/78125 [1:01:53<9:47:21,  2.01it/s]
  9%|███▏                              | 7279/78125 [1:01:53<9:46:54,  2.01it/s]
  9%|███▏                              | 7280/78125 [1:01:54<9:47:02,  2.01it/s]
  9%|███▏                              | 7281/78125 [1:01:54<9:47:28,  2.01it/s]
  9%|███▏                              | 7282/78125 [1:01:55<9:46:53,  2.01it/s]
  9%|███▏                              | 7283/78125 [1:01:55<9:47:00,  2.01it/s]
  9%|███▏                              | 7284/78125 [1:01:56<9:47:35,  2.01it/s]
  9%|███▏                              | 7285/78125 [1:01:56<9:47:24,  2.01it/s]
  9%|███▏                              | 7286/78125 [1:01:57<9:47:00,  2.01it/s]
  9%|███▏                              | 7287/78125 [1:01:57<9:46:50,  2.01it/s]
  9%|███▏                              | 7288/78125 [1:01:58<9:47:59,  2.01it/s]
  9%|███▏                              | 7289/78125 [1:01:58<9:47:43,  2.01it/s]
  9%|███▏                              | 7290/78125 [1:01:59<9:46:38,  2.01it/s]
  9%|███▏                              | 7291/78125 [1:01:59<9:47:09,  2.01it/s]
  9%|███▏                              | 7292/78125 [1:02:00<9:46:53,  2.01it/s]
  9%|███▏                              | 7293/78125 [1:02:00<9:46:31,  2.01it/s]
  9%|███▏                              | 7294/78125 [1:02:01<9:46:48,  2.01it/s]
  9%|███▏                              | 7295/78125 [1:02:01<9:46:58,  2.01it/s]
  9%|███▏                              | 7296/78125 [1:02:02<9:46:52,  2.01it/s]
  9%|███▏                              | 7297/78125 [1:02:02<9:46:57,  2.01it/s]
  9%|███▏                              | 7298/78125 [1:02:03<9:47:25,  2.01it/s]
  9%|███▏                              | 7299/78125 [1:02:03<9:47:41,  2.01it/s]
  9%|███▏                              | 7300/78125 [1:02:04<9:47:02,  2.01it/s]
  9%|███▏                              | 7301/78125 [1:02:04<9:46:01,  2.01it/s]
  9%|███▏                              | 7302/78125 [1:02:05<9:46:26,  2.01it/s]
  9%|███▏                              | 7303/78125 [1:02:05<9:47:46,  2.01it/s]
  9%|███▏                              | 7304/78125 [1:02:06<9:46:59,  2.01it/s]
  9%|███▏                              | 7305/78125 [1:02:06<9:46:51,  2.01it/s]
  9%|███▏                              | 7306/78125 [1:02:07<9:46:56,  2.01it/s]
  9%|███▏                              | 7307/78125 [1:02:07<9:46:35,  2.01it/s]
  9%|███▏                              | 7308/78125 [1:02:08<9:46:13,  2.01it/s]
  9%|███▏                              | 7309/78125 [1:02:08<9:46:00,  2.01it/s]
  9%|███▏                              | 7310/78125 [1:02:09<9:46:08,  2.01it/s]
  9%|███▏                              | 7311/78125 [1:02:09<9:46:03,  2.01it/s]
  9%|███▏                              | 7312/78125 [1:02:10<9:46:11,  2.01it/s]
  9%|███▏                              | 7313/78125 [1:02:10<9:46:11,  2.01it/s]
  9%|███▏                              | 7314/78125 [1:02:11<9:45:55,  2.01it/s]
  9%|███▏                              | 7315/78125 [1:02:11<9:45:43,  2.01it/s]
  9%|███▏                              | 7316/78125 [1:02:12<9:46:29,  2.01it/s]
  9%|███▏                              | 7317/78125 [1:02:12<9:46:38,  2.01it/s]
  9%|███▏                              | 7318/78125 [1:02:13<9:46:50,  2.01it/s]
  9%|███▏                              | 7319/78125 [1:02:13<9:46:45,  2.01it/s]
  9%|███▏                              | 7320/78125 [1:02:14<9:46:20,  2.01it/s]
  9%|███▏                              | 7321/78125 [1:02:14<9:46:13,  2.01it/s]
  9%|███▏                              | 7322/78125 [1:02:15<9:46:01,  2.01it/s]
  9%|███▏                              | 7323/78125 [1:02:15<9:46:16,  2.01it/s]
  9%|███▏                              | 7324/78125 [1:02:16<9:46:32,  2.01it/s]
  9%|███▏                              | 7325/78125 [1:02:16<9:46:41,  2.01it/s]
  9%|███▏                              | 7326/78125 [1:02:17<9:46:40,  2.01it/s]
  9%|███▏                              | 7327/78125 [1:02:17<9:46:10,  2.01it/s]
  9%|███▏                              | 7328/78125 [1:02:18<9:46:37,  2.01it/s]
  9%|███▏                              | 7329/78125 [1:02:18<9:46:54,  2.01it/s]
  9%|███▏                              | 7330/78125 [1:02:19<9:47:30,  2.01it/s]
  9%|███▏                              | 7331/78125 [1:02:19<9:47:15,  2.01it/s]
  9%|███▏                              | 7332/78125 [1:02:20<9:46:46,  2.01it/s]
  9%|███▏                              | 7333/78125 [1:02:20<9:47:16,  2.01it/s]
  9%|███▏                              | 7334/78125 [1:02:21<9:47:22,  2.01it/s]
  9%|███▏                              | 7335/78125 [1:02:21<9:47:14,  2.01it/s]
  9%|███▏                              | 7336/78125 [1:02:22<9:47:24,  2.01it/s]
  9%|███▏                              | 7337/78125 [1:02:22<9:46:46,  2.01it/s]
  9%|███▏                              | 7338/78125 [1:02:23<9:46:59,  2.01it/s]
  9%|███▏                              | 7339/78125 [1:02:23<9:46:58,  2.01it/s]
  9%|███▏                              | 7340/78125 [1:02:24<9:46:21,  2.01it/s]
  9%|███▏                              | 7341/78125 [1:02:24<9:47:49,  2.01it/s]
  9%|███▏                              | 7342/78125 [1:02:25<9:47:11,  2.01it/s]
  9%|███▏                              | 7343/78125 [1:02:25<9:46:02,  2.01it/s]
  9%|███▏                              | 7344/78125 [1:02:26<9:46:25,  2.01it/s]
  9%|███▏                              | 7345/78125 [1:02:26<9:45:57,  2.01it/s]
  9%|███▏                              | 7346/78125 [1:02:27<9:45:31,  2.01it/s]
  9%|███▏                              | 7347/78125 [1:02:27<9:44:47,  2.02it/s]
  9%|███▏                              | 7348/78125 [1:02:28<9:45:46,  2.01it/s]
  9%|███▏                              | 7349/78125 [1:02:28<9:46:34,  2.01it/s]
  9%|███▏                              | 7350/78125 [1:02:29<9:46:33,  2.01it/s]
  9%|███▏                              | 7351/78125 [1:02:29<9:46:47,  2.01it/s]
  9%|███▏                              | 7352/78125 [1:02:30<9:47:42,  2.01it/s]
  9%|███▏                              | 7353/78125 [1:02:30<9:47:35,  2.01it/s]
  9%|███▏                              | 7354/78125 [1:02:31<9:47:06,  2.01it/s]
  9%|███▏                              | 7355/78125 [1:02:31<9:47:04,  2.01it/s]
  9%|███▏                              | 7356/78125 [1:02:32<9:46:08,  2.01it/s]
  9%|███▏                              | 7357/78125 [1:02:32<9:45:55,  2.01it/s]
  9%|███▏                              | 7358/78125 [1:02:33<9:46:18,  2.01it/s]
  9%|███▏                              | 7359/78125 [1:02:33<9:45:52,  2.01it/s]
  9%|███▏                              | 7360/78125 [1:02:34<9:46:26,  2.01it/s]
  9%|███▏                              | 7361/78125 [1:02:34<9:46:15,  2.01it/s]
  9%|███▏                              | 7362/78125 [1:02:35<9:45:41,  2.01it/s]
  9%|███▏                              | 7363/78125 [1:02:35<9:45:11,  2.02it/s]
  9%|███▏                              | 7364/78125 [1:02:36<9:44:53,  2.02it/s]
  9%|███▏                              | 7365/78125 [1:02:36<9:45:23,  2.01it/s]
  9%|███▏                              | 7366/78125 [1:02:37<9:45:59,  2.01it/s]
  9%|███▏                              | 7367/78125 [1:02:37<9:45:35,  2.01it/s]
  9%|███▏                              | 7368/78125 [1:02:38<9:45:13,  2.02it/s]
  9%|███▏                              | 7369/78125 [1:02:38<9:45:41,  2.01it/s]
  9%|███▏                              | 7370/78125 [1:02:39<9:45:57,  2.01it/s]
  9%|███▏                              | 7371/78125 [1:02:39<9:46:13,  2.01it/s]
  9%|███▏                              | 7372/78125 [1:02:40<9:46:18,  2.01it/s]
  9%|███▏                              | 7373/78125 [1:02:40<9:45:57,  2.01it/s]
  9%|███▏                              | 7374/78125 [1:02:41<9:46:14,  2.01it/s]
  9%|███▏                              | 7375/78125 [1:02:41<9:46:24,  2.01it/s]
  9%|███▏                              | 7376/78125 [1:02:42<9:46:12,  2.01it/s]
  9%|███▏                              | 7377/78125 [1:02:42<9:46:28,  2.01it/s]
  9%|███▏                              | 7378/78125 [1:02:43<9:46:57,  2.01it/s]
  9%|███▏                              | 7379/78125 [1:02:43<9:46:00,  2.01it/s]
  9%|███▏                              | 7380/78125 [1:02:44<9:46:00,  2.01it/s]
  9%|███▏                              | 7381/78125 [1:02:44<9:45:48,  2.01it/s]
  9%|███▏                              | 7382/78125 [1:02:45<9:46:11,  2.01it/s]
  9%|███▏                              | 7383/78125 [1:02:45<9:46:00,  2.01it/s]
  9%|███▏                              | 7384/78125 [1:02:46<9:45:54,  2.01it/s]
  9%|███▏                              | 7385/78125 [1:02:46<9:45:23,  2.01it/s]
  9%|███▏                              | 7386/78125 [1:02:47<9:45:19,  2.01it/s]
  9%|███▏                              | 7387/78125 [1:02:47<9:45:59,  2.01it/s]
  9%|███▏                              | 7388/78125 [1:02:48<9:46:02,  2.01it/s]
  9%|███▏                              | 7389/78125 [1:02:48<9:46:08,  2.01it/s]
  9%|███▏                              | 7390/78125 [1:02:49<9:46:26,  2.01it/s]
  9%|███▏                              | 7391/78125 [1:02:49<9:46:47,  2.01it/s]
  9%|███▏                              | 7392/78125 [1:02:50<9:46:22,  2.01it/s]
  9%|███▏                              | 7393/78125 [1:02:50<9:45:09,  2.01it/s]
  9%|███▏                              | 7394/78125 [1:02:51<9:45:48,  2.01it/s]
  9%|███▏                              | 7395/78125 [1:02:51<9:46:19,  2.01it/s]
  9%|███▏                              | 7396/78125 [1:02:52<9:46:07,  2.01it/s]
  9%|███▏                              | 7397/78125 [1:02:52<9:45:48,  2.01it/s]
  9%|███▏                              | 7398/78125 [1:02:53<9:45:59,  2.01it/s]
  9%|███▏                              | 7399/78125 [1:02:53<9:47:06,  2.01it/s]
  9%|███▏                              | 7400/78125 [1:02:54<9:46:32,  2.01it/s]
  9%|███▏                              | 7401/78125 [1:02:54<9:46:42,  2.01it/s]
  9%|███▏                              | 7402/78125 [1:02:55<9:46:43,  2.01it/s]
  9%|███▏                              | 7403/78125 [1:02:55<9:46:14,  2.01it/s]
  9%|███▏                              | 7404/78125 [1:02:56<9:46:12,  2.01it/s]
  9%|███▏                              | 7405/78125 [1:02:56<9:46:13,  2.01it/s]
  9%|███▏                              | 7406/78125 [1:02:57<9:47:05,  2.01it/s]
  9%|███▏                              | 7407/78125 [1:02:57<9:46:10,  2.01it/s]
  9%|███▏                              | 7408/78125 [1:02:58<9:46:20,  2.01it/s]
  9%|███▏                              | 7409/78125 [1:02:58<9:46:31,  2.01it/s]
  9%|███▏                              | 7410/78125 [1:02:59<9:46:38,  2.01it/s]
  9%|███▏                              | 7411/78125 [1:02:59<9:45:35,  2.01it/s]
  9%|███▏                              | 7412/78125 [1:03:00<9:46:17,  2.01it/s]
  9%|███▏                              | 7413/78125 [1:03:00<9:46:14,  2.01it/s]
  9%|███▏                              | 7414/78125 [1:03:01<9:46:31,  2.01it/s]
  9%|███▏                              | 7415/78125 [1:03:01<9:46:22,  2.01it/s]
  9%|███▏                              | 7416/78125 [1:03:02<9:46:33,  2.01it/s]
  9%|███▏                              | 7417/78125 [1:03:02<9:46:50,  2.01it/s]
  9%|███▏                              | 7418/78125 [1:03:03<9:46:24,  2.01it/s]
  9%|███▏                              | 7419/78125 [1:03:03<9:46:14,  2.01it/s]
  9%|███▏                              | 7420/78125 [1:03:04<9:45:42,  2.01it/s]
  9%|███▏                              | 7421/78125 [1:03:04<9:45:47,  2.01it/s]
 10%|███▏                              | 7422/78125 [1:03:05<9:46:27,  2.01it/s]
 10%|███▏                              | 7423/78125 [1:03:05<9:46:22,  2.01it/s]
 10%|███▏                              | 7424/78125 [1:03:06<9:46:45,  2.01it/s]
 10%|███▏                              | 7425/78125 [1:03:06<9:46:02,  2.01it/s]
 10%|███▏                              | 7426/78125 [1:03:07<9:46:38,  2.01it/s]
 10%|███▏                              | 7427/78125 [1:03:07<9:46:07,  2.01it/s]
 10%|███▏                              | 7428/78125 [1:03:08<9:46:14,  2.01it/s]
 10%|███▏                              | 7429/78125 [1:03:08<9:46:02,  2.01it/s]
 10%|███▏                              | 7430/78125 [1:03:09<9:46:01,  2.01it/s]
 10%|███▏                              | 7431/78125 [1:03:09<9:45:55,  2.01it/s]
 10%|███▏                              | 7432/78125 [1:03:09<9:45:17,  2.01it/s]
 10%|███▏                              | 7433/78125 [1:03:10<9:46:21,  2.01it/s]
 10%|███▏                              | 7434/78125 [1:03:10<9:45:47,  2.01it/s]
 10%|███▏                              | 7435/78125 [1:03:11<9:46:04,  2.01it/s]
 10%|███▏                              | 7436/78125 [1:03:11<9:46:13,  2.01it/s]
 10%|███▏                              | 7437/78125 [1:03:12<9:46:28,  2.01it/s]
 10%|███▏                              | 7438/78125 [1:03:12<9:46:20,  2.01it/s]
 10%|███▏                              | 7439/78125 [1:03:13<9:45:52,  2.01it/s]
 10%|███▏                              | 7440/78125 [1:03:13<9:46:50,  2.01it/s]
 10%|███▏                              | 7441/78125 [1:03:14<9:46:50,  2.01it/s]
 10%|███▏                              | 7442/78125 [1:03:14<9:45:33,  2.01it/s]
 10%|███▏                              | 7443/78125 [1:03:15<9:45:01,  2.01it/s]
 10%|███▏                              | 7444/78125 [1:03:15<9:44:43,  2.01it/s]
 10%|███▏                              | 7445/78125 [1:03:16<9:45:07,  2.01it/s]
 10%|███▏                              | 7446/78125 [1:03:16<9:45:52,  2.01it/s]
 10%|███▏                              | 7447/78125 [1:03:17<9:45:47,  2.01it/s]
 10%|███▏                              | 7448/78125 [1:03:17<9:45:54,  2.01it/s]
 10%|███▏                              | 7449/78125 [1:03:18<9:45:49,  2.01it/s]
 10%|███▏                              | 7450/78125 [1:03:18<9:46:10,  2.01it/s]
 10%|███▏                              | 7451/78125 [1:03:19<9:45:34,  2.01it/s]
 10%|███▏                              | 7452/78125 [1:03:19<9:45:37,  2.01it/s]
 10%|███▏                              | 7453/78125 [1:03:20<9:46:26,  2.01it/s]
 10%|███▏                              | 7454/78125 [1:03:20<9:45:30,  2.01it/s]
 10%|███▏                              | 7455/78125 [1:03:21<9:45:36,  2.01it/s]
 10%|███▏                              | 7456/78125 [1:03:21<9:45:45,  2.01it/s]
 10%|███▏                              | 7457/78125 [1:03:22<9:46:11,  2.01it/s]
 10%|███▏                              | 7458/78125 [1:03:22<9:46:11,  2.01it/s]
 10%|███▏                              | 7459/78125 [1:03:23<9:46:22,  2.01it/s]
 10%|███▏                              | 7460/78125 [1:03:23<9:46:21,  2.01it/s]
 10%|███▏                              | 7461/78125 [1:03:24<9:46:33,  2.01it/s]
 10%|███▏                              | 7462/78125 [1:03:24<9:46:03,  2.01it/s]
 10%|███▏                              | 7463/78125 [1:03:25<9:45:47,  2.01it/s]
 10%|███▏                              | 7464/78125 [1:03:25<9:45:42,  2.01it/s]
 10%|███▏                              | 7465/78125 [1:03:26<9:45:34,  2.01it/s]
 10%|███▏                              | 7466/78125 [1:03:26<9:45:59,  2.01it/s]
 10%|███▏                              | 7467/78125 [1:03:27<9:45:23,  2.01it/s]
 10%|███▎                              | 7468/78125 [1:03:27<9:44:52,  2.01it/s]
 10%|███▎                              | 7469/78125 [1:03:28<9:44:46,  2.01it/s]
 10%|███▎                              | 7470/78125 [1:03:28<9:45:02,  2.01it/s]
 10%|███▎                              | 7471/78125 [1:03:29<9:44:51,  2.01it/s]
 10%|███▎                              | 7472/78125 [1:03:29<9:44:49,  2.01it/s]
 10%|███▎                              | 7473/78125 [1:03:30<9:44:26,  2.01it/s]
 10%|███▎                              | 7474/78125 [1:03:30<9:45:15,  2.01it/s]
 10%|███▎                              | 7475/78125 [1:03:31<9:44:50,  2.01it/s]
 10%|███▎                              | 7476/78125 [1:03:31<9:45:34,  2.01it/s]
 10%|███▎                              | 7477/78125 [1:03:32<9:45:31,  2.01it/s]
 10%|███▎                              | 7478/78125 [1:03:32<9:44:54,  2.01it/s]
 10%|███▎                              | 7479/78125 [1:03:33<9:45:29,  2.01it/s]
 10%|███▎                              | 7480/78125 [1:03:33<9:46:18,  2.01it/s]
 10%|███▎                              | 7481/78125 [1:03:34<9:46:07,  2.01it/s]
 10%|███▎                              | 7482/78125 [1:03:34<9:45:40,  2.01it/s]
 10%|███▎                              | 7483/78125 [1:03:35<9:45:46,  2.01it/s]
 10%|███▎                              | 7484/78125 [1:03:35<9:45:40,  2.01it/s]
 10%|███▎                              | 7485/78125 [1:03:36<9:44:58,  2.01it/s]
 10%|███▎                              | 7486/78125 [1:03:36<9:44:51,  2.01it/s]
 10%|███▎                              | 7487/78125 [1:03:37<9:44:00,  2.02it/s]
 10%|███▎                              | 7488/78125 [1:03:37<9:44:54,  2.01it/s]
 10%|███▎                              | 7489/78125 [1:03:38<9:45:04,  2.01it/s]
 10%|███▎                              | 7490/78125 [1:03:38<9:45:03,  2.01it/s]
 10%|███▎                              | 7491/78125 [1:03:39<9:45:10,  2.01it/s]
 10%|███▎                              | 7492/78125 [1:03:39<9:45:21,  2.01it/s]
 10%|███▎                              | 7493/78125 [1:03:40<9:45:17,  2.01it/s]
 10%|███▎                              | 7494/78125 [1:03:40<9:44:40,  2.01it/s]
 10%|███▎                              | 7495/78125 [1:03:41<9:44:39,  2.01it/s]
 10%|███▎                              | 7496/78125 [1:03:41<9:44:33,  2.01it/s]
 10%|███▎                              | 7497/78125 [1:03:42<9:44:44,  2.01it/s]
 10%|███▎                              | 7498/78125 [1:03:42<9:45:39,  2.01it/s]
 10%|███▎                              | 7499/78125 [1:03:43<9:45:19,  2.01it/s]
 10%|███▎                              | 7500/78125 [1:03:43<9:45:15,  2.01it/s]
 10%|███▎                              | 7501/78125 [1:03:44<9:44:52,  2.01it/s]
 10%|███▎                              | 7502/78125 [1:03:44<9:43:44,  2.02it/s]
 10%|███▎                              | 7503/78125 [1:03:45<9:43:22,  2.02it/s]
 10%|███▎                              | 7504/78125 [1:03:45<9:43:32,  2.02it/s]
 10%|███▎                              | 7505/78125 [1:03:46<9:44:01,  2.02it/s]
 10%|███▎                              | 7506/78125 [1:03:46<9:44:36,  2.01it/s]
 10%|███▎                              | 7507/78125 [1:03:47<9:44:45,  2.01it/s]
 10%|███▎                              | 7508/78125 [1:03:47<9:44:25,  2.01it/s]
 10%|███▎                              | 7509/78125 [1:03:48<9:45:00,  2.01it/s]
 10%|███▎                              | 7510/78125 [1:03:48<9:45:19,  2.01it/s]
 10%|███▎                              | 7511/78125 [1:03:49<9:45:15,  2.01it/s]
 10%|███▎                              | 7512/78125 [1:03:49<9:44:53,  2.01it/s]
 10%|███▎                              | 7513/78125 [1:03:50<9:45:17,  2.01it/s]
 10%|███▎                              | 7514/78125 [1:03:50<9:44:55,  2.01it/s]
 10%|███▎                              | 7515/78125 [1:03:51<9:44:28,  2.01it/s]
 10%|███▎                              | 7516/78125 [1:03:51<9:44:31,  2.01it/s]
 10%|███▎                              | 7517/78125 [1:03:52<9:44:47,  2.01it/s]
 10%|███▎                              | 7518/78125 [1:03:52<9:44:41,  2.01it/s]
 10%|███▎                              | 7519/78125 [1:03:53<9:45:03,  2.01it/s]
 10%|███▎                              | 7520/78125 [1:03:53<9:45:30,  2.01it/s]
 10%|███▎                              | 7521/78125 [1:03:54<9:45:19,  2.01it/s]
 10%|███▎                              | 7522/78125 [1:03:54<9:45:04,  2.01it/s]
 10%|███▎                              | 7523/78125 [1:03:55<9:45:26,  2.01it/s]
 10%|███▎                              | 7524/78125 [1:03:55<9:44:57,  2.01it/s]
 10%|███▎                              | 7525/78125 [1:03:56<9:44:16,  2.01it/s]
 10%|███▎                              | 7526/78125 [1:03:56<9:44:03,  2.01it/s]
 10%|███▎                              | 7527/78125 [1:03:57<9:44:18,  2.01it/s]
 10%|███▎                              | 7528/78125 [1:03:57<9:43:59,  2.01it/s]
 10%|███▎                              | 7529/78125 [1:03:58<9:43:30,  2.02it/s]
 10%|███▎                              | 7530/78125 [1:03:58<9:44:03,  2.01it/s]
 10%|███▎                              | 7531/78125 [1:03:59<9:44:34,  2.01it/s]
 10%|███▎                              | 7532/78125 [1:03:59<9:43:52,  2.02it/s]
 10%|███▎                              | 7533/78125 [1:04:00<9:44:17,  2.01it/s]
 10%|███▎                              | 7534/78125 [1:04:00<9:43:48,  2.02it/s]
 10%|███▎                              | 7535/78125 [1:04:01<9:43:42,  2.02it/s]
 10%|███▎                              | 7536/78125 [1:04:01<9:44:23,  2.01it/s]
 10%|███▎                              | 7537/78125 [1:04:02<9:44:26,  2.01it/s]
 10%|███▎                              | 7538/78125 [1:04:02<9:43:59,  2.01it/s]
 10%|███▎                              | 7539/78125 [1:04:03<9:43:43,  2.02it/s]
 10%|███▎                              | 7540/78125 [1:04:03<9:43:59,  2.01it/s]
 10%|███▎                              | 7541/78125 [1:04:04<9:44:45,  2.01it/s]
 10%|███▎                              | 7542/78125 [1:04:04<9:44:51,  2.01it/s]
 10%|███▎                              | 7543/78125 [1:04:05<9:44:54,  2.01it/s]
 10%|███▎                              | 7544/78125 [1:04:05<9:44:19,  2.01it/s]
 10%|███▎                              | 7545/78125 [1:04:06<9:44:19,  2.01it/s]
 10%|███▎                              | 7546/78125 [1:04:06<9:44:03,  2.01it/s]
 10%|███▎                              | 7547/78125 [1:04:07<9:44:08,  2.01it/s]
 10%|███▎                              | 7548/78125 [1:04:07<9:43:37,  2.02it/s]
 10%|███▎                              | 7549/78125 [1:04:08<9:43:26,  2.02it/s]
 10%|███▎                              | 7550/78125 [1:04:08<9:43:04,  2.02it/s]
 10%|███▎                              | 7551/78125 [1:04:09<9:42:52,  2.02it/s]
 10%|███▎                              | 7552/78125 [1:04:09<9:43:58,  2.01it/s]
 10%|███▎                              | 7553/78125 [1:04:10<9:43:36,  2.02it/s]
 10%|███▎                              | 7554/78125 [1:04:10<9:43:30,  2.02it/s]
 10%|███▎                              | 7555/78125 [1:04:11<9:44:17,  2.01it/s]
 10%|███▎                              | 7556/78125 [1:04:11<9:44:03,  2.01it/s]
 10%|███▎                              | 7557/78125 [1:04:12<9:43:36,  2.02it/s]
 10%|███▎                              | 7558/78125 [1:04:12<9:43:39,  2.02it/s]
 10%|███▎                              | 7559/78125 [1:04:13<9:43:48,  2.01it/s]
 10%|███▎                              | 7560/78125 [1:04:13<9:43:39,  2.02it/s]
 10%|███▎                              | 7561/78125 [1:04:14<9:43:50,  2.01it/s]
 10%|███▎                              | 7562/78125 [1:04:14<9:43:47,  2.01it/s]
 10%|███▎                              | 7563/78125 [1:04:15<9:43:49,  2.01it/s]
 10%|███▎                              | 7564/78125 [1:04:15<9:43:45,  2.01it/s]
 10%|███▎                              | 7565/78125 [1:04:16<9:43:13,  2.02it/s]
 10%|███▎                              | 7566/78125 [1:04:16<9:43:27,  2.02it/s]
 10%|███▎                              | 7567/78125 [1:04:17<9:42:56,  2.02it/s]
 10%|███▎                              | 7568/78125 [1:04:17<9:43:00,  2.02it/s]
 10%|███▎                              | 7569/78125 [1:04:18<9:43:25,  2.02it/s]
 10%|███▎                              | 7570/78125 [1:04:18<9:43:25,  2.02it/s]
 10%|███▎                              | 7571/78125 [1:04:19<9:43:32,  2.02it/s]
 10%|███▎                              | 7572/78125 [1:04:19<9:43:28,  2.02it/s]
 10%|███▎                              | 7573/78125 [1:04:20<9:43:54,  2.01it/s]
 10%|███▎                              | 7574/78125 [1:04:20<9:43:08,  2.02it/s]
 10%|███▎                              | 7575/78125 [1:04:21<9:43:45,  2.01it/s]
 10%|███▎                              | 7576/78125 [1:04:21<9:43:35,  2.01it/s]
 10%|███▎                              | 7577/78125 [1:04:22<9:43:28,  2.02it/s]
 10%|███▎                              | 7578/78125 [1:04:22<9:43:15,  2.02it/s]
 10%|███▎                              | 7579/78125 [1:04:23<9:43:49,  2.01it/s]
 10%|███▎                              | 7580/78125 [1:04:23<9:44:13,  2.01it/s]
 10%|███▎                              | 7581/78125 [1:04:24<9:44:28,  2.01it/s]
 10%|███▎                              | 7582/78125 [1:04:24<9:44:04,  2.01it/s]
 10%|███▎                              | 7583/78125 [1:04:25<9:43:59,  2.01it/s]
 10%|███▎                              | 7584/78125 [1:04:25<9:43:39,  2.01it/s]
 10%|███▎                              | 7585/78125 [1:04:26<9:43:39,  2.01it/s]
 10%|███▎                              | 7586/78125 [1:04:26<9:43:26,  2.02it/s]
 10%|███▎                              | 7587/78125 [1:04:26<9:43:11,  2.02it/s]
 10%|███▎                              | 7588/78125 [1:04:27<9:43:12,  2.02it/s]
 10%|███▎                              | 7589/78125 [1:04:27<9:43:20,  2.02it/s]
 10%|███▎                              | 7590/78125 [1:04:28<9:44:21,  2.01it/s]
 10%|███▎                              | 7591/78125 [1:04:28<9:44:31,  2.01it/s]
 10%|███▎                              | 7592/78125 [1:04:29<9:44:40,  2.01it/s]
 10%|███▎                              | 7593/78125 [1:04:29<9:44:17,  2.01it/s]
 10%|███▎                              | 7594/78125 [1:04:30<9:44:52,  2.01it/s]
 10%|███▎                              | 7595/78125 [1:04:30<9:44:55,  2.01it/s]
 10%|███▎                              | 7596/78125 [1:04:31<9:44:48,  2.01it/s]
 10%|███▎                              | 7597/78125 [1:04:31<9:44:21,  2.01it/s]
 10%|███▎                              | 7598/78125 [1:04:32<9:44:42,  2.01it/s]
 10%|███▎                              | 7599/78125 [1:04:32<9:44:04,  2.01it/s]
 10%|███▎                              | 7600/78125 [1:04:33<9:43:28,  2.01it/s]
 10%|███▎                              | 7601/78125 [1:04:33<9:44:15,  2.01it/s]
 10%|███▎                              | 7602/78125 [1:04:34<9:44:20,  2.01it/s]
 10%|███▎                              | 7603/78125 [1:04:34<9:44:14,  2.01it/s]
 10%|███▎                              | 7604/78125 [1:04:35<9:44:36,  2.01it/s]
 10%|███▎                              | 7605/78125 [1:04:35<9:44:31,  2.01it/s]
 10%|███▎                              | 7606/78125 [1:04:36<9:44:16,  2.01it/s]
 10%|███▎                              | 7607/78125 [1:04:36<9:43:18,  2.01it/s]
 10%|███▎                              | 7608/78125 [1:04:37<9:43:45,  2.01it/s]
 10%|███▎                              | 7609/78125 [1:04:37<9:43:34,  2.01it/s]
 10%|███▎                              | 7610/78125 [1:04:38<9:43:28,  2.01it/s]
 10%|███▎                              | 7611/78125 [1:04:38<9:42:52,  2.02it/s]
 10%|███▎                              | 7612/78125 [1:04:39<9:43:09,  2.02it/s]
 10%|███▎                              | 7613/78125 [1:04:39<9:43:18,  2.01it/s]
 10%|███▎                              | 7614/78125 [1:04:40<9:43:11,  2.02it/s]
 10%|███▎                              | 7615/78125 [1:04:40<9:43:49,  2.01it/s]
 10%|███▎                              | 7616/78125 [1:04:41<9:43:07,  2.02it/s]
 10%|███▎                              | 7617/78125 [1:04:41<9:42:58,  2.02it/s]
 10%|███▎                              | 7618/78125 [1:04:42<9:43:08,  2.02it/s]
 10%|███▎                              | 7619/78125 [1:04:42<9:42:50,  2.02it/s]
 10%|███▎                              | 7620/78125 [1:04:43<9:43:09,  2.02it/s]
 10%|███▎                              | 7621/78125 [1:04:43<9:43:22,  2.01it/s]
 10%|███▎                              | 7622/78125 [1:04:44<9:43:52,  2.01it/s]
 10%|███▎                              | 7623/78125 [1:04:44<9:44:15,  2.01it/s]
 10%|███▎                              | 7624/78125 [1:04:45<9:43:47,  2.01it/s]
 10%|███▎                              | 7625/78125 [1:04:45<9:43:19,  2.01it/s]
 10%|███▎                              | 7626/78125 [1:04:46<9:42:48,  2.02it/s]
 10%|███▎                              | 7627/78125 [1:04:46<9:43:15,  2.01it/s]
 10%|███▎                              | 7628/78125 [1:04:47<9:43:43,  2.01it/s]
 10%|███▎                              | 7629/78125 [1:04:47<9:43:14,  2.01it/s]
 10%|███▎                              | 7630/78125 [1:04:48<9:43:34,  2.01it/s]
 10%|███▎                              | 7631/78125 [1:04:48<9:43:21,  2.01it/s]
 10%|███▎                              | 7632/78125 [1:04:49<9:43:15,  2.01it/s]
 10%|███▎                              | 7633/78125 [1:04:49<9:43:44,  2.01it/s]
 10%|███▎                              | 7634/78125 [1:04:50<9:44:07,  2.01it/s]
 10%|███▎                              | 7635/78125 [1:04:50<9:43:55,  2.01it/s]
 10%|███▎                              | 7636/78125 [1:04:51<9:44:37,  2.01it/s]
 10%|███▎                              | 7637/78125 [1:04:51<9:43:43,  2.01it/s]
 10%|███▎                              | 7638/78125 [1:04:52<9:43:41,  2.01it/s]
 10%|███▎                              | 7639/78125 [1:04:52<9:43:30,  2.01it/s]
 10%|███▎                              | 7640/78125 [1:04:53<9:43:08,  2.01it/s]
 10%|███▎                              | 7641/78125 [1:04:53<9:42:49,  2.02it/s]
 10%|███▎                              | 7642/78125 [1:04:54<9:41:56,  2.02it/s]
 10%|███▎                              | 7643/78125 [1:04:54<9:42:51,  2.02it/s]
 10%|███▎                              | 7644/78125 [1:04:55<9:43:14,  2.01it/s]
 10%|███▎                              | 7645/78125 [1:04:55<9:43:08,  2.01it/s]
 10%|███▎                              | 7646/78125 [1:04:56<9:44:08,  2.01it/s]
 10%|███▎                              | 7647/78125 [1:04:56<9:44:36,  2.01it/s]
 10%|███▎                              | 7648/78125 [1:04:57<9:44:02,  2.01it/s]
 10%|███▎                              | 7649/78125 [1:04:57<9:43:33,  2.01it/s]
 10%|███▎                              | 7650/78125 [1:04:58<9:43:33,  2.01it/s]
 10%|███▎                              | 7651/78125 [1:04:58<9:42:54,  2.02it/s]
 10%|███▎                              | 7652/78125 [1:04:59<9:42:11,  2.02it/s]
 10%|███▎                              | 7653/78125 [1:04:59<9:42:34,  2.02it/s]
 10%|███▎                              | 7654/78125 [1:05:00<9:43:28,  2.01it/s]
 10%|███▎                              | 7655/78125 [1:05:00<9:43:06,  2.01it/s]
 10%|███▎                              | 7656/78125 [1:05:01<9:43:17,  2.01it/s]
 10%|███▎                              | 7657/78125 [1:05:01<9:43:03,  2.01it/s]
 10%|███▎                              | 7658/78125 [1:05:02<9:43:04,  2.01it/s]
 10%|███▎                              | 7659/78125 [1:05:02<9:43:18,  2.01it/s]
 10%|███▎                              | 7660/78125 [1:05:03<9:43:37,  2.01it/s]
 10%|███▎                              | 7661/78125 [1:05:03<9:43:04,  2.01it/s]
 10%|███▎                              | 7662/78125 [1:05:04<9:42:29,  2.02it/s]
 10%|███▎                              | 7663/78125 [1:05:04<9:42:33,  2.02it/s]
 10%|███▎                              | 7664/78125 [1:05:05<9:42:14,  2.02it/s]
 10%|███▎                              | 7665/78125 [1:05:05<9:43:03,  2.01it/s]
 10%|███▎                              | 7666/78125 [1:05:06<9:42:46,  2.02it/s]
 10%|███▎                              | 7667/78125 [1:05:06<9:42:28,  2.02it/s]
 10%|███▎                              | 7668/78125 [1:05:07<9:42:37,  2.02it/s]
 10%|███▎                              | 7669/78125 [1:05:07<9:42:06,  2.02it/s]
 10%|███▎                              | 7670/78125 [1:05:08<9:41:55,  2.02it/s]
 10%|███▎                              | 7671/78125 [1:05:08<9:42:23,  2.02it/s]
 10%|███▎                              | 7672/78125 [1:05:09<9:43:30,  2.01it/s]
 10%|███▎                              | 7673/78125 [1:05:09<9:43:13,  2.01it/s]
 10%|███▎                              | 7674/78125 [1:05:10<9:43:45,  2.01it/s]
 10%|███▎                              | 7675/78125 [1:05:10<9:43:07,  2.01it/s]
 10%|███▎                              | 7676/78125 [1:05:11<9:43:27,  2.01it/s]
 10%|███▎                              | 7677/78125 [1:05:11<9:43:29,  2.01it/s]
 10%|███▎                              | 7678/78125 [1:05:12<9:42:38,  2.02it/s]
 10%|███▎                              | 7679/78125 [1:05:12<9:42:21,  2.02it/s]
 10%|███▎                              | 7680/78125 [1:05:13<9:42:13,  2.02it/s]
 10%|███▎                              | 7681/78125 [1:05:13<9:42:19,  2.02it/s]
 10%|███▎                              | 7682/78125 [1:05:14<9:42:17,  2.02it/s]
 10%|███▎                              | 7683/78125 [1:05:14<9:43:24,  2.01it/s]
 10%|███▎                              | 7684/78125 [1:05:15<9:42:53,  2.01it/s]
 10%|███▎                              | 7685/78125 [1:05:15<9:42:43,  2.01it/s]
 10%|███▎                              | 7686/78125 [1:05:16<9:42:19,  2.02it/s]
 10%|███▎                              | 7687/78125 [1:05:16<9:42:15,  2.02it/s]
 10%|███▎                              | 7688/78125 [1:05:17<9:42:03,  2.02it/s]
 10%|███▎                              | 7689/78125 [1:05:17<9:41:57,  2.02it/s]
 10%|███▎                              | 7690/78125 [1:05:18<9:42:22,  2.02it/s]
 10%|███▎                              | 7691/78125 [1:05:18<9:41:53,  2.02it/s]
 10%|███▎                              | 7692/78125 [1:05:19<9:41:37,  2.02it/s]
 10%|███▎                              | 7693/78125 [1:05:19<9:41:37,  2.02it/s]
 10%|███▎                              | 7694/78125 [1:05:20<9:42:06,  2.02it/s]
 10%|███▎                              | 7695/78125 [1:05:20<9:41:34,  2.02it/s]
 10%|███▎                              | 7696/78125 [1:05:21<9:41:27,  2.02it/s]
 10%|███▎                              | 7697/78125 [1:05:21<9:41:49,  2.02it/s]
 10%|███▎                              | 7698/78125 [1:05:22<9:42:26,  2.02it/s]
 10%|███▎                              | 7699/78125 [1:05:22<9:42:45,  2.01it/s]
 10%|███▎                              | 7700/78125 [1:05:23<9:41:54,  2.02it/s]
 10%|███▎                              | 7701/78125 [1:05:23<9:42:21,  2.02it/s]
 10%|███▎                              | 7702/78125 [1:05:24<9:42:18,  2.02it/s]
 10%|███▎                              | 7703/78125 [1:05:24<9:42:29,  2.01it/s]
 10%|███▎                              | 7704/78125 [1:05:25<9:43:06,  2.01it/s]
 10%|███▎                              | 7705/78125 [1:05:25<9:42:57,  2.01it/s]
 10%|███▎                              | 7706/78125 [1:05:26<9:42:36,  2.01it/s]
 10%|███▎                              | 7707/78125 [1:05:26<9:43:09,  2.01it/s]
 10%|███▎                              | 7708/78125 [1:05:27<9:42:33,  2.01it/s]
 10%|███▎                              | 7709/78125 [1:05:27<9:42:06,  2.02it/s]
 10%|███▎                              | 7710/78125 [1:05:28<9:42:30,  2.01it/s]
 10%|███▎                              | 7711/78125 [1:05:28<9:42:30,  2.01it/s]
 10%|███▎                              | 7712/78125 [1:05:29<9:42:48,  2.01it/s]
 10%|███▎                              | 7713/78125 [1:05:29<9:43:03,  2.01it/s]
 10%|███▎                              | 7714/78125 [1:05:30<9:42:52,  2.01it/s]
 10%|███▎                              | 7715/78125 [1:05:30<9:42:10,  2.02it/s]
 10%|███▎                              | 7716/78125 [1:05:31<9:42:41,  2.01it/s]
 10%|███▎                              | 7717/78125 [1:05:31<9:42:10,  2.02it/s]
 10%|███▎                              | 7718/78125 [1:05:32<9:42:03,  2.02it/s]
 10%|███▎                              | 7719/78125 [1:05:32<9:41:48,  2.02it/s]
 10%|███▎                              | 7720/78125 [1:05:33<9:41:51,  2.02it/s]
 10%|███▎                              | 7721/78125 [1:05:33<9:42:00,  2.02it/s]
 10%|███▎                              | 7722/78125 [1:05:34<9:42:30,  2.01it/s]
 10%|███▎                              | 7723/78125 [1:05:34<9:41:35,  2.02it/s]
 10%|███▎                              | 7724/78125 [1:05:35<9:42:06,  2.02it/s]
 10%|███▎                              | 7725/78125 [1:05:35<9:42:18,  2.01it/s]
 10%|███▎                              | 7726/78125 [1:05:36<9:42:17,  2.01it/s]
 10%|███▎                              | 7727/78125 [1:05:36<9:41:58,  2.02it/s]
 10%|███▎                              | 7728/78125 [1:05:36<9:41:56,  2.02it/s]
 10%|███▎                              | 7729/78125 [1:05:37<9:42:03,  2.02it/s]
 10%|███▎                              | 7730/78125 [1:05:37<9:42:01,  2.02it/s]
 10%|███▎                              | 7731/78125 [1:05:38<9:41:39,  2.02it/s]
 10%|███▎                              | 7732/78125 [1:05:38<9:42:42,  2.01it/s]
 10%|███▎                              | 7733/78125 [1:05:39<9:42:37,  2.01it/s]
 10%|███▎                              | 7734/78125 [1:05:39<9:42:16,  2.01it/s]
 10%|███▎                              | 7735/78125 [1:05:40<9:41:57,  2.02it/s]
 10%|███▎                              | 7736/78125 [1:05:40<9:41:34,  2.02it/s]
 10%|███▎                              | 7737/78125 [1:05:41<9:40:58,  2.02it/s]
 10%|███▎                              | 7738/78125 [1:05:41<9:41:01,  2.02it/s]
 10%|███▎                              | 7739/78125 [1:05:42<9:40:53,  2.02it/s]
 10%|███▎                              | 7740/78125 [1:05:42<9:41:01,  2.02it/s]
 10%|███▎                              | 7741/78125 [1:05:43<9:41:14,  2.02it/s]
 10%|███▎                              | 7742/78125 [1:05:43<9:41:26,  2.02it/s]
 10%|███▎                              | 7743/78125 [1:05:44<9:42:36,  2.01it/s]
 10%|███▎                              | 7744/78125 [1:05:44<9:42:31,  2.01it/s]
 10%|███▎                              | 7745/78125 [1:05:45<9:42:32,  2.01it/s]
 10%|███▎                              | 7746/78125 [1:05:45<9:41:51,  2.02it/s]
 10%|███▎                              | 7747/78125 [1:05:46<9:41:53,  2.02it/s]
 10%|███▎                              | 7748/78125 [1:05:46<9:42:06,  2.02it/s]
 10%|███▎                              | 7749/78125 [1:05:47<9:42:00,  2.02it/s]
 10%|███▎                              | 7750/78125 [1:05:47<9:42:06,  2.01it/s]
 10%|███▎                              | 7751/78125 [1:05:48<9:41:13,  2.02it/s]
 10%|███▎                              | 7752/78125 [1:05:48<9:41:04,  2.02it/s]
 10%|███▎                              | 7753/78125 [1:05:49<9:42:13,  2.01it/s]
 10%|███▎                              | 7754/78125 [1:05:49<9:41:43,  2.02it/s]
 10%|███▎                              | 7755/78125 [1:05:50<9:41:49,  2.02it/s]
 10%|███▍                              | 7756/78125 [1:05:50<9:41:29,  2.02it/s]
 10%|███▍                              | 7757/78125 [1:05:51<9:42:00,  2.02it/s]
 10%|███▍                              | 7758/78125 [1:05:51<9:42:18,  2.01it/s]
 10%|███▍                              | 7759/78125 [1:05:52<9:41:40,  2.02it/s]
 10%|███▍                              | 7760/78125 [1:05:52<9:41:03,  2.02it/s]
 10%|███▍                              | 7761/78125 [1:05:53<9:40:32,  2.02it/s]
 10%|███▍                              | 7762/78125 [1:05:53<9:40:58,  2.02it/s]
 10%|███▍                              | 7763/78125 [1:05:54<9:41:00,  2.02it/s]
 10%|███▍                              | 7764/78125 [1:05:54<9:41:18,  2.02it/s]
 10%|███▍                              | 7765/78125 [1:05:55<9:41:48,  2.02it/s]
 10%|███▍                              | 7766/78125 [1:05:55<9:41:29,  2.02it/s]
 10%|███▍                              | 7767/78125 [1:05:56<9:41:25,  2.02it/s]
 10%|███▍                              | 7768/78125 [1:05:56<9:42:40,  2.01it/s]
 10%|███▍                              | 7769/78125 [1:05:57<9:42:38,  2.01it/s]
 10%|███▍                              | 7770/78125 [1:05:57<9:42:14,  2.01it/s]
 10%|███▍                              | 7771/78125 [1:05:58<9:41:47,  2.02it/s]
 10%|███▍                              | 7772/78125 [1:05:58<9:42:04,  2.01it/s]
 10%|███▍                              | 7773/78125 [1:05:59<9:42:53,  2.01it/s]
 10%|███▍                              | 7774/78125 [1:05:59<9:42:57,  2.01it/s]
 10%|███▍                              | 7775/78125 [1:06:00<9:42:04,  2.01it/s]
 10%|███▍                              | 7776/78125 [1:06:00<9:41:29,  2.02it/s]
 10%|███▍                              | 7777/78125 [1:06:01<9:40:57,  2.02it/s]
 10%|███▍                              | 7778/78125 [1:06:01<9:41:21,  2.02it/s]
 10%|███▍                              | 7779/78125 [1:06:02<9:41:22,  2.02it/s]
 10%|███▍                              | 7780/78125 [1:06:02<9:41:37,  2.02it/s]
 10%|███▍                              | 7781/78125 [1:06:03<9:41:52,  2.01it/s]
 10%|███▍                              | 7782/78125 [1:06:03<9:42:12,  2.01it/s]
 10%|███▍                              | 7783/78125 [1:06:04<9:41:48,  2.02it/s]
 10%|███▍                              | 7784/78125 [1:06:04<9:42:24,  2.01it/s]
 10%|███▍                              | 7785/78125 [1:06:05<9:42:22,  2.01it/s]
 10%|███▍                              | 7786/78125 [1:06:05<9:41:48,  2.01it/s]
 10%|███▍                              | 7787/78125 [1:06:06<9:41:28,  2.02it/s]
 10%|███▍                              | 7788/78125 [1:06:06<9:41:19,  2.02it/s]
 10%|███▍                              | 7789/78125 [1:06:07<9:42:01,  2.01it/s]
 10%|███▍                              | 7790/78125 [1:06:07<9:41:51,  2.01it/s]
 10%|███▍                              | 7791/78125 [1:06:08<9:41:31,  2.02it/s]
 10%|███▍                              | 7792/78125 [1:06:08<9:42:01,  2.01it/s]
 10%|███▍                              | 7793/78125 [1:06:09<9:41:36,  2.02it/s]
 10%|███▍                              | 7794/78125 [1:06:09<9:41:26,  2.02it/s]
 10%|███▍                              | 7795/78125 [1:06:10<9:41:47,  2.01it/s]
 10%|███▍                              | 7796/78125 [1:06:10<9:41:51,  2.01it/s]
 10%|███▍                              | 7797/78125 [1:06:11<9:42:05,  2.01it/s]
 10%|███▍                              | 7798/78125 [1:06:11<9:41:29,  2.02it/s]
 10%|███▍                              | 7799/78125 [1:06:12<9:41:23,  2.02it/s]
 10%|███▍                              | 7800/78125 [1:06:12<9:41:19,  2.02it/s]
 10%|███▍                              | 7801/78125 [1:06:13<9:41:28,  2.02it/s]
 10%|███▍                              | 7802/78125 [1:06:13<9:41:15,  2.02it/s]
 10%|███▍                              | 7803/78125 [1:06:14<9:40:25,  2.02it/s]
 10%|███▍                              | 7804/78125 [1:06:14<9:40:44,  2.02it/s]
 10%|███▍                              | 7805/78125 [1:06:15<9:40:44,  2.02it/s]
 10%|███▍                              | 7806/78125 [1:06:15<9:41:29,  2.02it/s]
 10%|███▍                              | 7807/78125 [1:06:16<9:41:45,  2.01it/s]
 10%|███▍                              | 7808/78125 [1:06:16<9:41:41,  2.01it/s]
 10%|███▍                              | 7809/78125 [1:06:17<9:41:03,  2.02it/s]
 10%|███▍                              | 7810/78125 [1:06:17<9:40:56,  2.02it/s]
 10%|███▍                              | 7811/78125 [1:06:18<9:41:07,  2.02it/s]
 10%|███▍                              | 7812/78125 [1:06:18<9:40:53,  2.02it/s]Error executing job with overrides: ['data_path=data/sudoku-6x6', 'epochs=5000', 'eval_interval=500', 'checkpoint_every_eval=True', 'global_batch_size=64', 'lr=7e-5', 'puzzle_emb_lr=7e-5', 'weight_decay=1.0', 'puzzle_emb_weight_decay=1.0']
4029.5s 153 Traceback (most recent call last):
4029.5s 154 File "/kaggle/working/pretrain.py", line 436, in launch
4029.5s 155 metrics = evaluate(config, train_state, eval_loader, eval_metadata, rank=RANK, world_size=WORLD_SIZE)
4029.5s 156 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 157 File "/kaggle/working/pretrain.py", line 284, in evaluate
4029.5s 158 carry, _, metrics, preds, all_finish = train_state.model(carry=carry, batch=batch, return_keys=config.eval_save_outputs)
4029.5s 159 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 160 File "/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py", line 465, in __call__
4029.5s 161 return super().__call__(*args, **kwargs)
4029.5s 162 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 163 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
4029.5s 164 return self._call_impl(*args, **kwargs)
4029.5s 165 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 166 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
4029.5s 167 return forward_call(*args, **kwargs)
4029.5s 168 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 169 File "/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py", line 953, in compile_wrapper
4029.5s 170 return fn(*args, **kwargs)
4029.5s 171 ^^^^^^^^^^^^^^^^^^^
4029.5s 172 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
4029.5s 173 return self._call_impl(*args, **kwargs)
4029.5s 174 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 175 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
4029.5s 176 return forward_call(*args, **kwargs)
4029.5s 177 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 178 File "/kaggle/working/models/losses.py", line 57, in forward
4029.5s 179 new_carry, outputs = self.model(**model_kwargs)
4029.5s 180 ^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 181 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
4029.5s 182 return self._call_impl(*args, **kwargs)
4029.5s 183 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 184 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
4029.5s 185 return forward_call(*args, **kwargs)
4029.5s 186 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 187 File "/kaggle/working/models/hrm/hrm_act_v1.py", line 249, in forward
4029.5s 188 new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)
4029.5s 189 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 190 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
4029.5s 191 return self._call_impl(*args, **kwargs)
4029.5s 192 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 193 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
4029.5s 194 return forward_call(*args, **kwargs)
4029.5s 195 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 196 File "/kaggle/working/models/hrm/hrm_act_v1.py", line 195, in forward
4029.5s 197 z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
4029.5s 198 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 199 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
4029.5s 200 return self._call_impl(*args, **kwargs)
4029.5s 201 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 202 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
4029.5s 203 return forward_call(*args, **kwargs)
4029.5s 204 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 205 File "/kaggle/working/models/hrm/hrm_act_v1.py", line 97, in forward
4029.5s 206 hidden_states = layer(hidden_states=hidden_states, **kwargs)
4029.5s 207 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 208 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
4029.5s 209 return self._call_impl(*args, **kwargs)
4029.5s 210 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 211 File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
4029.5s 212 return forward_call(*args, **kwargs)
4029.5s 213 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 214 File "/kaggle/working/models/hrm/hrm_act_v1.py", line 82, in forward
4029.5s 215 hidden_states = rms_norm(hidden_states + self.mlp(hidden_states), variance_epsilon=self.norm_eps)
4029.5s 216 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
4029.5s 217 File "/kaggle/working/models/layers.py", line 165, in rms_norm
4029.5s 218 hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
4029.5s 219 ~~~~~~~~~^~~~~~~~~~~~~~~~~~
4029.5s 220 torch.AcceleratorError: CUDA error: device-side assert triggered
4029.5s 221 Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
4029.5s 222 CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
4029.5s 223 For debugging consider passing CUDA_LAUNCH_BLOCKING=1
4029.5s 224 Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
4029.5s 225 
4029.5s 226 
4029.5s 227 Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.
4039.8s 228 [1;34mwandb[0m:
4039.8s 229 [1;34mwandb[0m: You can sync this run to the cloud by running:
4039.8s 230 [1;34mwandb[0m: [1mwandb sync /kaggle/working/wandb/offline-run-20260711_160237-7dx7a8sc[0m
4039.8s 231 [1;34mwandb[0m: Find logs at: [1;35mwandb/offline-run-20260711_160237-7dx7a8sc/logs[0m
4043.9s 232 10%|███▍                              | 7812/78125 [1:06:37<9:59:38,  1.95it/s]
4046.9s 233 ERROR: No checkpoint found! Training may not have completed.
4048.2s 234 Saved checkpoints:
4048.2s 235 ----------------------------------------
4052.1s 236 /usr/local/lib/python3.12/dist-packages/mistune.py:435: SyntaxWarning: invalid escape sequence '\|'
4052.1s 237 cells[i][c] = re.sub('\\\\\|', '|', cell)
4052.3s 238 /usr/local/lib/python3.12/dist-packages/nbconvert/filters/filter_links.py:36: SyntaxWarning: invalid escape sequence '\_'
4052.3s 239 text = re.sub(r'_', '\_', text) # Escape underscores in display text
4053.0s 240 [NbConvertApp] Converting notebook __notebook__.ipynb to notebook
4054.7s 241 [NbConvertApp] Writing 153142 bytes to __notebook__.ipynb
4057.2s 242 [NbConvertApp] Converting notebook __notebook__.ipynb to html
4058.4s 243 [NbConvertApp] Writing 733054 bytes to __results__.html


In [ ]:
%%writefile evaluate.py
from typing import List
import yaml
import os

import torch
import torch.distributed as dist

import pydantic
from omegaconf import OmegaConf
from pretrain import PretrainConfig, init_train_state, evaluate, create_dataloader


class EvalConfig(pydantic.BaseModel):
    checkpoint: str
    
    save_outputs: List[str] = ["inputs", "labels", "puzzle_identifiers", "logits", "q_halt_logits", "q_continue_logits"]


def launch():
    eval_cfg = EvalConfig(**OmegaConf.to_container(OmegaConf.from_cli()))  # type: ignore
    
    RANK = 0
    WORLD_SIZE = 1
    # Initialize distributed training if in distributed environment (e.g. torchrun)
    if "LOCAL_RANK" in os.environ:
        # Initialize distributed, default device and dtype
        dist.init_process_group(backend="nccl")

        RANK = dist.get_rank()
        WORLD_SIZE = dist.get_world_size()

        torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))

    with open(os.path.join(os.path.dirname(eval_cfg.checkpoint), "all_config.yaml"), "r") as f:
        config = PretrainConfig(**yaml.safe_load(f))

        config.eval_save_outputs = eval_cfg.save_outputs
        config.checkpoint_path = os.path.dirname(eval_cfg.checkpoint)

    # Dataloader
    train_loader, train_metadata = create_dataloader(config, "train", test_set_mode=False, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    eval_loader,  eval_metadata  = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)

    # Models
    train_state = init_train_state(config, train_metadata, world_size=WORLD_SIZE)
    # Try unwrap torch.compile
    try:
        train_state.model.load_state_dict(torch.load(eval_cfg.checkpoint, map_location="cuda"), assign=True)
    except:
        train_state.model.load_state_dict({k.removeprefix("_orig_mod."): v for k, v in torch.load(eval_cfg.checkpoint, map_location="cuda").items()}, assign=True)
    
    train_state.step = 0
    ckpt_filename = os.path.basename(eval_cfg.checkpoint)
    if ckpt_filename.startswith("step_"):
        train_state.step = int(ckpt_filename.removeprefix("step_"))

    # Evaluate
    print ("Starting evaluation")
    
    train_state.model.eval()
    metrics = evaluate(config, train_state, eval_loader, eval_metadata, rank=RANK, world_size=WORLD_SIZE)

    if metrics is not None:
        print (metrics)


if __name__ == "__main__":
    launch()



In [ ]:
%%writefile generate_6x6_sudoku_dataset_chatgpt.py

import random
import pandas as pd

BASE = [
    [1,2,3,4,5,6],
    [4,5,6,1,2,3],
    [2,3,4,5,6,1],
    [5,6,1,2,3,4],
    [3,4,5,6,1,2],
    [6,1,2,3,4,5],
]

def permute(grid):
    nums=list(range(1,7))
    random.shuffle(nums)
    mp={i+1:nums[i] for i in range(6)}
    return [[mp[v] for v in row] for row in grid]

def make_puzzle(sol, clues=18):
    cells=[(r,c) for r in range(6) for c in range(6)]
    random.shuffle(cells)
    keep=set(cells[:clues])
    puz=[]
    for r in range(6):
        row=[]
        for c in range(6):
            row.append(sol[r][c] if (r,c) in keep else 0)
        puz.append(row)
    return puz

def encode(g):
    return "".join(str(x) for row in g for x in row)

def generate_dataset(n=10000, clues=18):
    rows=[]
    for i in range(n):
        sol=permute(BASE)
        puz=make_puzzle(sol, clues)
        rows.append({
            "id": i,
            "puzzle": encode(puz),
            "solution": encode(sol),
            "clues": clues
        })
    return pd.DataFrame(rows)

if __name__ == "__main__":
    df=generate_dataset(100000,18)
    df.to_csv("sudoku6x6_train.csv", index=False)
    print("saved", len(df), "samples")



In [ ]:
%%writefile pdf_text.txt



In [ ]:
%%writefile pretrain.py
from typing import Optional, Any, Sequence, List
from dataclasses import dataclass
import os
import math
import yaml
import shutil

import torch
import torch.distributed as dist
from torch import nn
from torch.utils.data import DataLoader

import tqdm
import wandb
import coolname
import hydra
import pydantic
from omegaconf import DictConfig
from torch.optim import AdamW

from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig, PuzzleDatasetMetadata
from utils.functions import load_model_class, get_model_source_path
from models.sparse_embedding import CastedSparseEmbeddingSignSGD_Distributed


class LossConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')
    
    name: str


class ArchConfig(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(extra='allow')

    name: str
    loss: LossConfig


class PretrainConfig(pydantic.BaseModel):
    # Config
    arch: ArchConfig
    # Data
    data_path: str

    # Hyperparams
    global_batch_size: int
    epochs: int

    lr: float
    lr_min_ratio: float
    lr_warmup_steps: int

    weight_decay: float
    beta1: float
    beta2: float

    # Puzzle embedding
    puzzle_emb_lr: float
    puzzle_emb_weight_decay: float

    # Names
    project_name: Optional[str] = None
    run_name: Optional[str] = None
    checkpoint_path: Optional[str] = None

    # Extras
    seed: int = 0
    checkpoint_every_eval: bool = False
    eval_interval: Optional[int] = None
    eval_save_outputs: List[str] = []


@dataclass
class TrainState:
    model: nn.Module
    optimizers: Sequence[torch.optim.Optimizer]
    optimizer_lrs: Sequence[float]
    carry: Any

    step: int
    total_steps: int


def create_dataloader(config: PretrainConfig, split: str, rank: int, world_size: int, **kwargs):
    dataset = PuzzleDataset(PuzzleDatasetConfig(
        seed=config.seed,

        dataset_path=config.data_path,

        rank=rank,
        num_replicas=world_size,
        
        **kwargs
    ), split=split)
    dataloader = DataLoader(
        dataset,
        batch_size=None,

        num_workers=1,
        prefetch_factor=8,

        pin_memory=True,
        persistent_workers=True
    )
    return dataloader, dataset.metadata


def create_model(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, world_size: int):
    model_cfg = dict(
        **config.arch.__pydantic_extra__,  # type: ignore

        batch_size=config.global_batch_size // world_size,

        vocab_size=train_metadata.vocab_size,
        seq_len=train_metadata.seq_len,
        num_puzzle_identifiers=train_metadata.num_puzzle_identifiers,
        causal=False  # Non-autoregressive
    )

    # Instantiate model with loss head
    model_cls = load_model_class(config.arch.name)
    loss_head_cls = load_model_class(config.arch.loss.name)

    with torch.device("cuda"):
        model: nn.Module = model_cls(model_cfg)
        model = loss_head_cls(model, **config.arch.loss.__pydantic_extra__)  # type: ignore
        if "DISABLE_COMPILE" not in os.environ:
            model = torch.compile(model, dynamic=False)  # type: ignore

        # Broadcast parameters from rank 0
        if world_size > 1:
            with torch.no_grad():
                for param in list(model.parameters()) + list(model.buffers()):
                    dist.broadcast(param, src=0)

    # Optimizers and lr
    optimizers = [
        CastedSparseEmbeddingSignSGD_Distributed(
            model.model.puzzle_emb.buffers(),  # type: ignore
            
            lr=0,  # Needs to be set by scheduler
            weight_decay=config.puzzle_emb_weight_decay,

            world_size=world_size
        ),
        AdamW(
            model.parameters(),
            lr=0,  # Needs to be set by scheduler
            weight_decay=config.weight_decay,
            betas=(config.beta1, config.beta2)
        )
    ]
    optimizer_lrs = [
        config.puzzle_emb_lr,
        config.lr
    ]

    return model, optimizers, optimizer_lrs


def cosine_schedule_with_warmup_lr_lambda(
    current_step: int, *, base_lr: float, num_warmup_steps: int, num_training_steps: int, min_ratio: float = 0.0, num_cycles: float = 0.5
):
    if current_step < num_warmup_steps:
        return base_lr * float(current_step) / float(max(1, num_warmup_steps))

    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    return base_lr * (min_ratio + max(0.0, (1 - min_ratio) * 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))))


def init_train_state(config: PretrainConfig, train_metadata: PuzzleDatasetMetadata, world_size: int):
    # Estimated total training steps
    total_steps = int(config.epochs * train_metadata.total_groups * train_metadata.mean_puzzle_examples / config.global_batch_size)

    # Model
    model, optimizers, optimizer_lrs = create_model(config, train_metadata, world_size=world_size)

    return TrainState(
        step=0,
        total_steps=total_steps,

        model=model,
        optimizers=optimizers,
        optimizer_lrs=optimizer_lrs,
        carry=None
    )


def save_train_state(config: PretrainConfig, train_state: TrainState):
    # FIXME: Only saved model.
    if config.checkpoint_path is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)
    torch.save(train_state.model.state_dict(), os.path.join(config.checkpoint_path, f"step_{train_state.step}"))


def compute_lr(base_lr: float, config: PretrainConfig, train_state: TrainState):
    return cosine_schedule_with_warmup_lr_lambda(
        current_step=train_state.step,
        base_lr=base_lr,
        num_warmup_steps=round(config.lr_warmup_steps),
        num_training_steps=train_state.total_steps,
        min_ratio=config.lr_min_ratio
    )


def train_batch(config: PretrainConfig, train_state: TrainState, batch: Any, global_batch_size: int, rank: int, world_size: int):
    train_state.step += 1
    if train_state.step > train_state.total_steps:  # At most train_total_steps
        return

    # To device
    batch = {k: v.cuda() for k, v in batch.items()}

    # Init carry if it is None
    if train_state.carry is None:
        with torch.device("cuda"):
            train_state.carry = train_state.model.initial_carry(batch)  # type: ignore

    # Forward
    train_state.carry, loss, metrics, _, _ = train_state.model(carry=train_state.carry, batch=batch, return_keys=[])

    ((1 / global_batch_size) * loss).backward()

    # Allreduce
    if world_size > 1:
        for param in train_state.model.parameters():
            if param.grad is not None:
                dist.all_reduce(param.grad)
            
    # Apply optimizer
    lr_this_step = None    
    for optim, base_lr in zip(train_state.optimizers, train_state.optimizer_lrs):
        lr_this_step = compute_lr(base_lr, config, train_state)

        for param_group in optim.param_groups:
            param_group['lr'] = lr_this_step
            
        optim.step()
        optim.zero_grad()

    # Reduce metrics
    if len(metrics):
        assert not any(v.requires_grad for v in metrics.values())

        metric_keys = list(sorted(metrics.keys()))  # Sort keys to guarantee all processes use the same order.
        # Reduce and reconstruct
        metric_values = torch.stack([metrics[k] for k in metric_keys])
        if world_size > 1:
            dist.reduce(metric_values, dst=0)

        if rank == 0:
            metric_values = metric_values.cpu().numpy()
            reduced_metrics = {k: metric_values[i] for i, k in enumerate(metric_keys)}
            
            # Postprocess
            count = max(reduced_metrics["count"], 1)  # Avoid NaNs
            reduced_metrics = {f"train/{k}": v / (global_batch_size if k.endswith("loss") else count) for k, v in reduced_metrics.items()}

            reduced_metrics["train/lr"] = lr_this_step
            return reduced_metrics


def evaluate(config: PretrainConfig, train_state: TrainState, eval_loader: torch.utils.data.DataLoader, eval_metadata: PuzzleDatasetMetadata, rank: int, world_size: int):
    with torch.no_grad():
        set_ids = {k: idx for idx, k in enumerate(eval_metadata.sets)}
        
        all_preds = {}

        metric_keys = []
        metric_values = None
        metric_global_batch_size = [0 for _ in range(len(set_ids))]
        
        carry = None
        for set_name, batch, global_batch_size in eval_loader:
            # To device
            batch = {k: v.cuda() for k, v in batch.items()}
            with torch.device("cuda"):
                carry = train_state.model.initial_carry(batch)  # type: ignore

            # Forward
            while True:
                carry, _, metrics, preds, all_finish = train_state.model(carry=carry, batch=batch, return_keys=config.eval_save_outputs)
                
                if all_finish:
                    break

            for collection in (batch, preds):
                for k, v in collection.items():
                    if k in config.eval_save_outputs:
                        all_preds.setdefault(k, [])
                        all_preds[k].append(v.cpu())  # Move to CPU for saving GPU memory
                        
            del carry, preds, batch, all_finish

            # Aggregate
            set_id = set_ids[set_name]
            
            if metric_values is None:
                metric_keys = list(sorted(metrics.keys()))  # Sort keys to guarantee all processes use the same order.
                metric_values = torch.zeros((len(set_ids), len(metrics.values())), dtype=torch.float32, device="cuda")
                
            metric_values[set_id] += torch.stack([metrics[k] for k in metric_keys])
            metric_global_batch_size[set_id] += global_batch_size

        if len(all_preds) and config.checkpoint_path is not None:
            all_preds = {k: torch.cat(v, dim=0) for k, v in all_preds.items()}

            os.makedirs(config.checkpoint_path, exist_ok=True)
            torch.save(all_preds, os.path.join(config.checkpoint_path, f"step_{train_state.step}_all_preds.{rank}"))

        # Logging
        # Reduce to rank 0
        if metric_values is not None:
            if world_size > 1:
                dist.reduce(metric_values, dst=0)
            
            if rank == 0:
                reduced_metrics = metric_values.cpu().numpy()
                reduced_metrics = {set_name: {metric_name: reduced_metrics[set_id, metric_id] for metric_id, metric_name in enumerate(metric_keys)}
                                   for set_id, set_name in enumerate(set_ids)}
                
                # Postprocess
                for set_name, metrics in reduced_metrics.items():
                    count = metrics.pop("count")
                    reduced_metrics[set_name] = {k: v / count for k, v in metrics.items()}

                return reduced_metrics


def save_code_and_config(config: PretrainConfig):
    if config.checkpoint_path is None or wandb.run is None:
        return

    os.makedirs(config.checkpoint_path, exist_ok=True)

    # Copy code
    code_list = [
        get_model_source_path(config.arch.name),
        get_model_source_path(config.arch.loss.name)
    ]
    for code_file in code_list:
        if code_file is not None:
            code_name = os.path.basename(code_file)

            shutil.copy(code_file, os.path.join(config.checkpoint_path, code_name))

    # Dump config as yaml
    config_file = os.path.join(config.checkpoint_path, "all_config.yaml")
    with open(config_file, "wt") as f:
        yaml.dump(config.model_dump(), f)

    # Log code
    wandb.run.log_code(config.checkpoint_path)


def load_synced_config(hydra_config: DictConfig, rank: int, world_size: int) -> PretrainConfig:
    objects = [None]
    if rank == 0:
        config = PretrainConfig(**hydra_config)  # type: ignore

        # Naming
        if config.project_name is None:
            config.project_name = f"{os.path.basename(config.data_path).capitalize()} ACT-torch"
        if config.run_name is None:
            config.run_name = f"{config.arch.name.split('@')[-1]} {coolname.generate_slug(2)}"
        if config.checkpoint_path is None:
            config.checkpoint_path = os.path.join("checkpoints", config.project_name, config.run_name)

        objects = [config]

    if world_size > 1:
        dist.broadcast_object_list(objects, src=0)

    return objects[0]  # type: ignore


@hydra.main(config_path="config", config_name="cfg_pretrain", version_base=None)
def launch(hydra_config: DictConfig):
    RANK = 0
    WORLD_SIZE = 1

    # Initialize distributed training if in distributed environment (e.g. torchrun)
    if "LOCAL_RANK" in os.environ:
        # Initialize distributed, default device and dtype
        dist.init_process_group(backend="nccl")

        RANK = dist.get_rank()
        WORLD_SIZE = dist.get_world_size()

        torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
        
    # Load sync'ed config
    config = load_synced_config(hydra_config, rank=RANK, world_size=WORLD_SIZE)

    # Seed RNGs to ensure consistency
    torch.random.manual_seed(config.seed + RANK)

    # Dataset
    train_epochs_per_iter = config.eval_interval if config.eval_interval is not None else config.epochs
    total_iters = config.epochs // train_epochs_per_iter

    assert config.epochs % train_epochs_per_iter == 0, "Eval interval must be a divisor of total epochs."

    train_loader, train_metadata = create_dataloader(config, "train", test_set_mode=False, epochs_per_iter=train_epochs_per_iter, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)
    eval_loader,  eval_metadata  = create_dataloader(config, "test", test_set_mode=True, epochs_per_iter=1, global_batch_size=config.global_batch_size, rank=RANK, world_size=WORLD_SIZE)

    # Train state
    train_state = init_train_state(config, train_metadata, world_size=WORLD_SIZE)

    # Progress bar and logger
    progress_bar = None
    if RANK == 0:
        progress_bar = tqdm.tqdm(total=train_state.total_steps)

        wandb.init(project=config.project_name, name=config.run_name, config=config.model_dump(), settings=wandb.Settings(_disable_stats=True))  # type: ignore
        wandb.log({"num_params": sum(x.numel() for x in train_state.model.parameters())}, step=0)
        save_code_and_config(config)

    # Training Loop
    for _iter_id in range(total_iters):
        print (f"[Rank {RANK}, World Size {WORLD_SIZE}]: Epoch {_iter_id * train_epochs_per_iter}")

        ############ Train Iter
        train_state.model.train()
        for set_name, batch, global_batch_size in train_loader:
            metrics = train_batch(config, train_state, batch, global_batch_size, rank=RANK, world_size=WORLD_SIZE)

            if RANK == 0 and metrics is not None:
                wandb.log(metrics, step=train_state.step)
                progress_bar.update(train_state.step - progress_bar.n)  # type: ignore

        ############ Checkpointing (save BEFORE evaluation to prevent data loss)
        if RANK == 0 and (config.checkpoint_every_eval or (_iter_id == total_iters - 1)):
            save_train_state(config, train_state)

        ############ Evaluation (wrapped in try/except for T4 GPU stability)
        try:
            train_state.model.eval()
            metrics = evaluate(config, train_state, eval_loader, eval_metadata, rank=RANK, world_size=WORLD_SIZE)

            if RANK == 0 and metrics is not None:
                wandb.log(metrics, step=train_state.step)
                print(f"[Eval at step {train_state.step}] {metrics}")
        except Exception as e:
            print(f"WARNING: Evaluation failed at step {train_state.step}: {e}")
            print("Checkpoint was already saved. Training will continue.")

    # finalize
    if dist.is_initialized():
        dist.destroy_process_group()
    wandb.finish()


if __name__ == "__main__":
    launch()



In [ ]:
%%writefile puzzle_dataset.py
import os
import json

import numpy as np
import pydantic

import torch
from torch.utils.data import IterableDataset, get_worker_info

from models.losses import IGNORE_LABEL_ID
from dataset.common import PuzzleDatasetMetadata


def _sample_batch(rng: np.random.Generator, group_order: np.ndarray, puzzle_indices: np.ndarray, group_indices: np.ndarray, start_index: int, global_batch_size: int):
    # Pack examples into a full batch
    batch = []
    batch_puzzle_indices = []
    current_size = 0

    while (start_index < group_order.size) and (current_size < global_batch_size):
        # Pick a group and a puzzle from that group
        group_id = group_order[start_index]
        puzzle_id = rng.integers(group_indices[group_id], group_indices[group_id + 1])
        start_index += 1

        # Get range of the puzzle
        puzzle_start = puzzle_indices[puzzle_id]
        puzzle_size = int(puzzle_indices[puzzle_id + 1] - puzzle_start)

        append_size = min(puzzle_size, global_batch_size - current_size)

        # Put into batch
        batch_puzzle_indices.append(np.full(append_size, puzzle_id, dtype=np.int32))
        batch.append(puzzle_start + np.random.choice(puzzle_size, append_size, replace=False))

        current_size += append_size

    return start_index, np.concatenate(batch), np.concatenate(batch_puzzle_indices)


class PuzzleDatasetConfig(pydantic.BaseModel):
    seed: int
    dataset_path: str
    global_batch_size: int
    test_set_mode: bool

    epochs_per_iter: int  # Batch X epochs in an iteration to reduce overhead.

    rank: int
    num_replicas: int


class PuzzleDataset(IterableDataset):
    def __init__(self, config: PuzzleDatasetConfig, split: str = "train"):
        super().__init__()
        self.config = config
        self.split = split
        self.metadata = self._load_metadata()
        
        # Checks
        assert self.config.global_batch_size % self.config.num_replicas == 0, f"Global batch size {self.config.global_batch_size} must be multiples of nodes {self.config.num_replicas}."
        self.local_batch_size = self.config.global_batch_size // self.config.num_replicas

        # State
        self._data = None
        self._iters = 0

    def _load_metadata(self) -> PuzzleDatasetMetadata:
        with open(os.path.join(self.config.dataset_path, self.split, "dataset.json"), "r") as f:
            return PuzzleDatasetMetadata(**json.load(f))

    def _lazy_load_dataset(self):
        if self._data is not None:
            return

        field_mmap_modes = {
            "inputs": "r",
            "labels": "r",

            # Keep indices in memory
            "puzzle_identifiers": None,
            "puzzle_indices": None,
            "group_indices": None
        }

        # Load data
        self._data = {}
        for set_name in self.metadata.sets:
            # Load subset
            self._data[set_name] = {
                field_name: np.load(os.path.join(self.config.dataset_path, self.split, f"{set_name}__{field_name}.npy"), mmap_mode=mmap_mode)
                for field_name, mmap_mode in field_mmap_modes.items()
            }

    def _collate_batch(self, batch):
        # Convert dtype
        batch = {k: v.astype(np.int32) for k, v in batch.items()}

        # Convert ignore label IDs
        if self.metadata.ignore_label_id is not None:
            batch["labels"][batch["labels"] == self.metadata.ignore_label_id] = IGNORE_LABEL_ID

        # Pad
        if batch["puzzle_identifiers"].size < self.local_batch_size:
            pad_size = self.local_batch_size - batch["puzzle_identifiers"].size

            pad_values = {
                "inputs": self.metadata.pad_id,
                "labels": IGNORE_LABEL_ID,

                "puzzle_identifiers": self.metadata.blank_identifier_id
            }
            batch = {k: np.pad(v, ((0, pad_size), ) + ((0, 0), ) * (v.ndim - 1), constant_values=pad_values[k]) for k, v in batch.items()}

        # To tensor
        return {k: torch.from_numpy(v) for k, v in batch.items()}
    
    def _iter_test(self):
        for set_name, dataset in self._data.items():  # type: ignore
            total_examples = len(dataset["inputs"])

            # Load examples one by one
            start_index = 0
            while start_index < total_examples:
                # Compute indices
                end_index = min(total_examples, start_index + self.config.global_batch_size)
                
                local_start = start_index + self.config.rank * self.local_batch_size
                local_end   = min(start_index + (self.config.rank + 1) * self.local_batch_size, end_index)
                
                # Get batch of examples, and also puzzle IDs
                puzzle_indices = []
                puzzle_index = np.searchsorted(dataset["puzzle_indices"], local_start, side="right") - 1
                for i in range(local_start, local_end):
                    while puzzle_index + 1 < len(dataset["puzzle_indices"]) and i >= dataset["puzzle_indices"][puzzle_index + 1]:
                        puzzle_index += 1

                    puzzle_indices.append(puzzle_index)
                
                batch = self._collate_batch({
                    "inputs": dataset["inputs"][local_start: local_end],
                    "labels": dataset["labels"][local_start: local_end],
                    "puzzle_identifiers": dataset["puzzle_identifiers"][puzzle_indices]
                })

                yield set_name, batch, end_index - start_index
                
                # Advance to next batch
                start_index += self.config.global_batch_size

    def _iter_train(self):
        for set_name, dataset in self._data.items():  # type: ignore
            # Increase epoch count
            self._iters += 1

            # Randomly shuffle groups
            rng = np.random.Generator(np.random.Philox(seed=self.config.seed + self._iters))

            group_order = np.concatenate([rng.permutation(dataset["group_indices"].size - 1) for _i in range(self.config.epochs_per_iter)])
            start_index = 0
            
            while start_index < group_order.size:
                start_index, batch_indices, batch_puzzle_indices = _sample_batch(
                    rng,
                    group_order=group_order,
                    puzzle_indices=dataset["puzzle_indices"],
                    group_indices=dataset["group_indices"],
                    start_index=start_index,
                    global_batch_size=self.config.global_batch_size,
                )

                # Select current rank and collate
                global_effective_batch_size = batch_puzzle_indices.size  # Global effective batch size, excluding pads

                # Drop last batch
                if global_effective_batch_size < self.config.global_batch_size:
                    break

                batch_indices        = batch_indices       [self.config.rank * self.local_batch_size: (self.config.rank + 1) * self.local_batch_size]
                batch_puzzle_indices = batch_puzzle_indices[self.config.rank * self.local_batch_size: (self.config.rank + 1) * self.local_batch_size]
                batch = self._collate_batch({
                    "inputs": dataset["inputs"][batch_indices],
                    "labels": dataset["labels"][batch_indices],
                    "puzzle_identifiers": dataset["puzzle_identifiers"][batch_puzzle_indices]
                })

                yield set_name, batch, global_effective_batch_size
                
    def __iter__(self):
        worker_info = get_worker_info()
        assert worker_info is None or worker_info.num_workers == 1, "Multithreaded data loading is not currently supported."
        
        self._lazy_load_dataset()
        
        # Iterate using specified mode
        if self.config.test_set_mode:
            yield from self._iter_test()
        else:
            yield from self._iter_train()



In [ ]:
%%writefile requirements.txt
torch
einops
tqdm
coolname
pydantic
argdantic
wandb
omegaconf
hydra-core
huggingface_hub



In [ ]:
%%writefile config/cfg_pretrain.yaml
# ARC training config

defaults:
  - arch: hrm_v1
  - _self_

hydra:
  output_subdir: null

# Data path
data_path: data/arc-aug-1000

# Hyperparams - Training
global_batch_size: 768

epochs: 100000
eval_interval: 10000
checkpoint_every_eval: True

lr: 1e-4
lr_min_ratio: 1.0
lr_warmup_steps: 2000

# Standard hyperparameter settings for LM, as used in Llama
beta1: 0.9
beta2: 0.95
weight_decay: 0.1
puzzle_emb_weight_decay: 0.1

# Hyperparams - Puzzle embeddings training
puzzle_emb_lr: 1e-2



In [ ]:
%%writefile config/arch/hrm_v1.yaml
name: hrm.hrm_act_v1@HierarchicalReasoningModel_ACTV1
loss:
  name: losses@ACTLossHead
  loss_type: stablemax_cross_entropy

halt_exploration_prob: 0.1
halt_max_steps: 16

H_cycles: 2
L_cycles: 2

H_layers: 4
L_layers: 4

hidden_size: 512
num_heads: 8  # min(2, hidden_size // 64)
expansion: 4

puzzle_emb_ndim: ${.hidden_size}

pos_encodings: rope



In [ ]:
%%writefile dataset/build_6x6_sudoku_dataset.py
import os
import json
import random
import numpy as np
from tqdm import tqdm
from argdantic import ArgParser
from pydantic import BaseModel

try:
    from dataset.common import PuzzleDatasetMetadata
except ImportError:
    from common import PuzzleDatasetMetadata

cli = ArgParser()

class DataProcessConfig(BaseModel):
    output_dir: str = "data/sudoku-6x6"
    num_train: int = 1000
    num_test: int = 100
    holes: int = 20  # Number of blanks

def is_valid(board, r, c, val):
    if val in board[r]: return False
    if val in board[:, c]: return False
    br, bc = 2 * (r // 2), 3 * (c // 3)
    if val in board[br:br+2, bc:bc+3]: return False
    return True

def solve(board):
    for r in range(6):
        for c in range(6):
            if board[r, c] == 0:
                nums = list(range(1, 7))
                random.shuffle(nums)
                for val in nums:
                    if is_valid(board, r, c, val):
                        board[r, c] = val
                        if solve(board):
                            return True
                        board[r, c] = 0
                return False
    return True

def generate_6x6_board():
    board = np.zeros((6, 6), dtype=int)
    solve(board)
    return board

def generate_puzzles(num_puzzles, num_holes):
    inputs = []
    labels = []
    for _ in tqdm(range(num_puzzles), desc="Generating 6x6 puzzles"):
        solution = generate_6x6_board()
        puzzle = solution.copy()
        
        # Poke holes
        cells = [(r, c) for r in range(6) for c in range(6)]
        random.shuffle(cells)
        for r, c in cells[:num_holes]:
            puzzle[r, c] = 0
            
        inputs.append(puzzle)
        labels.append(solution)
    return inputs, labels

def convert_subset(set_name: str, config: DataProcessConfig):
    num_puzzles = config.num_train if set_name == "train" else config.num_test
    
    inputs, labels = generate_puzzles(num_puzzles, config.holes)
    
    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    
    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)
    
    for i, (inp, out) in enumerate(zip(inputs, labels)):
        # 0 becomes 1 (blank), 1-6 becomes 2-7
        results["inputs"].append(inp.flatten() + 1)
        results["labels"].append(out.flatten() + 1)
        
        results["puzzle_indices"].append(i + 1)
        results["puzzle_identifiers"].append(0)
        results["group_indices"].append(i + 1)
        
    results = {
        "inputs": np.array(results["inputs"], dtype=np.int32),
        "labels": np.array(results["labels"], dtype=np.int32),
        "group_indices": np.array(results["group_indices"], dtype=np.int32),
        "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
        "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
    }

    metadata = PuzzleDatasetMetadata(
        seq_len=36,
        vocab_size=8,  # PAD + "0" + "1"..."6"
        pad_id=0,
        ignore_label_id=0,
        blank_identifier_id=1,
        num_puzzle_identifiers=1,
        total_groups=len(results["group_indices"]) - 1,
        mean_puzzle_examples=1,
        sets=["all"]
    )

    save_dir = os.path.join(config.output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)
    
    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata.model_dump(), f)
        
    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)

    with open(os.path.join(config.output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)

@cli.command(singleton=True)
def preprocess_data(config: DataProcessConfig):
    convert_subset("train", config)
    convert_subset("test", config)

if __name__ == "__main__":
    cli()



In [ ]:
%%writefile dataset/build_arc_dataset.py
from typing import List, Optional, Tuple, Dict
from dataclasses import dataclass
from pathlib import Path
import os
import json
import hashlib
import numpy as np
from glob import glob

from argdantic import ArgParser
from pydantic import BaseModel

from common import PuzzleDatasetMetadata, dihedral_transform


cli = ArgParser()


class DataProcessConfig(BaseModel):
    # ARC-1
    dataset_dirs: List[str] = ["dataset/raw-data/ARC-AGI/data", "dataset/raw-data/ConceptARC/corpus"]
    output_dir: str = "data/arc-aug-1000"
    
    # ARC-2
    # dataset_dirs: List[str] = ["dataset/raw-data/ARC-AGI-2/data"]
    # output_dir: str = "data/arc-2-aug-1000"

    seed: int = 42
    num_aug: int = 1000
    
    
ARCMaxGridSize = 30
ARCAugmentRetriesFactor = 5
    

@dataclass
class ARCPuzzle:
    id: str

    examples: List[Tuple[np.ndarray, np.ndarray]]

    
def arc_grid_to_np(grid: List[List[int]]):
    arr = np.array(grid)

    # Shape check
    assert arr.ndim == 2
    assert arr.shape[0] <= ARCMaxGridSize and arr.shape[1] <= ARCMaxGridSize
    # Element check
    assert np.all((arr >= 0) & (arr <= 9))
    return arr.astype(np.uint8)


def np_grid_to_seq_translational_augment(inp: np.ndarray, out: np.ndarray, do_translation: bool):
    # PAD: 0, <eos>: 1, digits: 2 ... 11
    # Compute random top-left pad
    if do_translation:
        pad_r = np.random.randint(0, ARCMaxGridSize - max(inp.shape[0], out.shape[0]) + 1)
        pad_c = np.random.randint(0, ARCMaxGridSize - max(inp.shape[1], out.shape[1]) + 1)
    else:
        pad_r = pad_c = 0

    # Pad grid
    result = []
    for grid in [inp, out]:
        nrow, ncol = grid.shape
        grid = np.pad(grid + 2, ((pad_r, ARCMaxGridSize - pad_r - nrow), (pad_c, ARCMaxGridSize - pad_c - ncol)), constant_values=0)

        # Add <eos>
        eos_row, eos_col = pad_r + nrow, pad_c + ncol
        if eos_row < ARCMaxGridSize:
            grid[eos_row, pad_c:eos_col] = 1
        if eos_col < ARCMaxGridSize:
            grid[pad_r:eos_row, eos_col] = 1

        result.append(grid.flatten())

    return result


def puzzle_hash(puzzle: dict):
    # Hash the puzzle for checking equivalence
    def _grid_hash(grid: np.ndarray):
        buffer = [x.to_bytes(1) for x in grid.shape]
        buffer.append(grid.tobytes())
        
        return hashlib.sha256(b"".join(buffer)).hexdigest()
    
    hashes = []
    for example_type, example in puzzle.items():
        for input, label in example.examples:
            hashes.append(f"{_grid_hash(input)}|{_grid_hash(label)}")
            
    hashes.sort()
    return hashlib.sha256("|".join(hashes).encode()).hexdigest()


def convert_single_arc_puzzle(results: dict, default_name: str, puzzle: dict, aug_count: int, dest_mapping: Dict[str, Tuple[str, str]]):
    # Remove "name"
    name = puzzle.pop("name", default_name)
    
    # Convert
    dests = set(dest_mapping.values())
    converted = {dest: ARCPuzzle(name, []) for dest in dests}
    for example_type, examples in puzzle.items():
        dest = dest_mapping[example_type]
        converted[dest].examples.extend([(arc_grid_to_np(example["input"]), arc_grid_to_np(example["output"])) for example in examples])

    group = [converted]
    
    # Augment
    if aug_count > 0:
        hashes = {puzzle_hash(converted)}

        for _trial in range(ARCAugmentRetriesFactor * aug_count):
            # Augment plan
            trans_id = np.random.randint(0, 8)
            mapping = np.concatenate([np.arange(0, 1, dtype=np.uint8), np.random.permutation(np.arange(1, 10, dtype=np.uint8))])  # Permute colors, Excluding "0" (black)
            
            aug_repr = f"t{trans_id}_{''.join(str(x) for x in mapping)}"

            def _map_grid(grid: np.ndarray):
                return dihedral_transform(mapping[grid], trans_id)
            
            # Check duplicate
            augmented = {dest: ARCPuzzle(f"{puzzle.id}_{aug_repr}", [(_map_grid(input), _map_grid(label)) for (input, label) in puzzle.examples]) for dest, puzzle in converted.items()}
            h = puzzle_hash(augmented)
            if h not in hashes:
                hashes.add(h)
                group.append(augmented)
                
            if len(group) >= aug_count + 1:
                break
            
        if len(group) < aug_count + 1:
            print (f"[Puzzle {name}] augmentation not full, only {len(group)}")

    # Append
    for dest in dests:
        # Convert the examples
        dest_split, dest_set = dest

        results.setdefault(dest_split, {})
        results[dest_split].setdefault(dest_set, [])
        results[dest_split][dest_set].append([converted[dest] for converted in group])


def load_puzzles_arcagi(results: dict, dataset_path: str, config: DataProcessConfig):
    train_examples_dest = ("train", "all")
    test_examples_map = {
        "evaluation": [(1.0, ("test", "all"))],
        "_default": [(1.0, ("train", "all"))]
    }
    
    total_puzzles = 0
    for subdir in os.scandir(dataset_path):
        if subdir.is_dir():
            # Load all puzzles in this directory
            puzzles = []
            for filename in glob(os.path.join(subdir.path, "*.json")):
                with open(filename, "r") as f:
                    puzzles.append((Path(filename).stem, json.load(f)))
                    
            # Shuffle puzzles
            np.random.shuffle(puzzles)
            
            # Assign by fraction
            for idx, (default_name, puzzle) in enumerate(puzzles):
                fraction = idx / len(puzzles)
                test_examples_dest = None
                for f, dest in test_examples_map.get(subdir.name, test_examples_map["_default"]):
                    if fraction < f:
                        test_examples_dest = dest
                        break
                        
                assert test_examples_dest is not None
                
                convert_single_arc_puzzle(results, default_name, puzzle, config.num_aug, {"train": train_examples_dest, "test": test_examples_dest})
                total_puzzles += 1

    print (f"[{dataset_path}] total puzzles: {total_puzzles}")


def convert_dataset(config: DataProcessConfig):
    np.random.seed(config.seed)
    
    # Read dataset
    data = {}
    for dataset_dir in config.dataset_dirs:
        load_puzzles_arcagi(data, dataset_dir, config)
    
    # Map global puzzle identifiers
    num_identifiers = 1  # 0 is blank
    identifier_map = {}
    for split_name, split in data.items():
        for subset_name, subset in split.items():
            for group in subset:
                for puzzle in group:
                    if puzzle.id not in identifier_map:
                        identifier_map[puzzle.id] = num_identifiers
                        num_identifiers += 1

    print (f"Total puzzle IDs (including <blank>): {num_identifiers}")

    # Save
    for split_name, split in data.items():
        os.makedirs(os.path.join(config.output_dir, split_name), exist_ok=True)
        
        # Translational augmentations
        enable_translational_augment = split_name == "train"

        # Statistics
        total_examples = 0
        total_puzzles = 0
        total_groups = 0
        
        for subset_name, subset in split.items():
            # Construct subset
            results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
            results["puzzle_indices"].append(0)
            results["group_indices"].append(0)
            
            example_id = 0
            puzzle_id = 0
            
            for group in subset:
                for puzzle in group:
                    # Push puzzle
                    no_aug_id = np.random.randint(0, len(puzzle.examples))
                    for _idx_ex, (inp, out) in enumerate(puzzle.examples):
                        inp, out = np_grid_to_seq_translational_augment(inp, out, do_translation=enable_translational_augment and _idx_ex != no_aug_id)
                            
                        results["inputs"].append(inp)
                        results["labels"].append(out)
                        example_id += 1
                        
                        total_examples += 1

                    results["puzzle_indices"].append(example_id)
                    results["puzzle_identifiers"].append(identifier_map[puzzle.id])
                    
                    puzzle_id += 1
                    
                    total_puzzles += 1
                    
                # Push group
                results["group_indices"].append(puzzle_id)
                total_groups += 1
            
            for k, v in results.items():
                if k in {"inputs", "labels"}:
                    v = np.stack(v, 0)
                else:
                    v = np.array(v, dtype=np.int32)
                
                np.save(os.path.join(config.output_dir, split_name, f"{subset_name}__{k}.npy"), v)
        
        # Metadata
        metadata = PuzzleDatasetMetadata(
            seq_len=ARCMaxGridSize * ARCMaxGridSize,
            vocab_size=10 + 2,  # PAD + EOS + "0" ... "9"
            
            pad_id=0,
            ignore_label_id=0,
            
            blank_identifier_id=0,
            num_puzzle_identifiers=num_identifiers,
            
            total_groups=total_groups,
            mean_puzzle_examples=total_examples / total_puzzles,
            sets=list(split.keys())
        )

        # Save metadata as JSON.
        with open(os.path.join(config.output_dir, split_name, "dataset.json"), "w") as f:
            json.dump(metadata.model_dump(), f)
            
    # Save IDs mapping
    with open(os.path.join(config.output_dir, "identifiers.json"), "w") as f:
        ids_mapping = {v: k for k, v in identifier_map.items()}
        
        json.dump([ids_mapping.get(i, "<blank>") for i in range(num_identifiers)], f)


@cli.command(singleton=True)
def main(config: DataProcessConfig):
    convert_dataset(config)


if __name__ == "__main__":
    cli()



In [ ]:
%%writefile dataset/build_maze_dataset.py
from typing import Optional
import math
import os
import csv
import json
import numpy as np

from argdantic import ArgParser
from pydantic import BaseModel
from tqdm import tqdm
from huggingface_hub import hf_hub_download

from common import PuzzleDatasetMetadata, dihedral_transform


CHARSET = "# SGo"


cli = ArgParser()


class DataProcessConfig(BaseModel):
    source_repo: str = "sapientinc/maze-30x30-hard-1k"
    output_dir: str = "data/maze-30x30-hard-1k"

    subsample_size: Optional[int] = None
    aug: bool = False


def convert_subset(set_name: str, config: DataProcessConfig):
    # Read CSV
    all_chars = set()
    grid_size = None
    inputs = []
    labels = []
    
    with open(hf_hub_download(config.source_repo, f"{set_name}.csv", repo_type="dataset"), newline="") as csvfile:  # type: ignore
        reader = csv.reader(csvfile)
        next(reader)  # Skip header
        for source, q, a, rating in reader:
            all_chars.update(q)
            all_chars.update(a)

            if grid_size is None:
                n = int(len(q) ** 0.5)
                grid_size = (n, n)
                
            inputs.append(np.frombuffer(q.encode(), dtype=np.uint8).reshape(grid_size))
            labels.append(np.frombuffer(a.encode(), dtype=np.uint8).reshape(grid_size))

    # If subsample_size is specified for the training set,
    # randomly sample the desired number of examples.
    if set_name == "train" and config.subsample_size is not None:
        total_samples = len(inputs)
        if config.subsample_size < total_samples:
            indices = np.random.choice(total_samples, size=config.subsample_size, replace=False)
            inputs = [inputs[i] for i in indices]
            labels = [labels[i] for i in indices]

    # Generate dataset
    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    puzzle_id = 0
    example_id = 0
    
    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)
    
    for inp, out in zip(tqdm(inputs), labels):
        # Dihedral transformations for augmentation
        for aug_idx in range(8 if (set_name == "train" and config.aug) else 1):
            results["inputs"].append(dihedral_transform(inp, aug_idx))
            results["labels"].append(dihedral_transform(out, aug_idx))
            example_id += 1
            puzzle_id += 1
            
            results["puzzle_indices"].append(example_id)
            results["puzzle_identifiers"].append(0)
            
        # Push group
        results["group_indices"].append(puzzle_id)
            
    # Char mappings
    assert len(all_chars - set(CHARSET)) == 0
    
    char2id = np.zeros(256, np.uint8)
    char2id[np.array(list(map(ord, CHARSET)))] = np.arange(len(CHARSET)) + 1

    # To Numpy
    def _seq_to_numpy(seq):
        arr = np.vstack([char2id[s.reshape(-1)] for s in seq])
        
        return arr
    
    results = {
        "inputs": _seq_to_numpy(results["inputs"]),
        "labels": _seq_to_numpy(results["labels"]),
        
        "group_indices": np.array(results["group_indices"], dtype=np.int32),
        "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
        "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
    }

    # Metadata
    metadata = PuzzleDatasetMetadata(
        seq_len=int(math.prod(grid_size)),  # type: ignore
        vocab_size=len(CHARSET) + 1,  # PAD + Charset
        
        pad_id=0,
        ignore_label_id=0,
        
        blank_identifier_id=0,
        num_puzzle_identifiers=1,
        
        total_groups=len(results["group_indices"]) - 1,
        mean_puzzle_examples=1,
        sets=["all"]
    )

    # Save metadata as JSON.
    save_dir = os.path.join(config.output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)
    
    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata.model_dump(), f)
        
    # Save data
    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)
        
    # Save IDs mapping (for visualization only)
    with open(os.path.join(config.output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)


@cli.command(singleton=True)
def preprocess_data(config: DataProcessConfig):
    convert_subset("train", config)
    convert_subset("test", config)


if __name__ == "__main__":
    cli()



In [ ]:
%%writefile dataset/build_sudoku_dataset.py
from typing import Optional
import os
import csv
import json
import numpy as np

from argdantic import ArgParser
from pydantic import BaseModel
from tqdm import tqdm
from huggingface_hub import hf_hub_download

from common import PuzzleDatasetMetadata


cli = ArgParser()


class DataProcessConfig(BaseModel):
    source_repo: str = "sapientinc/sudoku-extreme"
    output_dir: str = "data/sudoku-extreme-full"

    subsample_size: Optional[int] = None
    min_difficulty: Optional[int] = None
    num_aug: int = 0


def shuffle_sudoku(board: np.ndarray, solution: np.ndarray):
    # Create a random digit mapping: a permutation of 1..9, with zero (blank) unchanged
    digit_map = np.pad(np.random.permutation(np.arange(1, 10)), (1, 0))
    
    # Randomly decide whether to transpose.
    transpose_flag = np.random.rand() < 0.5

    # Generate a valid row permutation:
    # - Shuffle the 3 bands (each band = 3 rows) and for each band, shuffle its 3 rows.
    bands = np.random.permutation(3)
    row_perm = np.concatenate([b * 3 + np.random.permutation(3) for b in bands])

    # Similarly for columns (stacks).
    stacks = np.random.permutation(3)
    col_perm = np.concatenate([s * 3 + np.random.permutation(3) for s in stacks])

    # Build an 81->81 mapping. For each new cell at (i, j)
    # (row index = i // 9, col index = i % 9),
    # its value comes from old row = row_perm[i//9] and old col = col_perm[i%9].
    mapping = np.array([row_perm[i // 9] * 9 + col_perm[i % 9] for i in range(81)])

    def apply_transformation(x: np.ndarray) -> np.ndarray:
        # Apply transpose flag
        if transpose_flag:
            x = x.T
        # Apply the position mapping.
        new_board = x.flatten()[mapping].reshape(9, 9).copy()
        # Apply digit mapping
        return digit_map[new_board]

    return apply_transformation(board), apply_transformation(solution)


def convert_subset(set_name: str, config: DataProcessConfig):
    # Read CSV
    inputs = []
    labels = []
    
    # Count total rows to sample indices without loading into memory
    csv_path = hf_hub_download(config.source_repo, f"{set_name}.csv", repo_type="dataset")
    
    selected_indices = None
    if set_name == "train" and config.subsample_size is not None:
        # Instead of counting (which takes time but no RAM), we assume a large number or just read the first N
        # Or better: count lines
        with open(csv_path, 'rb') as f:
            total_samples = sum(1 for _ in f) - 1 # minus header
            
        if config.subsample_size < total_samples:
            selected_indices = set(np.random.choice(total_samples, size=config.subsample_size, replace=False))

    with open(csv_path, newline="") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header
        for i, (source, q, a, rating) in enumerate(reader):
            if selected_indices is not None and i not in selected_indices:
                continue
                
            if (config.min_difficulty is None) or (int(rating) >= config.min_difficulty):
                assert len(q) == 81 and len(a) == 81
                
                inputs.append(np.frombuffer(q.replace('.', '0').encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))
                labels.append(np.frombuffer(a.encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))
                
            if selected_indices is None and config.subsample_size is not None and len(inputs) >= config.subsample_size:
                 break

    # Generate dataset
    num_augments = config.num_aug if set_name == "train" else 0

    results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
    puzzle_id = 0
    example_id = 0
    
    results["puzzle_indices"].append(0)
    results["group_indices"].append(0)
    
    for orig_inp, orig_out in zip(tqdm(inputs), labels):
        for aug_idx in range(1 + num_augments):
            # First index is not augmented
            if aug_idx == 0:
                inp, out = orig_inp, orig_out
            else:
                inp, out = shuffle_sudoku(orig_inp, orig_out)

            # Push puzzle (only single example)
            results["inputs"].append(inp)
            results["labels"].append(out)
            example_id += 1
            puzzle_id += 1
            
            results["puzzle_indices"].append(example_id)
            results["puzzle_identifiers"].append(0)
            
        # Push group
        results["group_indices"].append(puzzle_id)
        
    # To Numpy
    def _seq_to_numpy(seq):
        arr = np.concatenate(seq).reshape(len(seq), -1)
        
        assert np.all((arr >= 0) & (arr <= 9))
        return arr + 1
    
    results = {
        "inputs": _seq_to_numpy(results["inputs"]),
        "labels": _seq_to_numpy(results["labels"]),
        
        "group_indices": np.array(results["group_indices"], dtype=np.int32),
        "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
        "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
    }

    # Metadata
    metadata = PuzzleDatasetMetadata(
        seq_len=81,
        vocab_size=10 + 1,  # PAD + "0" ... "9"
        
        pad_id=0,
        ignore_label_id=0,
        
        blank_identifier_id=0,
        num_puzzle_identifiers=1,
        
        total_groups=len(results["group_indices"]) - 1,
        mean_puzzle_examples=1,
        sets=["all"]
    )

    # Save metadata as JSON.
    save_dir = os.path.join(config.output_dir, set_name)
    os.makedirs(save_dir, exist_ok=True)
    
    with open(os.path.join(save_dir, "dataset.json"), "w") as f:
        json.dump(metadata.model_dump(), f)
        
    # Save data
    for k, v in results.items():
        np.save(os.path.join(save_dir, f"all__{k}.npy"), v)
        
    # Save IDs mapping (for visualization only)
    with open(os.path.join(config.output_dir, "identifiers.json"), "w") as f:
        json.dump(["<blank>"], f)


@cli.command(singleton=True)
def preprocess_data(config: DataProcessConfig):
    convert_subset("train", config)
    convert_subset("test", config)


if __name__ == "__main__":
    cli()



In [ ]:
%%writefile dataset/common.py
from typing import List, Optional

import pydantic
import numpy as np


# Global list mapping each dihedral transform id to its inverse.
# Index corresponds to the original tid, and the value is its inverse.
DIHEDRAL_INVERSE = [0, 3, 2, 1, 4, 5, 6, 7]


class PuzzleDatasetMetadata(pydantic.BaseModel):
    pad_id: int
    ignore_label_id: Optional[int]
    blank_identifier_id: int
    
    vocab_size: int
    seq_len: int
    num_puzzle_identifiers: int
    
    total_groups: int
    mean_puzzle_examples: float

    sets: List[str]


def dihedral_transform(arr: np.ndarray, tid: int) -> np.ndarray:
    """8 dihedral symmetries by rotate, flip and mirror"""
    
    if tid == 0:
        return arr  # identity
    elif tid == 1:
        return np.rot90(arr, k=1)
    elif tid == 2:
        return np.rot90(arr, k=2)
    elif tid == 3:
        return np.rot90(arr, k=3)
    elif tid == 4:
        return np.fliplr(arr)       # horizontal flip
    elif tid == 5:
        return np.flipud(arr)       # vertical flip
    elif tid == 6:
        return arr.T                # transpose (reflection along main diagonal)
    elif tid == 7:
        return np.fliplr(np.rot90(arr, k=1))  # anti-diagonal reflection
    else:
        return arr
    
    
def inverse_dihedral_transform(arr: np.ndarray, tid: int) -> np.ndarray:
    return dihedral_transform(arr, DIHEDRAL_INVERSE[tid])



In [ ]:
%%writefile models/common.py
import math

import torch
from torch import nn


def trunc_normal_init_(tensor: torch.Tensor, std: float = 1.0, lower: float = -2.0, upper: float = 2.0):
    # NOTE: PyTorch nn.init.trunc_normal_ is not mathematically correct, the std dev is not actually the std dev of initialized tensor
    # This function is a PyTorch version of jax truncated normal init (default init method in flax)
    # https://github.com/jax-ml/jax/blob/main/jax/_src/random.py#L807-L848
    # https://github.com/jax-ml/jax/blob/main/jax/_src/nn/initializers.py#L162-L199

    with torch.no_grad():
        if std == 0:
            tensor.zero_()
        else:
            sqrt2 = math.sqrt(2)
            a = math.erf(lower / sqrt2)
            b = math.erf(upper / sqrt2)
            z = (b - a) / 2

            c = (2 * math.pi) ** -0.5
            pdf_u = c * math.exp(-0.5 * lower ** 2)
            pdf_l = c * math.exp(-0.5 * upper ** 2)
            comp_std = std / math.sqrt(1 - (upper * pdf_u - lower * pdf_l) / z - ((pdf_u - pdf_l) / z) ** 2)

            tensor.uniform_(a, b)
            tensor.erfinv_()
            tensor.mul_(sqrt2 * comp_std)
            tensor.clip_(lower * comp_std, upper * comp_std)

    return tensor



In [ ]:
%%writefile models/layers.py
from typing import Tuple

import torch
from torch import nn
import torch.nn.functional as F

try:
    from flash_attn_interface import flash_attn_func  # type: ignore[import]
except ImportError:
    # Fallback to FlashAttention 2
    try:
        from flash_attn import flash_attn_func  # type: ignore[import]
    except ImportError:
        # Fallback to PyTorch native SDPA
        def flash_attn_func(q, k, v, causal=False):
            q = q.transpose(1, 2)
            k = k.transpose(1, 2)
            v = v.transpose(1, 2)
            out = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
            return out.transpose(1, 2).contiguous()

from models.common import trunc_normal_init_


CosSin = Tuple[torch.Tensor, torch.Tensor]


def _find_multiple(a, b):
    return (-(a // -b)) * b


def rotate_half(x: torch.Tensor):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    # q, k: [bs, seq_len, num_heads, head_dim]
    # cos, sin: [seq_len, head_dim]
    orig_dtype = q.dtype
    q = q.to(cos.dtype)
    k = k.to(cos.dtype)

    q_embed = (q * cos.unsqueeze(-2)) + (rotate_half(q) * sin.unsqueeze(-2))
    k_embed = (k * cos.unsqueeze(-2)) + (rotate_half(k) * sin.unsqueeze(-2))

    return q_embed.to(orig_dtype), k_embed.to(orig_dtype)


class CastedLinear(nn.Module):
    def __init__(self,
                 in_features: int,
                 out_features: int,
                 bias: bool):
        super().__init__()
        # Truncated LeCun normal init
        self.weight = nn.Parameter(
            trunc_normal_init_(torch.empty((out_features, in_features)), std=1.0 / (in_features ** 0.5))
        )
        self.bias = None
        if bias:
            # Zero init bias
            self.bias = nn.Parameter(torch.zeros((out_features, )))

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return F.linear(input, self.weight.to(input.dtype), bias=self.bias.to(input.dtype) if self.bias is not None else None)


class CastedEmbedding(nn.Module):
    def __init__(self,
                 num_embeddings: int,
                 embedding_dim: int,
                 init_std: float,
                 cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to

        # Truncated LeCun normal init
        self.embedding_weight = nn.Parameter(
            trunc_normal_init_(torch.empty((num_embeddings, embedding_dim)), std=init_std)
        )
        
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        return F.embedding(input, self.embedding_weight.to(self.cast_to))


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings, base, device=None):
        super().__init__()

        # RoPE
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32, device=device) / dim))
        t = torch.arange(max_position_embeddings, dtype=torch.float32, device=device)
        freqs = torch.outer(t, inv_freq)

        # Different from paper, but it uses a different permutation in order to obtain the same calculation
        emb = torch.cat((freqs, freqs), dim=-1)
        self.cos_cached = nn.Buffer(emb.cos(), persistent=False)
        self.sin_cached = nn.Buffer(emb.sin(), persistent=False)

    def forward(self):
        return self.cos_cached, self.sin_cached


class Attention(nn.Module):
    def __init__(self, hidden_size, head_dim, num_heads, num_key_value_heads, causal=False):
        super().__init__()

        self.hidden_size = hidden_size
        self.head_dim = head_dim
        self.output_size = head_dim * num_heads
        self.num_heads = num_heads
        self.num_key_value_heads = num_key_value_heads
        self.causal = causal

        self.qkv_proj = CastedLinear(self.hidden_size, (self.num_heads + 2 * self.num_key_value_heads) * self.head_dim, bias=False)
        self.o_proj = CastedLinear(self.output_size, self.hidden_size, bias=False)

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = hidden_states.shape

        # hidden_states: [bs, seq_len, num_heads, head_dim]
        qkv = self.qkv_proj(hidden_states)

        # Split head
        qkv = qkv.view(batch_size, seq_len, self.num_heads + 2 * self.num_key_value_heads, self.head_dim)
        query = qkv[:, :, :self.num_heads]
        key = qkv[:, :, self.num_heads: self.num_heads + self.num_key_value_heads]
        value = qkv[:, :, self.num_heads + self.num_key_value_heads:]

        # RoPE
        if cos_sin is not None:
            cos, sin = cos_sin
            query, key = apply_rotary_pos_emb(query, key, cos, sin)

        # flash attn
        attn_output = flash_attn_func(q=query, k=key, v=value, causal=self.causal)
        if isinstance(attn_output, tuple):  # fa2 and fa3 compatibility
            attn_output = attn_output[0]

        attn_output = attn_output.view(batch_size, seq_len, self.output_size)  # type: ignore
        return self.o_proj(attn_output)


class SwiGLU(nn.Module):
    def __init__(self, hidden_size: int, expansion: float):
        super().__init__()
        inter = _find_multiple(round(expansion * hidden_size * 2 / 3), 256)

        self.gate_up_proj = CastedLinear(hidden_size, inter * 2, bias=False)
        self.down_proj    = CastedLinear(inter, hidden_size, bias=False)

    def forward(self, x):
        gate, up = self.gate_up_proj(x).chunk(2, dim=-1)
        return self.down_proj(F.silu(gate) * up)


def rms_norm(hidden_states: torch.Tensor, variance_epsilon: float) -> torch.Tensor:
    input_dtype = hidden_states.dtype
    hidden_states = hidden_states.to(torch.float32)

    variance = hidden_states.square().mean(-1, keepdim=True)
    hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
    return hidden_states.to(input_dtype)



In [ ]:
%%writefile models/losses.py
from typing import Any, Tuple, Dict, Sequence, Optional

import torch
import torch.nn.functional as F
from torch import nn


IGNORE_LABEL_ID = -100


def s(x, epsilon=1e-30):
    return torch.where(
        x<0,
        1/(1-x+ epsilon),
        x + 1
    )


def log_stablemax(x, dim=-1):
    s_x = s(x)
    return torch.log(s_x/torch.sum(s_x, dim=dim, keepdim=True))


def stablemax_cross_entropy(logits, labels, ignore_index: int = -100):
    logprobs = log_stablemax(logits.to(torch.float64), dim=-1)

    valid_mask = labels != ignore_index
    transformed_labels = torch.where(valid_mask, labels, 0)
    prediction_logprobs = torch.gather(logprobs, index=transformed_labels.to(torch.long).unsqueeze(-1), dim=-1).squeeze(-1)

    return -torch.where(valid_mask, prediction_logprobs, 0)


def softmax_cross_entropy(logits, labels, ignore_index: int = -100):
    # Cast logits to f32
    # Flatten logits
    return F.cross_entropy(logits.to(torch.float32).view(-1, logits.shape[-1]), labels.to(torch.long).view(-1), ignore_index=ignore_index, reduction="none").view(labels.shape)


class ACTLossHead(nn.Module):
    def __init__(self, model: nn.Module, loss_type: str):
        super().__init__()
        self.model = model
        self.loss_fn = globals()[loss_type]
        
    def initial_carry(self, *args, **kwargs):
        return self.model.initial_carry(*args, **kwargs)  # type: ignore

    def forward(
        self,
        return_keys: Sequence[str],
        # Model args
        **model_kwargs,
    ) -> Tuple[Any, torch.Tensor, Dict[str, torch.Tensor], Optional[Dict[str, torch.Tensor]], torch.Tensor]:
        # Model logits
        # B x SeqLen x D
        new_carry, outputs = self.model(**model_kwargs)
        labels = new_carry.current_data["labels"]

        # Correctness
        with torch.no_grad():
            mask = labels != IGNORE_LABEL_ID
            loss_counts = mask.sum(-1)
            loss_divisor = loss_counts.clamp_min(1).unsqueeze(-1)  # Avoid NaNs in division

            is_correct = mask & (torch.argmax(outputs["logits"], dim=-1) == labels)
            seq_is_correct = is_correct.sum(-1) == loss_counts
            
            # Metrics (halted)
            valid_metrics = new_carry.halted & (loss_counts > 0)
            metrics = {
                "count": valid_metrics.sum(),
                
                "accuracy":       torch.where(valid_metrics, (is_correct.to(torch.float32) / loss_divisor).sum(-1), 0).sum(),
                "exact_accuracy": (valid_metrics & seq_is_correct).sum(),

                "q_halt_accuracy": (valid_metrics & ((outputs["q_halt_logits"] >= 0) == seq_is_correct)).sum(),
                "steps":          torch.where(valid_metrics, new_carry.steps, 0).sum(),
            }

        # Losses
        # FIXME: Assuming the batch is always full
        lm_loss = (self.loss_fn(outputs["logits"], labels, ignore_index=IGNORE_LABEL_ID) / loss_divisor).sum()
        q_halt_loss = F.binary_cross_entropy_with_logits(outputs["q_halt_logits"], seq_is_correct.to(outputs["q_halt_logits"].dtype), reduction="sum")

        metrics.update({
            "lm_loss": lm_loss.detach(),
            "q_halt_loss": q_halt_loss.detach(),
        })

        # Q continue (bootstrapping target loss)
        q_continue_loss = 0
        if "target_q_continue" in outputs:
            q_continue_loss = F.binary_cross_entropy_with_logits(outputs["q_continue_logits"], outputs["target_q_continue"], reduction="sum")

            metrics["q_continue_loss"] = q_continue_loss.detach()

        # Filter outputs for return
        detached_outputs = {k: outputs[k].detach() for k in return_keys if k in outputs}

        return new_carry, lm_loss + 0.5 * (q_halt_loss + q_continue_loss), metrics, detached_outputs, new_carry.halted.all()



In [ ]:
%%writefile models/sparse_embedding.py
from typing import Union

import torch
from torch import nn
import torch.distributed as dist
from torch.optim.optimizer import Optimizer, ParamsT

from models.common import trunc_normal_init_


class CastedSparseEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, batch_size: int, init_std: float, cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to

        # Real Weights
        # Truncated LeCun normal init
        self.weights = nn.Buffer(
            trunc_normal_init_(torch.empty((num_embeddings, embedding_dim)), std=init_std), persistent=True
        )

        # Local weights and IDs
        # Local embeddings, with gradient, not persistent
        self.local_weights = nn.Buffer(torch.zeros(batch_size, embedding_dim, requires_grad=True), persistent=False)
        # Local embedding IDs, not persistent
        self.local_ids = nn.Buffer(torch.zeros(batch_size, dtype=torch.int32), persistent=False)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        if not self.training:
            # Test mode, no gradient
            return self.weights[inputs].to(self.cast_to)
            
        # Training mode, fill puzzle embedding from weights
        with torch.no_grad():
            self.local_weights.copy_(self.weights[inputs])
            self.local_ids.copy_(inputs)

        return self.local_weights.to(self.cast_to)


class CastedSparseEmbeddingSignSGD_Distributed(Optimizer):
    def __init__(
        self,
        params: ParamsT,

        world_size: int,
        lr: Union[float, torch.Tensor] = 1e-3,
        weight_decay: float = 1e-2,
    ):
        if not 0.0 <= lr:
            raise ValueError(f"Invalid learning rate: {lr}")
        if not 0.0 <= weight_decay:
            raise ValueError(f"Invalid weight_decay value: {weight_decay}")

        defaults = dict(
            lr=lr,
            weight_decay=weight_decay,
            world_size=world_size
        )
        super().__init__(params, defaults)

    @torch.no_grad
    def step(self, closure=None):  # type: ignore
        for group in self.param_groups:
            # Find the sparse embedding weights
            local_weights_grad = None
            local_ids = None
            weights = None
            
            assert len(group["params"]) == 3
            for p in group["params"]:
                if p.requires_grad:
                    local_weights_grad = p.grad
                elif p.ndim == 1:
                    local_ids = p
                elif p.ndim == 2:
                    weights = p
                else:
                    assert False
                
            assert local_weights_grad is not None
            assert local_ids is not None
            assert weights is not None
        
            # Apply SignSGD
            # Adam ≈ SignSGD if gradient is very sparse
            _sparse_emb_signsgd_dist(
                local_weights_grad,
                local_ids,
                weights,
                
                lr=group["lr"],
                weight_decay=group["weight_decay"],
                world_size=group["world_size"]
            )


def _sparse_emb_signsgd_dist(
    local_weights_grad: torch.Tensor,
    local_ids: torch.Tensor,
    weights: torch.Tensor,
    
    lr: float,
    weight_decay: float,
    world_size: int
) -> None:
    N, D = local_weights_grad.shape
    
    # All-gather
    all_weights_grad = local_weights_grad
    all_ids = local_ids

    if world_size > 1:
        all_weights_grad = torch.empty((world_size * N, D), dtype=local_weights_grad.dtype, device=local_weights_grad.device)
        all_ids = torch.empty(world_size * N,               dtype=local_ids.dtype,          device=local_ids.device)
    
        dist.all_gather_into_tensor(all_weights_grad, local_weights_grad)
        dist.all_gather_into_tensor(all_ids,          local_ids)

    # Unique
    grad_ids, inv = all_ids.unique(return_inverse=True)

    grad = torch.zeros((grad_ids.shape[0], D), dtype=all_weights_grad.dtype, device=all_weights_grad.device)
    grad.scatter_add_(0, inv.unsqueeze(-1).expand(-1, D), all_weights_grad)

    # SignSGD with decoupled weight decay
    p = weights[grad_ids]

    p.mul_(1.0 - lr * weight_decay).add_(torch.sign(grad), alpha=-lr)

    # Write updated slices back
    weights[grad_ids] = p



In [ ]:
%%writefile models/hrm/hrm_act_v1.py
from typing import Tuple, List, Dict, Optional
from dataclasses import dataclass
import math

import torch
import torch.nn.functional as F
from torch import nn
from pydantic import BaseModel

from models.common import trunc_normal_init_
from models.layers import rms_norm, SwiGLU, Attention, RotaryEmbedding, CosSin, CastedEmbedding, CastedLinear
from models.sparse_embedding import CastedSparseEmbedding


@dataclass
class HierarchicalReasoningModel_ACTV1InnerCarry:
    z_H: torch.Tensor
    z_L: torch.Tensor


@dataclass
class HierarchicalReasoningModel_ACTV1Carry:
    inner_carry: HierarchicalReasoningModel_ACTV1InnerCarry
    
    steps: torch.Tensor
    halted: torch.Tensor
    
    current_data: Dict[str, torch.Tensor]


class HierarchicalReasoningModel_ACTV1Config(BaseModel):
    batch_size: int
    seq_len: int
    puzzle_emb_ndim: int = 0
    num_puzzle_identifiers: int
    vocab_size: int

    H_cycles: int
    L_cycles: int

    H_layers: int
    L_layers: int

    # Transformer config
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str

    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    
    # Halting Q-learning config
    halt_max_steps: int
    halt_exploration_prob: float

    forward_dtype: str = "bfloat16"


class HierarchicalReasoningModel_ACTV1Block(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_ACTV1Config) -> None:
        super().__init__()

        self.self_attn = Attention(
            hidden_size=config.hidden_size,
            head_dim=config.hidden_size // config.num_heads,
            num_heads=config.num_heads,
            num_key_value_heads=config.num_heads,
            causal=False
        )
        self.mlp = SwiGLU(
            hidden_size=config.hidden_size,
            expansion=config.expansion,
        )
        self.norm_eps = config.rms_norm_eps

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor) -> torch.Tensor:
        # Post Norm
        # Self Attention
        hidden_states = rms_norm(hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states), variance_epsilon=self.norm_eps)
        # Fully Connected
        hidden_states = rms_norm(hidden_states + self.mlp(hidden_states), variance_epsilon=self.norm_eps)
        return hidden_states


class HierarchicalReasoningModel_ACTV1ReasoningModule(nn.Module):
    def __init__(self, layers: List[HierarchicalReasoningModel_ACTV1Block]):
        super().__init__()

        self.layers = torch.nn.ModuleList(layers)

    def forward(self, hidden_states: torch.Tensor, input_injection: torch.Tensor, **kwargs) -> torch.Tensor:
        # Input injection (add)
        hidden_states = hidden_states + input_injection
        # Layers
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, **kwargs)

        return hidden_states


class HierarchicalReasoningModel_ACTV1_Inner(nn.Module):
    def __init__(self, config: HierarchicalReasoningModel_ACTV1Config) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)

        # I/O
        self.embed_scale  = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.lm_head      = CastedLinear(self.config.hidden_size, self.config.vocab_size, bias=False)
        self.q_head       = CastedLinear(self.config.hidden_size, 2, bias=True)

        self.puzzle_emb_len = -(self.config.puzzle_emb_ndim // -self.config.hidden_size)  # ceil div
        if self.config.puzzle_emb_ndim > 0:
            # Zero init puzzle embeddings
            self.puzzle_emb = CastedSparseEmbedding(self.config.num_puzzle_identifiers, self.config.puzzle_emb_ndim,
                                                    batch_size=self.config.batch_size, init_std=0, cast_to=self.forward_dtype)

        # LM Blocks
        if self.config.pos_encodings == "rope":
            self.rotary_emb = RotaryEmbedding(dim=self.config.hidden_size // self.config.num_heads,
                                              max_position_embeddings=self.config.seq_len + self.puzzle_emb_len,
                                              base=self.config.rope_theta)
        elif self.config.pos_encodings == "learned":
            self.embed_pos = CastedEmbedding(self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        else:
            raise NotImplementedError()

        # Reasoning Layers
        self.H_level = HierarchicalReasoningModel_ACTV1ReasoningModule(layers=[HierarchicalReasoningModel_ACTV1Block(self.config) for _i in range(self.config.H_layers)])
        self.L_level = HierarchicalReasoningModel_ACTV1ReasoningModule(layers=[HierarchicalReasoningModel_ACTV1Block(self.config) for _i in range(self.config.L_layers)])
        
        # Initial states
        self.H_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)
        self.L_init = nn.Buffer(trunc_normal_init_(torch.empty(self.config.hidden_size, dtype=self.forward_dtype), std=1), persistent=True)

        # Q head special init
        # Init Q to (almost) zero for faster learning during bootstrapping
        with torch.no_grad():
            self.q_head.weight.zero_()
            self.q_head.bias.fill_(-5)  # type: ignore

    def _input_embeddings(self, input: torch.Tensor, puzzle_identifiers: torch.Tensor):
        # Token embedding
        embedding = self.embed_tokens(input.to(torch.int32))

        # Puzzle embeddings
        if self.config.puzzle_emb_ndim > 0:
            puzzle_embedding = self.puzzle_emb(puzzle_identifiers)
            
            pad_count = self.puzzle_emb_len * self.config.hidden_size - puzzle_embedding.shape[-1]
            if pad_count > 0:
                puzzle_embedding = F.pad(puzzle_embedding, (0, pad_count))

            embedding = torch.cat((puzzle_embedding.view(-1, self.puzzle_emb_len, self.config.hidden_size), embedding), dim=-2)

        # Position embeddings
        if self.config.pos_encodings == "learned":
            # scale by 1/sqrt(2) to maintain forward variance
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight.to(self.forward_dtype))

        # Scale
        return self.embed_scale * embedding

    def empty_carry(self, batch_size: int):
        return HierarchicalReasoningModel_ACTV1InnerCarry(
            z_H=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
            z_L=torch.empty(batch_size, self.config.seq_len + self.puzzle_emb_len, self.config.hidden_size, dtype=self.forward_dtype),
        )
        
    def reset_carry(self, reset_flag: torch.Tensor, carry: HierarchicalReasoningModel_ACTV1InnerCarry):
        return HierarchicalReasoningModel_ACTV1InnerCarry(
            z_H=torch.where(reset_flag.view(-1, 1, 1), self.H_init, carry.z_H),
            z_L=torch.where(reset_flag.view(-1, 1, 1), self.L_init, carry.z_L),
        )

    def forward(self, carry: HierarchicalReasoningModel_ACTV1InnerCarry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_ACTV1InnerCarry, torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        seq_info = dict(
            cos_sin=self.rotary_emb() if hasattr(self, "rotary_emb") else None,
        )

        # Input encoding
        input_embeddings = self._input_embeddings(batch["inputs"], batch["puzzle_identifiers"])

        # Forward iterations
        with torch.no_grad():
            z_H, z_L = carry.z_H, carry.z_L

            for _H_step in range(self.config.H_cycles):
                for _L_step in range(self.config.L_cycles):
                    if not ((_H_step == self.config.H_cycles - 1) and (_L_step == self.config.L_cycles - 1)):
                        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)

                if not (_H_step == self.config.H_cycles - 1):
                    z_H = self.H_level(z_H, z_L, **seq_info)

        assert not z_H.requires_grad and not z_L.requires_grad

        # 1-step grad
        z_L = self.L_level(z_L, z_H + input_embeddings, **seq_info)
        z_H = self.H_level(z_H, z_L, **seq_info)

        # LM Outputs
        new_carry = HierarchicalReasoningModel_ACTV1InnerCarry(z_H=z_H.detach(), z_L=z_L.detach())  # New carry no grad
        output = self.lm_head(z_H)[:, self.puzzle_emb_len:]

        # Q head
        q_logits = self.q_head(z_H[:, 0]).to(torch.float32)
        
        return new_carry, output, (q_logits[..., 0], q_logits[..., 1])


class HierarchicalReasoningModel_ACTV1(nn.Module):
    """ACT wrapper."""

    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = HierarchicalReasoningModel_ACTV1Config(**config_dict)
        self.inner = HierarchicalReasoningModel_ACTV1_Inner(self.config)

    @property
    def puzzle_emb(self):
        return self.inner.puzzle_emb

    def initial_carry(self, batch: Dict[str, torch.Tensor]):
        batch_size = batch["inputs"].shape[0]

        return HierarchicalReasoningModel_ACTV1Carry(
            inner_carry=self.inner.empty_carry(batch_size),  # Empty is expected, it will be reseted in first pass as all sequences are halted.
            
            steps=torch.zeros((batch_size, ), dtype=torch.int32),
            halted=torch.ones((batch_size, ), dtype=torch.bool),  # Default to halted
            
            current_data={k: torch.empty_like(v) for k, v in batch.items()}
        )
        
    def forward(self, carry: HierarchicalReasoningModel_ACTV1Carry, batch: Dict[str, torch.Tensor]) -> Tuple[HierarchicalReasoningModel_ACTV1Carry, Dict[str, torch.Tensor]]:
        # Update data, carry (removing halted sequences)
        new_inner_carry = self.inner.reset_carry(carry.halted, carry.inner_carry)
        
        new_steps = torch.where(carry.halted, 0, carry.steps)

        new_current_data = {k: torch.where(carry.halted.view((-1, ) + (1, ) * (batch[k].ndim - 1)), batch[k], v) for k, v in carry.current_data.items()}

        # Forward inner model
        new_inner_carry, logits, (q_halt_logits, q_continue_logits) = self.inner(new_inner_carry, new_current_data)

        outputs = {
            "logits": logits,
            "q_halt_logits": q_halt_logits,
            "q_continue_logits": q_continue_logits
        }
        
        with torch.no_grad():
            # Step
            new_steps = new_steps + 1
            is_last_step = new_steps >= self.config.halt_max_steps
            
            halted = is_last_step

            # if training, and ACT is enabled
            if self.training and (self.config.halt_max_steps > 1):
                # Halt signal
                # NOTE: During evaluation, always use max steps, this is to guarantee the same halting steps inside a batch for batching purposes
                halted = halted | (q_halt_logits > q_continue_logits)

                # Exploration
                min_halt_steps = (torch.rand_like(q_halt_logits) < self.config.halt_exploration_prob) * torch.randint_like(new_steps, low=2, high=self.config.halt_max_steps + 1)

                halted = halted & (new_steps >= min_halt_steps)

                # Compute target Q
                # NOTE: No replay buffer and target networks for computing target Q-value.
                # As batch_size is large, there're many parallel envs.
                # Similar concept as PQN https://arxiv.org/abs/2407.04811
                next_q_halt_logits, next_q_continue_logits = self.inner(new_inner_carry, new_current_data)[-1]
                
                outputs["target_q_continue"] = torch.sigmoid(torch.where(is_last_step, next_q_halt_logits, torch.maximum(next_q_halt_logits, next_q_continue_logits)))

        return HierarchicalReasoningModel_ACTV1Carry(new_inner_carry, new_steps, halted, new_current_data), outputs



In [ ]:
%%writefile utils/functions.py
import importlib
import inspect


def load_model_class(identifier: str, prefix: str = "models."):
    module_path, class_name = identifier.split('@')

    # Import the module
    module = importlib.import_module(prefix + module_path)
    cls = getattr(module, class_name)
    
    return cls


def get_model_source_path(identifier: str, prefix: str = "models."):
    module_path, class_name = identifier.split('@')

    module = importlib.import_module(prefix + module_path)
    return inspect.getsourcefile(module)



## Step 1: Install Dependencies

In [ ]:
!pip install -r requirements.txt


## Step 2: Generate 6x6 Sudoku Dataset

In [ ]:
!python dataset/build_6x6_sudoku_dataset.py --output-dir data/sudoku-6x6 --num-train 1000 --num-test 100 --holes 20


## Step 3: Train HRM on 6x6 Sudoku

- Checkpoints are saved BEFORE evaluation at each interval
- If evaluation crashes on T4 (bfloat16 issue), training continues
- ~11.5 hours training, checkpoints every ~69 minutes

In [ ]:
import os
os.environ['WANDB_MODE'] = 'offline'

!python pretrain.py \
    data_path=data/sudoku-6x6 \
    epochs=5000 \
    eval_interval=500 \
    checkpoint_every_eval=True \
    global_batch_size=64 \
    lr=7e-5 \
    puzzle_emb_lr=7e-5 \
    weight_decay=1.0 \
    puzzle_emb_weight_decay=1.0


## Step 4: Evaluate the Trained Model

Finds the latest checkpoint and runs evaluation.
Uses DISABLE_COMPILE to avoid T4 bfloat16 torch.compile issues during eval.

In [ ]:
import os

# Find the latest checkpoint
latest_step = -1
best_checkpoint_path = None

for root, dirs, files in os.walk('checkpoints'):
    for file in files:
        if file.startswith('step_') and 'all_preds' not in file:
            try:
                step_num = int(file.split('_')[1])
                if step_num > latest_step:
                    latest_step = step_num
                    best_checkpoint_path = os.path.join(root, file)
            except ValueError:
                pass

if best_checkpoint_path is None:
    print('ERROR: No checkpoint found!')
else:
    print(f'Latest checkpoint: {best_checkpoint_path}  (step {latest_step})')
    print('Running evaluation (with torch.compile disabled for T4 stability)...')
    print('=' * 60)
    # DISABLE_COMPILE avoids bfloat16 torch.compile issues on T4
    os.environ['DISABLE_COMPILE'] = '1'
    !python evaluate.py checkpoint="{best_checkpoint_path}"
    print('=' * 60)
    print('Evaluation complete!')


## Step 5: List All Saved Checkpoints

In [ ]:
import os

print('Saved checkpoints:')
print('-' * 40)
for root, dirs, files in os.walk('checkpoints'):
    for file in sorted(files):
        if file.startswith('step_') and 'all_preds' not in file:
            filepath = os.path.join(root, file)
            size_mb = os.path.getsize(filepath) / (1024 * 1024)
            print(f'  {file}  ({size_mb:.1f} MB)')
